<a href="https://colab.research.google.com/github/kbarakati/athena_camm_hackathon/blob/k4my4r/docs/day_17_18092026/Day_17_Sergei.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 2:  <font color = "red">**Structure → Property Challenge**</font>

Can local image structure predict the local EELS response?

For each location, we have:

**Image patch → EELS spectrum → Scalarizer**

Your task is to:

1. Build a <font color = "purple">**descriptor**</font> for the image patch.
2. Use a <font color = "purple">**regressor**</font> to predict a scalarizer or the full spectrum.
3. Compare methods by:
   - predictive power
   - complexity
   - explainability

<font color = "green">**Goal**</font>

> Find the simplest explainable descriptor that predicts the EELS response well.

## <font color = "purple">**1.**</font> Explore the Dataset

Each spatial location contains:

- one image intensity value
- one full EELS spectrum

The same position in the image corresponds to the same position in the EELS spectrum image.

Let's look at one dataset.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kbarakati/athena_camm_hackathon/blob/k4my4r/docs/day_17_18092026/notebook.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression
import os

In [ ]:
!gdown --fuzzy 1FKmlRTRDImQS91V4YQjcQHoJU19X6P_6

In [ ]:
file_name = "/content/Plasmonic_sets_hackathon.npy"

raw = np.load(
    file_name,
    allow_pickle=True
)

loadedfile = raw.tolist()
print("keys:", loadedfile.keys())

In [ ]:
# Choose dataset
data = loadedfile["2"]

image = data["image"]
spectra = data["spectrum image"]
energy = data["energy axis"]

H, W = image.shape

print(image.shape)
print(spectra.shape)
print(energy.shape)

In [ ]:
# Example locations
positions = [
    (H // 4, W // 4),
    (H // 2, W // 2),
    (3 * H // 4, W // 4),
    (3 * H // 4, 3 * W // 4),
]

colors = ["C0", "C1", "C2", "C3"]

# Integrated EELS map
eels_map = spectra.sum(axis=2)

plt.rcParams.update({"font.size": 14})

fig, ax = plt.subplots(1, 3, figsize=(15, 4))

# Structural image
ax[0].imshow(image, cmap="gray")
ax[0].set_title("Structural Image")

# Spectra
for i, ((y, x), color) in enumerate(zip(positions, colors)):

    ax[0].scatter(x, y, s=80, color=color)
    ax[0].text(x + 3, y, str(i + 2), color=color, fontsize=18)

    ax[1].plot(
        energy,
        spectra[y, x],
        color=color,
        label=f"Point {i + 1}"
    )

    ax[2].scatter(x, y, s=80, color=color)
    ax[2].text(x + 3, y, str(i + 2), color=color, fontsize=16)

ax[1].set_title("EELS Spectra")
ax[1].set_xlabel("Energy")
ax[1].set_ylabel("Intensity")
ax[1].legend()

# EELS map
ax[2].imshow(eels_map)
ax[2].set_title("Integrated EELS Map")

plt.tight_layout()
plt.show()

## <font color = "purple">**2.**</font> Define a Scalarizer

Each EELS spectrum can be reduced to a **single target value** called a scalarizer.

For example, the spectral intensity within an energy window:

$$
S(x,y)=\int_{E_1}^{E_2} I(x,y,E)\,dE
$$

where \(E_1\) and \(E_2\) define the spectral region of interest.

Other possible scalarizers include:

- peak intensity
- peak position
- peak area
- peak ratio

## <font color = "purple">**3.**</font> Choose a Spectral Region

Before defining the scalarizer, inspect the EELS spectrum and choose an energy range containing a feature of interest.

In [ ]:
# Mean spectrum over the whole image
mean_spectrum = spectra.mean(axis=(0, 1))

plt.figure(figsize=(7, 4))

plt.plot(energy, mean_spectrum)

plt.xlabel("Energy")
plt.ylabel("EELS Intensity")
plt.title("Mean EELS Spectrum")

plt.show()

## <font color = "purple">**4.**</font> Calculate the Scalarizer

We integrate the EELS intensity inside the selected energy window to obtain one target value at each spatial location.

In [ ]:
# Choose an energy range
E1 = 0.5
E2 = 1.0

mask = (energy >= E1) & (energy <= E2)

scalarizer_map = np.trapezoid(
    spectra[:, :, mask],
    energy[mask],
    axis=2
)

plt.figure(figsize=(5, 4))
plt.imshow(scalarizer_map)
plt.colorbar(label="Integrated spectral intensity")
plt.title(f"Scalarizer Map: {E1}–{E2}")
plt.show()

## <font color = "purple">**5.**</font> Create Patch–Spectrum–Scalarizer Pairs

For each location we collect:

**Image patch → EELS spectrum → Scalarizer**

The spectrum and scalarizer correspond to the center pixel of the patch.

In [ ]:
PATCH_SIZE = 15
half = PATCH_SIZE // 2

patches = []
targets = []

for y in range(half, H - half):
    for x in range(half, W - half):

        patch = image[y-half:y+half+1, x-half:x+half+1]

        patches.append(patch)
        targets.append(scalarizer_map[y, x])

patches = np.array(patches)
targets = np.array(targets)

print("Patches:", patches.shape)
print("Targets:", targets.shape)

## <font color = "purple">**6.**</font> Look at Example Pairs

Each patch has one target scalarizer value.

Can you see structural differences between low and high target values?

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(10, 3))

ids = np.linspace(0, len(patches) - 1, 4, dtype=int)

for i, idx in enumerate(ids):
    ax[i].imshow(patches[idx], cmap="gray")
    ax[i].set_title(f"Target = {targets[idx]:.2f}")
    ax[i].axis("off")

plt.tight_layout()
plt.show()

## The Challenge

You will design two parts of the prediction pipeline:

**Image Patch → Descriptor → Regressor → EELS Scalarizer**

### Your goal

Find a combination with:

- high predictive power
- low complexity
- good explainability

You can change:

1. the **descriptor** — how the image patch is represented
2. the **regressor** — how the descriptor is mapped to the EELS property

Try different combinations and compare their performance.

## <font color = "purple">**7.**</font> Define Your Descriptor and Regressor

Change only the code inside the marked sections.

The descriptor converts an image patch into features.

The regressor learns how those features relate to the EELS scalarizer.

In [ ]:
def descriptor(patch):

    # ============================================================
    # YOUR CODE STARTS HERE
    # ============================================================

    features = [
        patch.mean(),
        patch.std(),
        patch.max() - patch.min()
    ]

    # ============================================================
    # YOUR CODE ENDS HERE
    # ============================================================

    return features


def regressor():

    # ============================================================
    # YOUR CODE STARTS HERE
    # ============================================================

    model = LinearRegression()

    # ============================================================
    # YOUR CODE ENDS HERE
    # ============================================================

    return model

## <font color = "purple">**8.**</font> Run Your Pipeline

Now we test your descriptor–regressor combination.

The same train/test split is used each time so different combinations can be compared fairly.

In [ ]:
# Build descriptor matrix
X = np.array([descriptor(patch) for patch in patches])

# Same split for every experiment
X_train, X_test, y_train, y_test = train_test_split(
    X,
    targets,
    test_size=0.25,
    random_state=0
)

# Train
model = regressor()
model.fit(X_train, y_train)

# Predict
predictions = model.predict(X_test)

# Evaluate
r2 = r2_score(y_test, predictions)

print("Regressor:", type(model).__name__)
print("Number of features:", X.shape[1])
print(f"R²: {r2:.3f}")

## <font color = "purple">**9.**</font> Compare Prediction with Ground Truth

A good model should predict values close to the measured EELS scalarizer.

Points closer to the diagonal line indicate better agreement.

In [ ]:
# Size of valid prediction region
map_shape = (
    H - 2 * half,
    W - 2 * half
)

# Predicted values
predicted = model.predict(X).reshape(map_shape)

# Put predictions back into an image of the original size
prediction_map = np.full((H, W), np.nan)

prediction_map[
    half:H-half,
    half:W-half
] = predicted


fig, ax = plt.subplots(1, 3, figsize=(15, 4))

ax[0].imshow(scalarizer_map)
ax[0].set_title("Ground Truth")

ax[1].imshow(prediction_map)
ax[1].set_title("Prediction")

# Test data
ax[2].scatter(y_test, predictions, alpha=0.5)

low = min(y_test.min(), predictions.min())
high = max(y_test.max(), predictions.max())

ax[2].plot([low, high], [low, high], "--")
ax[2].set_xlabel("Ground Truth")
ax[2].set_ylabel("Prediction")
ax[2].set_title(f"Test Data — R² = {r2:.3f}")

plt.tight_layout()
plt.show()

## <font color = "purple">**10.**</font> Compare Descriptor–Regressor Combinations

Test several descriptors and regressors.

Create a heatmap where:

- rows = descriptors
- columns = regressors
- color/value = test R²

Use the same train/test split for every combination.

### <font color = "red">**Goal**</font>

Identify which descriptor–regressor combinations give the strongest predictive performance.

In [ ]:
# ============================================================
# YOUR CODE STARTS HERE
# ============================================================


# ============================================================
# YOUR CODE ENDS HERE
# ============================================================

## <font color = "purple">**11.**</font> Best Model and Explainability

From the heatmap, identify the best descriptor–regressor combination.

Report:

- best descriptor
- best regressor
- number of descriptor features
- test R²

Then explain:

- What does the descriptor measure?
- Which structural features are important?
- Why might those features be related to the EELS response?

In [ ]:
# ============================================================
# YOUR CODE STARTS HERE
# ============================================================


# ============================================================
# YOUR CODE ENDS HERE
# ============================================================

In [ ]:
# ============================================================
# PATCH SIZE × COMPRESSION BENCHMARK
#
# N = original patch width:       1, ..., 20
# m = average-pooling factor:     1, ..., 5
# complexity = compressed pixels: ceil(N/m)^2
#
# Requires:
#     image   : (H, W)
#     spectra : (H, W, n_energy)
#     energy  : (n_energy,)
# ============================================================

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score, mean_squared_error


# ------------------------------------------------------------
# 1. Experiment settings
# ------------------------------------------------------------

PATCH_SIZES = np.arange(1, 21)
COMPRESSION_FACTORS = np.arange(1, 6)

# Ridge regularization values tested for every model
RIDGE_ALPHAS = np.logspace(-3, 3, 9)

# Spatial train/test separation
TEST_FRACTION = 0.25
SPATIAL_BLOCK_SIZE = 80
RANDOM_STATE = 0

# Limit the number of spatial locations if the dataset is large.
# Set to None to use all available locations.
MAX_SAMPLES = 15000


# ------------------------------------------------------------
# 2. Construct a common set of valid patch centers
# ------------------------------------------------------------

H, W = image.shape
n_energy = spectra.shape[-1]

MAX_N = int(PATCH_SIZES.max())

# For even N, the patch extends asymmetrically around its
# nominal center:
#
# left = floor((N-1)/2)
# right = N - left
#
# This always produces an N × N patch.
max_left = (MAX_N - 1) // 2
max_right = MAX_N - max_left

coordinates = np.array([
    (y, x)
    for y in range(max_left, H - max_right + 1)
    for x in range(max_left, W - max_right + 1)
])

print("Available common patch centers:", len(coordinates))


# ------------------------------------------------------------
# 3. Define spatial groups
# ------------------------------------------------------------

n_blocks_x = int(np.ceil(W / SPATIAL_BLOCK_SIZE))

block_y = coordinates[:, 0] // SPATIAL_BLOCK_SIZE
block_x = coordinates[:, 1] // SPATIAL_BLOCK_SIZE

groups = block_y * n_blocks_x + block_x

unique_groups = np.unique(groups)

if len(unique_groups) < 4:
    raise ValueError(
        "Too few spatial groups. Reduce SPATIAL_BLOCK_SIZE."
    )


# ------------------------------------------------------------
# 4. Make one spatial train/test split for all experiments
# ------------------------------------------------------------

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=TEST_FRACTION,
    random_state=RANDOM_STATE
)

train_ids, test_ids = next(
    splitter.split(
        coordinates,
        groups=groups
    )
)

rng = np.random.default_rng(RANDOM_STATE)

# Optional subsampling after defining the spatial split
if MAX_SAMPLES is not None and len(coordinates) > MAX_SAMPLES:

    n_train = int(MAX_SAMPLES * (1 - TEST_FRACTION))
    n_test = MAX_SAMPLES - n_train

    train_ids = rng.choice(
        train_ids,
        size=min(n_train, len(train_ids)),
        replace=False
    )

    test_ids = rng.choice(
        test_ids,
        size=min(n_test, len(test_ids)),
        replace=False
    )

train_coordinates = coordinates[train_ids]
test_coordinates = coordinates[test_ids]

print("Training locations:", len(train_coordinates))
print("Test locations:", len(test_coordinates))


# ------------------------------------------------------------
# 5. Construct full-spectrum targets
# ------------------------------------------------------------

def spectra_at_coordinates(coords):
    return np.asarray([
        spectra[y, x, :]
        for y, x in coords
    ])


Y_train = spectra_at_coordinates(train_coordinates)
Y_test = spectra_at_coordinates(test_coordinates)

print("Training spectra:", Y_train.shape)
print("Test spectra:", Y_test.shape)


# ------------------------------------------------------------
# 6. Define patch extraction and compression
# ------------------------------------------------------------

def extract_patches(image, coordinates, patch_size):
    """
    Extract N × N patches at the supplied coordinates.
    Works for both odd and even patch sizes.
    """

    left = (patch_size - 1) // 2
    right = patch_size - left

    return np.asarray([
        image[
            y - left:y + right,
            x - left:x + right
        ]
        for y, x in coordinates
    ])


def compress_patches(patches, factor):
    """
    Compress patches using non-overlapping factor × factor
    average pooling.

    If N is not divisible by factor, the lower and right edges
    are padded using the nearest edge value.
    """

    if factor == 1:
        return patches.copy()

    n_samples, height, width = patches.shape

    output_height = int(np.ceil(height / factor))
    output_width = int(np.ceil(width / factor))

    padded_height = output_height * factor
    padded_width = output_width * factor

    pad_bottom = padded_height - height
    pad_right = padded_width - width

    padded = np.pad(
        patches,
        (
            (0, 0),
            (0, pad_bottom),
            (0, pad_right)
        ),
        mode="edge"
    )

    pooled = padded.reshape(
        n_samples,
        output_height,
        factor,
        output_width,
        factor
    ).mean(axis=(2, 4))

    return pooled


# ------------------------------------------------------------
# 7. Define a dimensionless full-spectrum error
# ------------------------------------------------------------

# Ordinary RMSE depends on the absolute units of the spectrum.
# Normalized RMSE makes the error easier to interpret.
#
# NRMSE = RMSE / RMS variation of the test spectra

target_scale = np.sqrt(
    np.mean(
        (Y_test - Y_test.mean(axis=0)) ** 2
    )
)

if target_scale == 0:
    target_scale = 1.0


# ------------------------------------------------------------
# 8. Run all 20 × 5 experiments
# ------------------------------------------------------------

results = []

total_models = (
    len(PATCH_SIZES)
    * len(COMPRESSION_FACTORS)
)

model_number = 0

for N in PATCH_SIZES:

    # Extract each patch size only once
    train_patches_N = extract_patches(
        image,
        train_coordinates,
        N
    )

    test_patches_N = extract_patches(
        image,
        test_coordinates,
        N
    )

    for m in COMPRESSION_FACTORS:

        model_number += 1
        start_time = time.perf_counter()

        # Compress patches
        train_compressed = compress_patches(
            train_patches_N,
            m
        )

        test_compressed = compress_patches(
            test_patches_N,
            m
        )

        compressed_height = train_compressed.shape[1]
        compressed_width = train_compressed.shape[2]

        n_features = (
            compressed_height
            * compressed_width
        )

        # Flatten compressed patches
        X_train = train_compressed.reshape(
            len(train_compressed),
            -1
        )

        X_test = test_compressed.reshape(
            len(test_compressed),
            -1
        )

        # Standardize pixel features using training data only
        scaler = StandardScaler()

        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        # Fit full-spectrum Ridge regressor
        model = RidgeCV(
            alphas=RIDGE_ALPHAS
        )

        model.fit(
            X_train_scaled,
            Y_train
        )

        Y_train_pred = model.predict(
            X_train_scaled
        )

        Y_test_pred = model.predict(
            X_test_scaled
        )

        # Full-spectrum errors
        train_rmse = np.sqrt(
            mean_squared_error(
                Y_train,
                Y_train_pred
            )
        )

        test_rmse = np.sqrt(
            mean_squared_error(
                Y_test,
                Y_test_pred
            )
        )

        train_nrmse = train_rmse / target_scale
        test_nrmse = test_rmse / target_scale

        train_r2 = r2_score(
            Y_train,
            Y_train_pred,
            multioutput="variance_weighted"
        )

        test_r2 = r2_score(
            Y_test,
            Y_test_pred,
            multioutput="variance_weighted"
        )

        elapsed = time.perf_counter() - start_time

        results.append({
            "patch_size": N,
            "compression_factor": m,
            "compressed_height": compressed_height,
            "compressed_width": compressed_width,
            "n_features": n_features,
            "alpha": model.alpha_,
            "train_rmse": train_rmse,
            "test_rmse": test_rmse,
            "train_nrmse": train_nrmse,
            "test_nrmse": test_nrmse,
            "train_r2": train_r2,
            "test_r2": test_r2,
            "runtime_seconds": elapsed
        })

        print(
            f"[{model_number:3d}/{total_models}] "
            f"N={N:2d}, m={m}, "
            f"pixels={n_features:3d}, "
            f"test NRMSE={test_nrmse:.4f}, "
            f"test R²={test_r2:.4f}"
        )

results_df = pd.DataFrame(results)

print("\nExperiment complete.")
display(results_df.head())


# ------------------------------------------------------------
# 9. Test error as a function of N and m
# ------------------------------------------------------------

error_matrix = (
    results_df
    .pivot(
        index="compression_factor",
        columns="patch_size",
        values="test_nrmse"
    )
    .sort_index()
)

fig, ax = plt.subplots(
    1,
    2,
    figsize=(17, 5)
)

# Heatmap
heatmap = ax[0].imshow(
    error_matrix.values,
    origin="lower",
    aspect="auto",
    cmap="viridis"
)

ax[0].set_xticks(
    np.arange(len(PATCH_SIZES))
)

ax[0].set_xticklabels(
    PATCH_SIZES
)

ax[0].set_yticks(
    np.arange(len(COMPRESSION_FACTORS))
)

ax[0].set_yticklabels(
    COMPRESSION_FACTORS
)

ax[0].set_xlabel("Original patch size N")
ax[0].set_ylabel("Compression factor m")
ax[0].set_title("Full-spectrum test NRMSE")

colorbar = fig.colorbar(
    heatmap,
    ax=ax[0]
)

colorbar.set_label(
    "Test NRMSE — lower is better"
)

# Error curves
for m in COMPRESSION_FACTORS:

    subset = results_df[
        results_df["compression_factor"] == m
    ].sort_values("patch_size")

    ax[1].plot(
        subset["patch_size"],
        subset["test_nrmse"],
        marker="o",
        linewidth=2,
        label=f"m = {m}"
    )

ax[1].set_xlabel("Original patch size N")
ax[1].set_ylabel("Test NRMSE")
ax[1].set_title("Prediction error versus patch size")
ax[1].grid(alpha=0.25)
ax[1].legend(
    title="Compression"
)

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 10. Calculate the Pareto front
# ------------------------------------------------------------

# A point is Pareto optimal if no other point has both:
#   1. an equal or smaller number of features, and
#   2. an equal or smaller test error,
# with at least one strict improvement.

def calculate_pareto_front(
    dataframe,
    complexity_column="n_features",
    error_column="test_nrmse"
):
    """
    Return one representative model at each Pareto-optimal point.
    Both complexity and error are minimized.
    """

    # First keep the lowest-error model for each complexity
    candidates = (
        dataframe
        .sort_values([
            complexity_column,
            error_column
        ])
        .groupby(
            complexity_column,
            as_index=False
        )
        .first()
        .sort_values(complexity_column)
    )

    pareto_rows = []
    best_error_so_far = np.inf

    for _, row in candidates.iterrows():

        current_error = row[error_column]

        if current_error < best_error_so_far:
            pareto_rows.append(row)
            best_error_so_far = current_error

    return pd.DataFrame(pareto_rows)


pareto_df = calculate_pareto_front(
    results_df
)

print("Pareto-optimal models:")

display(
    pareto_df[[
        "patch_size",
        "compression_factor",
        "compressed_height",
        "compressed_width",
        "n_features",
        "test_nrmse",
        "test_r2",
        "alpha"
    ]]
)


# ------------------------------------------------------------
# 11. Plot all models and the Pareto front
# ------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(10, 7)
)

# Plot every model, colored by compression factor
for m in COMPRESSION_FACTORS:

    subset = results_df[
        results_df["compression_factor"] == m
    ]

    ax.scatter(
        subset["n_features"],
        subset["test_nrmse"],
        s=65,
        alpha=0.65,
        label=f"m = {m}"
    )

# Connect Pareto-optimal points
ax.plot(
    pareto_df["n_features"],
    pareto_df["test_nrmse"],
    color="black",
    marker="o",
    markersize=8,
    linewidth=2.5,
    label="Pareto front",
    zorder=10
)

# Annotate Pareto models
for _, row in pareto_df.iterrows():

    label = (
        f"N={int(row['patch_size'])}, "
        f"m={int(row['compression_factor'])}"
    )

    ax.annotate(
        label,
        (
            row["n_features"],
            row["test_nrmse"]
        ),
        xytext=(5, 7),
        textcoords="offset points",
        fontsize=9
    )

ax.set_xscale("log")

ax.set_xlabel(
    "Predictor complexity: number of compressed pixels"
)

ax.set_ylabel(
    "Full-spectrum test NRMSE"
)

ax.set_title(
    "Predictive error versus image-descriptor complexity"
)

ax.grid(
    alpha=0.25,
    which="both"
)

ax.legend(
    title="Average-pooling factor",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 12. Report best and simplest competitive models
# ------------------------------------------------------------

best_model = results_df.loc[
    results_df["test_nrmse"].idxmin()
]

print("Lowest-error model")
print("------------------")
print(
    f"N = {int(best_model['patch_size'])}, "
    f"m = {int(best_model['compression_factor'])}"
)
print(
    f"Compressed size = "
    f"{int(best_model['compressed_height'])} × "
    f"{int(best_model['compressed_width'])}"
)
print(
    f"Number of pixels = "
    f"{int(best_model['n_features'])}"
)
print(
    f"Test NRMSE = "
    f"{best_model['test_nrmse']:.5f}"
)
print(
    f"Test R² = "
    f"{best_model['test_r2']:.5f}"
)

### Final question

> Which descriptor–regressor combination gives the best predictive performance while remaining simple and explainable?

Let's roll! First, let's expplore the significance of the features using regular SHAP

In [ ]:
# ============================================================
# PATCH -> FULL SPECTRUM REGRESSION WITH PIXEL-RESOLVED SHAP
# Requires:
#     image   : (H, W)
#     spectra : (H, W, n_energy)
#     energy  : (n_energy,)
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score, mean_squared_error


# ------------------------------------------------------------
# 1. User settings
# ------------------------------------------------------------

PATCH_SIZE = 15

# SHAP values will be integrated over this spectral interval.
# Change these values to analyze a different spectral feature.
SHAP_E1 = 0.5
SHAP_E2 = 1.0

# Number of examples to display
N_EXAMPLES = 5

# Spatial block size used for train/test separation
# Larger blocks provide a stricter spatial generalization test.
SPATIAL_BLOCK_SIZE = 4 * PATCH_SIZE

RANDOM_STATE = 0


# ------------------------------------------------------------
# 2. Construct patch-spectrum pairs
# ------------------------------------------------------------

H, W = image.shape
n_energy = spectra.shape[-1]

half = PATCH_SIZE // 2

patches = []
spectrum_targets = []
patch_centers = []

for y in range(half, H - half):
    for x in range(half, W - half):

        patch = image[
            y - half:y + half + 1,
            x - half:x + half + 1
        ]

        spectrum = spectra[y, x, :]

        patches.append(patch)
        spectrum_targets.append(spectrum)
        patch_centers.append((y, x))

patches = np.asarray(patches)
spectrum_targets = np.asarray(spectrum_targets)
patch_centers = np.asarray(patch_centers)

print("Patches:", patches.shape)
print("Spectrum targets:", spectrum_targets.shape)
print("Patch centers:", patch_centers.shape)


# ------------------------------------------------------------
# 3. Convert patches to pixel-feature vectors
# ------------------------------------------------------------

# Each pixel in the patch is treated as an input feature.
X = patches.reshape(len(patches), -1)
Y = spectrum_targets

feature_names = [
    f"pixel_y{iy}_x{ix}"
    for iy in range(PATCH_SIZE)
    for ix in range(PATCH_SIZE)
]

print("Feature matrix:", X.shape)
print("Target matrix:", Y.shape)


# ------------------------------------------------------------
# 4. Spatially separated train/test split
# ------------------------------------------------------------

# Assign neighboring patch centers to the same spatial block.
n_blocks_x = int(np.ceil(W / SPATIAL_BLOCK_SIZE))

block_y = patch_centers[:, 0] // SPATIAL_BLOCK_SIZE
block_x = patch_centers[:, 1] // SPATIAL_BLOCK_SIZE

spatial_groups = block_y * n_blocks_x + block_x

unique_groups = np.unique(spatial_groups)

if len(unique_groups) >= 4:

    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=0.25,
        random_state=RANDOM_STATE
    )

    train_ids, test_ids = next(
        splitter.split(X, Y, groups=spatial_groups)
    )

    split_name = "spatial-block split"

else:
    # Fallback for unusually small images
    from sklearn.model_selection import train_test_split

    all_ids = np.arange(len(X))

    train_ids, test_ids = train_test_split(
        all_ids,
        test_size=0.25,
        random_state=RANDOM_STATE
    )

    split_name = "random split (small-image fallback)"

X_train = X[train_ids]
X_test = X[test_ids]

Y_train = Y[train_ids]
Y_test = Y[test_ids]

print("Split:", split_name)
print("Training samples:", len(train_ids))
print("Test samples:", len(test_ids))


# ------------------------------------------------------------
# 5. Standardize the pixel features
# ------------------------------------------------------------

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# ------------------------------------------------------------
# 6. Train patch -> spectrum regressor
# ------------------------------------------------------------

# RidgeCV selects the regularization parameter using the
# training data. Ridge supports multi-output regression directly.
model = RidgeCV(
    alphas=np.logspace(-4, 4, 17)
)

model.fit(X_train_scaled, Y_train)

Y_train_pred = model.predict(X_train_scaled)
Y_test_pred = model.predict(X_test_scaled)

train_r2 = r2_score(
    Y_train,
    Y_train_pred,
    multioutput="variance_weighted"
)

test_r2 = r2_score(
    Y_test,
    Y_test_pred,
    multioutput="variance_weighted"
)

test_rmse = np.sqrt(
    mean_squared_error(Y_test, Y_test_pred)
)

# R² at every energy channel
r2_by_energy = np.array([
    r2_score(Y_test[:, i], Y_test_pred[:, i])
    if np.var(Y_test[:, i]) > 0
    else np.nan
    for i in range(n_energy)
])

print(f"Selected Ridge alpha: {model.alpha_:.4g}")
print(f"Train variance-weighted R²: {train_r2:.4f}")
print(f"Test variance-weighted R²:  {test_r2:.4f}")
print(f"Test RMSE:                  {test_rmse:.4g}")


# ------------------------------------------------------------
# 7. Plot predictive performance across the spectrum
# ------------------------------------------------------------

fig, ax = plt.subplots(1, 2, figsize=(13, 4))

ax[0].plot(
    energy,
    Y_test.mean(axis=0),
    color="black",
    linewidth=2,
    label="Measured test mean"
)

ax[0].plot(
    energy,
    Y_test_pred.mean(axis=0),
    color="tab:orange",
    linewidth=2,
    label="Predicted test mean"
)

ax[0].axvspan(
    SHAP_E1,
    SHAP_E2,
    color="tab:purple",
    alpha=0.15,
    label="SHAP integration window"
)

ax[0].set_xlabel("Energy")
ax[0].set_ylabel("Mean intensity")
ax[0].set_title("Mean measured and predicted spectra")
ax[0].legend()

ax[1].plot(energy, r2_by_energy, color="tab:blue")
ax[1].axhline(0, color="gray", linestyle="--")
ax[1].axvspan(
    SHAP_E1,
    SHAP_E2,
    color="tab:purple",
    alpha=0.15
)

ax[1].set_xlabel("Energy")
ax[1].set_ylabel("Test R²")
ax[1].set_title("Predictive power by energy channel")

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 8. Exact SHAP values for the linear multi-output model
# ------------------------------------------------------------

# For a linear model in standardized feature space:
#
# SHAP(sample, feature, energy) =
#     coefficient(feature, energy)
#     × [sample_feature - background_mean_feature]
#
# model.coef_ has shape:
#     (n_energy, n_features)

background_mean = X_train_scaled.mean(axis=0)

# Use examples spanning low to high prediction quality.
sample_errors = np.mean(
    (Y_test - Y_test_pred) ** 2,
    axis=1
)

ranked_ids = np.argsort(sample_errors)

# Select examples across the error distribution rather than
# showing only the best or worst cases.
positions = np.linspace(
    0,
    len(ranked_ids) - 1,
    min(N_EXAMPLES, len(ranked_ids)),
    dtype=int
)

example_local_ids = ranked_ids[positions]

X_examples_scaled = X_test_scaled[example_local_ids]
Y_examples = Y_test[example_local_ids]
Y_examples_pred = Y_test_pred[example_local_ids]

example_global_ids = test_ids[example_local_ids]
example_patches = patches[example_global_ids]
example_centers = patch_centers[example_global_ids]

# Shape:
# (n_examples, n_features, n_energy)
shap_spectral = (
    (X_examples_scaled - background_mean)[:, :, None]
    * model.coef_.T[None, :, :]
)

print("Spectral SHAP tensor:", shap_spectral.shape)


# ------------------------------------------------------------
# 9. Integrate spectral SHAP values over chosen energy window
# ------------------------------------------------------------

shap_energy_mask = (
    (energy >= SHAP_E1) &
    (energy <= SHAP_E2)
)

if shap_energy_mask.sum() < 2:
    raise ValueError(
        "The selected SHAP interval contains fewer than two "
        "energy points. Choose a wider interval."
    )

# Each value now tells us how one image pixel changes the
# predicted integrated spectral response in the selected window.
shap_window = np.trapezoid(
    shap_spectral[:, :, shap_energy_mask],
    energy[shap_energy_mask],
    axis=2
)

shap_images = shap_window.reshape(
    -1,
    PATCH_SIZE,
    PATCH_SIZE
)


# ------------------------------------------------------------
# 10. Verify SHAP additivity for the selected spectral window
# ------------------------------------------------------------

# Background prediction in standardized feature coordinates
background_spectrum = (
    model.intercept_
    + model.coef_ @ background_mean
)

background_window = np.trapezoid(
    background_spectrum[shap_energy_mask],
    energy[shap_energy_mask]
)

predicted_windows = np.trapezoid(
    Y_examples_pred[:, shap_energy_mask],
    energy[shap_energy_mask],
    axis=1
)

shap_reconstructed_windows = (
    background_window
    + shap_window.sum(axis=1)
)

max_additivity_error = np.max(
    np.abs(
        predicted_windows
        - shap_reconstructed_windows
    )
)

print(
    "Maximum SHAP additivity error:",
    f"{max_additivity_error:.3e}"
)


# ------------------------------------------------------------
# 11. Plot patch, SHAP image, and spectrum together
# ------------------------------------------------------------

n_show = len(example_local_ids)

fig, axes = plt.subplots(
    n_show,
    3,
    figsize=(14, 3.2 * n_show),
    squeeze=False
)

# Shared symmetric color range for comparable SHAP images
shap_limit = np.percentile(
    np.abs(shap_images),
    99
)

if shap_limit == 0:
    shap_limit = 1.0

for row in range(n_show):

    patch = example_patches[row]
    shap_image = shap_images[row]

    measured_spectrum = Y_examples[row]
    predicted_spectrum = Y_examples_pred[row]

    y_center, x_center = example_centers[row]

    # Original image patch
    axes[row, 0].imshow(
        patch,
        cmap="gray"
    )

    axes[row, 0].set_title(
        f"Image patch\ncenter = ({y_center}, {x_center})"
    )

    axes[row, 0].axis("off")

    # Pixel-resolved SHAP image
    shap_plot = axes[row, 1].imshow(
        shap_image,
        cmap="coolwarm",
        vmin=-shap_limit,
        vmax=shap_limit
    )

    axes[row, 1].set_title(
        f"SHAP image for {SHAP_E1:g}–{SHAP_E2:g}"
    )

    axes[row, 1].axis("off")

    # Measured and predicted spectra
    axes[row, 2].plot(
        energy,
        measured_spectrum,
        color="black",
        linewidth=2,
        label="Measured"
    )

    axes[row, 2].plot(
        energy,
        predicted_spectrum,
        color="tab:orange",
        linewidth=2,
        label="Predicted"
    )

    axes[row, 2].axvspan(
        SHAP_E1,
        SHAP_E2,
        color="tab:purple",
        alpha=0.15
    )

    example_r2 = r2_score(
        measured_spectrum,
        predicted_spectrum
    )

    axes[row, 2].set_title(
        f"Measured vs. predicted spectrum\n"
        f"single-spectrum R² = {example_r2:.3f}"
    )

    axes[row, 2].set_xlabel("Energy")
    axes[row, 2].set_ylabel("Intensity")
    axes[row, 2].legend()

colorbar = fig.colorbar(
    shap_plot,
    ax=axes[:, 1],
    fraction=0.025,
    pad=0.02
)

colorbar.set_label(
    "Pixel contribution to predicted\n"
    "integrated spectral response"
)

plt.tight_layout()
plt.show()

Now,
- can we analyze the SHAP images and see how rapidly they decay away from the center?
- or do PCA analysis on SHAP
- or do CCA between image and SHAP?
and us eit to learn what goes on?


Now, let's explroe simple compression and patch size

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

# 1. Settings and feasible limits
MAX_N, MAX_M = 20, 5
TEST_FRACTION = 0.25
MIN_TRAIN, MIN_TEST = 50, 20
ALPHAS = np.logspace(-3, 3, 9)

img = np.asarray(image, dtype=float)
cube = np.asarray(spectra, dtype=float)
H, W = img.shape
assert cube.shape[:2] == img.shape

# Fixed boundary: training on the left, testing on the right.
boundary = int(W * (1 - TEST_FRACTION))

feasible = [
    n for n in range(1, min(MAX_N, H, boundary, W - boundary) + 1)
    if (H - n + 1) * (boundary - n + 1) >= MIN_TRAIN
    and (H - n + 1) * (W - boundary - n + 1) >= MIN_TEST
]
if not feasible:
    raise ValueError(
        "Image too small for the requested patch counts. "
        "Reduce MIN_TRAIN / MIN_TEST or change TEST_FRACTION."
    )

N_MAX = max(feasible)
left = (N_MAX - 1) // 2
right = N_MAX - left

def centers(x_start, x_stop):
    yy, xx = np.meshgrid(
        np.arange(left, H - right + 1),
        np.arange(x_start + left, x_stop - right + 1),
        indexing="ij"
    )
    return yy.ravel(), xx.ravel()

train_xy = centers(0, boundary)
test_xy = centers(boundary, W)
Y_train, Y_test = cube[train_xy], cube[test_xy]

print(f"Image: {H} × {W}; fixed split at column {boundary}")
print(f"Feasible N: 1–{N_MAX}; m: 1–min({MAX_M}, N)")
print(f"Common patches: {len(Y_train)} train, {len(Y_test)} test")


# 2. Extract and compress patches
def patches_at(coords, n):
    a = (n - 1) // 2
    return np.stack([
        img[y-a:y-a+n, x-a:x-a+n]
        for y, x in zip(*coords)
    ])

def compress(patches, m):
    # Average pooling; replicate edges if N is not divisible by m.
    k = (patches.shape[1] + m - 1) // m
    pad = k * m - patches.shape[1]
    p = np.pad(patches, ((0, 0), (0, pad), (0, pad)), mode="edge")
    return p.reshape(-1, k, m, k, m).mean(axis=(2, 4)).reshape(-1, k*k)


# 3. Benchmark every feasible (N, m)
# Normalize by the error of the training-mean-spectrum baseline.
baseline_rmse = np.sqrt(np.mean((Y_test - Y_train.mean(axis=0))**2))
if baseline_rmse == 0:
    raise ValueError("Zero baseline error: normalized error is undefined.")

rows = []
for n in range(1, N_MAX + 1):
    train_patches = patches_at(train_xy, n)
    test_patches = patches_at(test_xy, n)

    for m in range(1, min(MAX_M, n) + 1):
        X_train = compress(train_patches, m)
        X_test = compress(test_patches, m)

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        model = RidgeCV(alphas=ALPHAS).fit(X_train, Y_train)
        rmse = np.sqrt(mean_squared_error(Y_test, model.predict(X_test)))

        rows.append({
            "N": n, "m": m, "pixels": X_train.shape[1],
            "RMSE": rmse, "NRMSE": rmse / baseline_rmse,
            "alpha": float(model.alpha_)
        })

    print(f"Completed N={n}/{N_MAX}")

results = pd.DataFrame(rows)


# 4. Pareto front: minimize pixel count and prediction error
candidates = (
    results.sort_values(["pixels", "NRMSE"])
    .drop_duplicates("pixels")
)
previous_best = candidates["NRMSE"].cummin().shift(fill_value=np.inf)
pareto = candidates[candidates["NRMSE"] < previous_best]


# 5. Error heatmap, error curves, and all points + Pareto front
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

grid = results.pivot(index="m", columns="N", values="NRMSE")
im = axes[0].imshow(
    grid.values, origin="lower", aspect="auto",
    extent=(0.5, N_MAX + 0.5, 0.5, len(grid) + 0.5),
    cmap="viridis"
)
axes[0].set(
    xlabel="Patch size N", ylabel="Compression m",
    title="Test error: lower is better"
)
axes[0].set_yticks(grid.index)
fig.colorbar(im, ax=axes[0], label="NRMSE")

for m, group in results.groupby("m"):
    axes[1].plot(group.N, group.NRMSE, "o-", ms=3, label=f"m={m}")
    axes[2].scatter(
        group.pixels, group.NRMSE, s=30, alpha=0.65, label=f"m={m}"
    )

axes[1].set(xlabel="Patch size N", ylabel="NRMSE", title="Error versus patch size")
axes[1].legend()

axes[2].plot(pareto.pixels, pareto.NRMSE, "ko-", label="Pareto front")
axes[2].set(
    xscale="log", xlabel="Number of compressed pixels",
    ylabel="NRMSE", title="Complexity–error trade-off"
)
axes[2].legend()
for ax in axes[1:]:
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

print("\nPareto-optimal configurations:")
display(pareto.reset_index(drop=True))

Now, let's explore different regressors and see how they behave

In [ ]:
# Requires Sections 1–2 from the previous block:
# N_MAX, MAX_M, train_xy, test_xy, Y_train, Y_test,
# patches_at(), compress()

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.cross_decomposition import PLSRegression
from sklearn.kernel_ridge import KernelRidge
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from IPython.display import display


# 3. Regressor definitions — fixed initial hyperparameters
SEED = 0
N_TREES = 200

def make_regressors(n_samples, n_features):
    return {
        "Ridge": Ridge(alpha=1.0),

        "PLS": PLSRegression(
            n_components=min(5, n_features, n_samples - 1),
            scale=False,
            max_iter=1000
        ),

        "RBF Kernel Ridge": KernelRidge(
            kernel="rbf",
            alpha=1.0,
            gamma=1.0 / n_features
        ),

        "Extra Trees": ExtraTreesRegressor(
            n_estimators=N_TREES,
            min_samples_leaf=3,
            max_features=1.0,
            random_state=SEED,
            n_jobs=-1
        ),

        "Random Forest": RandomForestRegressor(
            n_estimators=N_TREES,
            min_samples_leaf=3,
            max_features=1.0,
            random_state=SEED,
            n_jobs=-1
        )
    }


# 4. Run all regressors on identical patch representations
if not (np.isfinite(Y_train).all() and np.isfinite(Y_test).all()):
    raise ValueError("Spectra contain NaN or infinite values.")

baseline_rmse = np.sqrt(np.mean((Y_test - Y_train.mean(axis=0))**2))
if baseline_rmse <= 0:
    raise ValueError("Zero baseline error: NRMSE is undefined.")

# Center targets using training data only.
# Particularly important for kernel Ridge, which has no intercept.
target_mean = Y_train.mean(axis=0)
Y_fit = Y_train - target_mean

rows = []

for n in range(1, N_MAX + 1):
    train_patches = patches_at(train_xy, n)
    test_patches = patches_at(test_xy, n)

    for m in range(1, min(MAX_M, n) + 1):
        X_train = compress(train_patches, m)
        X_test = compress(test_patches, m)

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        if not (np.isfinite(X_train).all() and np.isfinite(X_test).all()):
            raise ValueError(f"Nonfinite image features at N={n}, m={m}.")

        models = make_regressors(*X_train.shape)

        # Avoid asking PLS for more components than the input rank.
        rank = np.linalg.matrix_rank(X_train)
        if rank:
            models["PLS"].set_params(
                n_components=min(5, rank, len(X_train) - 1)
            )

        for name, model in models.items():
            start = time.perf_counter()

            if rank == 0:
                # Constant patches cannot predict spatial variation.
                prediction = np.broadcast_to(target_mean, Y_test.shape)
                fit_seconds = 0.0
                predict_seconds = 0.0
            else:
                model.fit(X_train, Y_fit)
                fit_seconds = time.perf_counter() - start

                start = time.perf_counter()
                prediction = model.predict(X_test) + target_mean
                predict_seconds = time.perf_counter() - start

            rmse = np.sqrt(np.mean((Y_test - prediction)**2))
            if not np.isfinite(rmse):
                raise RuntimeError(f"Nonfinite error: {name}, N={n}, m={m}")

            rows.append({
                "method": name,
                "N": n,
                "m": m,
                "pixels": X_train.shape[1],
                "RMSE": rmse,
                "NRMSE": rmse / baseline_rmse,
                "fit_seconds": fit_seconds,
                "predict_seconds": predict_seconds
            })

    print(f"Completed N={n}/{N_MAX} for all five regressors")

results = pd.DataFrame(rows)
methods = list(results["method"].unique())


# 5. Pareto fronts: minimize both pixel count and error
def pareto_front(table):
    candidates = (
        table.sort_values(["pixels", "NRMSE", "method", "N", "m"])
        .drop_duplicates("pixels")
    )
    previous_best = candidates.NRMSE.cummin().shift(fill_value=np.inf)
    return candidates.loc[candidates.NRMSE < previous_best].copy()

fronts = {
    name: pareto_front(results[results.method == name])
    for name in methods
}
combined_front = pareto_front(results)


# 6. Three plots per method, with common scales
fig, axes = plt.subplots(
    len(methods), 3,
    figsize=(18, 4 * len(methods)),
    constrained_layout=True
)

error_min = results.NRMSE.min()
error_max = results.NRMSE.max()
margin = max(0.02, 0.05 * (error_max - error_min))
m_max = min(MAX_M, N_MAX)

pool_colors = {
    m: plt.get_cmap("tab10")(m - 1)
    for m in range(1, m_max + 1)
}

for row, name in enumerate(methods):
    subset = results[results.method == name]
    front = fronts[name]
    heat_ax, curve_ax, pareto_ax = axes[row]

    grid = subset.pivot(index="m", columns="N", values="NRMSE")
    grid = grid.reindex(
        index=range(1, m_max + 1),
        columns=range(1, N_MAX + 1)
    )

    im = heat_ax.imshow(
        grid.values,
        origin="lower",
        aspect="auto",
        extent=(0.5, N_MAX + 0.5, 0.5, m_max + 0.5),
        cmap="viridis",
        vmin=error_min,
        vmax=error_max
    )
    heat_ax.set(
        title=f"{name}: test-error heatmap",
        xlabel="Patch size N",
        ylabel="Compression factor m"
    )
    heat_ax.set_yticks(range(1, m_max + 1))
    fig.colorbar(im, ax=heat_ax, label="NRMSE")

    for m, group in subset.groupby("m"):
        color = pool_colors[m]
        curve_ax.plot(
            group.N, group.NRMSE, "o-",
            color=color, ms=3, label=f"m={m}"
        )
        pareto_ax.scatter(
            group.pixels, group.NRMSE,
            color=color, s=30, alpha=0.65
        )

    pareto_ax.plot(
        front.pixels, front.NRMSE,
        "ko-", lw=2, ms=5, label="Pareto front"
    )

    curve_ax.set(
        title=f"{name}: patch-size dependence",
        xlabel="Patch size N", ylabel="NRMSE"
    )
    pareto_ax.set(
        title=f"{name}: all configurations + Pareto front",
        xscale="log",
        xlabel="Number of compressed pixels",
        ylabel="NRMSE"
    )

    for ax in (curve_ax, pareto_ax):
        ax.set_ylim(max(0, error_min - margin), error_max + margin)
        ax.grid(alpha=0.2)
        ax.legend(fontsize=8)

plt.show()


# 7. Combined comparison across all five methods
fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)

for i, name in enumerate(methods):
    color = plt.get_cmap("tab10")(i)
    subset = results[results.method == name]
    front = fronts[name]

    axes[0].scatter(
        subset.pixels, subset.NRMSE,
        color=color, s=25, alpha=0.4, label=name
    )
    axes[1].plot(
        front.pixels, front.NRMSE,
        "o-", color=color, ms=5, label=name
    )

axes[0].plot(
    combined_front.pixels, combined_front.NRMSE,
    "k*-", lw=2, ms=10, label="Combined Pareto front"
)

axes[0].set_title("All configurations: global Pareto front")
axes[1].set_title("Pareto fronts by regressor")

for ax in axes:
    ax.set(
        xscale="log",
        xlabel="Number of compressed pixels",
        ylabel="Test NRMSE"
    )
    ax.grid(alpha=0.2)
    ax.legend(fontsize=9)

plt.show()


# 8. Summary tables
columns = ["method", "N", "m", "pixels", "NRMSE", "RMSE", "fit_seconds"]

print("Lowest-error configuration for each method:")
best_ids = results.groupby("method").NRMSE.idxmin()
display(results.loc[best_ids, columns].sort_values("NRMSE"))

print("Combined Pareto front:")
display(combined_front[columns].reset_index(drop=True))

print("Pareto configurations for each method:")
display(pd.concat(fronts.values(), ignore_index=True)[columns])

Now, let's introduce area scalarizer and try the same logic.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.cross_decomposition import PLSRegression
from sklearn.kernel_ridge import KernelRidge
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from IPython.display import display


# ============================================================
# 1. SETTINGS
# ============================================================
PEAK_E1, PEAK_E2 = 0.5, 1.0   # Change to encompass the plasmon peak
SUBTRACT_BACKGROUND = True    # Straight line between window endpoints

MAX_N, MAX_M = 20, 5
TEST_FRACTION = 0.25
MIN_TRAIN, MIN_TEST = 50, 20   # Patch counts, not independent samples

N_TREES = 200
SEED = 0


# ============================================================
# 2. VALIDATE DATA AND DEFINE THE PEAK-AREA SCALARIZER
# ============================================================
img = np.asarray(image, dtype=float)
cube = np.asarray(spectra, dtype=float)
E = np.asarray(energy, dtype=float).ravel()

if img.ndim != 2 or cube.ndim != 3:
    raise ValueError("Expected image (H,W) and spectra (H,W,n_energy).")
if cube.shape[:2] != img.shape or cube.shape[-1] != len(E):
    raise ValueError("Image, spectra, and energy shapes are inconsistent.")
if not all(np.isfinite(a).all() for a in (img, cube, E)):
    raise ValueError("Inputs contain NaN or infinite values.")

order = np.argsort(E)
E, cube = E[order], cube[..., order]
if np.any(np.diff(E) <= 0):
    raise ValueError("Energy values must be distinct.")

mask = (E >= PEAK_E1) & (E <= PEAK_E2)
if mask.sum() < 3:
    raise ValueError("Peak window must contain at least three energy channels.")

e = E[mask]
peak = cube[..., mask]
t = (e - e[0]) / (e[-1] - e[0])

background = (
    peak[..., :1] * (1 - t) + peak[..., -1:] * t
    if SUBTRACT_BACKGROUND else np.zeros_like(peak)
)
scalarizer_map = np.trapezoid(peak - background, x=e, axis=-1)

fig, ax = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
ax[0].plot(E, cube.mean(axis=(0, 1)), "k-", label="Mean spectrum")
ax[0].plot(
    e, background.mean(axis=(0, 1)), "--",
    color="tab:orange", label="Integration baseline"
)
ax[0].fill_between(
    e, background.mean(axis=(0, 1)), peak.mean(axis=(0, 1)),
    alpha=0.3, label="Peak area"
)
ax[0].set(xlabel="Energy", ylabel="Intensity", title="Check peak window")
ax[0].legend()

im = ax[1].imshow(scalarizer_map, cmap="viridis")
ax[1].set_title("Plasmon peak area")
fig.colorbar(im, ax=ax[1], label="Intensity × energy")
plt.show()


# ============================================================
# 3. FIXED SPATIAL SPLIT AND FEASIBLE PATCH LIMITS
# ============================================================
H, W = img.shape
boundary = int(W * (1 - TEST_FRACTION))

# Patches must fit entirely inside their training or test region.
feasible = [
    n for n in range(1, min(MAX_N, H, boundary, W - boundary) + 1)
    if (H - n + 1) * (boundary - n + 1) >= MIN_TRAIN
    and (H - n + 1) * (W - boundary - n + 1) >= MIN_TEST
]
if not feasible:
    raise ValueError(
        "Insufficient patches. Reduce MIN_TRAIN/MIN_TEST "
        "or change TEST_FRACTION."
    )

N_MAX = max(feasible)
left = (N_MAX - 1) // 2
right = N_MAX - left

def centers(x_start, x_stop):
    yy, xx = np.meshgrid(
        np.arange(left, H - right + 1),
        np.arange(x_start + left, x_stop - right + 1),
        indexing="ij"
    )
    return yy.ravel(), xx.ravel()

# Common centers for every N, m, and regressor.
train_xy = centers(0, boundary)
test_xy = centers(boundary, W)

y_train = scalarizer_map[train_xy]
y_test = scalarizer_map[test_xy]

print(f"Image: {H} × {W}; train/test boundary: column {boundary}")
print(f"Patch sizes: 1–{N_MAX}; compression: 1–min({MAX_M}, N)")
print(f"Training patches: {len(y_train)}; test patches: {len(y_test)}")
print(f"Integration interval sampled: {e[0]:g}–{e[-1]:g}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.imshow(img, cmap="gray")
ax.axvline(boundary - 0.5, color="red", linestyle="--")
ax.scatter(train_xy[1], train_xy[0], s=3, label="Training centers")
ax.scatter(test_xy[1], test_xy[0], s=3, label="Test centers")
ax.set_title("Fixed spatial split: no train/test patch overlap")
ax.legend()
plt.show()


# ============================================================
# 4. PATCH EXTRACTION, COMPRESSION, AND REGRESSORS
# ============================================================
def patches_at(coords, n):
    a = (n - 1) // 2
    return np.stack([
        img[y-a:y-a+n, x-a:x-a+n]
        for y, x in zip(*coords)
    ])

def compress(patches, m):
    """m × m average pooling; replicate bottom/right edges if needed."""
    k = (patches.shape[1] + m - 1) // m
    pad = k * m - patches.shape[1]
    p = np.pad(patches, ((0, 0), (0, pad), (0, pad)), mode="edge")
    return p.reshape(-1, k, m, k, m).mean(axis=(2, 4)).reshape(-1, k*k)

def make_regressors(d, rank, n_samples):
    return {
        "Ridge": Ridge(alpha=1.0),
        "PLS": PLSRegression(
            n_components=max(1, min(5, rank, n_samples - 1)),
            scale=False, max_iter=1000
        ),
        "RBF Kernel Ridge": KernelRidge(
            kernel="rbf", alpha=1.0, gamma=1.0 / d
        ),
        "Extra Trees": ExtraTreesRegressor(
            n_estimators=N_TREES, min_samples_leaf=3,
            max_features=1.0, random_state=SEED, n_jobs=-1
        ),
        "Random Forest": RandomForestRegressor(
            n_estimators=N_TREES, min_samples_leaf=3,
            max_features=1.0, random_state=SEED, n_jobs=-1
        )
    }


# ============================================================
# 5. BENCHMARK ALL FIVE METHODS
# ============================================================
target_mean = y_train.mean()
y_fit = y_train - target_mean

baseline_rmse = np.sqrt(np.mean((y_test - target_mean)**2))
if baseline_rmse == 0:
    raise ValueError("Zero baseline error: normalized error is undefined.")

test_variance = np.mean((y_test - y_test.mean())**2)
rows = []

for n in range(1, N_MAX + 1):
    train_patches = patches_at(train_xy, n)
    test_patches = patches_at(test_xy, n)

    for m in range(1, min(MAX_M, n) + 1):
        scaler = StandardScaler()
        X_train = scaler.fit_transform(compress(train_patches, m))
        X_test = scaler.transform(compress(test_patches, m))
        rank = np.linalg.matrix_rank(X_train)

        for name, model in make_regressors(
            X_train.shape[1], rank, len(X_train)
        ).items():
            start = time.perf_counter()

            if rank == 0 or np.ptp(y_train) == 0:
                prediction = np.full(len(y_test), target_mean)
                fit_seconds = predict_seconds = 0.0
            else:
                model.fit(X_train, y_fit)
                fit_seconds = time.perf_counter() - start

                start = time.perf_counter()
                prediction = np.asarray(model.predict(X_test)).ravel()
                prediction += target_mean
                predict_seconds = time.perf_counter() - start

            mse = np.mean((y_test - prediction)**2)
            if not np.isfinite(mse):
                raise RuntimeError(f"Nonfinite error: {name}, N={n}, m={m}")

            rows.append({
                "method": name, "N": n, "m": m,
                "pixels": X_train.shape[1],
                "RMSE": np.sqrt(mse),
                "NRMSE": np.sqrt(mse) / baseline_rmse,
                "R2": 1 - mse / test_variance if test_variance > 0 else np.nan,
                "fit_seconds": fit_seconds,
                "predict_seconds": predict_seconds
            })

    print(f"Completed N={n}/{N_MAX}")

results = pd.DataFrame(rows)
methods = list(results.method.unique())


# ============================================================
# 6. PARETO FRONTS
# ============================================================
def pareto_front(table):
    candidates = (
        table.sort_values(["pixels", "NRMSE", "method", "N", "m"])
        .drop_duplicates("pixels")
    )
    previous_best = candidates.NRMSE.cummin().shift(fill_value=np.inf)
    return candidates.loc[candidates.NRMSE < previous_best].copy()

fronts = {
    name: pareto_front(results[results.method == name])
    for name in methods
}
combined_front = pareto_front(results)


# ============================================================
# 7. HEATMAP, ERROR CURVES, AND PARETO PLOT FOR EACH METHOD
# ============================================================
fig, axes = plt.subplots(
    len(methods), 3, figsize=(18, 4 * len(methods)),
    constrained_layout=True
)

vmin, vmax = results.NRMSE.min(), results.NRMSE.max()
margin = max(0.02, 0.05 * (vmax - vmin))
m_max = min(MAX_M, N_MAX)
colors = {m: plt.get_cmap("tab10")(m - 1) for m in range(1, m_max + 1)}

for row, name in enumerate(methods):
    subset, front = results[results.method == name], fronts[name]
    heat_ax, curve_ax, pareto_ax = axes[row]

    grid = subset.pivot(index="m", columns="N", values="NRMSE").reindex(
        index=range(1, m_max + 1), columns=range(1, N_MAX + 1)
    )
    im = heat_ax.imshow(
        grid.values, origin="lower", aspect="auto",
        extent=(0.5, N_MAX + 0.5, 0.5, m_max + 0.5),
        cmap="viridis", vmin=vmin, vmax=vmax
    )
    heat_ax.set(
        title=f"{name}: peak-area error",
        xlabel="Patch size N", ylabel="Compression m"
    )
    heat_ax.set_yticks(range(1, m_max + 1))
    fig.colorbar(im, ax=heat_ax, label="NRMSE")

    for m, group in subset.groupby("m"):
        curve_ax.plot(
            group.N, group.NRMSE, "o-", color=colors[m],
            ms=3, label=f"m={m}"
        )
        pareto_ax.scatter(
            group.pixels, group.NRMSE, color=colors[m],
            s=30, alpha=0.65
        )

    pareto_ax.plot(front.pixels, front.NRMSE, "ko-", label="Pareto front")
    curve_ax.set(
        title=f"{name}: patch-size dependence",
        xlabel="Patch size N", ylabel="NRMSE"
    )
    pareto_ax.set(
        title=f"{name}: complexity–error trade-off",
        xscale="log", xlabel="Compressed pixels", ylabel="NRMSE"
    )

    for ax in (curve_ax, pareto_ax):
        ax.set_ylim(max(0, vmin - margin), vmax + margin)
        ax.grid(alpha=0.2)
        ax.legend(fontsize=8)

plt.show()


# ============================================================
# 8. COMBINED COMPARISON AND SUMMARY
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)

for i, name in enumerate(methods):
    color = plt.get_cmap("tab10")(i)
    subset, front = results[results.method == name], fronts[name]

    axes[0].scatter(
        subset.pixels, subset.NRMSE,
        color=color, s=25, alpha=0.4, label=name
    )
    axes[1].plot(
        front.pixels, front.NRMSE,
        "o-", color=color, ms=5, label=name
    )

axes[0].plot(
    combined_front.pixels, combined_front.NRMSE,
    "k*-", lw=2, ms=10, label="Combined Pareto front"
)
axes[0].set_title("All configurations and combined Pareto front")
axes[1].set_title("Pareto fronts by regressor")

for ax in axes:
    ax.set(
        xscale="log", xlabel="Number of compressed pixels",
        ylabel="Peak-area test NRMSE"
    )
    ax.grid(alpha=0.2)
    ax.legend(fontsize=8)

plt.show()

columns = ["method", "N", "m", "pixels", "NRMSE", "R2", "fit_seconds"]

print("\nLowest-error configuration per method:")
best_ids = results.groupby("method").NRMSE.idxmin()
display(results.loc[best_ids, columns].sort_values("NRMSE"))

print("\nCombined Pareto front:")
display(combined_front[columns].reset_index(drop=True))

print("\nIndividual Pareto fronts:")
display(pd.concat(fronts.values(), ignore_index=True)[columns])

## Lightweight symbolic descriptor evolution

### Goal

We search for a compact mathematical description of an image patch that predicts the **plasmon peak area at its center**. The regressor is fixed; the search changes the features supplied to it.

$$\text{Image patch }P \longrightarrow \Phi(P)=[f_1(P),\ldots,f_k(P)] \longrightarrow \text{Extra Trees} \longrightarrow \hat{y}$$

Here, $y$ is the previously defined peak-area scalarizer. This cell does not change the integration window or background subtraction.

### 1. Patches and validation

The patch size is fixed at `min(7, N_MAX)`. All candidate descriptors use the same patches and target values.

The existing training locations are divided into an inner fitting subset and a spatial validation subset along the image rows. Centers near the boundary are excluded so that fitting and validation patches share no image pixels. The code stops if either subset contains fewer than 15 patches.

The existing test region is reserved for evaluation after descriptor selection.

### 2. Primitive image measurements

Each patch is converted into a small bank of measurements:

- Intensity at the target pixel.
- Mean and standard deviation over the entire patch.
- Mean and standard deviation in three radial regions.
- Mean and standard deviation in the left, right, upper, and lower regions.

Radial boundaries are at $N/6$ and $N/3$ pixels from the target pixel. The outer region includes all remaining patch pixels, including the corners. Empty regions are omitted, and measurements constant on the inner fitting subset are removed.

These are intensity-based spatial measurements; this implementation does not segment the image or calculate explicit boundary geometry.

### 3. Constructed features

A feature is either one primitive measurement or a shallow expression combining two primitives:

| Operation | Expression |
|---|---|
| Primitive | $a$ |
| Difference | $a-b$ |
| Sum | $a+b$ |
| Product | $ab$ |
| Stabilized ratio | $a/(|b|+\epsilon_b)$ |

The ratio stabilizer is determined from the inner fitting data:

$$\epsilon_b=\max\left(0.1\,\mathrm{std}_{\mathrm{fit}}(b),10^{-12}\right)$$

For example, a feature could measure central-to-outer intensity contrast or the ratio of intensity fluctuations in two regions.

A descriptor contains between one and five distinct expressions. Expressions are not nested: an arithmetic result cannot become the input to another expression.

### 4. Evolutionary search

The search uses a population of eight descriptor sets over six generations.

For each candidate:

1. Calculate its feature matrix.
2. Train an Extra Trees regressor on the inner fitting subset.
3. Predict peak area on the spatial validation subset.
4. Calculate validation error and expression cost.

Every regressor uses 64 trees, a minimum leaf size of three, all input features as split candidates, and a fixed random seed. Regressor hyperparameters are not tuned.

Successful descriptors produce new candidates through:

- **Addition:** introduce a feature.
- **Deletion:** remove a feature.
- **Mutation:** replace an expression or change one of its primitive inputs.
- **Recombination:** combine features from two surviving descriptors.

Primitive calculations and evaluated descriptors are cached. There are at most 48 candidate evaluations, and usually fewer unique evaluations because candidates can repeat.

### 5. Error and complexity

Validation error is normalized by the error obtained by predicting the inner fitting subset's mean target:

$$\mathrm{NRMSE}_{\mathrm{val}}=\frac{\sqrt{\langle(y-\hat{y})^2\rangle_{\mathrm{val}}}}{\sqrt{\langle(y-\bar{y}_{\mathrm{fit}})^2\rangle_{\mathrm{val}}}}$$

An NRMSE below one outperforms this constant baseline.

Expression costs are:

- Primitive: 1.
- Sum, difference, or product: 3.
- Stabilized ratio: 5.

The search ranks descriptors using:

$$J=\mathrm{NRMSE}_{\mathrm{val}}+0.005\,C$$

where $C$ is the summed expression cost. This favors predictive descriptors with simple formulas.

### 6. Outputs

The cell produces:

- Progress reports showing the best validation error found.
- Explicit formulas for the selected descriptor.
- A validation Pareto table.
- A plot of best validation error versus generation.
- A plot of all evaluated descriptors: feature count versus validation error, colored by expression cost.
- A measured-versus-predicted peak-area plot for the final test evaluation.

The plotted Pareto front minimizes **feature count and validation error**. Expression cost is displayed separately. Final selection uses the penalized score $J$, so it need not select the lowest-error descriptor or a point on that two-objective front.

### 7. Final model

The selected descriptor is refitted using all original training locations, including those excluded from the inner split. Its ratio stabilizers remain fixed. It is then evaluated on the existing test region.

Useful output variables are:

- `evolution_results`: all evaluated descriptors and validation scores.
- `front`: the feature-count–error Pareto front.
- `selected`: the chosen symbolic expressions.
- `final_model`: the refitted Extra Trees regressor.
- `test_prediction`: predictions for the test locations.

### Scope and interpretation

This is a small, heuristic search—not an exhaustive search or a guarantee of the optimal descriptor. It selects and combines a fixed vocabulary of spatial measurements; it does not optimize patch size, continuous ring radii, or deeply nested formulas.

The code does not calculate SHAP values. The discovered formulas are interpretable predictive features, but their predictive success alone does not establish a physical mechanism.

The test score is independent of this search only if earlier results from that region have not influenced the descriptor design. Further changes guided by that score make the region a validation set.

In [ ]:
# Requires the previous code:
# img, scalarizer_map, train_xy, test_xy, N_MAX, patches_at()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import ExtraTreesRegressor
from IPython.display import display

# ------------------------------------------------------------
# 1. Small search budget
# ------------------------------------------------------------
PATCH_N = min(7, N_MAX)
POPULATION, GENERATIONS = 8, 6
MAX_FEATURES, N_TREES = 5, 64
SEED = 0
rng = np.random.default_rng(SEED)

def regressor():
    return ExtraTreesRegressor(
        n_estimators=N_TREES,
        min_samples_leaf=3,
        max_features=1.0,
        random_state=SEED,
        n_jobs=-1
    )

# Inner split along rows, with no shared pixels across patches.
cy = train_xy[0]
cut = (cy.min() + cy.max()) // 2
a = (PATCH_N - 1) // 2
b = PATCH_N - a

fit_ids = np.flatnonzero(cy + b <= cut)
val_ids = np.flatnonzero(cy - a >= cut)

if min(len(fit_ids), len(val_ids)) < 15:
    raise ValueError(
        "Too few inner training/validation patches. "
        "Reduce PATCH_N and rerun; no random-split fallback is used."
    )

P = patches_at(train_xy, PATCH_N)
y = scalarizer_map[train_xy]
validation_scale = np.sqrt(np.mean((y[val_ids] - y[fit_ids].mean())**2))
if validation_scale <= 0:
    raise ValueError("Validation baseline error is zero.")

print(f"Patch: {PATCH_N} × {PATCH_N}")
print(f"Inner fit: {len(fit_ids)}, validation: {len(val_ids)}")
print(f"Search budget: at most {POPULATION * GENERATIONS} evaluations")


# ------------------------------------------------------------
# 2. Primitive measurements: spatial means and deviations
# ------------------------------------------------------------
n = PATCH_N
yy, xx = np.indices((n, n))
yy, xx = yy - a, xx - a
radius = np.hypot(yy, xx)

regions = {"whole": np.ones((n, n), dtype=bool)}
for i, mask in enumerate([
    radius <= n / 6,
    (radius > n / 6) & (radius <= n / 3),
    radius > n / 3
]):
    if mask.any():
        regions[f"ring{i}"] = mask

for name, mask in {
    "left": xx < 0, "right": xx > 0,
    "upper": yy < 0, "lower": yy > 0
}.items():
    if mask.any():
        regions[name] = mask

def primitive_bank(patches):
    bank = {"center": patches[:, a, a]}
    for name, mask in regions.items():
        values = patches[:, mask]
        bank[f"mean_{name}"] = values.mean(axis=1)
        bank[f"std_{name}"] = values.std(axis=1)
    return bank

bank = primitive_bank(P)

# Remove primitives constant on the inner fit subset.
keys = [
    k for k, values in bank.items()
    if np.std(values[fit_ids]) > 0
]
if not keys:
    raise ValueError("No varying primitive features in the training patches.")

# Stabilizers learned exclusively from the inner fit subset.
eps = {
    k: max(0.1 * np.std(bank[k][fit_ids]), 1e-12)
    for k in keys
}


# ------------------------------------------------------------
# 3. Expression language
# Primitive, difference, sum, product, or stabilized ratio.
# Expressions are intentionally shallow.
# ------------------------------------------------------------
OPS = ("sub", "add", "mul", "ratio")

def random_expression():
    left = str(rng.choice(keys))
    if rng.random() < 0.4:
        return ("raw", left, "")
    return (str(rng.choice(OPS)), left, str(rng.choice(keys)))

def expression_value(expr, data):
    op, left, right = expr
    x = data[left]
    if op == "raw":
        return x
    z = data[right]
    if op == "sub":
        return x - z
    if op == "add":
        return x + z
    if op == "mul":
        return x * z
    return x / (np.abs(z) + eps[right])

def expression_label(expr):
    op, left, right = expr
    if op == "raw":
        return left
    if op == "ratio":
        return f"{left} / (abs({right}) + {eps[right]:.3g})"
    symbol = {"sub": "-", "add": "+", "mul": "*"}[op]
    return f"({left} {symbol} {right})"

def canonical(expressions):
    return tuple(sorted(set(expressions)))

def expression_cost(expr):
    # Primitive=1; binary expression=3; stabilized ratio=5.
    return 1 if expr[0] == "raw" else 5 if expr[0] == "ratio" else 3

feature_cache, archive = {}, {}

def feature_matrix(descriptor):
    for expr in descriptor:
        if expr not in feature_cache:
            feature_cache[expr] = expression_value(expr, bank)
    return np.column_stack([feature_cache[e] for e in descriptor])


# ------------------------------------------------------------
# 4. Evaluate, mutate, and recombine descriptor sets
# ------------------------------------------------------------
def evaluate(descriptor):
    if descriptor not in archive:
        X = feature_matrix(descriptor)
        model = regressor().fit(X[fit_ids], y[fit_ids])
        pred = model.predict(X[val_ids])
        error = np.sqrt(np.mean((pred - y[val_ids])**2)) / validation_scale
        archive[descriptor] = {
            "features": len(descriptor),
            "cost": sum(map(expression_cost, descriptor)),
            "val_NRMSE": error
        }
    return archive[descriptor]

def mutate(descriptor):
    child = list(descriptor)
    action = str(rng.choice(["add", "drop", "change"]))

    if action == "add" and len(child) < MAX_FEATURES:
        child.append(random_expression())
    elif action == "drop" and len(child) > 1:
        child.pop(int(rng.integers(len(child))))
    else:
        i = int(rng.integers(len(child)))
        op, left, right = child[i]
        if rng.random() < 0.5:
            child[i] = random_expression()
        else:
            child[i] = (
                op, str(rng.choice(keys)),
                right if op != "raw" else ""
            )
    return canonical(child)

# A small complexity penalty guides evolution; plots retain raw error.
def search_score(descriptor):
    result = evaluate(descriptor)
    return result["val_NRMSE"] + 0.005 * result["cost"]

population = [
    (("raw", k, ""),)
    for k in keys[:min(3, POPULATION)]
]
while len(population) < POPULATION:
    population.append(canonical([random_expression()]))

history = []

for generation in range(GENERATIONS):
    ranked = sorted(set(population), key=search_score)
    survivors = ranked[:max(2, POPULATION // 2)]

    best_error = min(v["val_NRMSE"] for v in archive.values())
    history.append(best_error)
    print(
        f"Generation {generation + 1}: "
        f"{len(archive)} unique descriptors, best NRMSE={best_error:.3f}"
    )

    population = list(survivors)
    while len(population) < POPULATION:
        parent = survivors[int(rng.integers(len(survivors)))]

        if rng.random() < 0.3 and len(survivors) > 1:
            other = survivors[int(rng.integers(len(survivors)))]
            pool = list(canonical(parent + other))
            count = min(len(pool), int(rng.integers(1, MAX_FEATURES + 1)))
            ids = rng.choice(len(pool), count, replace=False)
            child = canonical([pool[i] for i in ids])
        else:
            child = mutate(parent)

        population.append(child)


# ------------------------------------------------------------
# 5. Validation Pareto front and selected expressions
# ------------------------------------------------------------
records = [
    {
        **metrics, "descriptor": descriptor,
        "formula": "; ".join(map(expression_label, descriptor))
    }
    for descriptor, metrics in archive.items()
]
evolution_results = pd.DataFrame(records)

# Two-dimensional front: feature count versus validation error.
candidates = (
    evolution_results.sort_values(["features", "val_NRMSE", "cost"])
    .drop_duplicates("features")
)
previous = candidates.val_NRMSE.cummin().shift(fill_value=np.inf)
front = candidates[candidates.val_NRMSE < previous]

# Select by the predefined error + expression-cost objective.
selected = min(archive, key=search_score)

print("\nSelected descriptor:")
for i, expr in enumerate(selected, 1):
    print(f"  f{i} = {expression_label(expr)}")

display(front[["features", "cost", "val_NRMSE", "formula"]])

fig, ax = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
ax[0].plot(range(1, GENERATIONS + 1), history, "o-")
ax[0].set(
    xlabel="Generation", ylabel="Best validation NRMSE",
    title="Descriptor evolution"
)

points = ax[1].scatter(
    evolution_results.features, evolution_results.val_NRMSE,
    c=evolution_results.cost, cmap="viridis", alpha=0.7
)
ax[1].plot(front.features, front.val_NRMSE, "k*-", label="Pareto front")
ax[1].set(
    xlabel="Number of constructed features", ylabel="Validation NRMSE",
    title="All evaluated descriptors"
)
ax[1].legend()
fig.colorbar(points, ax=ax[1], label="Expression cost")
plt.show()


# ------------------------------------------------------------
# 6. Refit selected descriptor and evaluate test region once
# ------------------------------------------------------------
X_all = feature_matrix(selected)
test_bank = primitive_bank(patches_at(test_xy, PATCH_N))
X_test = np.column_stack([
    expression_value(expr, test_bank) for expr in selected
])
y_test_final = scalarizer_map[test_xy]

final_model = regressor().fit(X_all, y)
test_prediction = final_model.predict(X_test)

test_rmse = np.sqrt(np.mean((test_prediction - y_test_final)**2))
test_baseline = np.sqrt(np.mean((y_test_final - y.mean())**2))
test_nrmse = test_rmse / test_baseline if test_baseline > 0 else np.nan

print(f"\nSelected descriptor test RMSE:  {test_rmse:.4g}")
print(f"Selected descriptor test NRMSE: {test_nrmse:.3f}")

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(y_test_final, test_prediction, s=18, alpha=0.7)
lo = min(y_test_final.min(), test_prediction.min())
hi = max(y_test_final.max(), test_prediction.max())
ax.plot([lo, hi], [lo, hi], "k--")
ax.set(
    xlabel="Measured peak area", ylabel="Predicted peak area",
    title=f"Evolved descriptor: test NRMSE={test_nrmse:.3f}"
)
plt.tight_layout()
plt.show()

## Genetic optimization of symmetric image masks

### Objective

Discover which parts of a local image patch are needed to predict the **plasmon peak area at the target location**.

We fix the patch size at $8\times8$ pixels and use the same Extra Trees regressor for every candidate. A genetic algorithm (GA) changes the spatial mask used to construct the input features.

$$\text{Image patch} \longrightarrow \text{Evolved mask} \longrightarrow \text{Features} \longrightarrow \text{Extra Trees} \longrightarrow \text{Predicted peak area}$$

### 1. Symmetry and chromosome representation

The mask is constrained to be invariant under rotations by $90^\circ$ and reflections of the square.

For an $8\times8$ patch, these symmetries produce **10 independent pixel groups**, not eight. Pixels belong to the same group if a rotation or reflection maps one onto the other.

Each candidate mask is encoded by a binary chromosome:

$$\mathbf{g}=(g_1,\ldots,g_{10}),\qquad g_j\in\{0,1\}$$

A gene value of one activates all pixels in its symmetry group. Zero excludes that group. The all-zero chromosome is excluded because it supplies no features.

An eight-gene version could instead use eight predefined radial bands, but that would impose a different spatial grouping.

### 2. Two ways to construct features

We will compare two representations using the same mask chromosome.

**A. Symmetric pixel selection**

Retain each activated pixel intensity as a separate input feature:

$$\Phi_{\mathbf{g}}(P)=\{P_{uv}:M_{\mathbf{g}}(u,v)=1\}$$

Complexity is the number of retained pixels. One gene can activate several pixels, so the number of active genes is not the feature dimensionality.

The selection mask is symmetric, but the predictor is not necessarily rotation- or reflection-invariant: individual pixel positions remain separate features.

**B. Symmetric group averaging**

Calculate one mean intensity for each activated symmetry group:

$$f_j(P)=\frac{1}{|G_j|}\sum_{(u,v)\in G_j}P_{uv}$$

The descriptor contains the group means for which $g_j=1$. Complexity is the number of active groups, with a maximum of 10 features.

This representation is invariant to the specified square rotations and reflections because those transformations only permute pixels within each group.

### 3. Why use binary masks?

Binary masks change which spatial information reaches the regressor.

Multiplying individual pixel features by nonzero continuous weights generally does not change the threshold partitions available to a tree model. Binary selection avoids searching over weights that have little effect.

Learning continuous weights would be more meaningful if the weighted pixels were combined into pooled features.

### 4. Genetic search

Begin with a small population containing simple masks, the full mask, and random nonempty masks.

For each candidate:

1. Expand its chromosome into an $8\times8$ mask.
2. Construct the selected-pixel or group-average descriptor.
3. Fit Extra Trees on the fitting subset.
4. Calculate peak-area prediction error on the spatial validation subset.
5. Record prediction error and feature dimensionality.

Generate new candidates through:

- **Mutation:** flip one or more genes.
- **Crossover:** exchange genes between two parent masks.
- **Selection:** retain candidates offering useful error–complexity trade-offs.
- **Elitism:** preserve the best nondominated candidates.

Cache evaluated chromosomes to avoid repeated fits. The two feature representations are evaluated separately.

### 5. Fitness and Pareto optimization

The objectives are to minimize validation error and descriptor dimensionality.

Validation error is normalized against a constant prediction equal to the fitting subset's mean peak area:

$$\mathrm{NRMSE}_{\mathrm{val}}=\frac{\sqrt{\langle(y-\hat y)^2\rangle_{\mathrm{val}}}}{\sqrt{\langle(y-\bar y_{\mathrm{fit}})^2\rangle_{\mathrm{val}}}}$$

An NRMSE below one improves on this baseline.

A candidate is Pareto-optimal if no other evaluated candidate has both equal or lower error and equal or fewer features, with at least one strict improvement.

The Pareto front shows how much predictive accuracy is gained by retaining additional spatial information.

### 6. Validation and reproducibility

All masks use identical fitting and validation locations and fixed regressor hyperparameters. Patches must not share pixels across the fitting/validation boundary.

Mask evolution uses validation scores only. After choosing a mask, refit on the available training data and evaluate on an untouched test region or image.

Use a fixed random seed during the initial comparison. Repeated seeds and spatial splits can subsequently assess whether the selected masks are stable.

### 7. Patch-center convention

An $8\times8$ patch has its geometric center between four pixels. If the target spectrum belongs to one pixel, specify its index explicitly and use the same extraction convention throughout.

The imposed symmetry is about the patch's geometric center, not that target pixel. A $9\times9$ patch would allow exact symmetry around a central target pixel, but would change the chromosome grouping.

### 8. Expected outputs

- Validation error versus feature dimensionality for all evaluated masks.
- Separate and combined Pareto fronts for the two representations.
- Images of representative Pareto-optimal masks.
- Search progress versus the number of unique masks evaluated.
- Measured versus predicted peak area for the final selected model.

There are only $2^{10}-1=1023$ nonempty chromosomes. The GA samples this space economically; exhaustive evaluation remains possible as a later benchmark.

The selected masks reveal which spatial measurements are useful under the chosen data split and regressor. They do not, by themselves, establish a physical interaction range or causal mechanism.

In [ ]:
# Requires: image, scalarizer_map
# scalarizer_map contains your previously defined plasmon peak area.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import ExtraTreesRegressor
from IPython.display import display

# ============================================================
# 1. Configuration
# ============================================================
N = 8
POPULATION = 12
GENERATIONS = 8
TREES = 64
SEED = 0

TEST_FRACTION = 0.25
MIN_FIT, MIN_VAL, MIN_TEST = 20, 15, 15

rng = np.random.default_rng(SEED)
img = np.asarray(image, dtype=float)
target = np.asarray(scalarizer_map, dtype=float)

if img.ndim != 2 or target.shape != img.shape:
    raise ValueError("image and scalarizer_map must be matching 2D arrays.")
if not (np.isfinite(img).all() and np.isfinite(target).all()):
    raise ValueError("Inputs contain NaN or infinite values.")

H, W = img.shape
boundary = int(W * (1 - TEST_FRACTION))
a, b = (N - 1) // 2, N - (N - 1) // 2

# Fixed spatial split: left training region, right test region.
# Every patch fits entirely within its own region.
def extract_region(x0, x1):
    coords = np.array([
        (y, x)
        for y in range(a, H - b + 1)
        for x in range(x0 + a, x1 - b + 1)
    ], dtype=int).reshape(-1, 2)

    if len(coords) == 0:
        raise ValueError("Image region too small for an 8×8 patch.")

    patches = np.stack([
        img[y-a:y+b, x-a:x+b] for y, x in coords
    ])
    values = target[coords[:, 0], coords[:, 1]]
    return patches, values, coords

P_train, y_train, coords = extract_region(0, boundary)

# Inner fitting/validation split: upper/lower parts of training region.
# Exclude patches crossing the horizontal boundary.
row_boundary = H // 2
fit_ids = np.flatnonzero(coords[:, 0] + b <= row_boundary)
val_ids = np.flatnonzero(coords[:, 0] - a >= row_boundary)

n_test = max(0, H - N + 1) * max(0, W - boundary - N + 1)
if (
    len(fit_ids) < MIN_FIT
    or len(val_ids) < MIN_VAL
    or n_test < MIN_TEST
):
    raise ValueError(
        f"Insufficient 8×8 patches: fit={len(fit_ids)}, "
        f"validation={len(val_ids)}, test={n_test}. "
        "Use a larger image or revise the fixed split/count requirements."
    )

baseline = np.sqrt(np.mean(
    (y_train[val_ids] - y_train[fit_ids].mean())**2
))
if baseline <= 0:
    raise ValueError("Validation baseline error is zero.")

print(f"8×8 patches; target pixel within patch: ({a}, {a})")
print(f"Inner fit: {len(fit_ids)}; validation: {len(val_ids)}")
print(f"Final training: {len(y_train)}; reserved test: {n_test}")
print("No patch pixels are shared across fitting/validation/test regions.")


# ============================================================
# 2. Construct the 10 square-symmetry groups
# ============================================================
yy, xx = np.indices((N, N))
dy = np.abs(2 * yy - (N - 1))
dx = np.abs(2 * xx - (N - 1))

# Rotations and reflections preserve the sorted pair of distances.
pairs = np.stack([np.minimum(dy, dx), np.maximum(dy, dx)], axis=-1)
group_keys = sorted(
    set(map(tuple, pairs.reshape(-1, 2))),
    key=lambda p: (p[0]**2 + p[1]**2, p)
)
group_map = np.zeros((N, N), dtype=int)
for j, key in enumerate(group_keys):
    group_map[np.all(pairs == key, axis=-1)] = j

G = len(group_keys)
assert G == 10
group_sizes = np.bincount(group_map.ravel())

def bits(code):
    return ((int(code) >> np.arange(G)) & 1).astype(bool)

def mask_image(code):
    return bits(code)[group_map]

def chromosome(code):
    # Leftmost character corresponds to group 1.
    return "".join(bits(code).astype(int).astype(str))

flat_train = P_train.reshape(-1, N * N)
pooled_train = np.column_stack([
    flat_train[:, group_map.ravel() == j].mean(axis=1)
    for j in range(G)
])

fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
axes[0].imshow(img, cmap="gray")
axes[0].axvline(boundary - 0.5, color="red", linestyle="--")
axes[0].scatter(
    coords[fit_ids, 1], coords[fit_ids, 0], s=5, label="Fit"
)
axes[0].scatter(
    coords[val_ids, 1], coords[val_ids, 0], s=5, label="Validation"
)
axes[0].set_title("Training/validation centers; test region on right")
axes[0].legend()

axes[1].imshow(group_map, cmap="tab10", vmin=-0.5, vmax=9.5)
for iy in range(N):
    for ix in range(N):
        axes[1].text(
            ix, iy, str(group_map[iy, ix] + 1),
            ha="center", va="center", fontsize=10
        )
axes[1].plot(a, a, "rx", ms=12, mew=2)
axes[1].set_title("Symmetry groups; red × = target pixel")
plt.show()


# ============================================================
# 3. Candidate evaluation and Pareto ranking
# ============================================================
METHODS = ["Pixel selection", "Group averaging"]
archive = {}

def make_model():
    return ExtraTreesRegressor(
        n_estimators=TREES,
        min_samples_leaf=3,
        max_features=1.0,
        random_state=SEED,
        n_jobs=-1
    )

def features(code, method, flat=flat_train, pooled=pooled_train):
    active = bits(code)
    if method == "Pixel selection":
        return flat[:, active[group_map.ravel()]]
    return pooled[:, active]

def evaluate(code, generation):
    # Evaluate every new mask under both representations.
    for method in METHODS:
        key = (int(code), method)
        if key in archive:
            continue
        X = features(code, method)
        model = make_model().fit(X[fit_ids], y_train[fit_ids])
        pred = model.predict(X[val_ids])
        rmse = np.sqrt(np.mean((pred - y_train[val_ids])**2))

        archive[key] = {
            "code": int(code),
            "genes": chromosome(code),
            "method": method,
            "active_groups": int(bits(code).sum()),
            "features": X.shape[1],
            "val_NRMSE": rmse / baseline,
            "first_generation": generation
        }

def table(codes=None, method=None):
    rows = [
        row for (code, name), row in archive.items()
        if (codes is None or code in codes)
        and (method is None or name == method)
    ]
    return pd.DataFrame(rows)

def nondominated_layers(df):
    """Pareto layers minimizing both feature count and error."""
    values = df[["features", "val_NRMSE"]].to_numpy()
    remaining = list(range(len(df)))
    layers = []

    while remaining:
        front = []
        for i in remaining:
            dominated = any(
                np.all(values[j] <= values[i])
                and np.any(values[j] < values[i])
                for j in remaining if j != i
            )
            if not dominated:
                front.append(i)
        layers.append(front)
        remaining = [i for i in remaining if i not in front]
    return layers

def pareto(df):
    return df.iloc[nondominated_layers(df)[0]].sort_values(
        ["features", "val_NRMSE"]
    )

def select_parents(df, count=4):
    """Pareto rank followed by crowding distance for diversity."""
    selected = []
    for layer in nondominated_layers(df):
        if len(selected) + len(layer) <= count:
            selected.extend(layer)
            continue

        values = df.iloc[layer][["features", "val_NRMSE"]].to_numpy()
        crowding = np.zeros(len(layer))
        for column in range(2):
            order = np.argsort(values[:, column])
            span = np.ptp(values[:, column])
            if span == 0:
                continue
            crowding[order[[0, -1]]] = np.inf
            for k in range(1, len(order) - 1):
                crowding[order[k]] += (
                    values[order[k+1], column]
                    - values[order[k-1], column]
                ) / span

        chosen = np.argsort(-crowding)[:count - len(selected)]
        selected.extend(layer[i] for i in chosen)
        break

    return df.iloc[selected].code.astype(int).tolist()


# ============================================================
# 4. Detailed generation reports
# ============================================================
history = []
columns = [
    "code", "genes", "method", "active_groups",
    "features", "val_NRMSE", "first_generation"
]

def report(generation, population):
    current = table(set(population))
    print(f"\n{'=' * 65}")
    print(f"GENERATION {generation + 1}")
    print(
        f"Population: {len(population)} masks | "
        f"Unique masks evaluated: {len(archive) // 2}"
    )
    display(current[columns].sort_values(["method", "val_NRMSE"]))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    masks_to_show = []

    for ax, method in zip(axes, METHODS):
        all_rows = table(method=method)
        now = current[current.method == method]
        front = pareto(all_rows)

        ax.scatter(
            all_rows.features, all_rows.val_NRMSE,
            s=25, alpha=0.3, label="Archive"
        )
        ax.scatter(
            now.features, now.val_NRMSE,
            s=55, facecolors="none", edgecolors="tab:orange",
            label="Current population"
        )
        line = front.drop_duplicates("features")
        ax.plot(
            line.features, line.val_NRMSE, "k.-",
            label="Archive Pareto front"
        )
        ax.set(
            title=method, xlabel="Input features", ylabel="Validation NRMSE"
        )
        ax.grid(alpha=0.2)
        ax.legend(fontsize=8)

        best = all_rows.sort_values(["val_NRMSE", "features"]).iloc[0]
        simple = front.sort_values(["features", "val_NRMSE"]).iloc[0]
        history.append({
            "generation": generation + 1,
            "method": method,
            "best_NRMSE": best.val_NRMSE,
            "unique_masks": len(all_rows)
        })
        masks_to_show.extend([
            (f"{method}\nLowest error", best),
            (f"{method}\nSimplest Pareto mask", simple)
        ])

    plt.show()

    fig, axes = plt.subplots(1, 4, figsize=(13, 3), constrained_layout=True)
    for ax, (label, row) in zip(axes, masks_to_show):
        ax.imshow(mask_image(row.code), cmap="gray", vmin=0, vmax=1)
        ax.plot(a, a, "rx", ms=9, mew=2)
        ax.set_title(
            f"{label}\n"
            f"d={int(row.features)}, error={row.val_NRMSE:.3f}",
            fontsize=9
        )
        ax.axis("off")
    plt.show()


# ============================================================
# 5. Genetic evolution
# ============================================================
# Initial population: each single group plus the full mask.
population = [1 << j for j in range(G)] + [(1 << G) - 1]
while len(population) < POPULATION:
    code = int(rng.integers(1, 1 << G))
    if code not in population:
        population.append(code)

for generation in range(GENERATIONS):
    for code in population:
        evaluate(code, generation + 1)

    report(generation, population)

    if generation == GENERATIONS - 1:
        break

    # Preserve nondominated/diverse parents for each representation.
    elites = sorted(set(
        select_parents(table(method=METHODS[0]))
        + select_parents(table(method=METHODS[1]))
    ))
    next_population = elites.copy()

    attempts = 0
    while len(next_population) < POPULATION:
        attempts += 1

        # Occasional random immigrants preserve exploration.
        if rng.random() < 0.15 or attempts > 100:
            code = int(rng.integers(1, 1 << G))
        else:
            p1, p2 = rng.choice(elites, size=2, replace=True)
            child = np.where(rng.random(G) < 0.5, bits(p1), bits(p2))

            # Expected one bit flip per child.
            child ^= rng.random(G) < 1 / G
            code = int(np.sum(child.astype(int) * (1 << np.arange(G))))

        if code and code not in next_population:
            next_population.append(code)

    population = next_population


# ============================================================
# 6. Final validation fronts and search history
# ============================================================
ga_results = table()
ga_history = pd.DataFrame(history)
ga_fronts = {
    method: pareto(table(method=method)) for method in METHODS
}

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for method in METHODS:
    h = ga_history[ga_history.method == method]
    front = ga_fronts[method].drop_duplicates("features")

    axes[0].plot(h.generation, h.best_NRMSE, "o-", label=method)
    axes[1].plot(front.features, front.val_NRMSE, "o-", label=method)

axes[0].set(
    xlabel="Generation", ylabel="Best validation NRMSE",
    title="Search progress"
)
axes[1].set(
    xlabel="Input features", ylabel="Validation NRMSE",
    title="Final validation Pareto fronts"
)
for ax in axes:
    ax.grid(alpha=0.2)
    ax.legend()
plt.show()

for method, front in ga_fronts.items():
    print(f"\nFinal Pareto masks: {method}")
    display(front[columns])


# ============================================================
# 7. Select by validation, refit, and evaluate test region
# ============================================================
# Predefined choice: lowest validation error, then fewer features.
# Select one mask per representation before looking at test scores.
selected = {
    method: table(method=method).sort_values(
        ["val_NRMSE", "features", "code"]
    ).iloc[0]
    for method in METHODS
}

P_test, y_test, test_coords = extract_region(boundary, W)
flat_test = P_test.reshape(-1, N * N)
pooled_test = np.column_stack([
    flat_test[:, group_map.ravel() == j].mean(axis=1)
    for j in range(G)
])

test_baseline = np.sqrt(np.mean((y_test - y_train.mean())**2))
final_models, summaries = {}, []

fig, axes = plt.subplots(2, 2, figsize=(10, 8), constrained_layout=True)

for row_index, method in enumerate(METHODS):
    chosen = selected[method]
    code = int(chosen.code)

    model = make_model().fit(features(code, method), y_train)
    prediction = model.predict(features(code, method, flat_test, pooled_test))
    final_models[method] = model

    rmse = np.sqrt(np.mean((prediction - y_test)**2))
    nrmse = rmse / test_baseline if test_baseline > 0 else np.nan
    summaries.append({
        "method": method, "code": code,
        "features": int(chosen.features),
        "validation_NRMSE": chosen.val_NRMSE,
        "test_RMSE": rmse, "test_NRMSE": nrmse
    })

    axes[row_index, 0].imshow(mask_image(code), cmap="gray", vmin=0, vmax=1)
    axes[row_index, 0].plot(a, a, "rx", ms=12, mew=2)
    axes[row_index, 0].set_title(
        f"{method}\nSelected mask, d={int(chosen.features)}"
    )
    axes[row_index, 0].axis("off")

    ax = axes[row_index, 1]
    ax.scatter(y_test, prediction, s=20, alpha=0.7)
    lo = min(y_test.min(), prediction.min())
    hi = max(y_test.max(), prediction.max())
    ax.plot([lo, hi], [lo, hi], "k--")
    ax.set(
        xlabel="Measured peak area", ylabel="Predicted peak area",
        title=f"Test NRMSE = {nrmse:.3f}"
    )

plt.show()
test_summary = pd.DataFrame(summaries)
display(test_summary)

## Genetic optimization of an unconstrained 8 × 8 pixel mask

### Objective

Find a compact subset of image pixels that predicts the **plasmon peak area at the target location**.

Each candidate is an $8\times8$ binary mask with **64 independent genes**. No symmetry constraints are imposed. The population contains **25 candidate masks per generation**, not 25 genes.

The same mask is applied to every image patch:

$$\mathbf{x}(P,M)=\{P_{ij}:M_{ij}=1\}$$

Selected pixel intensities become separate input features. Excluded pixels are removed rather than replaced by zeros. A fixed Extra Trees regressor maps this feature vector to the peak-area scalarizer.

### 1. Data and spatial validation

The code reuses `image` and the previously calculated `scalarizer_map`. It does not change the peak integration window or background subtraction.

The image is divided into:

- A left training region.
- A right test region, occupying approximately 25% of the image width.

The training region is further divided horizontally into inner fitting and validation subsets. Patches crossing any split boundary are excluded, so patches from different subsets do not share pixels.

All masks use identical fitting and validation locations. The test region is evaluated only after the final mask has been selected.

For an $8\times8$ patch, the target pixel is at index `(3, 3)` using zero-based indexing. The patch therefore extends three pixels above/left and four pixels below/right of the target.

### 2. Chromosomes and initialization

Each chromosome is a sequence of 64 binary values:

$$\mathbf{g}=(g_1,\ldots,g_{64}),\qquad g_j\in\{0,1\}$$

A value of one retains the corresponding pixel; zero removes it. Feature dimensionality is:

$$d=\sum_{j=1}^{64}g_j$$

Empty masks are excluded.

The initial population includes:

- The full patch.
- The target pixel alone.
- A central $4\times4$ region.
- A cross through the target pixel.
- Random masks with different sparsity levels.

### 3. Candidate evaluation

For each previously unseen mask:

1. Extract the selected pixel features.
2. Fit Extra Trees on the inner fitting subset.
3. Predict peak area on the validation subset.
4. Record validation error and selected-pixel count.

The regressor uses 64 trees, a minimum leaf size of three, all selected features as split candidates, and a fixed random seed. Hyperparameters remain unchanged throughout evolution.

Validation error is normalized against predicting the inner fitting subset's mean peak area:

$$\mathrm{NRMSE}_{\mathrm{val}}=\frac{\sqrt{\langle(y-\hat y)^2\rangle_{\mathrm{val}}}}{\sqrt{\langle(y-\bar y_{\mathrm{fit}})^2\rangle_{\mathrm{val}}}}$$

NRMSE below one improves on this constant baseline.

Evaluations are cached, so a mask encountered again does not require another model fit.

### 4. Selection: retain 20% of the population

Five of the 25 masks survive unchanged into the next generation.

**One elite** is preserved automatically: the current mask with the lowest validation error, breaking ties by fewer selected pixels.

The other four survivors are chosen through tournaments:

1. Randomly sample three candidates from the remaining population.
2. Prefer the candidate with the lower Pareto rank.
3. Break rank ties using larger crowding distance.
4. Remove the winner from the available survivor candidates and repeat.

Pareto ranking minimizes both validation error and selected-pixel count. Rank zero contains nondominated candidates; later ranks contain successive nondominated layers.

Crowding distance favors candidates in less densely occupied parts of the error–complexity trade-off. Selection is based on the current generation; the archive records all evaluated masks.

### 5. Crossover, mutation, and random immigrants

The remaining 20 population positions are filled with offspring or random immigrants.

**Uniform crossover:** choose two survivors randomly, with replacement. Each gene is inherited independently from either parent with equal probability.

**Mutation:** independently flip each offspring gene with probability $1/64$:

$$p_{\mathrm{mutation}}=\frac{1}{64}$$

This produces one flipped pixel on average, although an offspring can have zero, one, or several mutations.

**Random immigrants:** approximately 15% of offspring-generation attempts create a random mask instead of applying crossover and mutation.

Empty masks and duplicate masks within the next population are rejected. Masks from earlier generations may reappear and reuse cached evaluations.

### 6. Intermediate outputs

Every generation displays:

- The number of new and cumulative unique mask evaluations.
- Best and median validation NRMSE.
- The five selected survivor IDs.
- A table of all 25 candidates, including pixel count, error, Pareto rank, and survival status.
- Images of the four lowest-validation-error masks.
- The current population and cumulative Pareto front on an error-versus-pixel-count plot.

In mask images, white pixels are retained and black pixels are excluded. The red cross marks the target pixel.

The four displayed masks are ranked by validation error, with pixel count used to break ties. They are not necessarily the simplest masks or all tournament survivors.

### 7. Pareto front and final selection

The cumulative Pareto front identifies evaluated masks for which no alternative has both fewer or equal pixels and lower or equal error, with at least one strict improvement.

After evolution, the final mask is selected from the entire archive by:

1. Lowest validation NRMSE.
2. Fewest selected pixels if errors tie.
3. Mask ID as a final deterministic tie-breaker.

The model is then refitted on all training-region patches, including patches excluded from the inner fitting/validation split, and evaluated on the test region.

Final outputs include test RMSE, normalized RMSE, $R^2$, the selected mask, and a measured-versus-predicted peak-area plot.

### 8. Output variables

| Variable | Contents |
|---|---|
| `ga_results` | All unique evaluated masks and validation scores |
| `ga_history` | Performance and complexity summaries by generation |
| `ga_generations` | Candidate tables for every generation |
| `ga_front` | Final validation error–complexity Pareto front |
| `selected_mask` | Final 64-element Boolean pixel mask |
| `final_model` | Regressor refitted on all training patches |
| `test_prediction` | Predictions at the reserved test locations |

### Interpretation

The search explores a small subset of the $2^{64}-1$ possible nonempty masks. It is a heuristic search and does not guarantee a globally optimal mask.

A selected pixel is useful in combination with the other selected pixels under this regressor and validation split. Selection alone does not establish causal importance.

Repeated seeds and spatial splits can assess mask stability. If the test results guide subsequent design choices, that region becomes validation data; an additional untouched region or image is needed for an independent final test.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import ExtraTreesRegressor
from IPython.display import display


# ============================================================
# 1. Settings
# ============================================================
N = 8
N_GENES = N * N
POPULATION = 25
GENERATIONS = 100

SURVIVAL_FRACTION = 0.20       # Exactly 5 survivors
TOURNAMENT_SIZE = 3
MUTATION_RATE = 1 / N_GENES    # One flipped pixel on average
IMMIGRANT_RATE = 0.15

N_TREES = 64
SEED = 0

TEST_FRACTION = 0.25
MIN_FIT, MIN_VAL, MIN_TEST = 20, 15, 15

rng = np.random.default_rng(SEED)
N_SURVIVORS = int(np.ceil(POPULATION * SURVIVAL_FRACTION))

img = np.asarray(image, dtype=float)
target = np.asarray(scalarizer_map, dtype=float)

if img.ndim != 2 or target.shape != img.shape:
    raise ValueError("image and scalarizer_map must be matching 2D arrays.")
if not (np.isfinite(img).all() and np.isfinite(target).all()):
    raise ValueError("Inputs contain NaN or infinite values.")

H, W = img.shape
boundary = int(W * (1 - TEST_FRACTION))
a, b = (N - 1) // 2, N - (N - 1) // 2


# ============================================================
# 2. Fixed spatial splits with no cross-boundary patch overlap
# ============================================================
def extract_region(x0, x1):
    coordinates = np.array([
        (y, x)
        for y in range(a, H - b + 1)
        for x in range(x0 + a, x1 - b + 1)
    ], dtype=int).reshape(-1, 2)

    if len(coordinates) == 0:
        raise ValueError("Region too small for 8×8 patches.")

    patches = np.stack([
        img[y-a:y+b, x-a:x+b] for y, x in coordinates
    ])
    values = target[coordinates[:, 0], coordinates[:, 1]]
    return patches, values, coordinates


P_train, y_train, train_coords = extract_region(0, boundary)
X_all = P_train.reshape(-1, N_GENES)

row_boundary = H // 2
fit_ids = np.flatnonzero(train_coords[:, 0] + b <= row_boundary)
val_ids = np.flatnonzero(train_coords[:, 0] - a >= row_boundary)

n_test = max(0, H - N + 1) * max(0, W - boundary - N + 1)

if (
    len(fit_ids) < MIN_FIT
    or len(val_ids) < MIN_VAL
    or n_test < MIN_TEST
):
    raise ValueError(
        f"Too few patches: fit={len(fit_ids)}, "
        f"validation={len(val_ids)}, test={n_test}. "
        "Use a larger image or revise the fixed split/count requirements."
    )

validation_baseline = np.sqrt(np.mean(
    (y_train[val_ids] - y_train[fit_ids].mean())**2
))
if validation_baseline <= 0:
    raise ValueError("Validation baseline error is zero.")

print(f"Chromosome: {N_GENES} independently selected pixels")
print(f"Population: {POPULATION}; survivors: {N_SURVIVORS}")
print(f"Inner fit: {len(fit_ids)}; validation: {len(val_ids)}")
print(f"Final training: {len(y_train)}; reserved test: {n_test}")
print(f"Target pixel within each patch: row {a}, column {a} (zero-based)")

fig, ax = plt.subplots(figsize=(7, 4))
ax.imshow(img, cmap="gray")
ax.axvline(boundary - 0.5, color="red", linestyle="--")
ax.scatter(
    train_coords[fit_ids, 1], train_coords[fit_ids, 0],
    s=6, label="Inner fit"
)
ax.scatter(
    train_coords[val_ids, 1], train_coords[val_ids, 0],
    s=6, label="Validation"
)
ax.set_title("Fixed spatial split; test region to the right")
ax.legend()
plt.show()


# ============================================================
# 3. Chromosomes and cached model evaluation
# ============================================================
# A chromosome is a tuple of 64 zeros/ones.
# This avoids integer overflow when encoding 64-bit masks.
def chromosome(mask):
    return tuple(np.asarray(mask, dtype=np.uint8).tolist())


def random_mask():
    # Sample different sparsity levels, not only 50%-filled masks.
    probability = rng.uniform(0.05, 0.95)
    mask = rng.random(N_GENES) < probability
    if not mask.any():
        mask[int(rng.integers(N_GENES))] = True
    return chromosome(mask)


def make_model():
    return ExtraTreesRegressor(
        n_estimators=N_TREES,
        min_samples_leaf=3,
        max_features=1.0,
        random_state=SEED,
        n_jobs=-1
    )


archive = {}


def evaluate(mask, generation):
    if mask not in archive:
        selected = np.asarray(mask, dtype=bool)
        X = X_all[:, selected]

        model = make_model().fit(X[fit_ids], y_train[fit_ids])
        prediction = model.predict(X[val_ids])
        rmse = np.sqrt(np.mean(
            (prediction - y_train[val_ids])**2
        ))

        archive[mask] = {
            "mask_id": len(archive) + 1,
            "pixels": int(selected.sum()),
            "val_RMSE": rmse,
            "val_NRMSE": rmse / validation_baseline,
            "first_generation": generation
        }

    return archive[mask]


def results_table(masks=None):
    if masks is None:
        masks = list(archive)
    return pd.DataFrame([
        {"mask": mask, **archive[mask]} for mask in masks
    ])


# ============================================================
# 4. Pareto ranks and crowding distances
# ============================================================
def rank_population(df):
    """Lower Pareto rank and larger crowding distance are preferred."""
    df = df.reset_index(drop=True).copy()
    values = df[["pixels", "val_NRMSE"]].to_numpy()

    ranks = np.zeros(len(df), dtype=int)
    crowding = np.zeros(len(df), dtype=float)
    remaining = list(range(len(df)))
    rank = 0

    while remaining:
        front = [
            i for i in remaining
            if not any(
                np.all(values[j] <= values[i])
                and np.any(values[j] < values[i])
                for j in remaining if j != i
            )
        ]
        ranks[front] = rank

        # Normalize crowding independently for each objective.
        for column in range(2):
            ordered = sorted(front, key=lambda i: values[i, column])
            span = values[ordered[-1], column] - values[ordered[0], column]
            if span == 0:
                continue

            crowding[ordered[0]] = np.inf
            crowding[ordered[-1]] = np.inf

            for k in range(1, len(ordered) - 1):
                crowding[ordered[k]] += (
                    values[ordered[k+1], column]
                    - values[ordered[k-1], column]
                ) / span

        remaining = [i for i in remaining if i not in front]
        rank += 1

    df["pareto_rank"] = ranks
    df["crowding"] = crowding
    return df


def pareto_front(df):
    # One representative per pixel count, for plotting the lower envelope.
    candidates = (
        df.sort_values(["pixels", "val_NRMSE"])
        .drop_duplicates("pixels")
    )
    previous = candidates.val_NRMSE.cummin().shift(fill_value=np.inf)
    return candidates[candidates.val_NRMSE < previous]


# ============================================================
# 5. Tournament survival, crossover, and mutation
# ============================================================
def select_survivors(population):
    ranked = rank_population(results_table(population))

    # Preserve one elite: lowest validation error, then fewest pixels.
    elite_index = ranked.sort_values(
        ["val_NRMSE", "pixels"]
    ).index[0]
    chosen = [int(elite_index)]
    available = [i for i in ranked.index if i != elite_index]

    while len(chosen) < N_SURVIVORS:
        contestants = rng.choice(
            available,
            size=min(TOURNAMENT_SIZE, len(available)),
            replace=False
        )

        # Random contestant order provides random resolution of exact ties.
        winner = min(
            contestants,
            key=lambda i: (
                ranked.loc[i, "pareto_rank"],
                -ranked.loc[i, "crowding"]
            )
        )

        chosen.append(int(winner))
        available.remove(int(winner))

    return ranked.loc[chosen, "mask"].tolist()


def next_generation(survivors):
    population = list(survivors)
    seen = set(population)
    attempts = 0

    while len(population) < POPULATION:
        attempts += 1

        if rng.random() < IMMIGRANT_RATE or attempts > 200:
            child = random_mask()
        else:
            parent_ids = rng.choice(len(survivors), size=2, replace=True)
            p1 = np.asarray(survivors[parent_ids[0]], dtype=bool)
            p2 = np.asarray(survivors[parent_ids[1]], dtype=bool)

            # Uniform crossover.
            mask = np.where(rng.random(N_GENES) < 0.5, p1, p2)

            # Independent pixel mutations.
            mask ^= rng.random(N_GENES) < MUTATION_RATE

            if not mask.any():
                continue
            child = chromosome(mask)

        if child not in seen:
            population.append(child)
            seen.add(child)

    return population


# ============================================================
# 6. Initial population
# ============================================================
yy, xx = np.indices((N, N))

initial_masks = [
    np.ones((N, N), dtype=bool),               # Entire patch
    (yy == a) & (xx == a),                    # Target pixel
    (yy >= 2) & (yy < 6) & (xx >= 2) & (xx < 6),
    (yy == a) | (xx == a),                    # Cross through target
]

population = list(dict.fromkeys(
    chromosome(mask.ravel()) for mask in initial_masks
))
while len(population) < POPULATION:
    candidate = random_mask()
    if candidate not in population:
        population.append(candidate)


# ============================================================
# 7. Evolve and display four best masks in every generation
# ============================================================
history = []
generation_tables = []

for generation in range(1, GENERATIONS + 1):
    before = len(archive)
    for mask in population:
        evaluate(mask, generation)

    current = rank_population(results_table(population))
    survivors = select_survivors(population)
    survivor_set = set(survivors)
    current["survives"] = [
        mask in survivor_set for mask in current["mask"]
    ]
    current["generation"] = generation
    generation_tables.append(current.copy())

    # "Best four" means lowest validation error, not lowest Pareto rank.
    best_four = current.sort_values(
        ["val_NRMSE", "pixels"]
    ).head(4)

    all_results = results_table()
    front = pareto_front(all_results)

    history.append({
        "generation": generation,
        "best_validation": current.val_NRMSE.min(),
        "median_validation": current.val_NRMSE.median(),
        "mean_pixels": current.pixels.mean(),
        "unique_evaluations": len(archive)
    })

    print(f"\n{'=' * 65}\nGENERATION {generation}/{GENERATIONS}")
    print(
        f"New evaluations: {len(archive) - before}; "
        f"total unique masks: {len(archive)}"
    )
    print(
        f"Best NRMSE: {current.val_NRMSE.min():.4f}; "
        f"median: {current.val_NRMSE.median():.4f}"
    )
    print(
        "Five selected survivors"
        + (" (would seed the next generation):" if generation == GENERATIONS
           else ":")
    )
    print([archive[m]["mask_id"] for m in survivors])

    display(current[[
        "mask_id", "pixels", "val_NRMSE",
        "pareto_rank", "survives"
    ]].sort_values(["val_NRMSE", "pixels"]))

    # Four best masks in this generation.
    fig, axes = plt.subplots(
        1, 4, figsize=(13, 3.2), constrained_layout=True
    )
    for ax, (_, row) in zip(axes, best_four.iterrows()):
        mask = np.asarray(row["mask"]).reshape(N, N)
        ax.imshow(mask, cmap="gray", vmin=0, vmax=1)
        ax.plot(a, a, "rx", ms=10, mew=2)
        ax.set_title(
            f"Mask {int(row.mask_id)} | {int(row.pixels)} pixels\n"
            f"Validation NRMSE={row.val_NRMSE:.3f}\n"
            f"Pareto rank={int(row.pareto_rank)}",
            fontsize=10
        )
        ax.set_xticks([])
        ax.set_yticks([])
    fig.suptitle(
        f"Generation {generation}: four lowest-error masks",
        fontsize=13
    )
    plt.show()

    # Current population against all previously evaluated masks.
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.scatter(
        all_results.pixels, all_results.val_NRMSE,
        s=25, alpha=0.25, label="All evaluated masks"
    )
    ax.scatter(
        current.pixels, current.val_NRMSE,
        s=55, facecolors="none", edgecolors="tab:orange",
        label="Current generation"
    )
    ax.plot(
        front.pixels, front.val_NRMSE,
        "k.-", label="Cumulative Pareto front"
    )
    ax.set(
        xlabel="Selected pixels",
        ylabel="Validation NRMSE",
        title=f"Generation {generation}: complexity–error trade-off"
    )
    ax.grid(alpha=0.2)
    ax.legend()
    plt.tight_layout()
    plt.show()

    if generation < GENERATIONS:
        population = next_generation(survivors)


# ============================================================
# 8. Final search summary
# ============================================================
ga_results = results_table()
ga_history = pd.DataFrame(history)
ga_generations = pd.concat(generation_tables, ignore_index=True)
ga_front = pareto_front(ga_results)

print("\nFinal validation Pareto front:")
display(ga_front[[
    "mask_id", "pixels", "val_NRMSE", "first_generation"
]])

fig, axes = plt.subplots(
    1, 2, figsize=(12, 4), constrained_layout=True
)
axes[0].plot(
    ga_history.generation, ga_history.best_validation,
    "o-", label="Best"
)
axes[0].plot(
    ga_history.generation, ga_history.median_validation,
    "o-", label="Median"
)
axes[0].set(
    xlabel="Generation", ylabel="Validation NRMSE",
    title="Prediction performance"
)
axes[0].legend()

axes[1].plot(
    ga_history.generation, ga_history.mean_pixels, "o-"
)
axes[1].set(
    xlabel="Generation", ylabel="Mean selected pixels",
    title="Population complexity"
)
for ax in axes:
    ax.grid(alpha=0.2)
plt.show()


# ============================================================
# 9. Select by validation; evaluate test region once
# ============================================================
best = ga_results.sort_values(
    ["val_NRMSE", "pixels", "mask_id"]
).iloc[0]

selected_mask = np.asarray(best["mask"], dtype=bool)
final_model = make_model().fit(X_all[:, selected_mask], y_train)

P_test, y_test, test_coords = extract_region(boundary, W)
X_test = P_test.reshape(-1, N_GENES)[:, selected_mask]
test_prediction = final_model.predict(X_test)

test_rmse = np.sqrt(np.mean((test_prediction - y_test)**2))
test_baseline = np.sqrt(np.mean((y_test - y_train.mean())**2))
test_nrmse = test_rmse / test_baseline if test_baseline > 0 else np.nan

test_variance = np.mean((y_test - y_test.mean())**2)
test_r2 = (
    1 - test_rmse**2 / test_variance
    if test_variance > 0 else np.nan
)

print(f"\nSelected mask: {int(best.mask_id)}")
print(f"Selected pixels: {int(best.pixels)}/64")
print(f"Validation NRMSE: {best.val_NRMSE:.4f}")
print(f"Test RMSE: {test_rmse:.4g}")
print(f"Test NRMSE: {test_nrmse:.4f}")
print(f"Test R²: {test_r2:.4f}")

fig, axes = plt.subplots(
    1, 2, figsize=(10, 4), constrained_layout=True
)
axes[0].imshow(
    selected_mask.reshape(N, N), cmap="gray", vmin=0, vmax=1
)
axes[0].plot(a, a, "rx", ms=12, mew=2)
axes[0].set_title("Final selected mask")
axes[0].axis("off")

axes[1].scatter(y_test, test_prediction, s=20, alpha=0.7)
lo = min(y_test.min(), test_prediction.min())
hi = max(y_test.max(), test_prediction.max())
axes[1].plot([lo, hi], [lo, hi], "k--")
axes[1].set(
    xlabel="Measured peak area",
    ylabel="Predicted peak area",
    title=f"Test NRMSE={test_nrmse:.3f}"
)
plt.show()

## Genetic construction of a spatial descriptor for 12 × 12 patches

### Objective

Learn where an image patch needs individual pixel detail, where pooling is sufficient, and which regions can be discarded when predicting the **plasmon peak area**.

The genetic algorithm evolves a spatial descriptor. The Extra Trees regressor and peak-area scalarizer remain fixed.

$$P \longrightarrow \Phi_T(P) \longrightarrow \text{Extra Trees} \longrightarrow \hat y$$

Here, $P$ is a patch and $T$ is a tree specifying how its regions are converted into features. The same tree is applied to every patch.

### 1. Hierarchical representation

The patch is divided using the hierarchy:

$$12\times12 \rightarrow 6\times6 \rightarrow 3\times3 \rightarrow 1\times1$$

A $12\times12$ or $6\times6$ region splits into four quadrants. A $3\times3$ region splits into nine individual pixels. This is therefore a hierarchical partition tree, with nine-way branching at the final subdivision.

Each region has one operation:

| Operation | Action | Output features |
|---|---|---:|
| `DROP` | Ignore the region | 0 |
| `MEAN` | Average its intensities | 1 |
| `MAX` | Take its maximum intensity | 1 |
| `SPLIT` | Process its children separately | Sum of child outputs |
| `PIXEL` | Retain a single pixel intensity | 1 |

`PIXEL` is used only for $1\times1$ regions. Regions do not overlap, and features are emitted recursively in a fixed top-to-bottom, left-to-right order.

Unlike a binary pixel mask, this chromosome has a variable tree structure and a variable number of output features.

### 2. Example descriptor

The four $6\times6$ quadrants could be represented as:

- Upper left: retain all 36 pixels.
- Upper right: one mean intensity.
- Lower left: four maxima from its $3\times3$ children.
- Lower right: discard.

This produces **41 features from 108 source pixels**.

The two complexity measures answer different questions:

- **Feature count:** how many numbers reach the regressor?
- **Source-pixel count:** how many image pixels contribute to those numbers?

A mean over the entire patch produces one feature but uses all 144 pixels.

### 3. Spatial fitting, validation, and testing

The code reuses `image` and `scalarizer_map`.

The left part of the image is the training region; the right approximately 25% is reserved for testing. The training region is divided horizontally into inner fitting and validation subsets.

Patches crossing these boundaries are excluded, so subsets do not share patch pixels. All candidate descriptors use the same locations.

The code requires at least 20 inner fitting patches, 15 validation patches, and 15 test patches. It stops if the fixed geometry cannot provide them.

The target pixel is at zero-based index `(5, 5)` inside each patch. A $12\times12$ patch therefore extends five pixels above/left and six below/right of its target.

### 4. Initialization and evaluation

The population contains **25 descriptors**. Initial candidates include:

- One mean over the full patch.
- One maximum over the full patch.
- All 144 individual pixels.
- A mixed quadrant descriptor.
- Random valid trees.

For each previously unseen descriptor, the code extracts its features, trains Extra Trees on the inner fitting subset, and evaluates it on the validation subset.

The regressor uses 64 trees, a minimum leaf size of three, all available features as split candidates, and a fixed random seed. Its hyperparameters are not tuned during evolution.

Repeated region measurements and complete descriptor evaluations are cached.

### 5. Mutation rules

Each non-immigrant offspring undergoes one attempted mutation:

| Mutation | Modification | Nominal probability |
|---|---|---:|
| Refine | Replace a pooled region with pooled children | 25% |
| Coarsen | Replace a split subtree with one mean or maximum | 25% |
| Change pooling | Switch `MEAN` and `MAX` | 20% |
| Drop | Remove a region or subtree | 15% |
| Activate | Restore a dropped region as a pool or pixel | 15% |

Probabilities are renormalized over mutations valid for the current tree.

Refinement preserves the parent pooling type in its children. At the final subdivision, a $3\times3$ pool becomes nine individual pixels.

A subtree whose children are all dropped is simplified to `DROP`. A mutation that would discard the entire descriptor is rejected by retaining the parent.

### 6. Crossover and immigrants

For non-immigrant offspring, crossover is attempted with probability 0.8.

Two survivors are sampled with replacement. A non-root spatial region represented in both parents is selected, and the first parent's subtree is replaced with the second parent's subtree at that region.

This preserves spatial compatibility: a region can only receive a subtree describing the same location and size.

If no compatible non-root region exists, the first parent proceeds directly to mutation.

Approximately 15% of offspring-generation attempts instead produce a random tree. Empty descriptors and duplicates within the next population are rejected.

### 7. Tournament selection and survival

Each generation retains **five survivors: 20% of the population**.

One elite survives automatically: the descriptor with the lowest validation error, breaking ties by fewer features.

The remaining four survivors are selected through tournaments of three candidates. Preference is given to:

1. Lower Pareto rank.
2. Larger crowding distance when ranks tie.

Pareto ranking minimizes both validation error and output feature count. Crowding distance encourages diversity along that trade-off.

Selection uses the current population. The cumulative archive records all evaluated descriptors and is used for reporting and final selection.

### 8. Validation error and Pareto front

Validation RMSE is normalized by the error of predicting the inner fitting subset's mean target:

$$\mathrm{NRMSE}_{\mathrm{val}}=\frac{\sqrt{\langle(y-\hat y)^2\rangle_{\mathrm{val}}}}{\sqrt{\langle(y-\bar y_{\mathrm{fit}})^2\rangle_{\mathrm{val}}}}$$

An NRMSE below one improves on this constant baseline.

The Pareto front contains descriptors for which no evaluated alternative provides both equal or lower error and equal or fewer features, with at least one strict improvement.

Source-pixel count is reported but is not an optimization objective in this implementation.

### 9. Intermediate visualizations

Every generation displays:

- New and cumulative descriptor evaluation counts.
- A candidate table with feature count, source pixels, operation counts, validation error, Pareto rank, and survival status.
- The four descriptors with the lowest validation errors.
- The current population and cumulative Pareto front.

Descriptor colors are:

| Color | Operation |
|---|---|
| Blue | Mean pooling |
| Orange | Maximum pooling |
| Green | Individual pixel |
| Gray | Discarded region |
| Red cross | Target pixel |

The four displayed descriptors are ranked by error, not by simplicity or tournament outcome.

### 10. Final selection and outputs

After evolution, the lowest-validation-error descriptor is selected from the entire archive, breaking ties by fewer features and then descriptor ID.

It is refitted on all training-region patches and evaluated on the reserved test region. The code reports test RMSE and NRMSE, plots measured versus predicted peak area, and lists the selected features with their operations, locations, and region sizes.

| Variable | Contents |
|---|---|
| `ga_results` | All unique evaluated descriptors |
| `ga_history` | Generation-level progress |
| `ga_generations` | Candidate tables for every generation |
| `ga_front` | Final validation Pareto front |
| `selected_tree` | Chosen hierarchical descriptor |
| `feature_definitions` | Ordered spatial feature definitions |
| `final_model` | Regressor refitted on all training patches |
| `test_prediction` | Predictions in the test region |

### Interpretation

The algorithm searches for a useful spatial compression under a fixed regressor and validation split. It can discover mixtures of fine detail, pooled context, and discarded regions, but does not guarantee a globally optimal descriptor.

The tree restricts region boundaries to the specified hierarchy. It does not learn arbitrary shapes or continuously movable pooling windows.

Selected regions indicate predictive usefulness, not causal importance. Repeated seeds and spatial splits can assess stability. If test results guide subsequent changes, an additional untouched region or image is needed for an independent final evaluation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.patches import Rectangle
from sklearn.ensemble import ExtraTreesRegressor
from IPython.display import display


# ============================================================
# 1. Configuration
# ============================================================
N = 12
POPULATION = 25
GENERATIONS = 100
SURVIVORS = 15
TOURNAMENT_SIZE = 5
CROSSOVER_RATE = 0.8
IMMIGRANT_RATE = 0.15
TREES = 64
SEED = 0

TEST_FRACTION = 0.25
MIN_FIT, MIN_VAL, MIN_TEST = 20, 15, 15

rng = np.random.default_rng(SEED)
img = np.asarray(image, dtype=float)
target = np.asarray(scalarizer_map, dtype=float)

if img.ndim != 2 or target.shape != img.shape:
    raise ValueError("image and scalarizer_map must be matching 2D arrays.")
if not (np.isfinite(img).all() and np.isfinite(target).all()):
    raise ValueError("Inputs contain NaN or infinite values.")

H, W = img.shape
boundary = int(W * (1 - TEST_FRACTION))
a, b = (N - 1) // 2, N - (N - 1) // 2


# ============================================================
# 2. Spatial splitting and patch extraction
# ============================================================
def extract_region(x0, x1):
    coords = np.array([
        (y, x)
        for y in range(a, H - b + 1)
        for x in range(x0 + a, x1 - b + 1)
    ], dtype=int).reshape(-1, 2)

    if not len(coords):
        raise ValueError("Region too small for 12×12 patches.")

    patches = np.stack([
        img[y-a:y+b, x-a:x+b] for y, x in coords
    ])
    return patches, target[coords[:, 0], coords[:, 1]], coords


P_train, y_train, coordinates = extract_region(0, boundary)
row_boundary = H // 2

fit_ids = np.flatnonzero(coordinates[:, 0] + b <= row_boundary)
val_ids = np.flatnonzero(coordinates[:, 0] - a >= row_boundary)

n_test = max(0, H - N + 1) * max(0, W - boundary - N + 1)

if (
    len(fit_ids) < MIN_FIT
    or len(val_ids) < MIN_VAL
    or n_test < MIN_TEST
):
    raise ValueError(
        f"Insufficient 12×12 patches: fit={len(fit_ids)}, "
        f"validation={len(val_ids)}, test={n_test}. "
        "Use a larger image or revise the split/count requirements."
    )

baseline = np.sqrt(np.mean(
    (y_train[val_ids] - y_train[fit_ids].mean())**2
))
if baseline <= 0:
    raise ValueError("Validation baseline error is zero.")

print(f"Fit={len(fit_ids)}, validation={len(val_ids)}, test={n_test}")
print(f"Target pixel inside patch: ({a}, {a}), zero-based")
print("Region sizes: 12 → 6 → 3 → individual pixels")


# ============================================================
# 3. Tree representation
# ============================================================
# Leaf: ("DROP",), ("MEAN",), ("MAX",), ("PIXEL",)
# Branch: ("SPLIT", child1, child2, ...)
#
# Children are ordered top-to-bottom, left-to-right.
DROP = ("DROP",)
MEAN = ("MEAN",)
MAX = ("MAX",)
PIXEL = ("PIXEL",)

ROOT = (0, 0, N)


def child_regions(region):
    y, x, size = region
    if size == 1:
        return []
    step = size // 2 if size % 2 == 0 else 1
    return [
        (y + dy, x + dx, step)
        for dy in range(0, size, step)
        for dx in range(0, size, step)
    ]


def simplify(tree):
    if tree[0] != "SPLIT":
        return tree
    children = tuple(simplify(t) for t in tree[1:])
    if all(t == DROP for t in children):
        return DROP
    return ("SPLIT",) + children


def random_tree(region=ROOT):
    size = region[2]
    if size == 1:
        return PIXEL if rng.random() < 0.7 else DROP

    op = str(rng.choice(
        ["DROP", "MEAN", "MAX", "SPLIT"],
        p=[0.15, 0.25, 0.20, 0.40]
    ))
    if op != "SPLIT":
        return (op,)
    return simplify(
        ("SPLIT",) + tuple(random_tree(r) for r in child_regions(region))
    )


def nodes(tree, region=ROOT, path=()):
    yield path, region, tree
    if tree[0] == "SPLIT":
        for i, (child, box) in enumerate(zip(tree[1:], child_regions(region))):
            yield from nodes(child, box, path + (i,))


def replace(tree, path, subtree):
    if not path:
        return simplify(subtree)
    children = list(tree[1:])
    i = path[0]
    children[i] = replace(children[i], path[1:], subtree)
    return simplify(("SPLIT",) + tuple(children))


def active_leaves(tree):
    return [
        (region, node[0])
        for _, region, node in nodes(tree)
        if node[0] not in ("DROP", "SPLIT")
    ]


def full_pixels(region=ROOT):
    if region[2] == 1:
        return PIXEL
    return ("SPLIT",) + tuple(full_pixels(r) for r in child_regions(region))


# ============================================================
# 4. Feature extraction and cached validation
# ============================================================
feature_cache = {}
archive = {}


def measure(patches, region, op):
    y, x, size = region
    values = patches[:, y:y+size, x:x+size]
    if op in ("MEAN", "PIXEL"):
        return values.mean(axis=(1, 2))
    return values.max(axis=(1, 2))


def feature_matrix(tree, patches=None):
    columns = []
    for region, op in active_leaves(tree):
        if patches is None:
            key = (region, op)
            if key not in feature_cache:
                feature_cache[key] = measure(P_train, region, op)
            columns.append(feature_cache[key])
        else:
            columns.append(measure(patches, region, op))
    return np.column_stack(columns)


def make_model():
    return ExtraTreesRegressor(
        n_estimators=TREES,
        min_samples_leaf=3,
        max_features=1.0,
        random_state=SEED,
        n_jobs=-1
    )


def evaluate(tree, generation):
    if tree not in archive:
        X = feature_matrix(tree)
        model = make_model().fit(X[fit_ids], y_train[fit_ids])
        pred = model.predict(X[val_ids])
        leaves = active_leaves(tree)

        archive[tree] = {
            "id": len(archive) + 1,
            "features": X.shape[1],
            "source_pixels": sum(region[2]**2 for region, _ in leaves),
            "means": sum(op == "MEAN" for _, op in leaves),
            "maxima": sum(op == "MAX" for _, op in leaves),
            "pixels": sum(op == "PIXEL" for _, op in leaves),
            "val_NRMSE": np.sqrt(np.mean(
                (pred - y_train[val_ids])**2
            )) / baseline,
            "first_generation": generation
        }


def table(trees=None):
    if trees is None:
        trees = list(archive)
    return pd.DataFrame([
        {"tree": t, **archive[t]} for t in trees
    ])


# ============================================================
# 5. Mutation and same-region crossover
# ============================================================
def mutate(tree):
    items = list(nodes(tree))
    eligible = {
        "refine": [
            item for item in items
            if item[1][2] > 1 and item[2][0] in ("MEAN", "MAX")
        ],
        "coarsen": [item for item in items if item[2][0] == "SPLIT"],
        "pool": [item for item in items if item[2][0] in ("MEAN", "MAX")],
        "drop": [item for item in items if item[2][0] != "DROP"],
        "activate": [item for item in items if item[2][0] == "DROP"]
    }
    weights = {
        "refine": 0.25, "coarsen": 0.25, "pool": 0.20,
        "drop": 0.15, "activate": 0.15
    }
    choices = [name for name, entries in eligible.items() if entries]
    probabilities = np.array([weights[name] for name in choices])
    probabilities /= probabilities.sum()

    action = str(rng.choice(choices, p=probabilities))
    options = eligible[action]
    path, region, node = options[int(rng.integers(len(options)))]

    if action == "refine":
        replacement = ("SPLIT",) + tuple(
            PIXEL if r[2] == 1 else node for r in child_regions(region)
        )
    elif action == "coarsen":
        replacement = MEAN if rng.random() < 0.5 else MAX
    elif action == "pool":
        replacement = MAX if node == MEAN else MEAN
    elif action == "drop":
        replacement = DROP
    else:
        replacement = (
            PIXEL if region[2] == 1
            else MEAN if rng.random() < 0.5 else MAX
        )

    child = replace(tree, path, replacement)
    return child if active_leaves(child) else tree


def crossover(parent1, parent2):
    first = {path: node for path, _, node in nodes(parent1)}
    second = {path: node for path, _, node in nodes(parent2)}
    common = sorted((set(first) & set(second)) - {()})

    if not common:
        return parent1   # Mutation follows below.

    path = common[int(rng.integers(len(common)))]
    child = replace(parent1, path, second[path])
    return child if active_leaves(child) else parent1


# ============================================================
# 6. Pareto ranking and tournament selection
# ============================================================
def rank_population(df):
    df = df.reset_index(drop=True).copy()
    values = df[["features", "val_NRMSE"]].to_numpy()
    ranks = np.zeros(len(df), dtype=int)
    crowding = np.zeros(len(df))
    remaining = list(range(len(df)))
    rank = 0

    while remaining:
        front = [
            i for i in remaining
            if not any(
                np.all(values[j] <= values[i])
                and np.any(values[j] < values[i])
                for j in remaining if j != i
            )
        ]
        ranks[front] = rank

        for column in range(2):
            ordered = sorted(front, key=lambda i: values[i, column])
            span = values[ordered[-1], column] - values[ordered[0], column]
            if span == 0:
                continue
            crowding[ordered[0]] = crowding[ordered[-1]] = np.inf
            for k in range(1, len(ordered) - 1):
                crowding[ordered[k]] += (
                    values[ordered[k+1], column]
                    - values[ordered[k-1], column]
                ) / span

        remaining = [i for i in remaining if i not in front]
        rank += 1

    df["rank"] = ranks
    df["crowding"] = crowding
    return df


def select_survivors(population):
    ranked = rank_population(table(population))
    elite = int(ranked.sort_values(["val_NRMSE", "features"]).index[0])
    selected = [elite]
    available = [i for i in ranked.index if i != elite]

    while len(selected) < SURVIVORS:
        contestants = rng.choice(
            available, min(TOURNAMENT_SIZE, len(available)), replace=False
        )
        winner = int(min(
            contestants,
            key=lambda i: (ranked.loc[i, "rank"], -ranked.loc[i, "crowding"])
        ))
        selected.append(winner)
        available.remove(winner)

    return ranked.loc[selected, "tree"].tolist()


def pareto_front(df):
    candidates = (
        df.sort_values(["features", "val_NRMSE"])
        .drop_duplicates("features")
    )
    previous = candidates.val_NRMSE.cummin().shift(fill_value=np.inf)
    return candidates[candidates.val_NRMSE < previous]


def breed(survivors):
    population = list(survivors)
    seen = set(population)
    attempts = 0

    while len(population) < POPULATION:
        attempts += 1
        if rng.random() < IMMIGRANT_RATE or attempts > 200:
            child = random_tree()
        else:
            i, j = rng.integers(len(survivors), size=2)
            child = survivors[i]
            if rng.random() < CROSSOVER_RATE:
                child = crossover(child, survivors[j])
            child = mutate(child)

        child = simplify(child)
        if active_leaves(child) and child not in seen:
            population.append(child)
            seen.add(child)

    return population


# ============================================================
# 7. Draw descriptor layouts
# ============================================================
COLORS = {
    "DROP": "#eeeeee", "MEAN": "#56b4e9",
    "MAX": "#e69f00", "PIXEL": "#009e73"
}


def draw_tree(ax, tree):
    for _, (y, x, size), node in nodes(tree):
        op = node[0]
        if op == "SPLIT":
            continue

        ax.add_patch(Rectangle(
            (x - 0.5, y - 0.5), size, size,
            facecolor=COLORS[op], edgecolor="white", linewidth=0.8
        ))
        if size > 1:
            ax.text(
                x + (size - 1)/2, y + (size - 1)/2,
                {"DROP": "–", "MEAN": "mean", "MAX": "max"}[op],
                ha="center", va="center", fontsize=8
            )

    ax.plot(a, a, "rx", ms=9, mew=2)
    ax.set_xlim(-0.5, N - 0.5)
    ax.set_ylim(N - 0.5, -0.5)
    ax.set_aspect("equal")
    ax.axis("off")


# ============================================================
# 8. Initialization and evolution
# ============================================================
population = [
    MEAN,
    MAX,
    full_pixels(),
    ("SPLIT", MEAN, MAX, DROP, MEAN)
]

while len(population) < POPULATION:
    candidate = random_tree()
    if active_leaves(candidate) and candidate not in population:
        population.append(candidate)

history = []
generation_tables = []

for generation in range(1, GENERATIONS + 1):
    before = len(archive)
    for tree in population:
        evaluate(tree, generation)

    current = rank_population(table(population))
    survivors = select_survivors(population)
    survivor_set = set(survivors)
    current["survives"] = [t in survivor_set for t in current.tree]
    current["generation"] = generation
    generation_tables.append(current.copy())

    best_four = current.sort_values(["val_NRMSE", "features"]).head(4)
    all_results = table()
    front = pareto_front(all_results)

    history.append({
        "generation": generation,
        "best_NRMSE": current.val_NRMSE.min(),
        "median_NRMSE": current.val_NRMSE.median(),
        "mean_features": current.features.mean(),
        "unique_descriptors": len(archive)
    })

    print(f"\n{'=' * 60}\nGENERATION {generation}/{GENERATIONS}")
    print(
        f"New descriptors: {len(archive) - before}; "
        f"total evaluated: {len(archive)}"
    )
    print("Blue=mean, orange=max, green=pixel, gray=drop; red ×=target")

    display(current[[
        "id", "features", "source_pixels", "means", "maxima", "pixels",
        "val_NRMSE", "rank", "survives"
    ]].sort_values(["val_NRMSE", "features"]))

    fig, axes = plt.subplots(
        1, 4, figsize=(14, 3.5), constrained_layout=True
    )
    for ax, (_, row) in zip(axes, best_four.iterrows()):
        draw_tree(ax, row.tree)
        ax.set_title(
            f"Descriptor {int(row.id)}\n"
            f"{int(row.features)} features; "
            f"{int(row.source_pixels)} source pixels\n"
            f"Validation NRMSE={row.val_NRMSE:.3f}",
            fontsize=10
        )
    fig.suptitle(f"Generation {generation}: four lowest-error descriptors")
    plt.show()

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.scatter(
        all_results.features, all_results.val_NRMSE,
        alpha=0.25, s=25, label="Archive"
    )
    ax.scatter(
        current.features, current.val_NRMSE,
        facecolors="none", edgecolors="tab:orange",
        s=55, label="Current generation"
    )
    ax.plot(
        front.features, front.val_NRMSE,
        "k.-", label="Cumulative Pareto front"
    )
    ax.set(
        xlabel="Number of output features",
        ylabel="Validation NRMSE",
        title="Descriptor complexity versus prediction error"
    )
    ax.grid(alpha=0.2)
    ax.legend()
    plt.show()

    if generation < GENERATIONS:
        population = breed(survivors)


# ============================================================
# 9. Final validation summary
# ============================================================
ga_results = table()
ga_history = pd.DataFrame(history)
ga_generations = pd.concat(generation_tables, ignore_index=True)
ga_front = pareto_front(ga_results)

print("\nFinal validation Pareto front:")
display(ga_front[[
    "id", "features", "source_pixels", "means", "maxima", "pixels",
    "val_NRMSE"
]])

fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
axes[0].plot(
    ga_history.generation, ga_history.best_NRMSE, "o-", label="Best"
)
axes[0].plot(
    ga_history.generation, ga_history.median_NRMSE, "o-", label="Median"
)
axes[0].set(xlabel="Generation", ylabel="Validation NRMSE")
axes[0].legend()

axes[1].plot(ga_history.generation, ga_history.mean_features, "o-")
axes[1].set(xlabel="Generation", ylabel="Mean feature count")

for ax in axes:
    ax.grid(alpha=0.2)
plt.show()


# ============================================================
# 10. Select descriptor, refit, and evaluate reserved test region
# ============================================================
best = ga_results.sort_values(["val_NRMSE", "features", "id"]).iloc[0]
selected_tree = best.tree
final_model = make_model().fit(feature_matrix(selected_tree), y_train)

P_test, y_test, test_coordinates = extract_region(boundary, W)
test_prediction = final_model.predict(feature_matrix(selected_tree, P_test))

test_rmse = np.sqrt(np.mean((test_prediction - y_test)**2))
test_baseline = np.sqrt(np.mean((y_test - y_train.mean())**2))
test_nrmse = test_rmse / test_baseline if test_baseline > 0 else np.nan

print(f"\nSelected descriptor: {int(best.id)}")
print(f"Features: {int(best.features)}; source pixels: {int(best.source_pixels)}")
print(f"Validation NRMSE: {best.val_NRMSE:.4f}")
print(f"Test RMSE: {test_rmse:.4g}; test NRMSE: {test_nrmse:.4f}")

# Explicit feature definitions, in the order passed to the regressor.
feature_definitions = pd.DataFrame([
    {
        "feature": i + 1, "operation": op,
        "row_start": region[0], "column_start": region[1],
        "region_size": region[2]
    }
    for i, (region, op) in enumerate(active_leaves(selected_tree))
])
display(feature_definitions)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
draw_tree(axes[0], selected_tree)
axes[0].set_title("Selected spatial descriptor")

axes[1].scatter(y_test, test_prediction, s=20, alpha=0.7)
lo = min(y_test.min(), test_prediction.min())
hi = max(y_test.max(), test_prediction.max())
axes[1].plot([lo, hi], [lo, hi], "k--")
axes[1].set(
    xlabel="Measured peak area",
    ylabel="Predicted peak area",
    title=f"Test NRMSE={test_nrmse:.3f}"
)
plt.show()

Now let's do it for the whole spectrum

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.patches import Rectangle
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from IPython.display import display


# ============================================================
# 1. Settings
# ============================================================
N = 12
POPULATION, GENERATIONS = 100, 100
SURVIVORS, TOURNAMENT_SIZE = 15, 5
CROSSOVER_RATE, IMMIGRANT_RATE = 0.8, 0.15
RIDGE_ALPHA = 10.0             # Fixed throughout evolution
SEED = 0

TEST_FRACTION = 0.25
MIN_FIT, MIN_VAL, MIN_TEST = 20, 15, 15
N_EXAMPLES = 3

rng = np.random.default_rng(SEED)
img = np.asarray(image, dtype=float)
cube = np.asarray(spectra, dtype=float)
E = np.asarray(energy, dtype=float).ravel()

if img.ndim != 2 or cube.shape != (*img.shape, len(E)):
    raise ValueError("Expected image (H,W), spectra (H,W,n_energy), energy (n_energy,).")
if not all(np.isfinite(v).all() for v in (img, cube, E)):
    raise ValueError("Inputs contain NaN or infinite values.")

order = np.argsort(E)
E, cube = E[order], cube[..., order]
if np.any(np.diff(E) <= 0):
    raise ValueError("Energy values must be distinct.")

H, W = img.shape
boundary = int(W * (1 - TEST_FRACTION))
a, b = (N - 1) // 2, N - (N - 1) // 2


# ============================================================
# 2. Fixed spatial split; targets are complete spectra
# ============================================================
def extract_region(x0, x1):
    coords = np.array([
        (y, x)
        for y in range(a, H - b + 1)
        for x in range(x0 + a, x1 - b + 1)
    ], dtype=int).reshape(-1, 2)

    if not len(coords):
        raise ValueError("Region too small for 12×12 patches.")

    patches = np.stack([
        img[y-a:y+b, x-a:x+b] for y, x in coords
    ])
    targets = cube[coords[:, 0], coords[:, 1], :]
    return patches, targets, coords


P_train, Y_train, coordinates = extract_region(0, boundary)

row_boundary = H // 2
fit_ids = np.flatnonzero(coordinates[:, 0] + b <= row_boundary)
val_ids = np.flatnonzero(coordinates[:, 0] - a >= row_boundary)
n_test = max(0, H - N + 1) * max(0, W - boundary - N + 1)

if (
    len(fit_ids) < MIN_FIT
    or len(val_ids) < MIN_VAL
    or n_test < MIN_TEST
):
    raise ValueError(
        f"Insufficient patches: fit={len(fit_ids)}, "
        f"validation={len(val_ids)}, test={n_test}. "
        "Use a larger image or revise the fixed split/count requirements."
    )

mean_fit_spectrum = Y_train[fit_ids].mean(axis=0)
baseline = np.sqrt(np.mean(
    (Y_train[val_ids] - mean_fit_spectrum)**2
))
if baseline <= 0:
    raise ValueError("Validation baseline error is zero.")

# Fixed, spatially spread example locations; no target-based selection.
def spread_indices(coords, count):
    count = min(count, len(coords))
    chosen = [int(np.argmin(np.sum((coords - coords.mean(axis=0))**2, axis=1)))]
    distances = np.full(len(coords), np.inf)
    while len(chosen) < count:
        distances = np.minimum(
            distances,
            np.sum((coords - coords[chosen[-1]])**2, axis=1)
        )
        distances[chosen] = -1
        chosen.append(int(np.argmax(distances)))
    return np.array(chosen)


example_local = spread_indices(coordinates[val_ids], N_EXAMPLES)
example_ids = val_ids[example_local]

print(f"Patch: {N}×{N}; spectrum channels: {len(E)}")
print(f"Fit={len(fit_ids)}, validation={len(val_ids)}, reserved test={n_test}")
print(f"Ridge alpha={RIDGE_ALPHA}; target pixel in patch=({a},{a})")

fig, ax = plt.subplots(figsize=(7, 4))
ax.imshow(img, cmap="gray")
ax.axvline(boundary - 0.5, color="red", linestyle="--")
ax.scatter(
    coordinates[fit_ids, 1], coordinates[fit_ids, 0], s=5, label="Fit"
)
ax.scatter(
    coordinates[val_ids, 1], coordinates[val_ids, 0], s=5, label="Validation"
)
for i, idx in enumerate(example_ids, 1):
    y, x = coordinates[idx]
    ax.plot(x, y, "rx")
    ax.text(x + 1, y, str(i), color="red")
ax.set_title("Fixed splits and validation spectrum examples")
ax.legend()
plt.show()


# ============================================================
# 3. Descriptor trees: 12 → 6 → 3 → 1
# ============================================================
DROP, MEAN, MAX, PIXEL = ("DROP",), ("MEAN",), ("MAX",), ("PIXEL",)
ROOT = (0, 0, N)


def child_regions(region):
    y, x, size = region
    if size == 1:
        return []
    step = size // 2 if size % 2 == 0 else 1
    return [
        (y + dy, x + dx, step)
        for dy in range(0, size, step)
        for dx in range(0, size, step)
    ]


def simplify(tree):
    if tree[0] != "SPLIT":
        return tree
    children = tuple(simplify(t) for t in tree[1:])
    return DROP if all(t == DROP for t in children) else ("SPLIT",) + children


def random_tree(region=ROOT):
    if region[2] == 1:
        return PIXEL if rng.random() < 0.7 else DROP
    op = str(rng.choice(
        ["DROP", "MEAN", "MAX", "SPLIT"], p=[0.15, 0.25, 0.20, 0.40]
    ))
    if op != "SPLIT":
        return (op,)
    return simplify(
        ("SPLIT",) + tuple(random_tree(r) for r in child_regions(region))
    )


def nodes(tree, region=ROOT, path=()):
    yield path, region, tree
    if tree[0] == "SPLIT":
        for i, (child, box) in enumerate(zip(tree[1:], child_regions(region))):
            yield from nodes(child, box, path + (i,))


def replace(tree, path, subtree):
    if not path:
        return simplify(subtree)
    children = list(tree[1:])
    children[path[0]] = replace(children[path[0]], path[1:], subtree)
    return simplify(("SPLIT",) + tuple(children))


def active_leaves(tree):
    return [
        (region, node[0])
        for _, region, node in nodes(tree)
        if node[0] not in ("DROP", "SPLIT")
    ]


def full_pixels(region=ROOT):
    if region[2] == 1:
        return PIXEL
    return ("SPLIT",) + tuple(full_pixels(r) for r in child_regions(region))


# ============================================================
# 4. Features and multi-output Ridge evaluation
# ============================================================
feature_cache, archive = {}, {}


def measure(patches, region, op):
    y, x, size = region
    values = patches[:, y:y+size, x:x+size]
    if op in ("MEAN", "PIXEL"):
        return values.mean(axis=(1, 2))
    return values.max(axis=(1, 2))


def feature_matrix(tree, patches=None):
    columns = []
    for region, op in active_leaves(tree):
        if patches is None:
            key = (region, op)
            if key not in feature_cache:
                feature_cache[key] = measure(P_train, region, op)
            columns.append(feature_cache[key])
        else:
            columns.append(measure(patches, region, op))
    return np.column_stack(columns)


def make_model():
    # Input scaling is fitted only on the data passed to fit().
    # Spectral targets remain in their original intensity units.
    return make_pipeline(
        StandardScaler(),
        Ridge(alpha=RIDGE_ALPHA, solver="svd")
    )


def evaluate(tree, generation):
    if tree not in archive:
        X = feature_matrix(tree)
        model = make_model().fit(X[fit_ids], Y_train[fit_ids])
        prediction = model.predict(X[val_ids])
        rmse = np.sqrt(np.mean((prediction - Y_train[val_ids])**2))

        if not np.isfinite(rmse):
            raise RuntimeError("Nonfinite prediction error.")

        leaves = active_leaves(tree)
        archive[tree] = {
            "id": len(archive) + 1,
            "features": X.shape[1],
            "source_pixels": sum(r[2]**2 for r, _ in leaves),
            "val_NRMSE": rmse / baseline,
            "first_generation": generation,
            # Store only the example spectra to keep memory modest.
            "example_predictions": prediction[example_local].copy()
        }


def table(trees=None):
    if trees is None:
        trees = list(archive)
    return pd.DataFrame([
        {
            "tree": tree,
            **{k: v for k, v in archive[tree].items()
               if k != "example_predictions"}
        }
        for tree in trees
    ])


# ============================================================
# 5. Mutation and crossover
# ============================================================
def mutate(tree):
    items = list(nodes(tree))
    eligible = {
        "refine": [
            v for v in items
            if v[1][2] > 1 and v[2][0] in ("MEAN", "MAX")
        ],
        "coarsen": [v for v in items if v[2][0] == "SPLIT"],
        "pool": [v for v in items if v[2][0] in ("MEAN", "MAX")],
        "drop": [v for v in items if v[2][0] != "DROP"],
        "activate": [v for v in items if v[2][0] == "DROP"]
    }
    weights = {
        "refine": 0.25, "coarsen": 0.25, "pool": 0.20,
        "drop": 0.15, "activate": 0.15
    }
    choices = [name for name, options in eligible.items() if options]
    probabilities = np.array([weights[name] for name in choices])
    probabilities /= probabilities.sum()

    action = str(rng.choice(choices, p=probabilities))
    options = eligible[action]
    path, region, node = options[int(rng.integers(len(options)))]

    if action == "refine":
        replacement = ("SPLIT",) + tuple(
            PIXEL if r[2] == 1 else node for r in child_regions(region)
        )
    elif action == "coarsen":
        replacement = MEAN if rng.random() < 0.5 else MAX
    elif action == "pool":
        replacement = MAX if node == MEAN else MEAN
    elif action == "drop":
        replacement = DROP
    else:
        replacement = (
            PIXEL if region[2] == 1
            else MEAN if rng.random() < 0.5 else MAX
        )

    child = replace(tree, path, replacement)
    return child if active_leaves(child) else tree


def crossover(parent1, parent2):
    first = {p: t for p, _, t in nodes(parent1)}
    second = {p: t for p, _, t in nodes(parent2)}
    common = sorted((set(first) & set(second)) - {()})
    if not common:
        return parent1
    path = common[int(rng.integers(len(common)))]
    child = replace(parent1, path, second[path])
    return child if active_leaves(child) else parent1


# ============================================================
# 6. Pareto tournaments and reproduction
# ============================================================
def rank_population(df):
    df = df.reset_index(drop=True).copy()
    values = df[["features", "val_NRMSE"]].to_numpy()
    ranks = np.zeros(len(df), dtype=int)
    crowding = np.zeros(len(df))
    remaining = list(range(len(df)))
    rank = 0

    while remaining:
        front = [
            i for i in remaining
            if not any(
                np.all(values[j] <= values[i])
                and np.any(values[j] < values[i])
                for j in remaining if j != i
            )
        ]
        ranks[front] = rank

        for column in range(2):
            ordered = sorted(front, key=lambda i: values[i, column])
            span = values[ordered[-1], column] - values[ordered[0], column]
            if span == 0:
                continue
            crowding[ordered[0]] = crowding[ordered[-1]] = np.inf
            for k in range(1, len(ordered) - 1):
                crowding[ordered[k]] += (
                    values[ordered[k+1], column]
                    - values[ordered[k-1], column]
                ) / span

        remaining = [i for i in remaining if i not in front]
        rank += 1

    df["rank"], df["crowding"] = ranks, crowding
    return df


def select_survivors(population):
    ranked = rank_population(table(population))
    elite = int(ranked.sort_values(["val_NRMSE", "features"]).index[0])
    selected = [elite]
    available = [i for i in ranked.index if i != elite]

    while len(selected) < SURVIVORS:
        contestants = rng.choice(
            available, min(TOURNAMENT_SIZE, len(available)), replace=False
        )
        winner = int(min(
            contestants,
            key=lambda i: (ranked.loc[i, "rank"], -ranked.loc[i, "crowding"])
        ))
        selected.append(winner)
        available.remove(winner)

    return ranked.loc[selected, "tree"].tolist()


def pareto_front(df):
    candidates = (
        df.sort_values(["features", "val_NRMSE"])
        .drop_duplicates("features")
    )
    previous = candidates.val_NRMSE.cummin().shift(fill_value=np.inf)
    return candidates[candidates.val_NRMSE < previous]


def breed(survivors):
    population = list(survivors)
    seen = set(population)
    attempts = 0

    while len(population) < POPULATION:
        attempts += 1
        if rng.random() < IMMIGRANT_RATE or attempts > 200:
            child = random_tree()
        else:
            i, j = rng.integers(len(survivors), size=2)
            child = survivors[i]
            if rng.random() < CROSSOVER_RATE:
                child = crossover(child, survivors[j])
            child = mutate(child)

        child = simplify(child)
        if active_leaves(child) and child not in seen:
            population.append(child)
            seen.add(child)

    return population


# ============================================================
# 7. Plot helpers
# ============================================================
COLORS = {
    "DROP": "#eeeeee", "MEAN": "#56b4e9",
    "MAX": "#e69f00", "PIXEL": "#009e73"
}


def draw_tree(ax, tree):
    for _, (y, x, size), node in nodes(tree):
        op = node[0]
        if op == "SPLIT":
            continue
        ax.add_patch(Rectangle(
            (x - 0.5, y - 0.5), size, size,
            facecolor=COLORS[op], edgecolor="white", linewidth=0.7
        ))
        if size > 1:
            ax.text(
                x + (size - 1)/2, y + (size - 1)/2,
                {"DROP": "–", "MEAN": "mean", "MAX": "max"}[op],
                ha="center", va="center", fontsize=8
            )
    ax.plot(a, a, "rx", ms=9, mew=2)
    ax.set_xlim(-0.5, N - 0.5)
    ax.set_ylim(N - 0.5, -0.5)
    ax.set_aspect("equal")
    ax.axis("off")


def plot_example_spectra(actual, predictions, coords, title):
    # predictions: (n_models, n_examples, n_energy)
    mean_prediction = predictions.mean(axis=0)
    disagreement = predictions.std(axis=0)

    fig, axes = plt.subplots(
        1, len(coords), figsize=(5 * len(coords), 3.5),
        squeeze=False, constrained_layout=True
    )
    for i, ax in enumerate(axes[0]):
        for model_prediction in predictions:
            ax.plot(E, model_prediction[i], color="tab:blue", alpha=0.15)

        ax.plot(E, actual[i], "k-", lw=2, label="Measured")
        ax.plot(
            E, mean_prediction[i], color="tab:orange", lw=2,
            label=f"Mean of {len(predictions)} predictions"
        )
        ax.fill_between(
            E,
            mean_prediction[i] - disagreement[i],
            mean_prediction[i] + disagreement[i],
            color="tab:orange", alpha=0.2, label="±1 model SD"
        )
        y, x = coords[i]
        ax.set(
            xlabel="Energy", ylabel="Intensity",
            title=f"Location ({y}, {x})"
        )
        ax.legend(fontsize=8)

    fig.suptitle(title)
    plt.show()


# ============================================================
# 8. Evolution with generation-by-generation spectra
# ============================================================
population = [
    MEAN, MAX, full_pixels(),
    ("SPLIT", MEAN, MAX, DROP, MEAN)
]
while len(population) < POPULATION:
    candidate = random_tree()
    if active_leaves(candidate) and candidate not in population:
        population.append(candidate)

history, generation_tables = [], []

for generation in range(1, GENERATIONS + 1):
    before = len(archive)
    for tree in population:
        evaluate(tree, generation)

    current = rank_population(table(population))
    survivors = select_survivors(population)
    survivor_set = set(survivors)

    current["survives"] = [t in survivor_set for t in current.tree]
    current["generation"] = generation
    generation_tables.append(current.copy())

    best_four = current.sort_values(["val_NRMSE", "features"]).head(4)
    all_results = table()
    front = pareto_front(all_results)

    history.append({
        "generation": generation,
        "best_NRMSE": current.val_NRMSE.min(),
        "median_NRMSE": current.val_NRMSE.median(),
        "mean_features": current.features.mean(),
        "unique_descriptors": len(archive)
    })

    print(f"\nGENERATION {generation}/{GENERATIONS}")
    print(
        f"New descriptors: {len(archive) - before}; "
        f"total: {len(archive)}"
    )
    display(current[[
        "id", "features", "source_pixels", "val_NRMSE", "rank", "survives"
    ]].sort_values(["val_NRMSE", "features"]))

    fig, axes = plt.subplots(
        1, 4, figsize=(14, 3.3), constrained_layout=True
    )
    for ax, (_, row) in zip(axes, best_four.iterrows()):
        draw_tree(ax, row.tree)
        ax.set_title(
            f"ID {int(row.id)} | {int(row.features)} features\n"
            f"Validation NRMSE={row.val_NRMSE:.3f}",
            fontsize=10
        )
    fig.suptitle(
        f"Generation {generation}: blue=mean, orange=max, "
        "green=pixel, gray=drop"
    )
    plt.show()

    predictions = np.stack([
        archive[t]["example_predictions"] for t in best_four.tree
    ])
    plot_example_spectra(
        Y_train[example_ids], predictions, coordinates[example_ids],
        f"Generation {generation}: four best descriptors at fixed validation pixels"
    )

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.scatter(
        all_results.features, all_results.val_NRMSE,
        s=25, alpha=0.25, label="Archive"
    )
    ax.scatter(
        current.features, current.val_NRMSE,
        s=55, facecolors="none", edgecolors="tab:orange",
        label="Current generation"
    )
    ax.plot(
        front.features, front.val_NRMSE,
        "k.-", label="Cumulative Pareto front"
    )
    ax.set(xlabel="Output features", ylabel="Whole-spectrum validation NRMSE")
    ax.grid(alpha=0.2)
    ax.legend()
    plt.show()

    if generation < GENERATIONS:
        population = breed(survivors)


# ============================================================
# 9. Final selection using validation only
# ============================================================
ga_results = table()
ga_history = pd.DataFrame(history)
ga_generations = pd.concat(generation_tables, ignore_index=True)
ga_front = pareto_front(ga_results)

# Choose four descriptors before accessing test targets.
final_four = ga_results.sort_values(
    ["val_NRMSE", "features", "id"]
).head(4)
selected_tree = final_four.iloc[0].tree

print("\nFinal validation Pareto front:")
display(ga_front[["id", "features", "source_pixels", "val_NRMSE"]])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ga_history.generation, ga_history.best_NRMSE, "o-", label="Best")
ax.plot(ga_history.generation, ga_history.median_NRMSE, "o-", label="Median")
ax.set(xlabel="Generation", ylabel="Validation NRMSE")
ax.legend()
ax.grid(alpha=0.2)
plt.show()


# ============================================================
# 10. Refit top four and evaluate test spectra
# ============================================================
P_test, Y_test, test_coordinates = extract_region(boundary, W)
final_models, test_predictions = [], []

for tree in final_four.tree:
    model = make_model().fit(feature_matrix(tree), Y_train)
    prediction = model.predict(feature_matrix(tree, P_test))
    final_models.append(model)
    test_predictions.append(prediction)

test_predictions = np.stack(test_predictions)
ensemble_prediction = test_predictions.mean(axis=0)
final_model = final_models[0]

test_baseline = np.sqrt(np.mean(
    (Y_test - Y_train.mean(axis=0))**2
))

summary = []
for label, prediction in [
    ("Best individual descriptor", test_predictions[0]),
    ("Mean of four selected descriptors", ensemble_prediction)
]:
    rmse = np.sqrt(np.mean((prediction - Y_test)**2))
    summary.append({
        "predictor": label,
        "test_RMSE": rmse,
        "test_NRMSE": rmse / test_baseline if test_baseline > 0 else np.nan
    })
display(pd.DataFrame(summary))

test_examples = spread_indices(test_coordinates, N_EXAMPLES)
plot_example_spectra(
    Y_test[test_examples],
    test_predictions[:, test_examples, :],
    test_coordinates[test_examples],
    "Final refitted models: reserved test pixels"
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(
    E, np.sqrt(np.mean((test_predictions[0] - Y_test)**2, axis=0)),
    label="Best individual descriptor"
)
ax.plot(
    E, np.sqrt(np.mean((ensemble_prediction - Y_test)**2, axis=0)),
    label="Mean of four"
)
ax.set(xlabel="Energy", ylabel="Test RMSE", title="Prediction error by energy channel")
ax.legend()
ax.grid(alpha=0.2)
plt.show()

In [ ]:
# ============================================================
# All Pareto descriptors, their layouts, and the estimated knee
# Requires: ga_results, draw_tree()
# ============================================================

def inspect_pareto_descriptors(results, layouts_per_row=5):
    df = results.copy().reset_index(drop=True)
    values = df[["features", "val_NRMSE"]].to_numpy()

    # Keep ALL nondominated descriptors, including exact objective ties.
    keep = np.array([
        not np.any(
            np.all(values <= values[i], axis=1)
            & np.any(values < values[i], axis=1)
        )
        for i in range(len(df))
    ])

    all_front = df.loc[keep].sort_values(
        ["features", "val_NRMSE", "id"]
    ).reset_index(drop=True)

    # One representative per objective point for geometry.
    curve = all_front.drop_duplicates(
        ["features", "val_NRMSE"]
    ).reset_index(drop=True)

    knee_index = None
    curve["knee_score"] = np.nan

    if (
        len(curve) >= 3
        and np.ptp(curve.features) > 0
        and np.ptp(curve.val_NRMSE) > 0
    ):
        # Linear feature-count scale; both axes normalized to [0, 1].
        x = (
            (curve.features.to_numpy() - curve.features.min())
            / np.ptp(curve.features)
        )
        y = (
            (curve.val_NRMSE.to_numpy() - curve.val_NRMSE.min())
            / np.ptp(curve.val_NRMSE)
        )

        # Endpoint chord is y = 1 - x.
        score = (1 - x - y) / np.sqrt(2)
        curve["knee_score"] = score

        candidate = int(np.argmax(score[1:-1])) + 1
        if score[candidate] > 1e-6:
            knee_index = candidate

    all_front["estimated_knee"] = False
    if knee_index is not None:
        knee = curve.iloc[knee_index]
        all_front["estimated_knee"] = (
            (all_front.features == knee.features)
            & (all_front.val_NRMSE == knee.val_NRMSE)
        )

    columns = [
        c for c in [
            "id", "features", "source_pixels",
            "val_NRMSE", "first_generation", "estimated_knee"
        ]
        if c in all_front.columns
    ]

    print(f"All nondominated descriptors: {len(all_front)}")
    display(all_front[columns])

    # Quantify how much error rises when moving toward fewer features.
    transitions = []
    for i in range(len(curve) - 1):
        smaller = curve.iloc[i]
        larger = curve.iloc[i + 1]
        removed = int(larger.features - smaller.features)
        increase = float(smaller.val_NRMSE - larger.val_NRMSE)

        transitions.append({
            "from_id": int(larger.id),
            "to_id": int(smaller.id),
            "from_features": int(larger.features),
            "to_features": int(smaller.features),
            "features_removed": removed,
            "NRMSE_increase": increase,
            "increase_per_removed_feature": increase / removed
        })

    transition_table = pd.DataFrame(transitions)
    print("\nError increase when simplifying along the front:")
    display(transition_table)

    # Plot archive, front, and knee.
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.scatter(
        df.features, df.val_NRMSE,
        s=25, alpha=0.2, label="All evaluated descriptors"
    )
    ax.plot(
        curve.features, curve.val_NRMSE,
        "ko-", ms=5, label="Pareto front"
    )

    for _, row in curve.iterrows():
        ax.annotate(
            f"ID {int(row.id)}",
            (row.features, row.val_NRMSE),
            xytext=(4, 6), textcoords="offset points", fontsize=8
        )

    if knee_index is not None:
        knee = curve.iloc[knee_index]
        ax.scatter(
            [knee.features], [knee.val_NRMSE],
            marker="*", s=250, color="red", zorder=5,
            label="Estimated knee"
        )

        # Highlight the next simplification step from the knee.
        simpler = curve.iloc[knee_index - 1]
        ax.plot(
            [simpler.features, knee.features],
            [simpler.val_NRMSE, knee.val_NRMSE],
            color="red", linewidth=3,
            label="Next step toward fewer features"
        )
        print(
            f"\nEstimated knee: ID {int(knee.id)}, "
            f"{int(knee.features)} features, "
            f"validation NRMSE={knee.val_NRMSE:.4f}"
        )
        print("Knee and adjacent Pareto configurations:")
        display(curve.iloc[knee_index-1:knee_index+2][
            ["id", "features", "val_NRMSE"]
        ])
    else:
        print("\nNo distinct interior knee detected on this front.")

    ax.set(
        xlabel="Number of output features",
        ylabel="Whole-spectrum validation NRMSE",
        title="Pareto descriptors and onset of increasing error"
    )
    ax.grid(alpha=0.2)
    ax.legend()
    plt.tight_layout()
    plt.show()

    # Display EVERY Pareto layout in batches of at most 15.
    batch_size = 3 * layouts_per_row

    for start in range(0, len(all_front), batch_size):
        batch = all_front.iloc[start:start + batch_size]
        ncols = min(layouts_per_row, len(batch))
        nrows = int(np.ceil(len(batch) / ncols))

        fig, axes = plt.subplots(
            nrows, ncols,
            figsize=(3.2 * ncols, 3.5 * nrows),
            squeeze=False, constrained_layout=True
        )
        for ax, (_, row) in zip(axes.ravel(), batch.iterrows()):
            draw_tree(ax, row["tree"])
            label = " — KNEE" if row.estimated_knee else ""
            ax.set_title(
                f"ID {int(row.id)}{label}\n"
                f"{int(row.features)} features | "
                f"NRMSE={row.val_NRMSE:.3f}",
                color="red" if row.estimated_knee else "black",
                fontsize=10
            )
        for ax in axes.ravel()[len(batch):]:
            ax.axis("off")

        fig.suptitle(
            "Pareto layouts: increasing feature count\n"
            "Blue=mean; orange=max; green=pixel; gray=drop"
        )
        plt.show()

    return all_front, curve, transition_table


pareto_descriptors, pareto_curve, pareto_error_steps = (
    inspect_pareto_descriptors(ga_results)
)

# Trees for every Pareto descriptor, indexed by descriptor ID.
pareto_trees = {
    int(row["id"]): row["tree"]
    for _, row in pareto_descriptors.iterrows()
}

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.patches import Rectangle
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.decomposition import PCA
from IPython.display import display


# ============================================================
# 1. SETTINGS AND INPUTS
# ============================================================
N = 12
POPULATION, GENERATIONS = 100, 100
SURVIVORS, TOURNAMENT_SIZE = 20, 5
CROSSOVER_RATE, IMMIGRANT_RATE = 0.8, 0.15
RIDGE_ALPHA = 10.0
SEED = 0

TEST_FRACTION = 0.25
MIN_FIT, MIN_VAL, MIN_TEST = 20, 15, 15
N_EXAMPLES, PCA_COMPONENTS = 3, 3

rng = np.random.default_rng(SEED)
img = np.asarray(image, dtype=float)
cube = np.asarray(spectra, dtype=float)
E = np.asarray(energy, dtype=float).ravel()

if img.ndim != 2 or cube.shape != (*img.shape, len(E)):
    raise ValueError("Expected image (H,W), spectra (H,W,n_energy), energy (n_energy,).")
if not all(np.isfinite(v).all() for v in (img, cube, E)):
    raise ValueError("Inputs contain NaN or infinite values.")

order = np.argsort(E)
E, cube = E[order], cube[..., order]
if np.any(np.diff(E) <= 0):
    raise ValueError("Energy values must be distinct.")

H, W = img.shape
boundary = int(W * (1 - TEST_FRACTION))
a, b = (N - 1) // 2, N - (N - 1) // 2


# ============================================================
# 2. SPATIAL SPLITS: NO PATCH OVERLAP ACROSS BOUNDARIES
# ============================================================
def extract_region(x0, x1):
    coords = np.array([
        (y, x)
        for y in range(a, H - b + 1)
        for x in range(x0 + a, x1 - b + 1)
    ], dtype=int).reshape(-1, 2)
    if not len(coords):
        raise ValueError("Region too small for 12×12 patches.")
    patches = np.stack([img[y-a:y+b, x-a:x+b] for y, x in coords])
    return patches, cube[coords[:, 0], coords[:, 1]], coords


P_train, Y_train, coordinates = extract_region(0, boundary)
row_boundary = H // 2
fit_ids = np.flatnonzero(coordinates[:, 0] + b <= row_boundary)
val_ids = np.flatnonzero(coordinates[:, 0] - a >= row_boundary)

n_test = max(0, H - N + 1) * max(0, W - boundary - N + 1)
if (
    len(fit_ids) < MIN_FIT or len(val_ids) < MIN_VAL
    or n_test < MIN_TEST
):
    raise ValueError(
        f"Too few patches: fit={len(fit_ids)}, validation={len(val_ids)}, "
        f"test={n_test}. Use a larger image or revise the fixed split."
    )

baseline = np.sqrt(np.mean(
    (Y_train[val_ids] - Y_train[fit_ids].mean(axis=0))**2
))
if baseline <= 0:
    raise ValueError("Validation baseline error is zero.")


def spread_indices(coords, count):
    """Choose fixed, spatially spread examples without inspecting targets."""
    count = min(count, len(coords))
    chosen = [int(np.argmin(np.sum((coords - coords.mean(axis=0))**2, axis=1)))]
    distance = np.full(len(coords), np.inf)
    while len(chosen) < count:
        distance = np.minimum(
            distance, np.sum((coords - coords[chosen[-1]])**2, axis=1)
        )
        distance[chosen] = -1
        chosen.append(int(np.argmax(distance)))
    return np.asarray(chosen)


example_local = spread_indices(coordinates[val_ids], N_EXAMPLES)
example_ids = val_ids[example_local]

print(f"Patch={N}×{N}; target index=({a},{a}); channels={len(E)}")
print(f"Fit={len(fit_ids)}, validation={len(val_ids)}, reserved test={n_test}")
print(f"Population={POPULATION}, survivors={SURVIVORS}, Ridge alpha={RIDGE_ALPHA}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.imshow(img, cmap="gray")
ax.axvline(boundary - 0.5, color="red", linestyle="--")
for ids, label in [(fit_ids, "Fit"), (val_ids, "Validation")]:
    ax.scatter(coordinates[ids, 1], coordinates[ids, 0], s=5, label=label)
for i, idx in enumerate(example_ids, 1):
    y, x = coordinates[idx]
    ax.plot(x, y, "rx")
    ax.text(x + 1, y, str(i), color="red")
ax.set_title("Fixed spatial split and validation examples")
ax.legend()
plt.show()


# ============================================================
# 3. DESCRIPTOR TREES: 12 → 6 → 3 → 1
# ============================================================
# Larger squares split into four quadrants; 3×3 splits into nine pixels.
DROP, MEAN, MAX, PIXEL = ("DROP",), ("MEAN",), ("MAX",), ("PIXEL",)
ROOT = (0, 0, N)


def child_regions(region):
    y, x, size = region
    if size == 1:
        return []
    step = size // 2 if size % 2 == 0 else 1
    return [
        (y + dy, x + dx, step)
        for dy in range(0, size, step)
        for dx in range(0, size, step)
    ]


def simplify(tree):
    if tree[0] != "SPLIT":
        return tree
    children = tuple(simplify(t) for t in tree[1:])
    return DROP if all(t == DROP for t in children) else ("SPLIT",) + children


def random_tree(region=ROOT):
    if region[2] == 1:
        return PIXEL if rng.random() < 0.7 else DROP
    op = str(rng.choice(
        ["DROP", "MEAN", "MAX", "SPLIT"], p=[0.15, 0.25, 0.20, 0.40]
    ))
    if op != "SPLIT":
        return (op,)
    return simplify(
        ("SPLIT",) + tuple(random_tree(r) for r in child_regions(region))
    )


def nodes(tree, region=ROOT, path=()):
    yield path, region, tree
    if tree[0] == "SPLIT":
        for i, (child, box) in enumerate(zip(tree[1:], child_regions(region))):
            yield from nodes(child, box, path + (i,))


def replace(tree, path, subtree):
    if not path:
        return simplify(subtree)
    children = list(tree[1:])
    children[path[0]] = replace(children[path[0]], path[1:], subtree)
    return simplify(("SPLIT",) + tuple(children))


def active_leaves(tree):
    return [
        (region, node[0]) for _, region, node in nodes(tree)
        if node[0] not in ("DROP", "SPLIT")
    ]


def full_pixels(region=ROOT):
    if region[2] == 1:
        return PIXEL
    return ("SPLIT",) + tuple(full_pixels(r) for r in child_regions(region))


# ============================================================
# 4. FEATURES, RIDGE, AND CACHED VALIDATION
# ============================================================
feature_cache, archive = {}, {}


def measure(patches, region, op):
    y, x, size = region
    values = patches[:, y:y+size, x:x+size]
    return (
        values.mean(axis=(1, 2))
        if op in ("MEAN", "PIXEL") else values.max(axis=(1, 2))
    )


def feature_matrix(tree, patches=None):
    columns = []
    for region, op in active_leaves(tree):
        if patches is None:
            key = (region, op)
            if key not in feature_cache:
                feature_cache[key] = measure(P_train, region, op)
            columns.append(feature_cache[key])
        else:
            columns.append(measure(patches, region, op))
    return np.column_stack(columns)


def make_model():
    # Fit input scaling within each fit. Leave spectra in original units.
    return make_pipeline(
        StandardScaler(), Ridge(alpha=RIDGE_ALPHA, solver="svd")
    )


def evaluate(tree, generation):
    if tree in archive:
        return
    X = feature_matrix(tree)
    model = make_model().fit(X[fit_ids], Y_train[fit_ids])
    prediction = model.predict(X[val_ids])
    pixel_rmse = np.sqrt(np.mean((prediction - Y_train[val_ids])**2, axis=1))
    score = np.sqrt(np.mean(pixel_rmse**2)) / baseline
    if not np.isfinite(score):
        raise RuntimeError("Nonfinite validation error.")

    leaves = active_leaves(tree)
    archive[tree] = {
        "id": len(archive) + 1,
        "features": X.shape[1],
        "source_pixels": sum(region[2]**2 for region, _ in leaves),
        "val_NRMSE": score,
        "first_generation": generation,
        "pixel_rmse": pixel_rmse,
        "example_predictions": prediction[example_local].copy()
    }


def table(trees=None):
    if trees is None:
        trees = list(archive)
    return pd.DataFrame([
        {
            "tree": tree,
            **{k: v for k, v in archive[tree].items()
               if k not in ("pixel_rmse", "example_predictions")}
        } for tree in trees
    ])


# ============================================================
# 5. MUTATION AND CROSSOVER
# ============================================================
def mutate(tree):
    items = list(nodes(tree))
    eligible = {
        "refine": [v for v in items if v[1][2] > 1 and v[2][0] in ("MEAN", "MAX")],
        "coarsen": [v for v in items if v[2][0] == "SPLIT"],
        "pool": [v for v in items if v[2][0] in ("MEAN", "MAX")],
        "drop": [v for v in items if v[2][0] != "DROP"],
        "activate": [v for v in items if v[2][0] == "DROP"]
    }
    weights = dict(refine=.25, coarsen=.25, pool=.20, drop=.15, activate=.15)
    choices = [name for name in eligible if eligible[name]]
    probability = np.array([weights[name] for name in choices])
    action = str(rng.choice(choices, p=probability / probability.sum()))
    options = eligible[action]
    path, region, node = options[int(rng.integers(len(options)))]

    if action == "refine":
        new = ("SPLIT",) + tuple(
            PIXEL if r[2] == 1 else node for r in child_regions(region)
        )
    elif action == "coarsen":
        new = MEAN if rng.random() < .5 else MAX
    elif action == "pool":
        new = MAX if node == MEAN else MEAN
    elif action == "drop":
        new = DROP
    else:
        new = PIXEL if region[2] == 1 else (MEAN if rng.random() < .5 else MAX)

    child = replace(tree, path, new)
    return child if active_leaves(child) else tree


def crossover(parent1, parent2):
    first = {p: t for p, _, t in nodes(parent1)}
    second = {p: t for p, _, t in nodes(parent2)}
    common = sorted((set(first) & set(second)) - {()})
    if not common:
        return parent1
    path = common[int(rng.integers(len(common)))]
    child = replace(parent1, path, second[path])
    return child if active_leaves(child) else parent1


# ============================================================
# 6. PARETO RANKING, TOURNAMENTS, AND REPRODUCTION
# ============================================================
def rank_population(df):
    df = df.reset_index(drop=True).copy()
    values = df[["features", "val_NRMSE"]].to_numpy()
    ranks, crowding = np.zeros(len(df), int), np.zeros(len(df))
    remaining, rank = list(range(len(df))), 0

    while remaining:
        front = [
            i for i in remaining if not any(
                np.all(values[j] <= values[i]) and np.any(values[j] < values[i])
                for j in remaining if j != i
            )
        ]
        ranks[front] = rank
        for column in range(2):
            ordered = sorted(front, key=lambda i: values[i, column])
            span = values[ordered[-1], column] - values[ordered[0], column]
            if span == 0:
                continue
            crowding[ordered[0]] = crowding[ordered[-1]] = np.inf
            for k in range(1, len(ordered) - 1):
                crowding[ordered[k]] += (
                    values[ordered[k+1], column] - values[ordered[k-1], column]
                ) / span
        remaining = [i for i in remaining if i not in front]
        rank += 1

    df["rank"], df["crowding"] = ranks, crowding
    return df


def all_pareto(df):
    values = df[["features", "val_NRMSE"]].to_numpy()
    keep = [
        not np.any(
            np.all(values <= v, axis=1) & np.any(values < v, axis=1)
        ) for v in values
    ]
    return df.loc[keep].sort_values(
        ["features", "val_NRMSE", "id"]
    ).reset_index(drop=True)


def front_curve(df):
    return all_pareto(df).drop_duplicates(
        ["features", "val_NRMSE"]
    ).reset_index(drop=True)


def select_survivors(population):
    ranked = rank_population(table(population))
    elite = int(ranked.sort_values(["val_NRMSE", "features"]).index[0])
    chosen = [elite]
    available = [i for i in ranked.index if i != elite]
    while len(chosen) < SURVIVORS:
        contestants = rng.choice(
            available, min(TOURNAMENT_SIZE, len(available)), replace=False
        )
        winner = int(min(
            contestants,
            key=lambda i: (ranked.loc[i, "rank"], -ranked.loc[i, "crowding"])
        ))
        chosen.append(winner)
        available.remove(winner)
    return ranked.loc[chosen, "tree"].tolist()


def breed(survivors):
    population, seen, attempts = list(survivors), set(survivors), 0
    while len(population) < POPULATION:
        attempts += 1
        if rng.random() < IMMIGRANT_RATE or attempts > 200:
            child = random_tree()
        else:
            i, j = rng.integers(len(survivors), size=2)
            child = survivors[i]
            if rng.random() < CROSSOVER_RATE:
                child = crossover(child, survivors[j])
            child = mutate(child)
        child = simplify(child)
        if active_leaves(child) and child not in seen:
            population.append(child)
            seen.add(child)
    return population


# ============================================================
# 7. PLOTTING HELPERS
# ============================================================
COLORS = dict(DROP="#eeeeee", MEAN="#56b4e9", MAX="#e69f00", PIXEL="#009e73")


def draw_tree(ax, tree):
    for _, (y, x, size), node in nodes(tree):
        op = node[0]
        if op == "SPLIT":
            continue
        ax.add_patch(Rectangle(
            (x-.5, y-.5), size, size,
            facecolor=COLORS[op], edgecolor="white", linewidth=.7
        ))
        if size > 1:
            ax.text(
                x+(size-1)/2, y+(size-1)/2,
                {"DROP": "–", "MEAN": "mean", "MAX": "max"}[op],
                ha="center", va="center", fontsize=8
            )
    ax.plot(a, a, "rx", ms=9, mew=2)
    ax.set(xlim=(-.5, N-.5), ylim=(N-.5, -.5), aspect="equal")
    ax.axis("off")


def plot_layouts(rows, title, knee_ids=()):
    # Show every descriptor, in batches to avoid excessively tall figures.
    for start in range(0, len(rows), 12):
        batch = rows.iloc[start:start+12]
        cols = min(4, len(batch))
        fig, axes = plt.subplots(
            int(np.ceil(len(batch)/cols)), cols,
            figsize=(3.5*cols, 3.3*int(np.ceil(len(batch)/cols))),
            squeeze=False, constrained_layout=True
        )
        for ax, (_, row) in zip(axes.ravel(), batch.iterrows()):
            draw_tree(ax, row["tree"])
            knee = int(row["id"]) in knee_ids
            ax.set_title(
                f"ID {int(row['id'])}" + (" — KNEE" if knee else "") +
                f"\n{int(row.features)} features; error={row.val_NRMSE:.3f}",
                fontsize=10, color="red" if knee else "black"
            )
        for ax in axes.ravel()[len(batch):]:
            ax.axis("off")
        fig.suptitle(title)
        plt.show()


def plot_spectra(actual, predictions, coords, title):
    # predictions: model × example × energy
    mean, sd = predictions.mean(axis=0), predictions.std(axis=0)
    fig, axes = plt.subplots(
        1, len(coords), figsize=(5*len(coords), 3.5),
        squeeze=False, constrained_layout=True
    )
    for i, ax in enumerate(axes[0]):
        for p in predictions:
            ax.plot(E, p[i], color="tab:blue", alpha=.15)
        ax.plot(E, actual[i], "k-", lw=2, label="Measured")
        ax.plot(E, mean[i], color="tab:orange", lw=2, label="Mean prediction")
        ax.fill_between(
            E, mean[i]-sd[i], mean[i]+sd[i],
            color="tab:orange", alpha=.2, label="±1 model SD"
        )
        ax.set(
            xlabel="Energy", ylabel="Intensity",
            title=f"Location {tuple(coords[i])}"
        )
        ax.legend(fontsize=8)
    fig.suptitle(title)
    plt.show()


def error_map_for_tree(tree):
    """Validation spectral RMSE at each pixel; other locations are NaN."""
    result = np.full((H, W), np.nan)
    xy = coordinates[val_ids]
    result[xy[:, 0], xy[:, 1]] = archive[tree]["pixel_rmse"]
    return result


# ============================================================
# 8. EVOLUTION AND GENERATION-BY-GENERATION DIAGNOSTICS
# ============================================================
population = [MEAN, MAX, full_pixels(), ("SPLIT", MEAN, MAX, DROP, MEAN)]
while len(population) < POPULATION:
    candidate = random_tree()
    if active_leaves(candidate) and candidate not in population:
        population.append(candidate)

history, generation_tables = [], []
error_maps, error_records = [], []

for generation in range(1, GENERATIONS + 1):
    before = len(archive)
    for tree in population:
        evaluate(tree, generation)

    current = rank_population(table(population))
    survivors = select_survivors(population)
    survivor_set = set(survivors)
    current["survives"] = [t in survivor_set for t in current.tree]
    current["generation"] = generation
    generation_tables.append(current.copy())

    best_four = current.sort_values(["val_NRMSE", "features"]).head(4)
    winner = best_four.iloc[0]
    winner_map = error_map_for_tree(winner.tree)
    error_maps.append(winner_map.copy())
    error_records.append({
        "generation": generation, "descriptor_id": int(winner["id"]),
        "features": int(winner.features), "val_NRMSE": winner.val_NRMSE,
        "tree": winner.tree
    })

    history.append({
        "generation": generation,
        "best_NRMSE": current.val_NRMSE.min(),
        "median_NRMSE": current.val_NRMSE.median(),
        "mean_features": current.features.mean(),
        "unique_descriptors": len(archive)
    })

    print(f"\nGENERATION {generation}/{GENERATIONS}")
    print(f"New evaluations={len(archive)-before}; total={len(archive)}")
    display(current[[
        "id", "features", "source_pixels", "val_NRMSE", "rank", "survives"
    ]].sort_values(["val_NRMSE", "features"]))

    plot_layouts(
        best_four,
        f"Generation {generation}: four best descriptors\n"
        "Blue=mean; orange=max; green=pixel; gray=drop"
    )
    predictions = np.stack([
        archive[t]["example_predictions"] for t in best_four.tree
    ])
    plot_spectra(
        Y_train[example_ids], predictions, coordinates[example_ids],
        f"Generation {generation}: top-four predictions at fixed validation pixels"
    )

    all_results = table()
    curve = front_curve(all_results)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)

    axes[0].scatter(
        all_results.features, all_results.val_NRMSE,
        s=25, alpha=.25, label="Archive"
    )
    axes[0].scatter(
        current.features, current.val_NRMSE,
        facecolors="none", edgecolors="tab:orange", s=50,
        label="Current generation"
    )
    axes[0].plot(curve.features, curve.val_NRMSE, "k.-", label="Pareto front")
    axes[0].set(xlabel="Output features", ylabel="Validation NRMSE")
    axes[0].legend()
    axes[0].grid(alpha=.2)

    vmax = max(max(np.nanmax(m) for m in error_maps), 1e-12)
    im = axes[1].imshow(winner_map, cmap="magma", vmin=0, vmax=vmax)
    axes[1].set_title(f"Best descriptor: pixelwise spectral RMSE, gen {generation}")
    fig.colorbar(im, ax=axes[1], label="Spectral RMSE")
    plt.show()

    if generation < GENERATIONS:
        population = breed(survivors)

ga_results = table()
ga_history = pd.DataFrame(history)
ga_generations = pd.concat(generation_tables, ignore_index=True)
error_map_history = np.stack(error_maps)       # generation × H × W
error_map_metadata = pd.DataFrame(error_records)


# ============================================================
# 9. ERROR MAP HISTORY AND PCA
# ============================================================
def analyze_error_history(stack, n_components=3):
    # Display maps using one final, common scale.
    cols = min(5, len(stack))
    rows = int(np.ceil(len(stack)/cols))
    fig, axes = plt.subplots(
        rows, cols, figsize=(3*cols, 3*rows),
        squeeze=False, constrained_layout=True
    )
    vmax = max(float(np.nanmax(stack)), 1e-12)
    for i, ax in enumerate(axes.ravel()):
        if i >= len(stack):
            ax.axis("off")
            continue
        im = ax.imshow(stack[i], cmap="magma", vmin=0, vmax=vmax)
        ax.set_title(f"Generation {i+1}")
        ax.axis("off")
    fig.colorbar(im, ax=list(axes.ravel()), label="Validation spectral RMSE")
    plt.show()

    valid = np.isfinite(stack).all(axis=0)
    if len(stack) < 2 or valid.sum() < 2:
        print("Insufficient generations or spatial locations for PCA.")
        return None

    # Observations=pixels; variables=generations.
    X = stack[:, valid].T
    spatial_mean = X.mean(axis=0)
    centered = X - spatial_mean
    if np.sum(centered**2) == 0:
        print("No spatial variation: PCA is undefined.")
        return None

    k = min(n_components, X.shape[0]-1, X.shape[1])
    pca = PCA(n_components=k, svd_solver="full")
    scores = pca.fit_transform(X)
    singular = pca.singular_values_
    tolerance = np.finfo(float).eps * max(X.shape) * singular[0]
    keep = singular > tolerance
    singular = singular[keep]

    # Error(g,p) = spatial_mean(g) + sum_k amplitudes(g,k)*loadings(p,k).
    loadings = scores[:, keep] / singular[None, :]
    amplitudes = pca.components_[keep].T * singular[None, :]
    explained = pca.explained_variance_ratio_[keep]

    for j in range(len(singular)):
        if loadings[np.argmax(np.abs(loadings[:, j])), j] < 0:
            loadings[:, j] *= -1
            amplitudes[:, j] *= -1

    loading_maps = np.full((len(singular), H, W), np.nan)
    reconstruction = np.full_like(stack, np.nan)
    for j in range(len(singular)):
        loading_maps[j, valid] = loadings[:, j]
    reconstruction[:, valid] = spatial_mean[:, None] + amplitudes @ loadings.T

    fig, ax = plt.subplots(figsize=(7, 3))
    ax.plot(np.arange(1, len(stack)+1), spatial_mean, "o-")
    ax.set(
        xlabel="Generation", ylabel="Mean pixel RMSE",
        title="Spatial mean error removed before PCA"
    )
    ax.grid(alpha=.2)
    plt.show()

    fig, axes = plt.subplots(
        len(singular), 2, figsize=(11, 3.5*len(singular)),
        squeeze=False, constrained_layout=True
    )
    for j in range(len(singular)):
        axes[j, 0].plot(np.arange(1, len(stack)+1), amplitudes[:, j], "o-")
        axes[j, 0].set(
            xlabel="Generation", ylabel="Component amplitude",
            title=f"Component {j+1}: {100*explained[j]:.1f}% variance"
        )
        axes[j, 0].grid(alpha=.2)
        limit = np.nanmax(np.abs(loading_maps[j]))
        im = axes[j, 1].imshow(
            loading_maps[j], cmap="RdBu_r", vmin=-limit, vmax=limit
        )
        axes[j, 1].set_title(f"Spatial loading {j+1}")
        fig.colorbar(im, ax=axes[j, 1], label="Unit-norm spatial loading")
    plt.show()

    return {
        "pca": pca,
        "valid_pixels": valid,
        "mean_error_by_generation": spatial_mean,
        "generation_amplitudes": amplitudes,
        "spatial_loadings": loading_maps,
        "explained_variance_ratio": explained,
        "reconstructed_error_maps": reconstruction
    }


error_pca = analyze_error_history(error_map_history, PCA_COMPONENTS)


# ============================================================
# 10. ALL PARETO DESCRIPTORS AND THE ESTIMATED KNEE
# ============================================================
pareto_descriptors = all_pareto(ga_results)  # Includes exact objective ties.
pareto_curve = front_curve(ga_results)
knee_index = None

if (
    len(pareto_curve) >= 3
    and np.ptp(pareto_curve.features) > 0
    and np.ptp(pareto_curve.val_NRMSE) > 0
):
    x = (
        (pareto_curve.features.to_numpy() - pareto_curve.features.min())
        / np.ptp(pareto_curve.features)
    )
    y = (
        (pareto_curve.val_NRMSE.to_numpy() - pareto_curve.val_NRMSE.min())
        / np.ptp(pareto_curve.val_NRMSE)
    )
    # Distance below the normalized endpoint chord y=1-x.
    score = (1-x-y) / np.sqrt(2)
    candidate = int(np.argmax(score[1:-1])) + 1
    if score[candidate] > 1e-6:
        knee_index = candidate

knee_ids = []
if knee_index is not None:
    knee = pareto_curve.iloc[knee_index]
    knee_ids = pareto_descriptors.loc[
        (pareto_descriptors.features == knee.features)
        & (pareto_descriptors.val_NRMSE == knee.val_NRMSE), "id"
    ].astype(int).tolist()

pareto_descriptors["estimated_knee"] = pareto_descriptors["id"].isin(knee_ids)

print("\nALL NONDOMINATED DESCRIPTORS")
display(pareto_descriptors[[
    "id", "features", "source_pixels", "val_NRMSE", "estimated_knee"
]])

steps = []
for i in range(len(pareto_curve)-1):
    smaller, larger = pareto_curve.iloc[i], pareto_curve.iloc[i+1]
    removed = int(larger.features - smaller.features)
    increase = float(smaller.val_NRMSE - larger.val_NRMSE)
    steps.append({
        "from_id": int(larger["id"]), "to_id": int(smaller["id"]),
        "from_features": int(larger.features),
        "to_features": int(smaller.features),
        "features_removed": removed,
        "NRMSE_increase": increase,
        "increase_per_removed_feature": increase / removed
    })
pareto_error_steps = pd.DataFrame(steps)
print("ERROR INCREASE WHEN MOVING TOWARD FEWER FEATURES")
display(pareto_error_steps)

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(ga_results.features, ga_results.val_NRMSE, s=25, alpha=.2)
ax.plot(pareto_curve.features, pareto_curve.val_NRMSE, "ko-", label="Pareto front")
for _, row in pareto_curve.iterrows():
    ax.annotate(
        f"ID {int(row['id'])}", (row.features, row.val_NRMSE),
        xytext=(4, 6), textcoords="offset points", fontsize=8
    )
if knee_index is not None:
    knee = pareto_curve.iloc[knee_index]
    simpler = pareto_curve.iloc[knee_index-1]
    ax.scatter(
        [knee.features], [knee.val_NRMSE],
        marker="*", s=250, color="red", zorder=5, label="Estimated knee"
    )
    ax.plot(
        [simpler.features, knee.features],
        [simpler.val_NRMSE, knee.val_NRMSE],
        color="red", linewidth=3, label="Next simplification step"
    )
    print("KNEE AND ADJACENT CONFIGURATIONS")
    display(pareto_curve.iloc[knee_index-1:knee_index+2][
        ["id", "features", "val_NRMSE"]
    ])
else:
    print("No distinct interior knee detected.")

ax.set(xlabel="Output features", ylabel="Validation NRMSE", title="Pareto knee")
ax.grid(alpha=.2)
ax.legend()
plt.show()

plot_layouts(
    pareto_descriptors,
    "All Pareto descriptors, ordered by increasing feature count\n"
    "Blue=mean; orange=max; green=pixel; gray=drop",
    knee_ids
)
pareto_trees = {
    int(row["id"]): row["tree"] for _, row in pareto_descriptors.iterrows()
}


# ============================================================
# 11. SEARCH HISTORY
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
axes[0].plot(ga_history.generation, ga_history.best_NRMSE, "o-", label="Best")
axes[0].plot(ga_history.generation, ga_history.median_NRMSE, "o-", label="Median")
axes[0].set(xlabel="Generation", ylabel="Validation NRMSE")
axes[0].legend()
axes[1].plot(ga_history.generation, ga_history.mean_features, "o-")
axes[1].set(xlabel="Generation", ylabel="Mean output features")
for ax in axes:
    ax.grid(alpha=.2)
plt.show()


# ============================================================
# 12. REFIT VALIDATION-SELECTED MODELS; EVALUATE TEST REGION
# ============================================================
# Selection is fixed before examining any test scores.
final_four = ga_results.sort_values(["val_NRMSE", "features", "id"]).head(4)
selected_tree = final_four.iloc[0].tree

P_test, Y_test, test_coordinates = extract_region(boundary, W)
final_models, predictions = [], []
for tree in final_four.tree:
    model = make_model().fit(feature_matrix(tree), Y_train)
    final_models.append(model)
    predictions.append(model.predict(feature_matrix(tree, P_test)))

test_predictions = np.stack(predictions)
ensemble_prediction = test_predictions.mean(axis=0)
final_model = final_models[0]
test_baseline = np.sqrt(np.mean(
    (Y_test - Y_train.mean(axis=0))**2
))

summary = []
for name, pred in [
    ("Best individual descriptor", test_predictions[0]),
    ("Mean of four descriptors", ensemble_prediction)
]:
    rmse = np.sqrt(np.mean((pred - Y_test)**2))
    summary.append({
        "predictor": name, "test_RMSE": rmse,
        "test_NRMSE": rmse/test_baseline if test_baseline > 0 else np.nan
    })
test_summary = pd.DataFrame(summary)
display(test_summary)

test_examples = spread_indices(test_coordinates, N_EXAMPLES)
plot_spectra(
    Y_test[test_examples], test_predictions[:, test_examples],
    test_coordinates[test_examples], "Reserved test pixels: final refitted models"
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for name, pred in [
    ("Best descriptor", test_predictions[0]),
    ("Mean of four", ensemble_prediction)
]:
    axes[0].plot(E, np.sqrt(np.mean((pred-Y_test)**2, axis=0)), label=name)
axes[0].set(xlabel="Energy", ylabel="Test RMSE", title="Error by energy channel")
axes[0].legend()

test_error_map = np.full((H, W), np.nan)
test_error_map[test_coordinates[:, 0], test_coordinates[:, 1]] = (
    np.sqrt(np.mean((test_predictions[0]-Y_test)**2, axis=1))
)
im = axes[1].imshow(test_error_map, cmap="magma", vmin=0)
axes[1].set_title("Best refitted descriptor: test spectral RMSE")
fig.colorbar(im, ax=axes[1], label="Spectral RMSE")
plt.show()

feature_definitions = pd.DataFrame([
    {
        "feature": i+1, "operation": op,
        "row_start": region[0], "column_start": region[1],
        "region_size": region[2]
    }
    for i, (region, op) in enumerate(active_leaves(selected_tree))
])
print("SELECTED DESCRIPTOR FEATURE DEFINITIONS")
display(feature_definitions)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.patches import Rectangle
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.decomposition import PCA
from IPython.display import display


# ============================================================
# 1. SETTINGS
# Requires: image, spectra, energy
# ============================================================
N = 16
POPULATION, GENERATIONS = 100, 100
SURVIVORS, TOURNAMENT_SIZE = 20, 5
CROSSOVER_RATE, IMMIGRANT_RATE = 0.8, 0.15
RIDGE_ALPHA = 10.0
SEED = 0

TEST_FRACTION = 0.25
VALIDATION_FRACTION = 0.25
MIN_FIT, MIN_VAL, MIN_TEST = 20, 15, 15

N_EXAMPLES = 3
PCA_COMPONENTS = 3
N_HIGH_COMPLEXITY = 12

rng = np.random.default_rng(SEED)

img = np.asarray(image, dtype=float)
cube = np.asarray(spectra, dtype=float)
E = np.asarray(energy, dtype=float).ravel()

if img.ndim != 2 or cube.shape != (*img.shape, len(E)):
    raise ValueError(
        "Expected image (H,W), spectra (H,W,n_energy), energy (n_energy,)."
    )
if not all(np.isfinite(v).all() for v in (img, cube, E)):
    raise ValueError("Inputs contain NaN or infinite values.")
if not 0 < TEST_FRACTION < 1:
    raise ValueError("Invalid test fraction.")
if not 0 < VALIDATION_FRACTION < 1 - TEST_FRACTION:
    raise ValueError("Invalid validation fraction.")
if not 2 <= SURVIVORS < POPULATION or GENERATIONS < 1:
    raise ValueError("Invalid population settings.")

order = np.argsort(E)
E, cube = E[order], cube[..., order]
if np.any(np.diff(E) <= 0):
    raise ValueError("Energy values must be distinct.")

H, W = img.shape
a, b = (N - 1) // 2, N - (N - 1) // 2


# ============================================================
# 2. ALL PATCHES AND FIXED RANDOM SPLITS
# ============================================================
all_coordinates = np.array([
    (y, x)
    for y in range(a, H - b + 1)
    for x in range(a, W - b + 1)
], dtype=int).reshape(-1, 2)

if len(all_coordinates) < MIN_FIT + MIN_VAL + MIN_TEST:
    raise ValueError("Too few complete 12×12 patches.")

P_all = np.stack([
    img[y-a:y+b, x-a:x+b] for y, x in all_coordinates
])
Y_all = cube[all_coordinates[:, 0], all_coordinates[:, 1]]

train_ids, test_ids = train_test_split(
    np.arange(len(all_coordinates)),
    test_size=TEST_FRACTION,
    random_state=SEED
)
fit_ids, val_ids = train_test_split(
    train_ids,
    test_size=VALIDATION_FRACTION / (1 - TEST_FRACTION),
    random_state=SEED + 1
)

# All indices refer to P_all / Y_all.
if (
    len(fit_ids) < MIN_FIT or len(val_ids) < MIN_VAL
    or len(test_ids) < MIN_TEST
):
    raise ValueError("Insufficient fitting, validation, or test samples.")

baseline = np.sqrt(np.mean(
    (Y_all[val_ids] - Y_all[fit_ids].mean(axis=0))**2
))
if baseline <= 0:
    raise ValueError("Validation baseline error is zero.")


def spread_indices(coords, count):
    """Spatially spread examples selected without inspecting spectra."""
    count = min(count, len(coords))
    chosen = [
        int(np.argmin(np.sum((coords - coords.mean(axis=0))**2, axis=1)))
    ]
    distance = np.full(len(coords), np.inf)
    while len(chosen) < count:
        distance = np.minimum(
            distance,
            np.sum((coords - coords[chosen[-1]])**2, axis=1)
        )
        distance[chosen] = -1
        chosen.append(int(np.argmax(distance)))
    return np.asarray(chosen)


example_local = spread_indices(all_coordinates[val_ids], N_EXAMPLES)
example_ids = val_ids[example_local]

print(f"All valid centers: {len(all_coordinates)}")
print(f"Fit={len(fit_ids)}, validation={len(val_ids)}, test={len(test_ids)}")
print(f"Patch={N}×{N}; target index=({a},{a}); channels={len(E)}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
for ax in axes:
    ax.imshow(img, cmap="gray", alpha=0.45)
    ax.set(xlabel="Image column", ylabel="Image row")

for ids, color, label in [
    (train_ids, "blue", "Training: fit + validation"),
    (test_ids, "red", "Test")
]:
    xy = all_coordinates[ids]
    axes[0].scatter(
        xy[:, 1], xy[:, 0], c=color, s=10, linewidths=0,
        label=f"{label} ({len(ids)})"
    )

for ids, color, label in [
    (fit_ids, "blue", "Fit"),
    (val_ids, "orange", "Validation")
]:
    xy = all_coordinates[ids]
    axes[1].scatter(
        xy[:, 1], xy[:, 0], c=color, s=10, linewidths=0,
        label=f"{label} ({len(ids)})"
    )

axes[0].set_title("Random training/test centers")
axes[1].set_title("Inner fitting/validation centers")
for ax in axes:
    ax.legend(fontsize=8)
plt.show()


# ============================================================
# 3. HIERARCHICAL DESCRIPTOR TREES: 12 → 6 → 3 → 1
# ============================================================
DROP, MEAN, MAX, PIXEL = ("DROP",), ("MEAN",), ("MAX",), ("PIXEL",)
ROOT = (0, 0, N)


def child_regions(region):
    y, x, size = region
    if size == 1:
        return []
    step = size // 2 if size % 2 == 0 else 1
    return [
        (y + dy, x + dx, step)
        for dy in range(0, size, step)
        for dx in range(0, size, step)
    ]


def simplify(tree):
    if tree[0] != "SPLIT":
        return tree
    children = tuple(simplify(t) for t in tree[1:])
    return DROP if all(t == DROP for t in children) else ("SPLIT",) + children


def random_tree(region=ROOT):
    if region[2] == 1:
        return PIXEL if rng.random() < 0.7 else DROP
    op = str(rng.choice(
        ["DROP", "MEAN", "MAX", "SPLIT"],
        p=[0.15, 0.25, 0.20, 0.40]
    ))
    if op != "SPLIT":
        return (op,)
    return simplify(
        ("SPLIT",) + tuple(random_tree(r) for r in child_regions(region))
    )


def nodes(tree, region=ROOT, path=()):
    yield path, region, tree
    if tree[0] == "SPLIT":
        for i, (child, box) in enumerate(zip(tree[1:], child_regions(region))):
            yield from nodes(child, box, path + (i,))


def replace(tree, path, subtree):
    if not path:
        return simplify(subtree)
    children = list(tree[1:])
    children[path[0]] = replace(children[path[0]], path[1:], subtree)
    return simplify(("SPLIT",) + tuple(children))


def active_leaves(tree):
    return [
        (region, node[0])
        for _, region, node in nodes(tree)
        if node[0] not in ("DROP", "SPLIT")
    ]


def full_pixels(region=ROOT):
    if region[2] == 1:
        return PIXEL
    return ("SPLIT",) + tuple(full_pixels(r) for r in child_regions(region))


# ============================================================
# 4. FEATURES, RIDGE, AND VALIDATION-ONLY FITNESS
# ============================================================
feature_cache, archive = {}, {}


def feature_matrix(tree):
    """Descriptor values for all valid patch centers."""
    columns = []
    for region, op in active_leaves(tree):
        key = (region, op)
        if key not in feature_cache:
            y, x, size = region
            values = P_all[:, y:y+size, x:x+size]
            feature_cache[key] = (
                values.mean(axis=(1, 2))
                if op in ("MEAN", "PIXEL")
                else values.max(axis=(1, 2))
            )
        columns.append(feature_cache[key])
    return np.column_stack(columns)


def make_model():
    # Scaling is fitted only on the subset passed to fit().
    # Spectra remain in their original intensity units.
    return make_pipeline(
        StandardScaler(),
        Ridge(alpha=RIDGE_ALPHA, solver="svd")
    )


def evaluate(tree, generation):
    if tree in archive:
        return

    X = feature_matrix(tree)
    model = make_model().fit(X[fit_ids], Y_all[fit_ids])
    prediction = model.predict(X[val_ids])

    pixel_rmse = np.sqrt(np.mean(
        (prediction - Y_all[val_ids])**2, axis=1
    ))
    score = np.sqrt(np.mean(pixel_rmse**2)) / baseline
    if not np.isfinite(score):
        raise RuntimeError("Nonfinite validation error.")

    leaves = active_leaves(tree)
    archive[tree] = {
        "id": len(archive) + 1,
        "features": X.shape[1],
        "source_pixels": sum(r[2]**2 for r, _ in leaves),
        "mean_regions": sum(op == "MEAN" for _, op in leaves),
        "max_regions": sum(op == "MAX" for _, op in leaves),
        "individual_pixels": sum(op == "PIXEL" for _, op in leaves),
        "val_NRMSE": score,
        "first_generation": generation,
        "example_predictions": prediction[example_local].copy()
    }


def table(trees=None):
    trees = list(archive) if trees is None else trees
    excluded = {"example_predictions", "all_pixel_rmse"}
    return pd.DataFrame([
        {
            "tree": tree,
            **{k: v for k, v in archive[tree].items() if k not in excluded}
        }
        for tree in trees
    ])


def error_map_for_tree(tree):
    """All-point RMSE for a model trained only on the inner fitting subset."""
    if "all_pixel_rmse" not in archive[tree]:
        X = feature_matrix(tree)
        model = make_model().fit(X[fit_ids], Y_all[fit_ids])
        prediction = model.predict(X)
        archive[tree]["all_pixel_rmse"] = np.sqrt(
            np.mean((prediction - Y_all)**2, axis=1)
        )

    result = np.full((H, W), np.nan)
    result[all_coordinates[:, 0], all_coordinates[:, 1]] = (
        archive[tree]["all_pixel_rmse"]
    )
    return result


# ============================================================
# 5. MUTATION AND SAME-REGION CROSSOVER
# ============================================================
def mutate(tree):
    items = list(nodes(tree))
    eligible = {
        "refine": [
            v for v in items
            if v[1][2] > 1 and v[2][0] in ("MEAN", "MAX")
        ],
        "coarsen": [v for v in items if v[2][0] == "SPLIT"],
        "pool": [v for v in items if v[2][0] in ("MEAN", "MAX")],
        "drop": [v for v in items if v[2][0] != "DROP"],
        "activate": [v for v in items if v[2][0] == "DROP"]
    }
    weights = dict(refine=.25, coarsen=.25, pool=.20, drop=.15, activate=.15)
    choices = [name for name in eligible if eligible[name]]
    probability = np.array([weights[name] for name in choices])
    action = str(rng.choice(choices, p=probability / probability.sum()))

    options = eligible[action]
    path, region, node = options[int(rng.integers(len(options)))]

    if action == "refine":
        new = ("SPLIT",) + tuple(
            PIXEL if r[2] == 1 else node
            for r in child_regions(region)
        )
    elif action == "coarsen":
        new = MEAN if rng.random() < .5 else MAX
    elif action == "pool":
        new = MAX if node == MEAN else MEAN
    elif action == "drop":
        new = DROP
    else:
        new = PIXEL if region[2] == 1 else (MEAN if rng.random() < .5 else MAX)

    child = replace(tree, path, new)
    return child if active_leaves(child) else tree


def crossover(parent1, parent2):
    first = {p: t for p, _, t in nodes(parent1)}
    second = {p: t for p, _, t in nodes(parent2)}
    common = sorted((set(first) & set(second)) - {()})
    if not common:
        return parent1
    path = common[int(rng.integers(len(common)))]
    child = replace(parent1, path, second[path])
    return child if active_leaves(child) else parent1


# ============================================================
# 6. PARETO RANKS, TOURNAMENTS, AND REPRODUCTION
# ============================================================
def rank_population(df):
    df = df.reset_index(drop=True).copy()
    values = df[["features", "val_NRMSE"]].to_numpy()
    ranks = np.zeros(len(df), dtype=int)
    crowding = np.zeros(len(df))
    remaining, rank = list(range(len(df))), 0

    while remaining:
        front = [
            i for i in remaining
            if not any(
                np.all(values[j] <= values[i])
                and np.any(values[j] < values[i])
                for j in remaining if j != i
            )
        ]
        ranks[front] = rank

        for column in range(2):
            ordered = sorted(front, key=lambda i: values[i, column])
            span = values[ordered[-1], column] - values[ordered[0], column]
            if span == 0:
                continue
            crowding[ordered[0]] = crowding[ordered[-1]] = np.inf
            for k in range(1, len(ordered) - 1):
                crowding[ordered[k]] += (
                    values[ordered[k+1], column]
                    - values[ordered[k-1], column]
                ) / span

        remaining = [i for i in remaining if i not in front]
        rank += 1

    df["rank"], df["crowding"] = ranks, crowding
    return df


def all_pareto(df):
    values = df[["features", "val_NRMSE"]].to_numpy()
    keep = [
        not np.any(
            np.all(values <= v, axis=1) & np.any(values < v, axis=1)
        )
        for v in values
    ]
    return df.loc[keep].sort_values(
        ["features", "val_NRMSE", "id"]
    ).reset_index(drop=True)


def front_curve(df):
    return all_pareto(df).drop_duplicates(
        ["features", "val_NRMSE"]
    ).reset_index(drop=True)


def select_survivors(population):
    ranked = rank_population(table(population))
    elite = int(ranked.sort_values(["val_NRMSE", "features"]).index[0])
    chosen = [elite]
    available = [i for i in ranked.index if i != elite]

    while len(chosen) < SURVIVORS:
        contestants = rng.choice(
            available,
            min(TOURNAMENT_SIZE, len(available)),
            replace=False
        )
        winner = int(min(
            contestants,
            key=lambda i: (ranked.loc[i, "rank"], -ranked.loc[i, "crowding"])
        ))
        chosen.append(winner)
        available.remove(winner)

    return ranked.loc[chosen, "tree"].tolist()


def breed(survivors):
    population, seen, attempts = list(survivors), set(survivors), 0

    while len(population) < POPULATION:
        attempts += 1
        if rng.random() < IMMIGRANT_RATE or attempts > 200:
            child = random_tree()
        else:
            i, j = rng.integers(len(survivors), size=2)
            child = survivors[i]
            if rng.random() < CROSSOVER_RATE:
                child = crossover(child, survivors[j])
            child = mutate(child)

        child = simplify(child)
        if active_leaves(child) and child not in seen:
            population.append(child)
            seen.add(child)

    return population


# ============================================================
# 7. PLOTTING HELPERS
# ============================================================
COLORS = dict(DROP="#eeeeee", MEAN="#56b4e9", MAX="#e69f00", PIXEL="#009e73")
SUMMARY_COLUMNS = [
    "id", "features", "source_pixels", "mean_regions",
    "max_regions", "individual_pixels", "val_NRMSE"
]


def draw_tree(ax, tree):
    for _, (y, x, size), node in nodes(tree):
        op = node[0]
        if op == "SPLIT":
            continue
        ax.add_patch(Rectangle(
            (x-.5, y-.5), size, size,
            facecolor=COLORS[op], edgecolor="white", linewidth=.7
        ))
        if size > 1:
            ax.text(
                x+(size-1)/2, y+(size-1)/2,
                {"DROP": "–", "MEAN": "mean", "MAX": "max"}[op],
                ha="center", va="center", fontsize=8
            )
    ax.plot(a, a, "rx", ms=9, mew=2)
    ax.set(xlim=(-.5, N-.5), ylim=(N-.5, -.5), aspect="equal")
    ax.axis("off")


def plot_layouts(rows, title, knee_ids=()):
    for start in range(0, len(rows), 12):
        batch = rows.iloc[start:start+12]
        cols = min(4, len(batch))
        nrows = int(np.ceil(len(batch)/cols))
        fig, axes = plt.subplots(
            nrows, cols, figsize=(3.5*cols, 3.5*nrows),
            squeeze=False, constrained_layout=True
        )

        for ax, (_, row) in zip(axes.ravel(), batch.iterrows()):
            draw_tree(ax, row["tree"])
            knee = int(row["id"]) in knee_ids
            ax.set_title(
                f"ID {int(row['id'])}" + (" — KNEE" if knee else "")
                + f"\n{int(row.features)} features; "
                f"{int(row.source_pixels)} source pixels"
                + f"\nValidation NRMSE={row.val_NRMSE:.3f}",
                fontsize=9, color="red" if knee else "black"
            )

        for ax in axes.ravel()[len(batch):]:
            ax.axis("off")

        fig.suptitle(
            title + "\nBlue=mean; orange=max; green=pixel; gray=drop"
        )
        plt.show()


def plot_spectra(actual, predictions, coords, title):
    mean = predictions.mean(axis=0)
    sd = predictions.std(axis=0)

    fig, axes = plt.subplots(
        1, len(coords), figsize=(5*len(coords), 3.5),
        squeeze=False, constrained_layout=True
    )
    for i, ax in enumerate(axes[0]):
        for p in predictions:
            ax.plot(E, p[i], color="tab:blue", alpha=.15)

        ax.plot(E, actual[i], "k-", lw=2, label="Measured")
        ax.plot(E, mean[i], color="tab:orange", lw=2, label="Mean prediction")
        ax.fill_between(
            E, mean[i]-sd[i], mean[i]+sd[i],
            color="tab:orange", alpha=.2, label="±1 model SD"
        )
        ax.set(
            xlabel="Energy", ylabel="Intensity",
            title=f"Location {tuple(coords[i])}"
        )
        ax.legend(fontsize=8)

    fig.suptitle(title)
    plt.show()


# ============================================================
# 8. EVOLUTION AND ALL-POINT ERROR MAPS
# ============================================================
population = [
    MEAN, MAX, full_pixels(),
    ("SPLIT", MEAN, MAX, DROP, MEAN)
]
while len(population) < POPULATION:
    candidate = random_tree()
    if active_leaves(candidate) and candidate not in population:
        population.append(candidate)

history, generation_tables = [], []
error_maps, error_records = [], []

for generation in range(1, GENERATIONS + 1):
    before = len(archive)
    for tree in population:
        evaluate(tree, generation)

    current = rank_population(table(population))
    survivors = select_survivors(population)
    survivor_set = set(survivors)

    current["survives"] = [t in survivor_set for t in current.tree]
    current["generation"] = generation
    generation_tables.append(current.copy())

    best_four = current.sort_values(["val_NRMSE", "features"]).head(4)
    winner = best_four.iloc[0]

    # All-point map is diagnostic only: not used for selection.
    winner_map = error_map_for_tree(winner.tree)
    all_errors = archive[winner.tree]["all_pixel_rmse"]
    error_maps.append(winner_map.copy())

    error_records.append({
        "generation": generation,
        "descriptor_id": int(winner["id"]),
        "features": int(winner.features),
        "val_NRMSE": winner.val_NRMSE,
        "all_point_RMSE": np.sqrt(np.mean(all_errors**2)),
        "tree": winner.tree
    })
    history.append({
        "generation": generation,
        "best_NRMSE": current.val_NRMSE.min(),
        "median_NRMSE": current.val_NRMSE.median(),
        "mean_features": current.features.mean(),
        "unique_descriptors": len(archive)
    })

    print(f"\nGENERATION {generation}/{GENERATIONS}")
    print(f"New evaluations={len(archive)-before}; total={len(archive)}")
    display(current[
        SUMMARY_COLUMNS + ["rank", "survives"]
    ].sort_values(["val_NRMSE", "features"]))

    plot_layouts(best_four, f"Generation {generation}: four best descriptors")
    predictions = np.stack([
        archive[t]["example_predictions"] for t in best_four.tree
    ])
    plot_spectra(
        Y_all[example_ids], predictions, all_coordinates[example_ids],
        f"Generation {generation}: fixed validation locations"
    )

    all_results = table()
    curve = front_curve(all_results)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    axes[0].scatter(
        all_results.features, all_results.val_NRMSE,
        s=25, alpha=.25, label="Archive"
    )
    axes[0].scatter(
        current.features, current.val_NRMSE,
        facecolors="none", edgecolors="tab:orange",
        s=50, label="Current generation"
    )
    axes[0].plot(curve.features, curve.val_NRMSE, "k.-", label="Pareto front")
    axes[0].set(xlabel="Output features", ylabel="Validation NRMSE")
    axes[0].legend()
    axes[0].grid(alpha=.2)

    vmax = max(max(np.nanmax(m) for m in error_maps), 1e-12)
    im = axes[1].imshow(
        winner_map, cmap="magma", vmin=0, vmax=vmax,
        interpolation="nearest"
    )
    axes[1].set_title(f"Best descriptor: ALL-point RMSE, generation {generation}")
    fig.colorbar(im, ax=axes[1], label="Spectral RMSE")
    plt.show()

    if generation < GENERATIONS:
        population = breed(survivors)

ga_results = table()
ga_history = pd.DataFrame(history)
ga_generations = pd.concat(generation_tables, ignore_index=True)
error_map_history = np.stack(error_maps)  # generation × H × W
error_map_metadata = pd.DataFrame(error_records)


# ============================================================
# 9. ALL-POINT ERROR HISTORY AND PCA
# ============================================================
def analyze_error_history(stack, n_components=3):
    cols = min(5, len(stack))
    rows = int(np.ceil(len(stack)/cols))

    fig, axes = plt.subplots(
        rows, cols, figsize=(3*cols, 3*rows),
        squeeze=False, constrained_layout=True
    )
    vmax = max(float(np.nanmax(stack)), 1e-12)
    for i, ax in enumerate(axes.ravel()):
        if i >= len(stack):
            ax.axis("off")
            continue
        im = ax.imshow(
            stack[i], cmap="magma", vmin=0, vmax=vmax,
            interpolation="nearest"
        )
        ax.set_title(f"Generation {i+1}")
        ax.axis("off")
    fig.colorbar(im, ax=list(axes.ravel()), label="All-point spectral RMSE")
    plt.show()

    valid = np.isfinite(stack).all(axis=0)
    if len(stack) < 2 or valid.sum() < 2:
        print("Insufficient data for PCA.")
        return None

    # Rows=pixels, columns=generations.
    X = stack[:, valid].T
    spatial_mean = X.mean(axis=0)

    if np.sum((X-spatial_mean)**2) == 0:
        print("No spatial variation: PCA is undefined.")
        return None

    k = min(n_components, X.shape[0]-1, X.shape[1])
    pca = PCA(n_components=k, svd_solver="full")
    scores = pca.fit_transform(X)
    singular = pca.singular_values_
    tolerance = np.finfo(float).eps * max(X.shape) * singular[0]
    keep = singular > tolerance
    singular = singular[keep]

    if len(singular) == 0:
        print("No numerically resolved PCA components.")
        return None

    # error(g,p) = mean(g) + sum_k amplitude(g,k)*loading(p,k)
    loadings = scores[:, keep] / singular[None, :]
    amplitudes = pca.components_[keep].T * singular[None, :]
    explained = pca.explained_variance_ratio_[keep]

    # Reproducible display sign.
    for j in range(len(singular)):
        if loadings[np.argmax(np.abs(loadings[:, j])), j] < 0:
            loadings[:, j] *= -1
            amplitudes[:, j] *= -1

    loading_maps = np.full((len(singular), H, W), np.nan)
    for j in range(len(singular)):
        loading_maps[j, valid] = loadings[:, j]

    reconstruction = np.full_like(stack, np.nan)
    reconstruction[:, valid] = (
        spatial_mean[:, None] + amplitudes @ loadings.T
    )

    fig, ax = plt.subplots(figsize=(7, 3))
    ax.plot(np.arange(1, len(stack)+1), spatial_mean, "o-")
    ax.set(
        xlabel="Generation", ylabel="Mean pixel RMSE",
        title="Spatial mean error removed before PCA"
    )
    ax.grid(alpha=.2)
    plt.show()

    fig, axes = plt.subplots(
        len(singular), 2, figsize=(11, 3.5*len(singular)),
        squeeze=False, constrained_layout=True
    )
    for j in range(len(singular)):
        axes[j, 0].plot(
            np.arange(1, len(stack)+1), amplitudes[:, j], "o-"
        )
        axes[j, 0].set(
            xlabel="Generation", ylabel="Component amplitude",
            title=f"Component {j+1}: {100*explained[j]:.1f}% variance"
        )
        axes[j, 0].grid(alpha=.2)

        limit = np.nanmax(np.abs(loading_maps[j]))
        im = axes[j, 1].imshow(
            loading_maps[j], cmap="RdBu_r",
            vmin=-limit, vmax=limit, interpolation="nearest"
        )
        axes[j, 1].set_title(f"Spatial loading {j+1}")
        fig.colorbar(im, ax=axes[j, 1], label="Unit-norm loading")
    plt.show()

    return {
        "pca": pca,
        "valid_pixels": valid,
        "mean_error_by_generation": spatial_mean,
        "generation_amplitudes": amplitudes,
        "spatial_loadings": loading_maps,
        "explained_variance_ratio": explained,
        "reconstructed_error_maps": reconstruction
    }


error_pca = analyze_error_history(error_map_history, PCA_COMPONENTS)


# ============================================================
# 10. ALL PARETO DESCRIPTORS AND KNEE
# ============================================================
pareto_descriptors = all_pareto(ga_results)
pareto_curve = front_curve(ga_results)
knee_index = None

if (
    len(pareto_curve) >= 3
    and np.ptp(pareto_curve.features) > 0
    and np.ptp(pareto_curve.val_NRMSE) > 0
):
    x = (
        (pareto_curve.features.to_numpy() - pareto_curve.features.min())
        / np.ptp(pareto_curve.features)
    )
    y = (
        (pareto_curve.val_NRMSE.to_numpy() - pareto_curve.val_NRMSE.min())
        / np.ptp(pareto_curve.val_NRMSE)
    )
    score = (1-x-y) / np.sqrt(2)
    candidate = int(np.argmax(score[1:-1])) + 1
    if score[candidate] > 1e-6:
        knee_index = candidate

knee_ids = []
if knee_index is not None:
    knee = pareto_curve.iloc[knee_index]
    knee_ids = pareto_descriptors.loc[
        (pareto_descriptors.features == knee.features)
        & (pareto_descriptors.val_NRMSE == knee.val_NRMSE),
        "id"
    ].astype(int).tolist()

pareto_descriptors["estimated_knee"] = pareto_descriptors["id"].isin(knee_ids)

print("\nALL NONDOMINATED DESCRIPTORS")
display(pareto_descriptors[SUMMARY_COLUMNS + ["estimated_knee"]])

steps = []
for i in range(len(pareto_curve)-1):
    smaller = pareto_curve.iloc[i]
    larger = pareto_curve.iloc[i+1]
    removed = int(larger.features - smaller.features)
    increase = float(smaller.val_NRMSE - larger.val_NRMSE)

    steps.append({
        "from_id": int(larger["id"]),
        "to_id": int(smaller["id"]),
        "from_features": int(larger.features),
        "to_features": int(smaller.features),
        "features_removed": removed,
        "NRMSE_increase": increase,
        "increase_per_removed_feature": increase / removed
    })

pareto_error_steps = pd.DataFrame(steps)
print("ERROR INCREASE WHEN FEATURES ARE REMOVED")
display(pareto_error_steps)

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(ga_results.features, ga_results.val_NRMSE, s=25, alpha=.2)
ax.plot(
    pareto_curve.features, pareto_curve.val_NRMSE,
    "ko-", label="Pareto front"
)
for _, row in pareto_curve.iterrows():
    ax.annotate(
        f"ID {int(row['id'])}",
        (row.features, row.val_NRMSE),
        xytext=(4, 6), textcoords="offset points", fontsize=8
    )

if knee_index is not None:
    knee = pareto_curve.iloc[knee_index]
    simpler = pareto_curve.iloc[knee_index-1]
    ax.scatter(
        [knee.features], [knee.val_NRMSE],
        marker="*", s=250, color="red", zorder=5,
        label="Estimated knee"
    )
    ax.plot(
        [simpler.features, knee.features],
        [simpler.val_NRMSE, knee.val_NRMSE],
        color="red", linewidth=3,
        label="Next simplification step"
    )
    print("KNEE AND ADJACENT CONFIGURATIONS")
    display(pareto_curve.iloc[knee_index-1:knee_index+2][SUMMARY_COLUMNS])
else:
    print("No distinct interior knee detected.")

ax.set(xlabel="Output features", ylabel="Validation NRMSE", title="Pareto knee")
ax.grid(alpha=.2)
ax.legend()
plt.show()

plot_layouts(
    pareto_descriptors,
    "All Pareto descriptors: increasing feature count",
    knee_ids
)
pareto_trees = {
    int(row["id"]): row["tree"]
    for _, row in pareto_descriptors.iterrows()
}


# ============================================================
# 11. HIGH-COMPLEXITY DESCRIPTORS
# ============================================================
high_complexity_descriptors = ga_results.sort_values(
    ["features", "val_NRMSE", "id"],
    ascending=[False, True, True]
).head(N_HIGH_COMPLEXITY)

print("\nHIGHEST-COMPLEXITY EVALUATED DESCRIPTORS")
display(high_complexity_descriptors[SUMMARY_COLUMNS])
plot_layouts(
    high_complexity_descriptors,
    "Highest-complexity evaluated descriptors"
)

high_complexity_pareto = pareto_descriptors.sort_values(
    ["features", "val_NRMSE"], ascending=[False, True]
).head(8)

print("\nHIGH-COMPLEXITY END OF THE PARETO FRONT")
display(high_complexity_pareto[SUMMARY_COLUMNS])
plot_layouts(
    high_complexity_pareto,
    "High-complexity Pareto descriptors",
    knee_ids
)

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(
    ga_results.features, ga_results.val_NRMSE,
    s=25, alpha=.2, label="All evaluated"
)
ax.plot(
    pareto_curve.features, pareto_curve.val_NRMSE,
    "k.-", label="Pareto front"
)
ax.scatter(
    high_complexity_descriptors.features,
    high_complexity_descriptors.val_NRMSE,
    s=75, facecolors="none", edgecolors="red",
    label="Displayed high-complexity descriptors"
)
for _, row in high_complexity_descriptors.iterrows():
    ax.annotate(
        str(int(row["id"])),
        (row.features, row.val_NRMSE),
        xytext=(4, 4), textcoords="offset points", fontsize=8
    )
ax.set(xlabel="Output features", ylabel="Validation NRMSE")
ax.grid(alpha=.2)
ax.legend()
plt.show()


# ============================================================
# 12. GENERATION HISTORY
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)

axes[0].plot(
    ga_history.generation, ga_history.best_NRMSE, "o-", label="Best"
)
axes[0].plot(
    ga_history.generation, ga_history.median_NRMSE, "o-", label="Median"
)
axes[0].set(xlabel="Generation", ylabel="Validation NRMSE")
axes[0].legend()

axes[1].plot(
    ga_history.generation, ga_history.mean_features, "o-"
)
axes[1].set(xlabel="Generation", ylabel="Mean output features")
for ax in axes:
    ax.grid(alpha=.2)
plt.show()


# ============================================================
# 13. FINAL REFIT AND ALL-POINT EVALUATION
# ============================================================
# Final descriptor selection remains validation-only.
final_four = ga_results.sort_values(
    ["val_NRMSE", "features", "id"]
).head(4)
selected_tree = final_four.iloc[0].tree

test_example_local = spread_indices(all_coordinates[test_ids], N_EXAMPLES)
test_example_ids = test_ids[test_example_local]

final_models = []
final_example_predictions = []
ensemble_prediction = np.zeros_like(Y_all)
best_all_prediction = None

# Refit on fit + validation (75%) and predict all valid locations.
for j, tree in enumerate(final_four.tree):
    X = feature_matrix(tree)
    model = make_model().fit(X[train_ids], Y_all[train_ids])
    prediction = model.predict(X)

    final_models.append(model)
    final_example_predictions.append(prediction[test_example_ids].copy())
    ensemble_prediction += prediction / len(final_four)

    if j == 0:
        best_all_prediction = prediction.copy()

final_model = final_models[0]

subsets = {
    "Fit": fit_ids,
    "Validation": val_ids,
    "Test": test_ids,
    "All valid centers": np.arange(len(all_coordinates))
}

test_baseline = np.sqrt(np.mean(
    (Y_all[test_ids] - Y_all[train_ids].mean(axis=0))**2
))

summary = []
for name, prediction in [
    ("Best descriptor", best_all_prediction),
    ("Mean of four descriptors", ensemble_prediction)
]:
    for subset_name, ids in subsets.items():
        rmse = np.sqrt(np.mean((prediction[ids] - Y_all[ids])**2))
        summary.append({
            "predictor": name,
            "subset": subset_name,
            "locations": len(ids),
            "spectral_RMSE": rmse,
            "test_NRMSE": (
                rmse / test_baseline
                if subset_name == "Test" and test_baseline > 0
                else np.nan
            )
        })

rmse_summary = pd.DataFrame(summary)
print("\nFINAL REFITTED MODELS: ERROR BY SUBSET")
display(rmse_summary)

plot_spectra(
    Y_all[test_example_ids],
    np.stack(final_example_predictions),
    all_coordinates[test_example_ids],
    "Final refitted models: randomly withheld test locations"
)

all_pixel_rmse = np.sqrt(np.mean(
    (best_all_prediction - Y_all)**2, axis=1
))
ensemble_pixel_rmse = np.sqrt(np.mean(
    (ensemble_prediction - Y_all)**2, axis=1
))

final_error_map = np.full((H, W), np.nan)
ensemble_error_map = np.full((H, W), np.nan)
final_error_map[
    all_coordinates[:, 0], all_coordinates[:, 1]
] = all_pixel_rmse
ensemble_error_map[
    all_coordinates[:, 0], all_coordinates[:, 1]
] = ensemble_pixel_rmse

fig, axes = plt.subplots(1, 3, figsize=(16, 4), constrained_layout=True)
for name, prediction in [
    ("Best descriptor", best_all_prediction),
    ("Mean of four", ensemble_prediction)
]:
    axes[0].plot(
        E,
        np.sqrt(np.mean(
            (prediction[test_ids] - Y_all[test_ids])**2, axis=0
        )),
        label=name
    )
axes[0].set(
    xlabel="Energy", ylabel="Test RMSE",
    title="Test error by energy channel"
)
axes[0].legend()

vmax = max(
    float(np.nanmax(final_error_map)),
    float(np.nanmax(ensemble_error_map)),
    1e-12
)
for ax, error_map, title in [
    (axes[1], final_error_map, "Best refitted descriptor: ALL points"),
    (axes[2], ensemble_error_map, "Mean of four: ALL points")
]:
    im = ax.imshow(
        error_map, cmap="magma",
        vmin=0, vmax=vmax, interpolation="nearest"
    )
    ax.set_title(title)
fig.colorbar(im, ax=list(axes[1:]), label="Spectral RMSE")
plt.show()

feature_definitions = pd.DataFrame([
    {
        "feature": i+1,
        "operation": op,
        "row_start": region[0],
        "column_start": region[1],
        "region_size": region[2]
    }
    for i, (region, op) in enumerate(active_leaves(selected_tree))
])
print("SELECTED DESCRIPTOR FEATURE DEFINITIONS")
display(feature_definitions)

# After hackathon

Done after hackathion. Let's add 4 fold rotational augmentation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.patches import Rectangle
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.decomposition import PCA
from IPython.display import display


# ============================================================
# 1. SETUP — START DIRECTLY AFTER ORIGINAL DATA LOADING
#
# The original opening must have created:
#     loadedfile = np.load(..., allow_pickle=True).tolist()
#
# No scalarizer, patches, regressors, or earlier GA cells required.
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.patches import Rectangle
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.decomposition import PCA
from IPython.display import display


# ------------------------------------------------------------
# Dataset selection
# ------------------------------------------------------------
DATASET_KEY = "2"

if "loadedfile" not in globals():
    raise RuntimeError(
        "Run the original data-loading cell first so that "
        "'loadedfile' contains the dataset dictionary."
    )

if DATASET_KEY not in loadedfile:
    raise KeyError(
        f"Dataset {DATASET_KEY!r} not found. "
        f"Available keys: {list(loadedfile.keys())}"
    )

data = loadedfile[DATASET_KEY]

# Explicitly extract inputs rather than reuse potentially stale variables.
image = np.asarray(data["image"], dtype=float)
spectra = np.asarray(data["spectrum image"], dtype=float)
energy = np.asarray(data["energy axis"], dtype=float).ravel()


# ------------------------------------------------------------
# GA and regression settings
# ------------------------------------------------------------
N = 12
POPULATION = 25
GENERATIONS = 10
SURVIVORS = 5
TOURNAMENT_SIZE = 3

CROSSOVER_RATE = 0.8
IMMIGRANT_RATE = 0.15
RIDGE_ALPHA = 10.0
SEED = 0

TEST_FRACTION = 0.25
VALIDATION_FRACTION = 0.25
MIN_FIT, MIN_VAL, MIN_TEST = 20, 15, 15

# Original patches plus this many distinct random quarter-turns.
# 0: no augmentation
# 1: one of 90°, 180°, 270° per training patch
# 3: all three rotations
AUGMENTATIONS_PER_PATCH = 1

N_EXAMPLES = 3
PCA_COMPONENTS = 3
N_HIGH_COMPLEXITY = 12


# ------------------------------------------------------------
# Validation and internal variables expected by Sections 2–12
# ------------------------------------------------------------
rng = np.random.default_rng(SEED)
aug_rng = np.random.default_rng(SEED + 1000)

img = image.copy()
cube = spectra.copy()
E = energy.copy()

if img.ndim != 2:
    raise ValueError(f"Expected a 2D image; received {img.shape}.")

if cube.shape != (*img.shape, len(E)):
    raise ValueError(
        f"Inconsistent shapes: image={img.shape}, "
        f"spectra={cube.shape}, energy={E.shape}. "
        "Expected spectra with shape (H, W, n_energy)."
    )

if not all(np.isfinite(v).all() for v in (img, cube, E)):
    raise ValueError("Inputs contain NaN or infinite values.")

if AUGMENTATIONS_PER_PATCH not in (0, 1, 2, 3):
    raise ValueError("AUGMENTATIONS_PER_PATCH must be 0, 1, 2, or 3.")

if not (
    0 < TEST_FRACTION < 1
    and 0 < VALIDATION_FRACTION < 1 - TEST_FRACTION
):
    raise ValueError("Invalid training/validation/test fractions.")

if not 2 <= SURVIVORS < POPULATION or GENERATIONS < 1:
    raise ValueError("Invalid GA settings.")

order = np.argsort(E)
E = E[order]
cube = cube[..., order]

if len(E) < 2 or np.any(np.diff(E) <= 0):
    raise ValueError("Energy axis must contain distinct values.")

H, W = img.shape
a = (N - 1) // 2
b = N - a

print(f"Dataset: {DATASET_KEY}")
print(f"Image: {img.shape}")
print(f"Spectra: {cube.shape}")
print(f"Energy range: {E[0]:g}–{E[-1]:g}")
print(f"Target: complete spectrum ({len(E)} channels)")
print(f"Patch: {N}×{N}; rotated copies per training patch: "
      f"{AUGMENTATIONS_PER_PATCH}")

# Quick inspection before starting the GA.
fig, axes = plt.subplots(
    1, 2, figsize=(11, 4), constrained_layout=True
)
axes[0].imshow(img, cmap="gray")
axes[0].set_title(f"Dataset {DATASET_KEY}: structural image")

axes[1].plot(E, cube.mean(axis=(0, 1)), color="black")
axes[1].set(
    xlabel="Energy",
    ylabel="Intensity",
    title="Mean spectrum — full prediction target"
)
plt.show()



# ============================================================
# 2. ORIGINAL PATCHES AND FIXED RANDOM SPLITS
# ============================================================
all_coordinates = np.array([
    (y, x)
    for y in range(a, H - b + 1)
    for x in range(a, W - b + 1)
], dtype=int).reshape(-1, 2)

if len(all_coordinates) < MIN_FIT + MIN_VAL + MIN_TEST:
    raise ValueError("Too few complete patches.")

P_all = np.stack([
    img[y-a:y+b, x-a:x+b] for y, x in all_coordinates
])
Y_all = cube[all_coordinates[:, 0], all_coordinates[:, 1]]

# Split ORIGINAL locations before making rotated copies.
train_ids, test_ids = train_test_split(
    np.arange(len(P_all)),
    test_size=TEST_FRACTION,
    random_state=SEED
)
fit_ids, val_ids = train_test_split(
    train_ids,
    test_size=VALIDATION_FRACTION / (1 - TEST_FRACTION),
    random_state=SEED + 1
)

if (
    len(fit_ids) < MIN_FIT or len(val_ids) < MIN_VAL
    or len(test_ids) < MIN_TEST
):
    raise ValueError("Insufficient samples in one or more subsets.")

baseline = np.sqrt(np.mean(
    (Y_all[val_ids] - Y_all[fit_ids].mean(axis=0))**2
))
if baseline <= 0:
    raise ValueError("Validation baseline error is zero.")


def spread_indices(coords, count):
    count = min(count, len(coords))
    chosen = [
        int(np.argmin(np.sum((coords - coords.mean(axis=0))**2, axis=1)))
    ]
    distance = np.full(len(coords), np.inf)
    while len(chosen) < count:
        distance = np.minimum(
            distance, np.sum((coords - coords[chosen[-1]])**2, axis=1)
        )
        distance[chosen] = -1
        chosen.append(int(np.argmax(distance)))
    return np.asarray(chosen)


example_local = spread_indices(all_coordinates[val_ids], N_EXAMPLES)
example_ids = val_ids[example_local]

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
for ax in axes:
    ax.imshow(img, cmap="gray", alpha=.45)
    ax.set(xlabel="Image column", ylabel="Image row")

for ids, color, label in [
    (train_ids, "blue", "Training: fit + validation"),
    (test_ids, "red", "Test")
]:
    xy = all_coordinates[ids]
    axes[0].scatter(
        xy[:, 1], xy[:, 0], c=color, s=10, linewidths=0,
        label=f"{label} ({len(ids)})"
    )
for ids, color, label in [
    (fit_ids, "blue", "Fit"), (val_ids, "orange", "Validation")
]:
    xy = all_coordinates[ids]
    axes[1].scatter(
        xy[:, 1], xy[:, 0], c=color, s=10, linewidths=0,
        label=f"{label} ({len(ids)})"
    )
axes[0].set_title("Random training/test centers")
axes[1].set_title("Inner fitting/validation centers")
for ax in axes:
    ax.legend(fontsize=8)
plt.show()


# ============================================================
# 3. FIXED ROTATION AUGMENTATION
# ============================================================
# Create copies for the final training set.
# During evolution, only copies whose source is in fit_ids are used.
# Validation copies are used only for final refitting.
aug_patches, aug_sources, aug_turns = [], [], []

for source_id in train_ids:
    turns = aug_rng.choice(
        [1, 2, 3], size=AUGMENTATIONS_PER_PATCH, replace=False
    )
    for k in turns:
        aug_patches.append(np.rot90(P_all[source_id], int(k)).copy())
        aug_sources.append(source_id)
        aug_turns.append(int(k))

P_aug = (
    np.stack(aug_patches)
    if aug_patches else np.empty((0, N, N), dtype=float)
)
aug_sources = np.asarray(aug_sources, dtype=int)
aug_turns = np.asarray(aug_turns, dtype=int)

augmentation_metadata = pd.DataFrame({
    "source_id": aug_sources,
    "rotation_degrees": 90 * aug_turns,
    "used_during_evolution": np.isin(aug_sources, fit_ids)
})

print(f"Original centers: {len(P_all)}")
print(f"Fit={len(fit_ids)}, validation={len(val_ids)}, test={len(test_ids)}")
print(
    f"Evolution fitting rows: "
    f"{len(fit_ids) * (1 + AUGMENTATIONS_PER_PATCH)}"
)
print(
    f"Final refitting rows: "
    f"{len(train_ids) * (1 + AUGMENTATIONS_PER_PATCH)}"
)
print("Validation/test scoring always uses original, unrotated patches.")

if len(P_aug):
    show = spread_indices(all_coordinates[fit_ids], N_EXAMPLES)
    fig, axes = plt.subplots(
        len(show), 2, figsize=(6, 3 * len(show)),
        squeeze=False, constrained_layout=True
    )
    for row, local_id in enumerate(show):
        source_id = fit_ids[local_id]
        j = np.flatnonzero(aug_sources == source_id)[0]
        vmin, vmax = P_all[source_id].min(), P_all[source_id].max()
        axes[row, 0].imshow(P_all[source_id], cmap="gray", vmin=vmin, vmax=vmax)
        axes[row, 1].imshow(P_aug[j], cmap="gray", vmin=vmin, vmax=vmax)
        axes[row, 0].set_title(f"Original: {tuple(all_coordinates[source_id])}")
        axes[row, 1].set_title(f"Rotated {90 * aug_turns[j]}°; same spectrum")
        for ax in axes[row]:
            ax.axis("off")
    plt.show()


# ============================================================
# 4. DESCRIPTOR TREES: 12 → 6 → 3 → 1
# ============================================================
DROP, MEAN, MAX, PIXEL = ("DROP",), ("MEAN",), ("MAX",), ("PIXEL",)
ROOT = (0, 0, N)


def child_regions(region):
    y, x, size = region
    if size == 1:
        return []
    step = size // 2 if size % 2 == 0 else 1
    return [
        (y + dy, x + dx, step)
        for dy in range(0, size, step)
        for dx in range(0, size, step)
    ]


def simplify(tree):
    if tree[0] != "SPLIT":
        return tree
    children = tuple(simplify(t) for t in tree[1:])
    return DROP if all(t == DROP for t in children) else ("SPLIT",) + children


def random_tree(region=ROOT):
    if region[2] == 1:
        return PIXEL if rng.random() < .7 else DROP
    op = str(rng.choice(
        ["DROP", "MEAN", "MAX", "SPLIT"], p=[.15, .25, .20, .40]
    ))
    if op != "SPLIT":
        return (op,)
    return simplify(
        ("SPLIT",) + tuple(random_tree(r) for r in child_regions(region))
    )


def nodes(tree, region=ROOT, path=()):
    yield path, region, tree
    if tree[0] == "SPLIT":
        for i, (child, box) in enumerate(zip(tree[1:], child_regions(region))):
            yield from nodes(child, box, path + (i,))


def replace(tree, path, subtree):
    if not path:
        return simplify(subtree)
    children = list(tree[1:])
    children[path[0]] = replace(children[path[0]], path[1:], subtree)
    return simplify(("SPLIT",) + tuple(children))


def active_leaves(tree):
    return [
        (region, node[0]) for _, region, node in nodes(tree)
        if node[0] not in ("DROP", "SPLIT")
    ]


def full_pixels(region=ROOT):
    if region[2] == 1:
        return PIXEL
    return ("SPLIT",) + tuple(full_pixels(r) for r in child_regions(region))


# ============================================================
# 5. FEATURES AND AUGMENTED RIDGE FITTING
# ============================================================
feature_cache, archive = {}, {}


def feature_matrix(tree, augmented=False):
    patches = P_aug if augmented else P_all
    columns = []
    for region, op in active_leaves(tree):
        key = (augmented, region, op)
        if key not in feature_cache:
            y, x, size = region
            values = patches[:, y:y+size, x:x+size]
            feature_cache[key] = (
                values.mean(axis=(1, 2))
                if op in ("MEAN", "PIXEL")
                else values.max(axis=(1, 2))
            )
        columns.append(feature_cache[key])
    return np.column_stack(columns)


def make_model():
    return make_pipeline(
        StandardScaler(),
        Ridge(alpha=RIDGE_ALPHA, solver="svd")
    )


def fit_augmented_model(tree, original_ids):
    X = feature_matrix(tree)
    X_fit = X[original_ids]
    Y_fit = Y_all[original_ids]

    if len(P_aug):
        selected = np.isin(aug_sources, original_ids)
        X_rot = feature_matrix(tree, augmented=True)[selected]
        Y_rot = Y_all[aug_sources[selected]]
        X_fit = np.concatenate([X_fit, X_rot], axis=0)
        Y_fit = np.concatenate([Y_fit, Y_rot], axis=0)

    # Keep regularization relative to sample count comparable when
    # adding augmented rows; Ridge uses a sum-of-squares objective.
    alpha = RIDGE_ALPHA * len(X_fit) / len(original_ids)
    model = make_model()
    model.set_params(ridge__alpha=alpha)
    return model.fit(X_fit, Y_fit)


def evaluate(tree, generation):
    if tree in archive:
        return

    model = fit_augmented_model(tree, fit_ids)
    prediction = model.predict(feature_matrix(tree)[val_ids])
    error = np.sqrt(np.mean((prediction - Y_all[val_ids])**2)) / baseline
    if not np.isfinite(error):
        raise RuntimeError("Nonfinite validation error.")

    leaves = active_leaves(tree)
    archive[tree] = {
        "id": len(archive) + 1,
        "features": len(leaves),
        "source_pixels": sum(r[2]**2 for r, _ in leaves),
        "mean_regions": sum(op == "MEAN" for _, op in leaves),
        "max_regions": sum(op == "MAX" for _, op in leaves),
        "individual_pixels": sum(op == "PIXEL" for _, op in leaves),
        "val_NRMSE": error,
        "first_generation": generation,
        "example_predictions": prediction[example_local].copy()
    }


def table(trees=None):
    trees = list(archive) if trees is None else trees
    excluded = {"example_predictions", "all_pixel_rmse"}
    return pd.DataFrame([
        {"tree": t, **{k: v for k, v in archive[t].items() if k not in excluded}}
        for t in trees
    ])


def error_map_for_tree(tree):
    if "all_pixel_rmse" not in archive[tree]:
        model = fit_augmented_model(tree, fit_ids)
        prediction = model.predict(feature_matrix(tree))
        archive[tree]["all_pixel_rmse"] = np.sqrt(
            np.mean((prediction - Y_all)**2, axis=1)
        )
    result = np.full((H, W), np.nan)
    result[all_coordinates[:, 0], all_coordinates[:, 1]] = (
        archive[tree]["all_pixel_rmse"]
    )
    return result


# ============================================================
# 6. MUTATION, CROSSOVER, TOURNAMENTS
# ============================================================
def mutate(tree):
    items = list(nodes(tree))
    eligible = {
        "refine": [
            v for v in items if v[1][2] > 1 and v[2][0] in ("MEAN", "MAX")
        ],
        "coarsen": [v for v in items if v[2][0] == "SPLIT"],
        "pool": [v for v in items if v[2][0] in ("MEAN", "MAX")],
        "drop": [v for v in items if v[2][0] != "DROP"],
        "activate": [v for v in items if v[2][0] == "DROP"]
    }
    weights = dict(refine=.25, coarsen=.25, pool=.20, drop=.15, activate=.15)
    choices = [name for name in eligible if eligible[name]]
    p = np.array([weights[name] for name in choices])
    action = str(rng.choice(choices, p=p / p.sum()))
    options = eligible[action]
    path, region, node = options[int(rng.integers(len(options)))]

    if action == "refine":
        new = ("SPLIT",) + tuple(
            PIXEL if r[2] == 1 else node for r in child_regions(region)
        )
    elif action == "coarsen":
        new = MEAN if rng.random() < .5 else MAX
    elif action == "pool":
        new = MAX if node == MEAN else MEAN
    elif action == "drop":
        new = DROP
    else:
        new = PIXEL if region[2] == 1 else (MEAN if rng.random() < .5 else MAX)

    child = replace(tree, path, new)
    return child if active_leaves(child) else tree


def crossover(parent1, parent2):
    first = {p: t for p, _, t in nodes(parent1)}
    second = {p: t for p, _, t in nodes(parent2)}
    common = sorted((set(first) & set(second)) - {()})
    if not common:
        return parent1
    path = common[int(rng.integers(len(common)))]
    child = replace(parent1, path, second[path])
    return child if active_leaves(child) else parent1


def rank_population(df):
    df = df.reset_index(drop=True).copy()
    values = df[["features", "val_NRMSE"]].to_numpy()
    ranks, crowding = np.zeros(len(df), int), np.zeros(len(df))
    remaining, rank = list(range(len(df))), 0

    while remaining:
        front = [
            i for i in remaining if not any(
                np.all(values[j] <= values[i]) and np.any(values[j] < values[i])
                for j in remaining if j != i
            )
        ]
        ranks[front] = rank
        for column in range(2):
            ordered = sorted(front, key=lambda i: values[i, column])
            span = values[ordered[-1], column] - values[ordered[0], column]
            if span == 0:
                continue
            crowding[ordered[0]] = crowding[ordered[-1]] = np.inf
            for k in range(1, len(ordered) - 1):
                crowding[ordered[k]] += (
                    values[ordered[k+1], column] - values[ordered[k-1], column]
                ) / span
        remaining = [i for i in remaining if i not in front]
        rank += 1

    df["rank"], df["crowding"] = ranks, crowding
    return df


def all_pareto(df):
    values = df[["features", "val_NRMSE"]].to_numpy()
    keep = [
        not np.any(np.all(values <= v, axis=1) & np.any(values < v, axis=1))
        for v in values
    ]
    return df.loc[keep].sort_values(
        ["features", "val_NRMSE", "id"]
    ).reset_index(drop=True)


def front_curve(df):
    return all_pareto(df).drop_duplicates(
        ["features", "val_NRMSE"]
    ).reset_index(drop=True)


def select_survivors(population):
    ranked = rank_population(table(population))
    elite = int(ranked.sort_values(["val_NRMSE", "features"]).index[0])
    chosen = [elite]
    available = [i for i in ranked.index if i != elite]

    while len(chosen) < SURVIVORS:
        contestants = rng.choice(
            available, min(TOURNAMENT_SIZE, len(available)), replace=False
        )
        winner = int(min(
            contestants,
            key=lambda i: (ranked.loc[i, "rank"], -ranked.loc[i, "crowding"])
        ))
        chosen.append(winner)
        available.remove(winner)

    return ranked.loc[chosen, "tree"].tolist()


def breed(survivors):
    population, seen, attempts = list(survivors), set(survivors), 0
    while len(population) < POPULATION:
        attempts += 1
        if rng.random() < IMMIGRANT_RATE or attempts > 200:
            child = random_tree()
        else:
            i, j = rng.integers(len(survivors), size=2)
            child = survivors[i]
            if rng.random() < CROSSOVER_RATE:
                child = crossover(child, survivors[j])
            child = mutate(child)
        child = simplify(child)
        if active_leaves(child) and child not in seen:
            population.append(child)
            seen.add(child)
    return population


# ============================================================
# 7. PLOTTING
# ============================================================
COLORS = dict(DROP="#eeeeee", MEAN="#56b4e9", MAX="#e69f00", PIXEL="#009e73")
SUMMARY_COLUMNS = [
    "id", "features", "source_pixels", "mean_regions",
    "max_regions", "individual_pixels", "val_NRMSE"
]


def draw_tree(ax, tree):
    for _, (y, x, size), node in nodes(tree):
        op = node[0]
        if op == "SPLIT":
            continue
        ax.add_patch(Rectangle(
            (x-.5, y-.5), size, size,
            facecolor=COLORS[op], edgecolor="white", linewidth=.7
        ))
        if size > 1:
            ax.text(
                x+(size-1)/2, y+(size-1)/2,
                {"DROP": "–", "MEAN": "mean", "MAX": "max"}[op],
                ha="center", va="center", fontsize=8
            )
    ax.plot(a, a, "rx", ms=9, mew=2)
    ax.set(xlim=(-.5, N-.5), ylim=(N-.5, -.5), aspect="equal")
    ax.axis("off")


def plot_layouts(rows, title, knee_ids=()):
    for start in range(0, len(rows), 12):
        batch = rows.iloc[start:start+12]
        cols = min(4, len(batch))
        nrows = int(np.ceil(len(batch)/cols))
        fig, axes = plt.subplots(
            nrows, cols, figsize=(3.5*cols, 3.5*nrows),
            squeeze=False, constrained_layout=True
        )
        for ax, (_, row) in zip(axes.ravel(), batch.iterrows()):
            draw_tree(ax, row["tree"])
            knee = int(row["id"]) in knee_ids
            ax.set_title(
                f"ID {int(row['id'])}" + (" — KNEE" if knee else "")
                + f"\n{int(row.features)} features; {int(row.source_pixels)} source pixels"
                + f"\nValidation NRMSE={row.val_NRMSE:.3f}",
                fontsize=9, color="red" if knee else "black"
            )
        for ax in axes.ravel()[len(batch):]:
            ax.axis("off")
        fig.suptitle(title + "\nBlue=mean; orange=max; green=pixel; gray=drop")
        plt.show()


def plot_spectra(actual, predictions, coords, title):
    mean, sd = predictions.mean(axis=0), predictions.std(axis=0)
    fig, axes = plt.subplots(
        1, len(coords), figsize=(5*len(coords), 3.5),
        squeeze=False, constrained_layout=True
    )
    for i, ax in enumerate(axes[0]):
        for p in predictions:
            ax.plot(E, p[i], color="tab:blue", alpha=.15)
        ax.plot(E, actual[i], "k-", lw=2, label="Measured")
        ax.plot(E, mean[i], color="tab:orange", lw=2, label="Mean prediction")
        ax.fill_between(
            E, mean[i]-sd[i], mean[i]+sd[i],
            color="tab:orange", alpha=.2, label="±1 model SD"
        )
        ax.set(
            xlabel="Energy", ylabel="Intensity",
            title=f"Location {tuple(coords[i])}"
        )
        ax.legend(fontsize=8)
    fig.suptitle(title)
    plt.show()


# ============================================================
# 8. EVOLUTION AND ALL-POINT ERROR HISTORY
# ============================================================
population = [MEAN, MAX, full_pixels(), ("SPLIT", MEAN, MAX, DROP, MEAN)]
while len(population) < POPULATION:
    candidate = random_tree()
    if active_leaves(candidate) and candidate not in population:
        population.append(candidate)

history, generation_tables, error_maps, error_records = [], [], [], []

for generation in range(1, GENERATIONS + 1):
    before = len(archive)
    for tree in population:
        evaluate(tree, generation)

    current = rank_population(table(population))
    survivors = select_survivors(population)
    survivor_set = set(survivors)
    current["survives"] = [t in survivor_set for t in current.tree]
    current["generation"] = generation
    generation_tables.append(current.copy())

    best_four = current.sort_values(["val_NRMSE", "features"]).head(4)
    winner = best_four.iloc[0]
    winner_map = error_map_for_tree(winner.tree)
    error_maps.append(winner_map.copy())
    error_records.append({
        "generation": generation,
        "descriptor_id": int(winner["id"]),
        "features": int(winner.features),
        "val_NRMSE": winner.val_NRMSE,
        "all_point_RMSE": np.sqrt(np.mean(
            archive[winner.tree]["all_pixel_rmse"]**2
        )),
        "tree": winner.tree
    })
    history.append({
        "generation": generation,
        "best_NRMSE": current.val_NRMSE.min(),
        "median_NRMSE": current.val_NRMSE.median(),
        "mean_features": current.features.mean(),
        "unique_descriptors": len(archive)
    })

    print(f"\nGENERATION {generation}/{GENERATIONS}")
    print(f"New evaluations={len(archive)-before}; total={len(archive)}")
    display(current[
        SUMMARY_COLUMNS + ["rank", "survives"]
    ].sort_values(["val_NRMSE", "features"]))

    plot_layouts(best_four, f"Generation {generation}: four best descriptors")
    plot_spectra(
        Y_all[example_ids],
        np.stack([archive[t]["example_predictions"] for t in best_four.tree]),
        all_coordinates[example_ids],
        f"Generation {generation}: original validation patches"
    )

    all_results = table()
    curve = front_curve(all_results)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    axes[0].scatter(
        all_results.features, all_results.val_NRMSE, s=25, alpha=.25,
        label="Archive"
    )
    axes[0].scatter(
        current.features, current.val_NRMSE, s=50,
        facecolors="none", edgecolors="tab:orange", label="Current generation"
    )
    axes[0].plot(curve.features, curve.val_NRMSE, "k.-", label="Pareto front")
    axes[0].set(xlabel="Output features", ylabel="Validation NRMSE")
    axes[0].legend()
    axes[0].grid(alpha=.2)

    vmax = max(max(np.nanmax(m) for m in error_maps), 1e-12)
    im = axes[1].imshow(
        winner_map, cmap="magma", vmin=0, vmax=vmax, interpolation="nearest"
    )
    axes[1].set_title(f"All-point spectral RMSE: generation {generation}")
    fig.colorbar(im, ax=axes[1], label="Spectral RMSE")
    plt.show()

    if generation < GENERATIONS:
        population = breed(survivors)

ga_results = table()
ga_history = pd.DataFrame(history)
ga_generations = pd.concat(generation_tables, ignore_index=True)
error_map_history = np.stack(error_maps)
error_map_metadata = pd.DataFrame(error_records)


# ============================================================
# 9. ERROR MAPS AND PCA
# ============================================================
def analyze_error_history(stack, n_components=3):
    cols = min(5, len(stack))
    rows = int(np.ceil(len(stack)/cols))
    fig, axes = plt.subplots(
        rows, cols, figsize=(3*cols, 3*rows),
        squeeze=False, constrained_layout=True
    )
    vmax = max(float(np.nanmax(stack)), 1e-12)
    for i, ax in enumerate(axes.ravel()):
        if i >= len(stack):
            ax.axis("off")
            continue
        im = ax.imshow(
            stack[i], cmap="magma", vmin=0, vmax=vmax, interpolation="nearest"
        )
        ax.set_title(f"Generation {i+1}")
        ax.axis("off")
    fig.colorbar(im, ax=list(axes.ravel()), label="All-point spectral RMSE")
    plt.show()

    valid = np.isfinite(stack).all(axis=0)
    if len(stack) < 2 or valid.sum() < 2:
        print("Insufficient data for PCA.")
        return None

    X = stack[:, valid].T
    spatial_mean = X.mean(axis=0)
    if np.sum((X-spatial_mean)**2) == 0:
        print("No spatial variation for PCA.")
        return None

    k = min(n_components, X.shape[0]-1, X.shape[1])
    pca = PCA(n_components=k, svd_solver="full")
    scores = pca.fit_transform(X)
    singular = pca.singular_values_
    keep = singular > np.finfo(float).eps * max(X.shape) * singular[0]
    singular = singular[keep]
    if not len(singular):
        return None

    loadings = scores[:, keep] / singular[None, :]
    amplitudes = pca.components_[keep].T * singular[None, :]
    explained = pca.explained_variance_ratio_[keep]

    for j in range(len(singular)):
        if loadings[np.argmax(np.abs(loadings[:, j])), j] < 0:
            loadings[:, j] *= -1
            amplitudes[:, j] *= -1

    loading_maps = np.full((len(singular), H, W), np.nan)
    for j in range(len(singular)):
        loading_maps[j, valid] = loadings[:, j]
    reconstruction = np.full_like(stack, np.nan)
    reconstruction[:, valid] = spatial_mean[:, None] + amplitudes @ loadings.T

    fig, ax = plt.subplots(figsize=(7, 3))
    ax.plot(np.arange(1, len(stack)+1), spatial_mean, "o-")
    ax.set(
        xlabel="Generation", ylabel="Mean pixel RMSE",
        title="Spatial mean error removed before PCA"
    )
    ax.grid(alpha=.2)
    plt.show()

    fig, axes = plt.subplots(
        len(singular), 2, figsize=(11, 3.5*len(singular)),
        squeeze=False, constrained_layout=True
    )
    for j in range(len(singular)):
        axes[j, 0].plot(np.arange(1, len(stack)+1), amplitudes[:, j], "o-")
        axes[j, 0].set(
            xlabel="Generation", ylabel="Component amplitude",
            title=f"Component {j+1}: {100*explained[j]:.1f}% variance"
        )
        axes[j, 0].grid(alpha=.2)
        limit = np.nanmax(np.abs(loading_maps[j]))
        im = axes[j, 1].imshow(
            loading_maps[j], cmap="RdBu_r",
            vmin=-limit, vmax=limit, interpolation="nearest"
        )
        axes[j, 1].set_title(f"Spatial loading {j+1}")
        fig.colorbar(im, ax=axes[j, 1], label="Unit-norm loading")
    plt.show()

    return {
        "pca": pca, "valid_pixels": valid,
        "mean_error_by_generation": spatial_mean,
        "generation_amplitudes": amplitudes,
        "spatial_loadings": loading_maps,
        "explained_variance_ratio": explained,
        "reconstructed_error_maps": reconstruction
    }


error_pca = analyze_error_history(error_map_history, PCA_COMPONENTS)


# ============================================================
# 10. PARETO FRONT AND KNEE
# ============================================================
pareto_descriptors = all_pareto(ga_results)
pareto_curve = front_curve(ga_results)
knee_index = None

if (
    len(pareto_curve) >= 3
    and np.ptp(pareto_curve.features) > 0
    and np.ptp(pareto_curve.val_NRMSE) > 0
):
    x = (
        (pareto_curve.features.to_numpy() - pareto_curve.features.min())
        / np.ptp(pareto_curve.features)
    )
    y = (
        (pareto_curve.val_NRMSE.to_numpy() - pareto_curve.val_NRMSE.min())
        / np.ptp(pareto_curve.val_NRMSE)
    )
    score = (1-x-y) / np.sqrt(2)
    candidate = int(np.argmax(score[1:-1])) + 1
    if score[candidate] > 1e-6:
        knee_index = candidate

knee_ids = []
if knee_index is not None:
    knee = pareto_curve.iloc[knee_index]
    knee_ids = pareto_descriptors.loc[
        (pareto_descriptors.features == knee.features)
        & (pareto_descriptors.val_NRMSE == knee.val_NRMSE), "id"
    ].astype(int).tolist()

pareto_descriptors["estimated_knee"] = pareto_descriptors["id"].isin(knee_ids)
print("\nALL PARETO DESCRIPTORS")
display(pareto_descriptors[SUMMARY_COLUMNS + ["estimated_knee"]])

steps = []
for i in range(len(pareto_curve)-1):
    smaller, larger = pareto_curve.iloc[i], pareto_curve.iloc[i+1]
    removed = int(larger.features - smaller.features)
    increase = float(smaller.val_NRMSE - larger.val_NRMSE)
    steps.append({
        "from_id": int(larger["id"]), "to_id": int(smaller["id"]),
        "from_features": int(larger.features),
        "to_features": int(smaller.features),
        "features_removed": removed,
        "NRMSE_increase": increase,
        "increase_per_removed_feature": increase / removed
    })
pareto_error_steps = pd.DataFrame(steps)
display(pareto_error_steps)

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(ga_results.features, ga_results.val_NRMSE, s=25, alpha=.2)
ax.plot(pareto_curve.features, pareto_curve.val_NRMSE, "ko-", label="Pareto front")
for _, row in pareto_curve.iterrows():
    ax.annotate(
        f"ID {int(row['id'])}", (row.features, row.val_NRMSE),
        xytext=(4, 6), textcoords="offset points", fontsize=8
    )
if knee_index is not None:
    knee = pareto_curve.iloc[knee_index]
    simpler = pareto_curve.iloc[knee_index-1]
    ax.scatter(
        [knee.features], [knee.val_NRMSE],
        marker="*", s=250, color="red", zorder=5, label="Estimated knee"
    )
    ax.plot(
        [simpler.features, knee.features],
        [simpler.val_NRMSE, knee.val_NRMSE],
        color="red", linewidth=3, label="Next simplification step"
    )
    print("KNEE AND ADJACENT CONFIGURATIONS")
    display(pareto_curve.iloc[knee_index-1:knee_index+2][SUMMARY_COLUMNS])
else:
    print("No distinct interior knee detected.")
ax.set(xlabel="Output features", ylabel="Validation NRMSE")
ax.grid(alpha=.2)
ax.legend()
plt.show()

plot_layouts(pareto_descriptors, "All Pareto descriptors", knee_ids)
pareto_trees = {
    int(row["id"]): row["tree"] for _, row in pareto_descriptors.iterrows()
}


# ============================================================
# 11. HIGH-COMPLEXITY DESCRIPTORS AND SEARCH HISTORY
# ============================================================
high_complexity_descriptors = ga_results.sort_values(
    ["features", "val_NRMSE", "id"], ascending=[False, True, True]
).head(N_HIGH_COMPLEXITY)

print("\nHIGHEST-COMPLEXITY EVALUATED DESCRIPTORS")
display(high_complexity_descriptors[SUMMARY_COLUMNS])
plot_layouts(high_complexity_descriptors, "Highest-complexity evaluated descriptors")

high_complexity_pareto = pareto_descriptors.sort_values(
    ["features", "val_NRMSE"], ascending=[False, True]
).head(8)
print("\nHIGH-COMPLEXITY PARETO DESCRIPTORS")
display(high_complexity_pareto[SUMMARY_COLUMNS])
plot_layouts(high_complexity_pareto, "High-complexity Pareto descriptors", knee_ids)

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(
    ga_results.features, ga_results.val_NRMSE, s=25, alpha=.2, label="All evaluated"
)
ax.plot(pareto_curve.features, pareto_curve.val_NRMSE, "k.-", label="Pareto front")
ax.scatter(
    high_complexity_descriptors.features, high_complexity_descriptors.val_NRMSE,
    s=75, facecolors="none", edgecolors="red", label="High-complexity descriptors"
)
for _, row in high_complexity_descriptors.iterrows():
    ax.annotate(
        str(int(row["id"])), (row.features, row.val_NRMSE),
        xytext=(4, 4), textcoords="offset points", fontsize=8
    )
ax.set(xlabel="Output features", ylabel="Validation NRMSE")
ax.legend()
ax.grid(alpha=.2)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
axes[0].plot(ga_history.generation, ga_history.best_NRMSE, "o-", label="Best")
axes[0].plot(ga_history.generation, ga_history.median_NRMSE, "o-", label="Median")
axes[0].set(xlabel="Generation", ylabel="Validation NRMSE")
axes[0].legend()
axes[1].plot(ga_history.generation, ga_history.mean_features, "o-")
axes[1].set(xlabel="Generation", ylabel="Mean output features")
for ax in axes:
    ax.grid(alpha=.2)
plt.show()


# ============================================================
# 12. FINAL AUGMENTED REFIT AND ALL-POINT EVALUATION
# ============================================================
final_four = ga_results.sort_values(["val_NRMSE", "features", "id"]).head(4)
selected_tree = final_four.iloc[0].tree

test_example_local = spread_indices(all_coordinates[test_ids], N_EXAMPLES)
test_example_ids = test_ids[test_example_local]

final_models, final_example_predictions = [], []
ensemble_prediction = np.zeros_like(Y_all)
best_all_prediction = None

for j, tree in enumerate(final_four.tree):
    # Original fit + validation locations, plus their rotated copies.
    model = fit_augmented_model(tree, train_ids)
    prediction = model.predict(feature_matrix(tree))  # Original patches only.
    final_models.append(model)
    final_example_predictions.append(prediction[test_example_ids].copy())
    ensemble_prediction += prediction / len(final_four)
    if j == 0:
        best_all_prediction = prediction.copy()

final_model = final_models[0]
subsets = {
    "Fit": fit_ids, "Validation": val_ids, "Test": test_ids,
    "All valid centers": np.arange(len(P_all))
}
test_baseline = np.sqrt(np.mean(
    (Y_all[test_ids] - Y_all[train_ids].mean(axis=0))**2
))

summary = []
for name, prediction in [
    ("Best descriptor", best_all_prediction),
    ("Mean of four descriptors", ensemble_prediction)
]:
    for subset_name, ids in subsets.items():
        rmse = np.sqrt(np.mean((prediction[ids] - Y_all[ids])**2))
        summary.append({
            "predictor": name, "subset": subset_name, "locations": len(ids),
            "spectral_RMSE": rmse,
            "test_NRMSE": (
                rmse/test_baseline
                if subset_name == "Test" and test_baseline > 0 else np.nan
            )
        })
rmse_summary = pd.DataFrame(summary)
print("\nFINAL MODELS: ORIGINAL-PATCH ERRORS")
display(rmse_summary)

plot_spectra(
    Y_all[test_example_ids], np.stack(final_example_predictions),
    all_coordinates[test_example_ids], "Final models: unrotated test patches"
)

all_pixel_rmse = np.sqrt(np.mean((best_all_prediction-Y_all)**2, axis=1))
ensemble_pixel_rmse = np.sqrt(np.mean((ensemble_prediction-Y_all)**2, axis=1))
final_error_map = np.full((H, W), np.nan)
ensemble_error_map = np.full((H, W), np.nan)
final_error_map[all_coordinates[:, 0], all_coordinates[:, 1]] = all_pixel_rmse
ensemble_error_map[all_coordinates[:, 0], all_coordinates[:, 1]] = ensemble_pixel_rmse

fig, axes = plt.subplots(1, 3, figsize=(16, 4), constrained_layout=True)
for name, prediction in [
    ("Best descriptor", best_all_prediction),
    ("Mean of four", ensemble_prediction)
]:
    axes[0].plot(
        E, np.sqrt(np.mean((prediction[test_ids]-Y_all[test_ids])**2, axis=0)),
        label=name
    )
axes[0].set(xlabel="Energy", ylabel="Test RMSE", title="Error by energy channel")
axes[0].legend()

vmax = max(np.nanmax(final_error_map), np.nanmax(ensemble_error_map), 1e-12)
for ax, error_map, title in [
    (axes[1], final_error_map, "Best descriptor: all original locations"),
    (axes[2], ensemble_error_map, "Mean of four: all original locations")
]:
    im = ax.imshow(
        error_map, cmap="magma", vmin=0, vmax=vmax, interpolation="nearest"
    )
    ax.set_title(title)
fig.colorbar(im, ax=list(axes[1:]), label="Spectral RMSE")
plt.show()

feature_definitions = pd.DataFrame([
    {
        "feature": i+1, "operation": op,
        "row_start": region[0], "column_start": region[1],
        "region_size": region[2]
    }
    for i, (region, op) in enumerate(active_leaves(selected_tree))
])
display(feature_definitions)

In [ ]:
# ============================================================
# GA SPATIAL DESCRIPTORS + x10 AUGMENTATION + TWO-LAYER CNN
#
# Inputs:
#   loadedfile["2"], or image / spectra / energy
#
# Outputs:
#   Whole-spectrum predictions
#   Four best descriptors and spectra each generation
#   All-point RMSE maps and PCA
#   Pareto layouts, knee, high-complexity layouts
# ============================================================

import copy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.patches import Rectangle
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from IPython.display import display

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader


# ============================================================
# 1. SETTINGS
# ============================================================
DATASET_KEY = "2"

N = 12
POPULATION, GENERATIONS = 100, 100
SURVIVORS, TOURNAMENT_SIZE = 25, 5
CROSSOVER_RATE, IMMIGRANT_RATE = 0.8, 0.15
SEED = 0

TEST_FRACTION = 0.25
VALIDATION_FRACTION = 0.25
MIN_FIT, MIN_VAL, MIN_TEST = 20, 15, 15

AUGMENTATION_FACTOR = 10
EPOCHS = 8                 # Fixed budget for every candidate
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
CHANNELS = (8, 16)

N_EXAMPLES = 3
PCA_COMPONENTS = 3
N_HIGH_COMPLEXITY = 12

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(2)

rng = np.random.default_rng(SEED)
aug_rng = np.random.default_rng(SEED + 1000)

if "loadedfile" in globals():
    data = loadedfile[DATASET_KEY]
    img = np.asarray(data["image"], dtype=float)
    cube = np.asarray(data["spectrum image"], dtype=float)
    E = np.asarray(data["energy axis"], dtype=float).ravel()
else:
    img = np.asarray(image, dtype=float)
    cube = np.asarray(spectra, dtype=float)
    E = np.asarray(energy, dtype=float).ravel()

if img.ndim != 2 or cube.shape != (*img.shape, len(E)):
    raise ValueError("Expected image (H,W), spectra (H,W,E), energy (E,).")
if not all(np.isfinite(v).all() for v in (img, cube, E)):
    raise ValueError("Inputs contain nonfinite values.")
if AUGMENTATION_FACTOR != 10:
    raise ValueError("This augmentation setup uses exactly 10 samples per source.")

order = np.argsort(E)
E, cube = E[order], cube[..., order]
if np.any(np.diff(E) <= 0):
    raise ValueError("Energy values must be distinct.")

H, W = img.shape
a, b = (N - 1)//2, N - (N - 1)//2


# ============================================================
# 2. ORIGINAL PATCHES AND RANDOM SPLITS
# ============================================================
all_coordinates = np.array([
    (y, x)
    for y in range(a, H-b+1)
    for x in range(a, W-b+1)
], dtype=int).reshape(-1, 2)

if len(all_coordinates) < MIN_FIT + MIN_VAL + MIN_TEST:
    raise ValueError("Too few valid patches.")

P_all = np.stack([
    img[y-a:y+b, x-a:x+b] for y, x in all_coordinates
]).astype(np.float32)

Y_all = cube[
    all_coordinates[:, 0], all_coordinates[:, 1]
].astype(np.float32)

train_ids, test_ids = train_test_split(
    np.arange(len(P_all)),
    test_size=TEST_FRACTION, random_state=SEED
)
fit_ids, val_ids = train_test_split(
    train_ids,
    test_size=VALIDATION_FRACTION/(1-TEST_FRACTION),
    random_state=SEED+1
)

if (
    len(fit_ids) < MIN_FIT or len(val_ids) < MIN_VAL
    or len(test_ids) < MIN_TEST
):
    raise ValueError("Insufficient fitting/validation/test locations.")

baseline = np.sqrt(np.mean(
    (Y_all[val_ids].astype(float)
     - Y_all[fit_ids].mean(axis=0, dtype=float))**2
))
if baseline <= 0:
    raise ValueError("Zero validation baseline error.")


def spread_indices(coords, count):
    chosen = [
        int(np.argmin(np.sum((coords-coords.mean(axis=0))**2, axis=1)))
    ]
    distance = np.full(len(coords), np.inf)
    while len(chosen) < min(count, len(coords)):
        distance = np.minimum(
            distance,
            np.sum((coords-coords[chosen[-1]])**2, axis=1)
        )
        distance[chosen] = -1
        chosen.append(int(np.argmax(distance)))
    return np.asarray(chosen)


example_ids = val_ids[
    spread_indices(all_coordinates[val_ids], N_EXAMPLES)
]

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
for ax in axes:
    ax.imshow(img, cmap="gray", alpha=.45)
    ax.set(xlabel="Column", ylabel="Row")

for ids, color, label in [
    (train_ids, "blue", "Train: fit + validation"),
    (test_ids, "red", "Test")
]:
    xy = all_coordinates[ids]
    axes[0].scatter(
        xy[:, 1], xy[:, 0], s=10, c=color,
        linewidths=0, label=f"{label}: {len(ids)}"
    )
for ids, color, label in [
    (fit_ids, "blue", "Fit"), (val_ids, "orange", "Validation")
]:
    xy = all_coordinates[ids]
    axes[1].scatter(
        xy[:, 1], xy[:, 0], s=10, c=color,
        linewidths=0, label=f"{label}: {len(ids)}"
    )
for ax in axes:
    ax.legend(fontsize=8)
axes[0].set_title("Random training/test split")
axes[1].set_title("Inner fitting/validation split")
plt.show()


# ============================================================
# 3. EIGHT SQUARE SYMMETRIES + TWO RANDOM REPEATS = x10
# ============================================================
TRANSFORM_NAMES = [
    "Identity", "90° CCW", "180°", "270° CCW",
    "Horizontal-axis mirror", "Vertical-axis mirror",
    "Main-diagonal mirror", "Other-diagonal mirror"
]


def transform_patch(p, code):
    if code < 4:
        return np.rot90(p, code)
    if code == 4:
        return np.flipud(p)
    if code == 5:
        return np.fliplr(p)
    if code == 6:
        return p.T
    return np.flip(p.T, axis=(0, 1))


# Split first; no test-source patches enter training augmentation.
aug_sources = np.repeat(train_ids, AUGMENTATION_FACTOR)
transform_codes = np.concatenate([
    np.r_[np.arange(8), aug_rng.choice(np.arange(1, 8), 2, replace=False)]
    for _ in train_ids
]).astype(int)

P_aug = np.stack([
    transform_patch(P_all[source], int(code))
    for source, code in zip(aug_sources, transform_codes)
]).astype(np.float32)

augmentation_metadata = pd.DataFrame({
    "source_id": aug_sources,
    "transform": [TRANSFORM_NAMES[k] for k in transform_codes],
    "used_in_evolution": np.isin(aug_sources, fit_ids)
})

print(f"Device: {DEVICE}")
print(f"Original locations: {len(P_all)}")
print(f"Evolution fitting rows: {len(fit_ids)} × 10 = {10*len(fit_ids)}")
print(f"Final fitting rows: {len(train_ids)} × 10 = {10*len(train_ids)}")
print("Validation/test predictions use original patches.")

# Show the actual 10-example augmentation of one fitting patch.
source = fit_ids[0]
positions = np.flatnonzero(aug_sources == source)
fig, axes = plt.subplots(2, 5, figsize=(13, 5), constrained_layout=True)
for ax, idx in zip(axes.ravel(), positions):
    ax.imshow(P_aug[idx], cmap="gray")
    ax.set_title(TRANSFORM_NAMES[transform_codes[idx]], fontsize=9)
    ax.axis("off")
fig.suptitle("One source patch: eight symmetries plus two random repeats")
plt.show()


# ============================================================
# 4. DESCRIPTOR TREES: 12 → 6 → 3 → 1
# ============================================================
DROP, MEAN, MAX, PIXEL = ("DROP",), ("MEAN",), ("MAX",), ("PIXEL",)
ROOT = (0, 0, N)


def child_regions(region):
    y, x, size = region
    if size == 1:
        return []
    step = size//2 if size % 2 == 0 else 1
    return [
        (y+dy, x+dx, step)
        for dy in range(0, size, step)
        for dx in range(0, size, step)
    ]


def simplify(tree):
    if tree[0] != "SPLIT":
        return tree
    children = tuple(simplify(t) for t in tree[1:])
    return DROP if all(t == DROP for t in children) else ("SPLIT",)+children


def random_tree(region=ROOT):
    if region[2] == 1:
        return PIXEL if rng.random() < .7 else DROP
    op = str(rng.choice(
        ["DROP", "MEAN", "MAX", "SPLIT"], p=[.15, .25, .20, .40]
    ))
    if op != "SPLIT":
        return (op,)
    return simplify(
        ("SPLIT",)+tuple(random_tree(r) for r in child_regions(region))
    )


def nodes(tree, region=ROOT, path=()):
    yield path, region, tree
    if tree[0] == "SPLIT":
        for i, (child, box) in enumerate(zip(tree[1:], child_regions(region))):
            yield from nodes(child, box, path+(i,))


def replace(tree, path, subtree):
    if not path:
        return simplify(subtree)
    children = list(tree[1:])
    children[path[0]] = replace(children[path[0]], path[1:], subtree)
    return simplify(("SPLIT",)+tuple(children))


def active_leaves(tree):
    return [
        (r, t[0]) for _, r, t in nodes(tree)
        if t[0] not in ("DROP", "SPLIT")
    ]


def full_pixels(region=ROOT):
    if region[2] == 1:
        return PIXEL
    return ("SPLIT",)+tuple(full_pixels(r) for r in child_regions(region))


# ============================================================
# 5. DESCRIPTOR → FOUR-CHANNEL SPATIAL INPUT
#
# Channel 0: normalized retained/pooled intensity
# Channel 1: PIXEL indicator
# Channel 2: MEAN indicator
# Channel 3: MAX indicator
#
# DROP is zero in all channels.
# Pooled values fill their region, without restoring discarded detail.
# ============================================================
def encode(tree, patches, mean, scale):
    result = np.zeros((len(patches), 4, N, N), dtype=np.float32)
    channel = {"PIXEL": 1, "MEAN": 2, "MAX": 3}

    for (y, x, size), op in active_leaves(tree):
        values = patches[:, y:y+size, x:x+size]
        if op == "MAX":
            values = values.max(axis=(1, 2))
        else:
            values = values.mean(axis=(1, 2))

        result[:, 0, y:y+size, x:x+size] = (
            (values-mean)/scale
        )[:, None, None]
        result[:, channel[op], y:y+size, x:x+size] = 1

    return result


class SpectrumCNN(nn.Module):
    """Two hidden convolutional layers and a linear spectral output."""
    def __init__(self, n_energy):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv2d(4, CHANNELS[0], kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AvgPool2d(2),          # 12 → 6
            nn.Conv2d(CHANNELS[0], CHANNELS[1], kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AvgPool2d(2),          # 6 → 3
            nn.Flatten(),
            nn.Linear(CHANNELS[1]*3*3, n_energy)
        )

    def forward(self, x):
        return self.network(x)


def fit_cnn(tree, original_ids):
    # Same initialization and minibatch order for comparable candidates.
    random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

    selected = np.isin(aug_sources, original_ids)
    mean = float(P_all[original_ids].mean(dtype=np.float64))
    scale = max(float(P_all[original_ids].std(dtype=np.float64)), 1e-12)

    # Center each energy channel, but use ONE global output scale.
    # This preserves the relative weighting of channels in raw MSE.
    y_mean = Y_all[original_ids].mean(axis=0, dtype=np.float64)
    y_scale = max(float(np.sqrt(np.mean(
        (Y_all[original_ids]-y_mean)**2
    ))), 1e-12)

    X = encode(tree, P_aug[selected], mean, scale)
    Y = ((Y_all[aug_sources[selected]]-y_mean)/y_scale).astype(np.float32)

    dataset = TensorDataset(torch.from_numpy(X), torch.from_numpy(Y))
    generator = torch.Generator().manual_seed(SEED)
    loader = DataLoader(
        dataset, batch_size=BATCH_SIZE, shuffle=True,
        generator=generator, num_workers=0
    )

    model = SpectrumCNN(len(E)).to(DEVICE)
    optimizer = torch.optim.Adam(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    loss_function = nn.MSELoss()

    model.train()
    for _ in range(EPOCHS):
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = loss_function(model(xb), yb)
            if not torch.isfinite(loss):
                raise RuntimeError("Nonfinite CNN training loss.")
            loss.backward()
            optimizer.step()

    return {
        "model": model.cpu(),
        "tree": tree,
        "x_mean": mean, "x_scale": scale,
        "y_mean": y_mean, "y_scale": y_scale
    }


def predict_cnn(bundle, patches):
    X = encode(
        bundle["tree"], patches, bundle["x_mean"], bundle["x_scale"]
    )
    model = bundle["model"].to(DEVICE).eval()
    outputs = []
    with torch.no_grad():
        for start in range(0, len(X), BATCH_SIZE):
            xb = torch.from_numpy(X[start:start+BATCH_SIZE]).to(DEVICE)
            outputs.append(model(xb).cpu().numpy())
    model.cpu()
    return (
        np.concatenate(outputs)*bundle["y_scale"] + bundle["y_mean"]
    )


# ============================================================
# 6. VALIDATION FITNESS AND ALL-POINT DIAGNOSTICS
# ============================================================
archive = {}


def evaluate(tree, generation):
    if tree in archive:
        return

    bundle = fit_cnn(tree, fit_ids)
    prediction = predict_cnn(bundle, P_all)

    # All-point errors are diagnostic; only validation errors select models.
    errors = np.sqrt(np.mean((prediction-Y_all)**2, axis=1))
    score = np.sqrt(np.mean(errors[val_ids]**2))/baseline
    if not np.isfinite(score):
        raise RuntimeError("Nonfinite candidate score.")

    leaves = active_leaves(tree)
    archive[tree] = {
        "id": len(archive)+1,
        "features": len(leaves),
        "source_pixels": sum(r[2]**2 for r, _ in leaves),
        "mean_regions": sum(op == "MEAN" for _, op in leaves),
        "max_regions": sum(op == "MAX" for _, op in leaves),
        "individual_pixels": sum(op == "PIXEL" for _, op in leaves),
        "val_NRMSE": score,
        "first_generation": generation,
        "all_pixel_rmse": errors,
        "example_predictions": prediction[example_ids]
    }


def table(trees=None):
    trees = list(archive) if trees is None else trees
    excluded = {"all_pixel_rmse", "example_predictions"}
    return pd.DataFrame([
        {"tree": t, **{k: v for k, v in archive[t].items() if k not in excluded}}
        for t in trees
    ])


def to_map(values):
    result = np.full((H, W), np.nan)
    result[all_coordinates[:, 0], all_coordinates[:, 1]] = values
    return result


# ============================================================
# 7. GA OPERATORS
# ============================================================
def mutate(tree):
    items = list(nodes(tree))
    eligible = {
        "refine": [v for v in items if v[1][2] > 1 and v[2][0] in ("MEAN", "MAX")],
        "coarsen": [v for v in items if v[2][0] == "SPLIT"],
        "pool": [v for v in items if v[2][0] in ("MEAN", "MAX")],
        "drop": [v for v in items if v[2][0] != "DROP"],
        "activate": [v for v in items if v[2][0] == "DROP"]
    }
    weights = dict(refine=.25, coarsen=.25, pool=.20, drop=.15, activate=.15)
    choices = [k for k in eligible if eligible[k]]
    p = np.array([weights[k] for k in choices])
    action = str(rng.choice(choices, p=p/p.sum()))
    options = eligible[action]
    path, region, node = options[int(rng.integers(len(options)))]

    if action == "refine":
        new = ("SPLIT",)+tuple(
            PIXEL if r[2] == 1 else node for r in child_regions(region)
        )
    elif action == "coarsen":
        new = MEAN if rng.random() < .5 else MAX
    elif action == "pool":
        new = MAX if node == MEAN else MEAN
    elif action == "drop":
        new = DROP
    else:
        new = PIXEL if region[2] == 1 else (MEAN if rng.random() < .5 else MAX)

    child = replace(tree, path, new)
    return child if active_leaves(child) else tree


def crossover(p1, p2):
    first = {p: t for p, _, t in nodes(p1)}
    second = {p: t for p, _, t in nodes(p2)}
    common = sorted((set(first)&set(second))-{()})
    if not common:
        return p1
    path = common[int(rng.integers(len(common)))]
    child = replace(p1, path, second[path])
    return child if active_leaves(child) else p1


def rank_population(df):
    df = df.reset_index(drop=True).copy()
    values = df[["features", "val_NRMSE"]].to_numpy()
    rank_values, crowding = np.zeros(len(df), int), np.zeros(len(df))
    remaining, rank = list(range(len(df))), 0

    while remaining:
        front = [
            i for i in remaining if not any(
                np.all(values[j] <= values[i]) and np.any(values[j] < values[i])
                for j in remaining if j != i
            )
        ]
        rank_values[front] = rank
        for column in range(2):
            ordered = sorted(front, key=lambda i: values[i, column])
            span = values[ordered[-1], column]-values[ordered[0], column]
            if span == 0:
                continue
            crowding[ordered[0]] = crowding[ordered[-1]] = np.inf
            for k in range(1, len(ordered)-1):
                crowding[ordered[k]] += (
                    values[ordered[k+1], column]-values[ordered[k-1], column]
                )/span
        remaining = [i for i in remaining if i not in front]
        rank += 1

    df["rank"], df["crowding"] = rank_values, crowding
    return df


def all_pareto(df):
    values = df[["features", "val_NRMSE"]].to_numpy()
    keep = [
        not np.any(np.all(values <= v, axis=1)&np.any(values < v, axis=1))
        for v in values
    ]
    return df.loc[keep].sort_values(
        ["features", "val_NRMSE", "id"]
    ).reset_index(drop=True)


def front_curve(df):
    return all_pareto(df).drop_duplicates(
        ["features", "val_NRMSE"]
    ).reset_index(drop=True)


def select_survivors(population):
    df = rank_population(table(population))
    elite = int(df.sort_values(["val_NRMSE", "features"]).index[0])
    chosen, available = [elite], [i for i in df.index if i != elite]
    while len(chosen) < SURVIVORS:
        contestants = rng.choice(
            available, min(TOURNAMENT_SIZE, len(available)), replace=False
        )
        winner = int(min(
            contestants, key=lambda i: (df.loc[i, "rank"], -df.loc[i, "crowding"])
        ))
        chosen.append(winner)
        available.remove(winner)
    return df.loc[chosen, "tree"].tolist()


def breed(survivors):
    population, seen, attempts = list(survivors), set(survivors), 0
    while len(population) < POPULATION:
        attempts += 1
        if rng.random() < IMMIGRANT_RATE or attempts > 200:
            child = random_tree()
        else:
            i, j = rng.integers(len(survivors), size=2)
            child = survivors[i]
            if rng.random() < CROSSOVER_RATE:
                child = crossover(child, survivors[j])
            child = mutate(child)
        child = simplify(child)
        if active_leaves(child) and child not in seen:
            population.append(child)
            seen.add(child)
    return population


# ============================================================
# 8. VISUALIZATION HELPERS
# ============================================================
COLORS = dict(DROP="#eeeeee", MEAN="#56b4e9", MAX="#e69f00", PIXEL="#009e73")
COLUMNS = [
    "id", "features", "source_pixels", "mean_regions",
    "max_regions", "individual_pixels", "val_NRMSE"
]


def draw_tree(ax, tree):
    for _, (y, x, size), node in nodes(tree):
        op = node[0]
        if op == "SPLIT":
            continue
        ax.add_patch(Rectangle(
            (x-.5, y-.5), size, size,
            facecolor=COLORS[op], edgecolor="white", linewidth=.7
        ))
        if size > 1:
            ax.text(
                x+(size-1)/2, y+(size-1)/2,
                {"DROP": "–", "MEAN": "mean", "MAX": "max"}[op],
                ha="center", va="center", fontsize=8
            )
    ax.plot(a, a, "rx", ms=9, mew=2)
    ax.set(xlim=(-.5, N-.5), ylim=(N-.5, -.5), aspect="equal")
    ax.axis("off")


def plot_layouts(rows, title, knee_ids=()):
    for start in range(0, len(rows), 12):
        batch = rows.iloc[start:start+12]
        cols, nrows = min(4, len(batch)), int(np.ceil(len(batch)/4))
        fig, axes = plt.subplots(
            nrows, cols, figsize=(3.5*cols, 3.4*nrows),
            squeeze=False, constrained_layout=True
        )
        for ax, (_, row) in zip(axes.ravel(), batch.iterrows()):
            draw_tree(ax, row.tree)
            knee = int(row["id"]) in knee_ids
            ax.set_title(
                f"ID {int(row['id'])}" + (" — KNEE" if knee else "")
                + f"\n{int(row.features)} features; error={row.val_NRMSE:.3f}",
                fontsize=9, color="red" if knee else "black"
            )
        for ax in axes.ravel()[len(batch):]:
            ax.axis("off")
        fig.suptitle(title+"\nBlue=mean; orange=max; green=pixel; gray=drop")
        plt.show()


def plot_spectra(actual, predictions, coords, title):
    mean, sd = predictions.mean(axis=0), predictions.std(axis=0)
    fig, axes = plt.subplots(
        1, len(coords), figsize=(5*len(coords), 3.5),
        squeeze=False, constrained_layout=True
    )
    for i, ax in enumerate(axes[0]):
        for pred in predictions:
            ax.plot(E, pred[i], color="tab:blue", alpha=.15)
        ax.plot(E, actual[i], "k-", lw=2, label="Measured")
        ax.plot(E, mean[i], color="tab:orange", lw=2, label="Mean prediction")
        ax.fill_between(
            E, mean[i]-sd[i], mean[i]+sd[i],
            color="tab:orange", alpha=.2, label="±1 model SD"
        )
        ax.set(
            xlabel="Energy", ylabel="Intensity",
            title=f"Location {tuple(coords[i])}"
        )
        ax.legend(fontsize=8)
    fig.suptitle(title)
    plt.show()


# ============================================================
# 9. EVOLUTION
# ============================================================
population = [MEAN, MAX, full_pixels(), ("SPLIT", MEAN, MAX, DROP, MEAN)]
while len(population) < POPULATION:
    tree = random_tree()
    if active_leaves(tree) and tree not in population:
        population.append(tree)

history, generation_tables, error_maps, error_records = [], [], [], []

for generation in range(1, GENERATIONS+1):
    before = len(archive)
    for i, tree in enumerate(population, 1):
        evaluate(tree, generation)
        if i % 5 == 0:
            print(f"Generation {generation}: evaluated {i}/{POPULATION}")

    current = rank_population(table(population))
    survivors = select_survivors(population)
    current["survives"] = [t in set(survivors) for t in current.tree]
    current["generation"] = generation
    generation_tables.append(current.copy())

    best_four = current.sort_values(["val_NRMSE", "features"]).head(4)
    winner = best_four.iloc[0]
    winner_errors = archive[winner.tree]["all_pixel_rmse"]
    error_maps.append(to_map(winner_errors))
    error_records.append({
        "generation": generation, "descriptor_id": int(winner["id"]),
        "features": int(winner.features), "val_NRMSE": winner.val_NRMSE,
        "all_point_RMSE": np.sqrt(np.mean(winner_errors**2)),
        "tree": winner.tree
    })
    history.append({
        "generation": generation,
        "best_NRMSE": current.val_NRMSE.min(),
        "median_NRMSE": current.val_NRMSE.median(),
        "mean_features": current.features.mean()
    })

    print(f"\nGeneration {generation}: {len(archive)-before} new models")
    display(current[COLUMNS+["rank", "survives"]].sort_values("val_NRMSE"))
    plot_layouts(best_four, f"Generation {generation}: four best descriptors")
    plot_spectra(
        Y_all[example_ids],
        np.stack([archive[t]["example_predictions"] for t in best_four.tree]),
        all_coordinates[example_ids],
        f"Generation {generation}: original validation patches"
    )

    results = table()
    curve = front_curve(results)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    axes[0].scatter(results.features, results.val_NRMSE, alpha=.25, label="Archive")
    axes[0].scatter(
        current.features, current.val_NRMSE,
        facecolors="none", edgecolors="orange", label="Current"
    )
    axes[0].plot(curve.features, curve.val_NRMSE, "k.-", label="Pareto")
    axes[0].set(xlabel="Descriptor information count", ylabel="Validation NRMSE")
    axes[0].legend()
    axes[0].grid(alpha=.2)

    vmax = max(max(np.nanmax(m) for m in error_maps), 1e-12)
    im = axes[1].imshow(error_maps[-1], cmap="magma", vmin=0, vmax=vmax)
    axes[1].set_title(f"All-point spectral RMSE: generation {generation}")
    fig.colorbar(im, ax=axes[1])
    plt.show()

    if generation < GENERATIONS:
        population = breed(survivors)

ga_results = table()
ga_history = pd.DataFrame(history)
ga_generations = pd.concat(generation_tables, ignore_index=True)
error_map_history = np.stack(error_maps)
error_map_metadata = pd.DataFrame(error_records)


# ============================================================
# 10. ERROR HISTORY AND PCA
# ============================================================
def analyze_error_history(stack):
    cols = min(5, len(stack))
    rows = int(np.ceil(len(stack)/cols))
    fig, axes = plt.subplots(
        rows, cols, figsize=(3*cols, 3*rows),
        squeeze=False, constrained_layout=True
    )
    vmax = max(float(np.nanmax(stack)), 1e-12)
    for i, ax in enumerate(axes.ravel()):
        if i >= len(stack):
            ax.axis("off")
            continue
        im = ax.imshow(stack[i], cmap="magma", vmin=0, vmax=vmax)
        ax.set_title(f"Generation {i+1}")
        ax.axis("off")
    fig.colorbar(im, ax=list(axes.ravel()), label="All-point RMSE")
    plt.show()

    valid = np.isfinite(stack).all(axis=0)
    if len(stack) < 2 or valid.sum() < 2:
        return None
    X = stack[:, valid].T
    mean = X.mean(axis=0)
    if np.sum((X-mean)**2) == 0:
        return None

    pca = PCA(
        n_components=min(PCA_COMPONENTS, X.shape[0]-1, X.shape[1]),
        svd_solver="full"
    )
    scores = pca.fit_transform(X)
    s = pca.singular_values_
    keep = s > np.finfo(float).eps*max(X.shape)*s[0]
    s = s[keep]
    if not len(s):
        return None

    spatial = scores[:, keep]/s[None, :]
    amplitude = pca.components_[keep].T*s[None, :]
    explained = pca.explained_variance_ratio_[keep]

    for j in range(len(s)):
        if spatial[np.argmax(np.abs(spatial[:, j])), j] < 0:
            spatial[:, j] *= -1
            amplitude[:, j] *= -1

    maps = np.full((len(s), H, W), np.nan)
    for j in range(len(s)):
        maps[j, valid] = spatial[:, j]

    fig, ax = plt.subplots(figsize=(7, 3))
    ax.plot(np.arange(1, len(stack)+1), mean, "o-")
    ax.set(xlabel="Generation", ylabel="Mean pixel RMSE")
    plt.show()

    fig, axes = plt.subplots(
        len(s), 2, figsize=(11, 3.5*len(s)),
        squeeze=False, constrained_layout=True
    )
    for j in range(len(s)):
        axes[j, 0].plot(np.arange(1, len(stack)+1), amplitude[:, j], "o-")
        axes[j, 0].set(
            xlabel="Generation", ylabel="Amplitude",
            title=f"Component {j+1}: {100*explained[j]:.1f}% variance"
        )
        limit = np.nanmax(np.abs(maps[j]))
        im = axes[j, 1].imshow(maps[j], cmap="RdBu_r", vmin=-limit, vmax=limit)
        axes[j, 1].set_title(f"Spatial loading {j+1}")
        fig.colorbar(im, ax=axes[j, 1])
    plt.show()

    reconstruction = np.full_like(stack, np.nan)
    reconstruction[:, valid] = mean[:, None]+amplitude@spatial.T
    return {
        "pca": pca, "mean_error_by_generation": mean,
        "generation_amplitudes": amplitude,
        "spatial_loadings": maps,
        "explained_variance_ratio": explained,
        "reconstructed_error_maps": reconstruction
    }


error_pca = analyze_error_history(error_map_history)


# ============================================================
# 11. PARETO FRONT, KNEE, AND HIGH-COMPLEXITY DESCRIPTORS
# ============================================================
pareto_descriptors = all_pareto(ga_results)
pareto_curve = front_curve(ga_results)
knee_index = None

if (
    len(pareto_curve) >= 3
    and np.ptp(pareto_curve.features) > 0
    and np.ptp(pareto_curve.val_NRMSE) > 0
):
    x = (
        (pareto_curve.features.to_numpy()-pareto_curve.features.min())
        / np.ptp(pareto_curve.features)
    )
    y = (
        (pareto_curve.val_NRMSE.to_numpy()-pareto_curve.val_NRMSE.min())
        / np.ptp(pareto_curve.val_NRMSE)
    )
    score = (1-x-y)/np.sqrt(2)
    candidate = int(np.argmax(score[1:-1]))+1
    if score[candidate] > 1e-6:
        knee_index = candidate

knee_ids = []
if knee_index is not None:
    knee = pareto_curve.iloc[knee_index]
    knee_ids = pareto_descriptors.loc[
        (pareto_descriptors.features == knee.features)
        & (pareto_descriptors.val_NRMSE == knee.val_NRMSE), "id"
    ].astype(int).tolist()

pareto_descriptors["estimated_knee"] = pareto_descriptors["id"].isin(knee_ids)
display(pareto_descriptors[COLUMNS+["estimated_knee"]])

steps = []
for i in range(len(pareto_curve)-1):
    small, large = pareto_curve.iloc[i], pareto_curve.iloc[i+1]
    removed = int(large.features-small.features)
    increase = float(small.val_NRMSE-large.val_NRMSE)
    steps.append({
        "from_id": int(large["id"]), "to_id": int(small["id"]),
        "features_removed": removed, "NRMSE_increase": increase,
        "increase_per_removed_feature": increase/removed
    })
pareto_error_steps = pd.DataFrame(steps)
display(pareto_error_steps)

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(ga_results.features, ga_results.val_NRMSE, alpha=.2)
ax.plot(pareto_curve.features, pareto_curve.val_NRMSE, "ko-", label="Pareto")
for _, row in pareto_curve.iterrows():
    ax.annotate(
        str(int(row["id"])), (row.features, row.val_NRMSE),
        xytext=(4, 5), textcoords="offset points", fontsize=8
    )
if knee_index is not None:
    knee = pareto_curve.iloc[knee_index]
    simpler = pareto_curve.iloc[knee_index-1]
    ax.scatter(
        [knee.features], [knee.val_NRMSE], s=220,
        marker="*", color="red", label="Estimated knee"
    )
    ax.plot(
        [simpler.features, knee.features],
        [simpler.val_NRMSE, knee.val_NRMSE], "r-", lw=3
    )
    display(pareto_curve.iloc[knee_index-1:knee_index+2][COLUMNS])
else:
    print("No distinct interior knee detected.")
ax.set(xlabel="Descriptor information count", ylabel="Validation NRMSE")
ax.legend()
ax.grid(alpha=.2)
plt.show()

plot_layouts(pareto_descriptors, "All Pareto descriptors", knee_ids)

high_complexity_descriptors = ga_results.sort_values(
    ["features", "val_NRMSE"], ascending=[False, True]
).head(N_HIGH_COMPLEXITY)
display(high_complexity_descriptors[COLUMNS])
plot_layouts(high_complexity_descriptors, "Highest-complexity evaluated descriptors")

high_complexity_pareto = pareto_descriptors.sort_values(
    ["features", "val_NRMSE"], ascending=[False, True]
).head(8)
plot_layouts(high_complexity_pareto, "High-complexity Pareto descriptors", knee_ids)

pareto_trees = {
    int(row["id"]): row.tree for _, row in pareto_descriptors.iterrows()
}

fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
axes[0].plot(ga_history.generation, ga_history.best_NRMSE, "o-", label="Best")
axes[0].plot(ga_history.generation, ga_history.median_NRMSE, "o-", label="Median")
axes[0].set(xlabel="Generation", ylabel="Validation NRMSE")
axes[0].legend()
axes[1].plot(ga_history.generation, ga_history.mean_features, "o-")
axes[1].set(xlabel="Generation", ylabel="Mean information count")
plt.show()


# ============================================================
# 12. FINAL x10 REFIT AND ALL-POINT EVALUATION
# ============================================================
final_four = ga_results.sort_values(["val_NRMSE", "features", "id"]).head(4)
selected_tree = final_four.iloc[0].tree
test_example_ids = test_ids[
    spread_indices(all_coordinates[test_ids], N_EXAMPLES)
]

final_models, final_examples = [], []
ensemble_prediction = np.zeros_like(Y_all, dtype=float)
best_all_prediction = None

for j, tree in enumerate(final_four.tree):
    # Uses fit + validation originals and all ten variants per source.
    bundle = fit_cnn(tree, train_ids)
    prediction = predict_cnn(bundle, P_all)
    final_models.append(bundle)
    final_examples.append(prediction[test_example_ids])
    ensemble_prediction += prediction/len(final_four)
    if j == 0:
        best_all_prediction = prediction.copy()

final_model = final_models[0]
test_baseline = np.sqrt(np.mean(
    (Y_all[test_ids]-Y_all[train_ids].mean(axis=0, dtype=float))**2
))
subsets = {
    "Fit": fit_ids, "Validation": val_ids, "Test": test_ids,
    "All valid centers": np.arange(len(P_all))
}

summary = []
for name, pred in [
    ("Best descriptor", best_all_prediction),
    ("Mean of four", ensemble_prediction)
]:
    for subset, ids in subsets.items():
        rmse = np.sqrt(np.mean((pred[ids]-Y_all[ids])**2))
        summary.append({
            "predictor": name, "subset": subset, "locations": len(ids),
            "RMSE": rmse,
            "test_NRMSE": (
                rmse/test_baseline
                if subset == "Test" and test_baseline > 0 else np.nan
            )
        })
rmse_summary = pd.DataFrame(summary)
display(rmse_summary)

plot_spectra(
    Y_all[test_example_ids], np.stack(final_examples),
    all_coordinates[test_example_ids], "Final CNNs: original test patches"
)

final_error_map = to_map(np.sqrt(np.mean(
    (best_all_prediction-Y_all)**2, axis=1
)))
ensemble_error_map = to_map(np.sqrt(np.mean(
    (ensemble_prediction-Y_all)**2, axis=1
)))

fig, axes = plt.subplots(1, 3, figsize=(16, 4), constrained_layout=True)
for name, pred in [
    ("Best descriptor", best_all_prediction),
    ("Mean of four", ensemble_prediction)
]:
    axes[0].plot(
        E, np.sqrt(np.mean((pred[test_ids]-Y_all[test_ids])**2, axis=0)),
        label=name
    )
axes[0].set(xlabel="Energy", ylabel="Test RMSE")
axes[0].legend()

vmax = max(np.nanmax(final_error_map), np.nanmax(ensemble_error_map), 1e-12)
for ax, error_map, title in [
    (axes[1], final_error_map, "Best CNN: all points"),
    (axes[2], ensemble_error_map, "Mean of four CNNs: all points")
]:
    im = ax.imshow(error_map, cmap="magma", vmin=0, vmax=vmax)
    ax.set_title(title)
fig.colorbar(im, ax=list(axes[1:]), label="Spectral RMSE")
plt.show()

feature_definitions = pd.DataFrame([
    {
        "feature": i+1, "operation": op,
        "row_start": region[0], "column_start": region[1],
        "region_size": region[2]
    }
    for i, (region, op) in enumerate(active_leaves(selected_tree))
])
display(feature_definitions)

print(
    "CNN parameters:",
    sum(p.numel() for p in final_model["model"].parameters())
)

# Predict additional original patches:
# new_spectra = predict_cnn(final_model, new_patches)

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.patches import Rectangle
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from IPython.display import display

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader


# ============================================================
# 1. SETTINGS
# Start after loading loadedfile, or image / spectra / energy.
# ============================================================
DATASET_KEY = "2"

N = 12
POPULATION, GENERATIONS = 50, 50
SURVIVORS, TOURNAMENT_SIZE = 20, 5
CROSSOVER_RATE, IMMIGRANT_RATE = 0.8, 0.15
SEED = 0

TEST_FRACTION = 0.25
VALIDATION_FRACTION = 0.25
MIN_FIT, MIN_VAL, MIN_TEST = 20, 15, 15

AUGMENTATION_FACTOR = 10
EPOCHS = 8
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
CHANNELS = (8, 16)

N_EXAMPLES = 3
PCA_COMPONENTS = 3
N_HIGH_COMPLEXITY = 12

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(2)

rng = np.random.default_rng(SEED)
aug_rng = np.random.default_rng(SEED + 1000)

if "loadedfile" in globals():
    data = loadedfile[DATASET_KEY]
    img = np.asarray(data["image"], dtype=float)
    cube = np.asarray(data["spectrum image"], dtype=float)
    E = np.asarray(data["energy axis"], dtype=float).ravel()
else:
    img = np.asarray(image, dtype=float)
    cube = np.asarray(spectra, dtype=float)
    E = np.asarray(energy, dtype=float).ravel()

if img.ndim != 2 or cube.shape != (*img.shape, len(E)):
    raise ValueError("Expected image (H,W), spectra (H,W,E), energy (E,).")
if not all(np.isfinite(v).all() for v in (img, cube, E)):
    raise ValueError("Inputs contain nonfinite values.")
if not 0 < VALIDATION_FRACTION < 1 - TEST_FRACTION < 1:
    raise ValueError("Invalid split fractions.")
if not 2 <= SURVIVORS < POPULATION or POPULATION < 4:
    raise ValueError("Invalid GA population settings.")
if GENERATIONS < 1 or EPOCHS < 1:
    raise ValueError("GENERATIONS and EPOCHS must be positive.")
if AUGMENTATION_FACTOR != 10:
    raise ValueError("This augmentation setup uses exactly ten samples per source.")

order = np.argsort(E)
E, cube = E[order], cube[..., order]
if np.any(np.diff(E) <= 0):
    raise ValueError("Energy values must be distinct.")

H, W = img.shape
a, b = (N - 1) // 2, N - (N - 1) // 2


# ============================================================
# 2. ALL VALID PATCHES AND FIXED RANDOM SPLITS
# ============================================================
all_coordinates = np.array([
    (y, x)
    for y in range(a, H - b + 1)
    for x in range(a, W - b + 1)
], dtype=int).reshape(-1, 2)

if len(all_coordinates) < MIN_FIT + MIN_VAL + MIN_TEST:
    raise ValueError("Too few valid 12×12 patches.")

P_all = np.stack([
    img[y-a:y+b, x-a:x+b] for y, x in all_coordinates
]).astype(np.float32)

Y_all = cube[
    all_coordinates[:, 0], all_coordinates[:, 1]
].astype(np.float32)

train_ids, test_ids = train_test_split(
    np.arange(len(P_all)),
    test_size=TEST_FRACTION,
    random_state=SEED
)
fit_ids, val_ids = train_test_split(
    train_ids,
    test_size=VALIDATION_FRACTION / (1 - TEST_FRACTION),
    random_state=SEED + 1
)

if (
    len(fit_ids) < MIN_FIT
    or len(val_ids) < MIN_VAL
    or len(test_ids) < MIN_TEST
):
    raise ValueError("Insufficient fitting/validation/test locations.")

baseline = np.sqrt(np.mean(
    (
        Y_all[val_ids].astype(float)
        - Y_all[fit_ids].mean(axis=0, dtype=float)
    )**2
))
if baseline <= 0:
    raise ValueError("Zero validation baseline error.")


def spread_indices(coords, count):
    count = min(count, len(coords))
    chosen = [
        int(np.argmin(np.sum((coords - coords.mean(axis=0))**2, axis=1)))
    ]
    distance = np.full(len(coords), np.inf)
    while len(chosen) < count:
        distance = np.minimum(
            distance,
            np.sum((coords - coords[chosen[-1]])**2, axis=1)
        )
        distance[chosen] = -1
        chosen.append(int(np.argmax(distance)))
    return np.asarray(chosen)


example_ids = val_ids[
    spread_indices(all_coordinates[val_ids], N_EXAMPLES)
]

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
for ax in axes:
    ax.imshow(img, cmap="gray", alpha=.45)
    ax.set(xlabel="Column", ylabel="Row")

for ids, color, label in [
    (train_ids, "blue", "Train: fit + validation"),
    (test_ids, "red", "Test")
]:
    xy = all_coordinates[ids]
    axes[0].scatter(
        xy[:, 1], xy[:, 0], s=10, c=color,
        linewidths=0, label=f"{label}: {len(ids)}"
    )

for ids, color, label in [
    (fit_ids, "blue", "Fit"), (val_ids, "orange", "Validation")
]:
    xy = all_coordinates[ids]
    axes[1].scatter(
        xy[:, 1], xy[:, 0], s=10, c=color,
        linewidths=0, label=f"{label}: {len(ids)}"
    )

axes[0].set_title("Random training/test split")
axes[1].set_title("Inner fitting/validation split")
for ax in axes:
    ax.legend(fontsize=8)
plt.show()


# ============================================================
# 3. x10 AUGMENTATION
# Eight unique square symmetries + two random nonidentity repeats.
# ============================================================
TRANSFORM_NAMES = [
    "Identity", "90° CCW", "180°", "270° CCW",
    "Horizontal-axis mirror", "Vertical-axis mirror",
    "Main-diagonal mirror", "Other-diagonal mirror"
]


def transform_patch(patch, code):
    if code < 4:
        return np.rot90(patch, code)
    if code == 4:
        return np.flipud(patch)
    if code == 5:
        return np.fliplr(patch)
    if code == 6:
        return patch.T
    return np.flip(patch.T, axis=(0, 1))


# Original sources were split BEFORE augmentation.
aug_sources = np.repeat(train_ids, AUGMENTATION_FACTOR)
transform_codes = np.concatenate([
    np.r_[np.arange(8), aug_rng.choice(np.arange(1, 8), 2, replace=False)]
    for _ in train_ids
]).astype(int)

P_aug = np.stack([
    transform_patch(P_all[source], int(code))
    for source, code in zip(aug_sources, transform_codes)
]).astype(np.float32)

augmentation_metadata = pd.DataFrame({
    "source_id": aug_sources,
    "transform": [TRANSFORM_NAMES[k] for k in transform_codes],
    "used_during_evolution": np.isin(aug_sources, fit_ids)
})

print(f"Device: {DEVICE}")
print(f"Fit={len(fit_ids)}, validation={len(val_ids)}, test={len(test_ids)}")
print(f"Evolution fitting examples: {10 * len(fit_ids)}")
print(f"Final fitting examples: {10 * len(train_ids)}")
print("Minimum descriptor: four active quadrants (2×2 representation).")

source = fit_ids[0]
positions = np.flatnonzero(aug_sources == source)
fig, axes = plt.subplots(2, 5, figsize=(13, 5), constrained_layout=True)
for ax, idx in zip(axes.ravel(), positions):
    ax.imshow(P_aug[idx], cmap="gray")
    ax.set_title(TRANSFORM_NAMES[transform_codes[idx]], fontsize=9)
    ax.axis("off")
fig.suptitle("Eight square symmetries plus two random repeats; spectrum unchanged")
plt.show()


# ============================================================
# 4. HIERARCHICAL DESCRIPTORS WITH MINIMUM 2×2 SUPPORT
#
# Root must always split into four nonempty 6×6 quadrants.
# Within quadrants: 6 → 3 → 1.
# DROP is permitted inside a quadrant, but not for the entire quadrant.
# ============================================================
DROP, MEAN, MAX, PIXEL = ("DROP",), ("MEAN",), ("MAX",), ("PIXEL",)
ROOT = (0, 0, N)


def child_regions(region):
    y, x, size = region
    if size == 1:
        return []
    step = size // 2 if size % 2 == 0 else 1
    return [
        (y + dy, x + dx, step)
        for dy in range(0, size, step)
        for dx in range(0, size, step)
    ]


def simplify(tree):
    if tree[0] != "SPLIT":
        return tree
    children = tuple(simplify(t) for t in tree[1:])
    return DROP if all(t == DROP for t in children) else ("SPLIT",) + children


def nodes(tree, region=ROOT, path=()):
    yield path, region, tree
    if tree[0] == "SPLIT":
        for i, (child, box) in enumerate(zip(tree[1:], child_regions(region))):
            yield from nodes(child, box, path + (i,))


def active_leaves(tree):
    return [
        (region, node[0]) for _, region, node in nodes(tree)
        if node[0] not in ("DROP", "SPLIT")
    ]


def has_information(tree):
    if tree[0] == "DROP":
        return False
    if tree[0] != "SPLIT":
        return True
    return any(has_information(t) for t in tree[1:])


def valid_descriptor(tree):
    return (
        tree[0] == "SPLIT"
        and len(tree) == 5
        and all(has_information(child) for child in tree[1:])
    )


def random_subtree(region):
    if region[2] == 1:
        return PIXEL if rng.random() < .7 else DROP

    op = str(rng.choice(
        ["DROP", "MEAN", "MAX", "SPLIT"],
        p=[.15, .25, .20, .40]
    ))
    if op != "SPLIT":
        return (op,)
    return simplify(
        ("SPLIT",) + tuple(random_subtree(r) for r in child_regions(region))
    )


def random_tree():
    children = []
    for region in child_regions(ROOT):
        child = random_subtree(region)
        if not has_information(child):
            child = MEAN if rng.random() < .5 else MAX
        children.append(child)
    return ("SPLIT",) + tuple(children)


def replace(tree, path, subtree):
    if not path:
        return simplify(subtree)
    children = list(tree[1:])
    children[path[0]] = replace(children[path[0]], path[1:], subtree)
    return simplify(("SPLIT",) + tuple(children))


def full_pixels(region=ROOT):
    if region[2] == 1:
        return PIXEL
    return ("SPLIT",) + tuple(full_pixels(r) for r in child_regions(region))


MINIMUM_MEAN = ("SPLIT", MEAN, MEAN, MEAN, MEAN)
MINIMUM_MAX = ("SPLIT", MAX, MAX, MAX, MAX)


# ============================================================
# 5. SPATIAL ENCODING AND TWO-HIDDEN-LAYER CNN
#
# Channel 0: normalized retained/pooled intensity.
# Channels 1–3: PIXEL / MEAN / MAX indicators.
# DROP: zero in all channels.
#
# A pooled value fills its region; no discarded detail is restored.
# ============================================================
def encode(tree, patches, mean, scale):
    result = np.zeros((len(patches), 4, N, N), dtype=np.float32)
    operation_channel = {"PIXEL": 1, "MEAN": 2, "MAX": 3}

    for (y, x, size), op in active_leaves(tree):
        region = patches[:, y:y+size, x:x+size]
        values = (
            region.max(axis=(1, 2))
            if op == "MAX" else region.mean(axis=(1, 2))
        )
        result[:, 0, y:y+size, x:x+size] = (
            (values - mean) / scale
        )[:, None, None]
        result[:, operation_channel[op], y:y+size, x:x+size] = 1

    return result


class SpectrumCNN(nn.Module):
    def __init__(self, n_energy):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv2d(4, CHANNELS[0], kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AvgPool2d(2),  # 12 → 6
            nn.Conv2d(CHANNELS[0], CHANNELS[1], kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AvgPool2d(2),  # 6 → 3
            nn.Flatten(),
            nn.Linear(CHANNELS[1] * 3 * 3, n_energy)
        )

    def forward(self, x):
        return self.network(x)


def fit_cnn(tree, original_ids):
    if not valid_descriptor(tree):
        raise ValueError("Descriptor must retain information in all four quadrants.")

    random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

    selected = np.isin(aug_sources, original_ids)

    x_mean = float(P_all[original_ids].mean(dtype=np.float64))
    x_scale = max(float(P_all[original_ids].std(dtype=np.float64)), 1e-12)

    # One global spectral scale preserves raw-MSE channel weighting.
    y_mean = Y_all[original_ids].mean(axis=0, dtype=np.float64)
    y_scale = max(float(np.sqrt(np.mean(
        (Y_all[original_ids] - y_mean)**2
    ))), 1e-12)

    X = encode(tree, P_aug[selected], x_mean, x_scale)
    Y = (
        (Y_all[aug_sources[selected]] - y_mean) / y_scale
    ).astype(np.float32)

    loader = DataLoader(
        TensorDataset(torch.from_numpy(X), torch.from_numpy(Y)),
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
        num_workers=0
    )

    model = SpectrumCNN(len(E)).to(DEVICE)
    optimizer = torch.optim.Adam(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    loss_function = nn.MSELoss()

    model.train()
    for _ in range(EPOCHS):
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = loss_function(model(xb), yb)
            if not torch.isfinite(loss):
                raise RuntimeError("Nonfinite CNN loss.")
            loss.backward()
            optimizer.step()

    return {
        "model": model.cpu(),
        "tree": tree,
        "x_mean": x_mean,
        "x_scale": x_scale,
        "y_mean": y_mean,
        "y_scale": y_scale
    }


def predict_cnn(bundle, patches):
    model = bundle["model"].to(DEVICE).eval()
    outputs = []

    with torch.no_grad():
        for start in range(0, len(patches), BATCH_SIZE):
            X = encode(
                bundle["tree"], patches[start:start+BATCH_SIZE],
                bundle["x_mean"], bundle["x_scale"]
            )
            prediction = model(torch.from_numpy(X).to(DEVICE))
            outputs.append(prediction.cpu().numpy())

    model.cpu()
    return np.concatenate(outputs) * bundle["y_scale"] + bundle["y_mean"]


# ============================================================
# 6. VALIDATION FITNESS AND ALL-POINT ERROR CACHE
# ============================================================
archive = {}


def evaluate(tree, generation):
    if tree in archive:
        return

    bundle = fit_cnn(tree, fit_ids)
    prediction = predict_cnn(bundle, P_all)
    errors = np.sqrt(np.mean((prediction - Y_all)**2, axis=1))
    score = np.sqrt(np.mean(errors[val_ids]**2)) / baseline

    if not np.isfinite(score):
        raise RuntimeError("Nonfinite validation score.")

    leaves = active_leaves(tree)
    archive[tree] = {
        "id": len(archive) + 1,
        "features": len(leaves),
        "source_pixels": sum(r[2]**2 for r, _ in leaves),
        "mean_regions": sum(op == "MEAN" for _, op in leaves),
        "max_regions": sum(op == "MAX" for _, op in leaves),
        "individual_pixels": sum(op == "PIXEL" for _, op in leaves),
        "val_NRMSE": score,
        "first_generation": generation,
        "all_pixel_rmse": errors,
        "example_predictions": prediction[example_ids]
    }


def table(trees=None):
    trees = list(archive) if trees is None else trees
    excluded = {"all_pixel_rmse", "example_predictions"}
    return pd.DataFrame([
        {"tree": t, **{k: v for k, v in archive[t].items() if k not in excluded}}
        for t in trees
    ])


def to_map(values):
    result = np.full((H, W), np.nan)
    result[all_coordinates[:, 0], all_coordinates[:, 1]] = values
    return result


# ============================================================
# 7. MUTATION AND CROSSOVER
# Root cannot be coarsened, dropped, or replaced.
# Mutations that empty a quadrant are rejected.
# ============================================================
def mutate(tree):
    items = [item for item in nodes(tree) if item[0]]  # Exclude root.

    eligible = {
        "refine": [
            v for v in items
            if v[1][2] > 1 and v[2][0] in ("MEAN", "MAX")
        ],
        "coarsen": [v for v in items if v[2][0] == "SPLIT"],
        "pool": [v for v in items if v[2][0] in ("MEAN", "MAX")],
        # A whole quadrant cannot be dropped.
        "drop": [
            v for v in items
            if len(v[0]) >= 2 and v[2][0] != "DROP"
        ],
        "activate": [v for v in items if v[2][0] == "DROP"]
    }

    weights = dict(refine=.25, coarsen=.25, pool=.20, drop=.15, activate=.15)
    choices = [name for name in eligible if eligible[name]]
    probability = np.array([weights[name] for name in choices])
    action = str(rng.choice(choices, p=probability / probability.sum()))

    options = eligible[action]
    path, region, node = options[int(rng.integers(len(options)))]

    if action == "refine":
        replacement = ("SPLIT",) + tuple(
            PIXEL if r[2] == 1 else node for r in child_regions(region)
        )
    elif action == "coarsen":
        replacement = MEAN if rng.random() < .5 else MAX
    elif action == "pool":
        replacement = MAX if node == MEAN else MEAN
    elif action == "drop":
        replacement = DROP
    else:
        replacement = (
            PIXEL if region[2] == 1
            else MEAN if rng.random() < .5 else MAX
        )

    child = replace(tree, path, replacement)
    return child if valid_descriptor(child) else tree


def crossover(parent1, parent2):
    first = {p: t for p, _, t in nodes(parent1)}
    second = {p: t for p, _, t in nodes(parent2)}
    common = sorted((set(first) & set(second)) - {()})

    if not common:
        return parent1

    path = common[int(rng.integers(len(common)))]
    child = replace(parent1, path, second[path])
    return child if valid_descriptor(child) else parent1


# ============================================================
# 8. PARETO RANKING AND TOURNAMENT SELECTION
# ============================================================
def rank_population(df):
    df = df.reset_index(drop=True).copy()
    values = df[["features", "val_NRMSE"]].to_numpy()
    ranks = np.zeros(len(df), dtype=int)
    crowding = np.zeros(len(df))
    remaining, rank = list(range(len(df))), 0

    while remaining:
        front = [
            i for i in remaining if not any(
                np.all(values[j] <= values[i])
                and np.any(values[j] < values[i])
                for j in remaining if j != i
            )
        ]
        ranks[front] = rank

        for column in range(2):
            ordered = sorted(front, key=lambda i: values[i, column])
            span = values[ordered[-1], column] - values[ordered[0], column]
            if span == 0:
                continue
            crowding[ordered[0]] = crowding[ordered[-1]] = np.inf
            for k in range(1, len(ordered) - 1):
                crowding[ordered[k]] += (
                    values[ordered[k+1], column]
                    - values[ordered[k-1], column]
                ) / span

        remaining = [i for i in remaining if i not in front]
        rank += 1

    df["rank"], df["crowding"] = ranks, crowding
    return df


def all_pareto(df):
    values = df[["features", "val_NRMSE"]].to_numpy()
    keep = [
        not np.any(
            np.all(values <= v, axis=1) & np.any(values < v, axis=1)
        )
        for v in values
    ]
    return df.loc[keep].sort_values(
        ["features", "val_NRMSE", "id"]
    ).reset_index(drop=True)


def front_curve(df):
    return all_pareto(df).drop_duplicates(
        ["features", "val_NRMSE"]
    ).reset_index(drop=True)


def select_survivors(population):
    df = rank_population(table(population))

    elite = int(df.sort_values(["val_NRMSE", "features"]).index[0])
    chosen = [elite]
    available = [i for i in df.index if i != elite]

    while len(chosen) < SURVIVORS:
        contestants = rng.choice(
            available, min(TOURNAMENT_SIZE, len(available)), replace=False
        )
        winner = int(min(
            contestants,
            key=lambda i: (df.loc[i, "rank"], -df.loc[i, "crowding"])
        ))
        chosen.append(winner)
        available.remove(winner)

    return df.loc[chosen, "tree"].tolist()


def breed(survivors):
    population, seen, attempts = list(survivors), set(survivors), 0

    while len(population) < POPULATION:
        attempts += 1
        if rng.random() < IMMIGRANT_RATE or attempts > 200:
            child = random_tree()
        else:
            i, j = rng.integers(len(survivors), size=2)
            child = survivors[i]
            if rng.random() < CROSSOVER_RATE:
                child = crossover(child, survivors[j])
            child = mutate(child)

        if valid_descriptor(child) and child not in seen:
            population.append(child)
            seen.add(child)

    return population


# ============================================================
# 9. VISUALIZATION HELPERS
# ============================================================
COLORS = dict(DROP="#eeeeee", MEAN="#56b4e9", MAX="#e69f00", PIXEL="#009e73")
COLUMNS = [
    "id", "features", "source_pixels", "mean_regions",
    "max_regions", "individual_pixels", "val_NRMSE"
]


def draw_tree(ax, tree):
    for _, (y, x, size), node in nodes(tree):
        op = node[0]
        if op == "SPLIT":
            continue

        ax.add_patch(Rectangle(
            (x-.5, y-.5), size, size,
            facecolor=COLORS[op], edgecolor="white", linewidth=.7
        ))
        if size > 1:
            ax.text(
                x+(size-1)/2, y+(size-1)/2,
                {"DROP": "–", "MEAN": "mean", "MAX": "max"}[op],
                ha="center", va="center", fontsize=8
            )

    # Show the mandatory 2×2 top-level partition.
    ax.axvline(N/2 - .5, color="black", linewidth=.8)
    ax.axhline(N/2 - .5, color="black", linewidth=.8)
    ax.plot(a, a, "rx", ms=9, mew=2)
    ax.set(xlim=(-.5, N-.5), ylim=(N-.5, -.5), aspect="equal")
    ax.axis("off")


def plot_layouts(rows, title, knee_ids=()):
    for start in range(0, len(rows), 12):
        batch = rows.iloc[start:start+12]
        cols = min(4, len(batch))
        nrows = int(np.ceil(len(batch) / cols))

        fig, axes = plt.subplots(
            nrows, cols, figsize=(3.5*cols, 3.4*nrows),
            squeeze=False, constrained_layout=True
        )

        for ax, (_, row) in zip(axes.ravel(), batch.iterrows()):
            draw_tree(ax, row.tree)
            knee = int(row["id"]) in knee_ids
            ax.set_title(
                f"ID {int(row['id'])}" + (" — KNEE" if knee else "")
                + f"\n{int(row.features)} features; "
                f"{int(row.source_pixels)} source pixels"
                + f"\nValidation NRMSE={row.val_NRMSE:.3f}",
                fontsize=9, color="red" if knee else "black"
            )

        for ax in axes.ravel()[len(batch):]:
            ax.axis("off")

        fig.suptitle(title + "\nBlue=mean; orange=max; green=pixel; gray=drop")
        plt.show()


def plot_spectra(actual, predictions, coords, title):
    mean, sd = predictions.mean(axis=0), predictions.std(axis=0)
    fig, axes = plt.subplots(
        1, len(coords), figsize=(5*len(coords), 3.5),
        squeeze=False, constrained_layout=True
    )

    for i, ax in enumerate(axes[0]):
        for pred in predictions:
            ax.plot(E, pred[i], color="tab:blue", alpha=.15)

        ax.plot(E, actual[i], "k-", lw=2, label="Measured")
        ax.plot(E, mean[i], color="tab:orange", lw=2, label="Mean prediction")
        ax.fill_between(
            E, mean[i]-sd[i], mean[i]+sd[i],
            color="tab:orange", alpha=.2, label="±1 model SD"
        )
        ax.set(
            xlabel="Energy", ylabel="Intensity",
            title=f"Location {tuple(coords[i])}"
        )
        ax.legend(fontsize=8)

    fig.suptitle(title)
    plt.show()


# ============================================================
# 10. EVOLUTION
# ============================================================
population = [
    MINIMUM_MEAN,
    MINIMUM_MAX,
    ("SPLIT", MEAN, MAX, MEAN, MAX),
    full_pixels()
]
while len(population) < POPULATION:
    candidate = random_tree()
    if candidate not in population:
        population.append(candidate)

assert all(valid_descriptor(t) for t in population)

history, generation_tables, error_maps, error_records = [], [], [], []

for generation in range(1, GENERATIONS + 1):
    before = len(archive)

    for i, tree in enumerate(population, 1):
        evaluate(tree, generation)
        if i % 5 == 0:
            print(f"Generation {generation}: evaluated {i}/{POPULATION}")

    current = rank_population(table(population))
    survivors = select_survivors(population)
    survivor_set = set(survivors)

    current["survives"] = [t in survivor_set for t in current.tree]
    current["generation"] = generation
    generation_tables.append(current.copy())

    best_four = current.sort_values(["val_NRMSE", "features"]).head(4)
    winner = best_four.iloc[0]
    winner_errors = archive[winner.tree]["all_pixel_rmse"]

    error_maps.append(to_map(winner_errors))
    error_records.append({
        "generation": generation,
        "descriptor_id": int(winner["id"]),
        "features": int(winner.features),
        "val_NRMSE": winner.val_NRMSE,
        "all_point_RMSE": np.sqrt(np.mean(winner_errors**2)),
        "tree": winner.tree
    })
    history.append({
        "generation": generation,
        "best_NRMSE": current.val_NRMSE.min(),
        "median_NRMSE": current.val_NRMSE.median(),
        "mean_features": current.features.mean()
    })

    print(f"\nGeneration {generation}: {len(archive)-before} new models")
    display(current[
        COLUMNS + ["rank", "survives"]
    ].sort_values(["val_NRMSE", "features"]))

    plot_layouts(best_four, f"Generation {generation}: four best descriptors")
    plot_spectra(
        Y_all[example_ids],
        np.stack([archive[t]["example_predictions"] for t in best_four.tree]),
        all_coordinates[example_ids],
        f"Generation {generation}: original validation patches"
    )

    results = table()
    curve = front_curve(results)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)

    axes[0].scatter(
        results.features, results.val_NRMSE, alpha=.25, label="Archive"
    )
    axes[0].scatter(
        current.features, current.val_NRMSE,
        facecolors="none", edgecolors="orange", label="Current"
    )
    axes[0].plot(curve.features, curve.val_NRMSE, "k.-", label="Pareto")
    axes[0].set(xlabel="Descriptor feature count", ylabel="Validation NRMSE")
    axes[0].legend()
    axes[0].grid(alpha=.2)

    vmax = max(max(np.nanmax(m) for m in error_maps), 1e-12)
    im = axes[1].imshow(
        error_maps[-1], cmap="magma", vmin=0, vmax=vmax
    )
    axes[1].set_title(f"All-point spectral RMSE: generation {generation}")
    fig.colorbar(im, ax=axes[1], label="Spectral RMSE")
    plt.show()

    if generation < GENERATIONS:
        population = breed(survivors)

ga_results = table()
ga_history = pd.DataFrame(history)
ga_generations = pd.concat(generation_tables, ignore_index=True)
error_map_history = np.stack(error_maps)
error_map_metadata = pd.DataFrame(error_records)


# ============================================================
# 11. ERROR HISTORY AND PCA
# ============================================================
def analyze_error_history(stack):
    cols = min(5, len(stack))
    rows = int(np.ceil(len(stack) / cols))
    fig, axes = plt.subplots(
        rows, cols, figsize=(3*cols, 3*rows),
        squeeze=False, constrained_layout=True
    )

    vmax = max(float(np.nanmax(stack)), 1e-12)
    for i, ax in enumerate(axes.ravel()):
        if i >= len(stack):
            ax.axis("off")
            continue
        im = ax.imshow(stack[i], cmap="magma", vmin=0, vmax=vmax)
        ax.set_title(f"Generation {i+1}")
        ax.axis("off")

    fig.colorbar(im, ax=list(axes.ravel()), label="All-point RMSE")
    plt.show()

    valid = np.isfinite(stack).all(axis=0)
    if len(stack) < 2 or valid.sum() < 2:
        print("Insufficient data for PCA.")
        return None

    # Observations=pixels; variables=generations.
    X = stack[:, valid].T
    mean = X.mean(axis=0)

    if np.sum((X-mean)**2) == 0:
        print("No spatial variation for PCA.")
        return None

    pca = PCA(
        n_components=min(PCA_COMPONENTS, X.shape[0]-1, X.shape[1]),
        svd_solver="full"
    )
    scores = pca.fit_transform(X)
    singular = pca.singular_values_
    keep = singular > np.finfo(float).eps * max(X.shape) * singular[0]
    singular = singular[keep]

    if not len(singular):
        return None

    spatial = scores[:, keep] / singular[None, :]
    amplitude = pca.components_[keep].T * singular[None, :]
    explained = pca.explained_variance_ratio_[keep]

    for j in range(len(singular)):
        if spatial[np.argmax(np.abs(spatial[:, j])), j] < 0:
            spatial[:, j] *= -1
            amplitude[:, j] *= -1

    maps = np.full((len(singular), H, W), np.nan)
    for j in range(len(singular)):
        maps[j, valid] = spatial[:, j]

    fig, ax = plt.subplots(figsize=(7, 3))
    ax.plot(np.arange(1, len(stack)+1), mean, "o-")
    ax.set(
        xlabel="Generation", ylabel="Mean pixel RMSE",
        title="Spatial mean removed before PCA"
    )
    plt.show()

    fig, axes = plt.subplots(
        len(singular), 2,
        figsize=(11, 3.5*len(singular)),
        squeeze=False, constrained_layout=True
    )
    for j in range(len(singular)):
        axes[j, 0].plot(
            np.arange(1, len(stack)+1), amplitude[:, j], "o-"
        )
        axes[j, 0].set(
            xlabel="Generation", ylabel="Amplitude",
            title=f"Component {j+1}: {100*explained[j]:.1f}% variance"
        )
        limit = np.nanmax(np.abs(maps[j]))
        im = axes[j, 1].imshow(
            maps[j], cmap="RdBu_r", vmin=-limit, vmax=limit
        )
        axes[j, 1].set_title(f"Spatial loading {j+1}")
        fig.colorbar(im, ax=axes[j, 1])

    plt.show()

    reconstruction = np.full_like(stack, np.nan)
    reconstruction[:, valid] = mean[:, None] + amplitude @ spatial.T

    return {
        "pca": pca,
        "mean_error_by_generation": mean,
        "generation_amplitudes": amplitude,
        "spatial_loadings": maps,
        "explained_variance_ratio": explained,
        "reconstructed_error_maps": reconstruction
    }


error_pca = analyze_error_history(error_map_history)


# ============================================================
# 12. PARETO FRONT, KNEE, AND HIGH-COMPLEXITY LAYOUTS
# ============================================================
pareto_descriptors = all_pareto(ga_results)
pareto_curve = front_curve(ga_results)
knee_index = None

if (
    len(pareto_curve) >= 3
    and np.ptp(pareto_curve.features) > 0
    and np.ptp(pareto_curve.val_NRMSE) > 0
):
    x = (
        (pareto_curve.features.to_numpy() - pareto_curve.features.min())
        / np.ptp(pareto_curve.features)
    )
    y = (
        (pareto_curve.val_NRMSE.to_numpy() - pareto_curve.val_NRMSE.min())
        / np.ptp(pareto_curve.val_NRMSE)
    )
    score = (1-x-y) / np.sqrt(2)
    candidate = int(np.argmax(score[1:-1])) + 1
    if score[candidate] > 1e-6:
        knee_index = candidate

knee_ids = []
if knee_index is not None:
    knee = pareto_curve.iloc[knee_index]
    knee_ids = pareto_descriptors.loc[
        (pareto_descriptors.features == knee.features)
        & (pareto_descriptors.val_NRMSE == knee.val_NRMSE), "id"
    ].astype(int).tolist()

pareto_descriptors["estimated_knee"] = pareto_descriptors["id"].isin(knee_ids)

print("\nALL PARETO DESCRIPTORS")
display(pareto_descriptors[COLUMNS + ["estimated_knee"]])

steps = []
for i in range(len(pareto_curve) - 1):
    small, large = pareto_curve.iloc[i], pareto_curve.iloc[i+1]
    removed = int(large.features - small.features)
    increase = float(small.val_NRMSE - large.val_NRMSE)
    steps.append({
        "from_id": int(large["id"]),
        "to_id": int(small["id"]),
        "from_features": int(large.features),
        "to_features": int(small.features),
        "features_removed": removed,
        "NRMSE_increase": increase,
        "increase_per_removed_feature": increase / removed
    })

pareto_error_steps = pd.DataFrame(steps)
display(pareto_error_steps)

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(ga_results.features, ga_results.val_NRMSE, alpha=.2)
ax.plot(
    pareto_curve.features, pareto_curve.val_NRMSE, "ko-", label="Pareto"
)
for _, row in pareto_curve.iterrows():
    ax.annotate(
        str(int(row["id"])), (row.features, row.val_NRMSE),
        xytext=(4, 5), textcoords="offset points", fontsize=8
    )

if knee_index is not None:
    knee = pareto_curve.iloc[knee_index]
    simpler = pareto_curve.iloc[knee_index-1]
    ax.scatter(
        [knee.features], [knee.val_NRMSE],
        s=220, marker="*", color="red", label="Estimated knee"
    )
    ax.plot(
        [simpler.features, knee.features],
        [simpler.val_NRMSE, knee.val_NRMSE], "r-", lw=3
    )
    print("KNEE AND ADJACENT CONFIGURATIONS")
    display(pareto_curve.iloc[knee_index-1:knee_index+2][COLUMNS])
else:
    print("No distinct interior knee detected.")

ax.set(xlabel="Descriptor feature count", ylabel="Validation NRMSE")
ax.legend()
ax.grid(alpha=.2)
plt.show()

plot_layouts(pareto_descriptors, "All Pareto descriptors", knee_ids)

high_complexity_descriptors = ga_results.sort_values(
    ["features", "val_NRMSE"], ascending=[False, True]
).head(N_HIGH_COMPLEXITY)

print("\nHIGHEST-COMPLEXITY EVALUATED DESCRIPTORS")
display(high_complexity_descriptors[COLUMNS])
plot_layouts(
    high_complexity_descriptors,
    "Highest-complexity evaluated descriptors"
)

high_complexity_pareto = pareto_descriptors.sort_values(
    ["features", "val_NRMSE"], ascending=[False, True]
).head(8)
plot_layouts(
    high_complexity_pareto,
    "High-complexity Pareto descriptors",
    knee_ids
)

pareto_trees = {
    int(row["id"]): row.tree
    for _, row in pareto_descriptors.iterrows()
}

fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
axes[0].plot(
    ga_history.generation, ga_history.best_NRMSE, "o-", label="Best"
)
axes[0].plot(
    ga_history.generation, ga_history.median_NRMSE, "o-", label="Median"
)
axes[0].set(xlabel="Generation", ylabel="Validation NRMSE")
axes[0].legend()
axes[1].plot(
    ga_history.generation, ga_history.mean_features, "o-"
)
axes[1].set(xlabel="Generation", ylabel="Mean descriptor feature count")
plt.show()


# ============================================================
# 13. FINAL x10 REFIT AND ALL-POINT EVALUATION
# ============================================================
final_four = ga_results.sort_values(
    ["val_NRMSE", "features", "id"]
).head(4)
selected_tree = final_four.iloc[0].tree

test_example_ids = test_ids[
    spread_indices(all_coordinates[test_ids], N_EXAMPLES)
]

final_models, final_examples = [], []
ensemble_prediction = np.zeros_like(Y_all, dtype=float)
best_all_prediction = None

for j, tree in enumerate(final_four.tree):
    bundle = fit_cnn(tree, train_ids)
    prediction = predict_cnn(bundle, P_all)

    final_models.append(bundle)
    final_examples.append(prediction[test_example_ids])
    ensemble_prediction += prediction / len(final_four)

    if j == 0:
        best_all_prediction = prediction.copy()

final_model = final_models[0]

test_baseline = np.sqrt(np.mean(
    (Y_all[test_ids] - Y_all[train_ids].mean(axis=0, dtype=float))**2
))
subsets = {
    "Fit": fit_ids,
    "Validation": val_ids,
    "Test": test_ids,
    "All valid centers": np.arange(len(P_all))
}

summary = []
for name, prediction in [
    ("Best descriptor", best_all_prediction),
    ("Mean of four", ensemble_prediction)
]:
    for subset, ids in subsets.items():
        rmse = np.sqrt(np.mean((prediction[ids] - Y_all[ids])**2))
        summary.append({
            "predictor": name,
            "subset": subset,
            "locations": len(ids),
            "RMSE": rmse,
            "test_NRMSE": (
                rmse / test_baseline
                if subset == "Test" and test_baseline > 0 else np.nan
            )
        })

rmse_summary = pd.DataFrame(summary)
display(rmse_summary)

plot_spectra(
    Y_all[test_example_ids],
    np.stack(final_examples),
    all_coordinates[test_example_ids],
    "Final CNNs: original test patches"
)

final_error_map = to_map(np.sqrt(np.mean(
    (best_all_prediction - Y_all)**2, axis=1
)))
ensemble_error_map = to_map(np.sqrt(np.mean(
    (ensemble_prediction - Y_all)**2, axis=1
)))

fig, axes = plt.subplots(1, 3, figsize=(16, 4), constrained_layout=True)
for name, prediction in [
    ("Best descriptor", best_all_prediction),
    ("Mean of four", ensemble_prediction)
]:
    axes[0].plot(
        E,
        np.sqrt(np.mean(
            (prediction[test_ids] - Y_all[test_ids])**2, axis=0
        )),
        label=name
    )
axes[0].set(xlabel="Energy", ylabel="Test RMSE")
axes[0].legend()

vmax = max(
    np.nanmax(final_error_map),
    np.nanmax(ensemble_error_map),
    1e-12
)
for ax, error_map, title in [
    (axes[1], final_error_map, "Best CNN: all points"),
    (axes[2], ensemble_error_map, "Mean of four CNNs: all points")
]:
    im = ax.imshow(error_map, cmap="magma", vmin=0, vmax=vmax)
    ax.set_title(title)
fig.colorbar(im, ax=list(axes[1:]), label="Spectral RMSE")
plt.show()

feature_definitions = pd.DataFrame([
    {
        "feature": i+1,
        "operation": op,
        "row_start": region[0],
        "column_start": region[1],
        "region_size": region[2]
    }
    for i, (region, op) in enumerate(active_leaves(selected_tree))
])
display(feature_definitions)

print(
    "CNN parameters:",
    sum(p.numel() for p in final_model["model"].parameters())
)

# Additional predictions:
# prediction = predict_cnn(final_model, new_patches)

In [ ]:
# ============================================================
# POLAR DESCRIPTOR GA → RANDOM FOREST → FULL SPECTRUM
# Augmentation: all 8 square symmetries, before polar statistics
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.ndimage import map_coordinates
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from matplotlib.patches import Rectangle
from IPython.display import display


# ------------------------------------------------------------
# 1. SETTINGS
# ------------------------------------------------------------

PATCH_SIZE = 12                  # Change this to explore spatial scale
N_RADIAL, N_ANGULAR = 4, 8
SAMPLES_PER_BIN = 2               # Fine polar grid: 16 × 32
GENERATIONS, POPULATION = 10, 25
KEEP, TOURNAMENT = 5, 3
CROSSOVER_RATE = 0.8
IMMIGRANT_RATE = 0.15
SEED = 0

TEST_FRACTION = 0.25
VALIDATION_FRACTION = 0.25         # Fraction of ALL original centers
N_EXAMPLES = 3

RF_SETTINGS = dict(
    n_estimators=20,
    max_depth=10,
    min_samples_leaf=2,
    max_features=1.0,
    n_jobs=-1,
    random_state=SEED,
)

rng = np.random.default_rng(SEED)

img = np.asarray(image, dtype=np.float32)
Ycube = np.asarray(spectra, dtype=np.float32)
E = np.asarray(energy).ravel()

assert img.ndim == 2
assert Ycube.shape[:2] == img.shape
assert Ycube.shape[2] == len(E)
assert PATCH_SIZE >= 3
assert np.isfinite(E).all()

order = np.argsort(E)
E, Ycube = E[order], Ycube[..., order]
assert np.all(np.diff(E) > 0), "Energy coordinates must be distinct."

H, W = img.shape
RADIUS = (PATCH_SIZE - 1) / 2
MARGIN = int(np.ceil(RADIUS))

# Complete support for every rotated/reflected circular neighborhood.
# For even patch sizes, this centered interpolation support spans one
# extra integer-grid position; its physical diameter is PATCH_SIZE - 1.
yy, xx = np.meshgrid(
    np.arange(MARGIN, H - MARGIN),
    np.arange(MARGIN, W - MARGIN),
    indexing="ij",
)
coords = np.column_stack([yy.ravel(), xx.ravel()])
if len(coords) == 0:
    raise ValueError("PATCH_SIZE is too large for this image.")


# ------------------------------------------------------------
# 2. TRANSFORM IMAGE COORDINATES, THEN SAMPLE IN POLAR COORDINATES
# ------------------------------------------------------------

# Sample bin interiors rather than repeating angular endpoints.
nr = N_RADIAL * SAMPLES_PER_BIN
nt = N_ANGULAR * SAMPLES_PER_BIN

r = (np.arange(nr) + 0.5) * RADIUS / nr
theta = (np.arange(nt) + 0.5) * 2 * np.pi / nt

# Cartesian convention: x right, y up.
u = r[:, None] * np.cos(theta)[None, :]
v = r[:, None] * np.sin(theta)[None, :]

# Four rotations plus four reflected rotations.
# Sampling transformed coordinates is equivalent to transforming
# the image about the target pixel BEFORE polar resampling.
transforms = []
for mirror in [False, True]:
    for k in range(4):
        a, b = u.copy(), v.copy()
        if mirror:
            a = -a
        for _ in range(k):
            a, b = -b, a
        transforms.append((a, b))

augmentation_names = [
    "Identity", "Rotation 90°", "Rotation 180°", "Rotation 270°",
    "Reflection", "Reflection + 90°",
    "Reflection + 180°", "Reflection + 270°",
]

# P: original centers × 8 transformations × fine radius × fine angle.
# Build in chunks to limit temporary memory.
P = np.empty((len(coords), 8, nr, nt), dtype=np.float32)

for start in range(0, len(coords), 256):
    stop = min(start + 256, len(coords))
    cy = coords[start:stop, 0, None, None]
    cx = coords[start:stop, 1, None, None]

    for g, (dx, dy) in enumerate(transforms):
        sampling_coords = np.array([
            np.broadcast_to(cy - dy, (stop-start, nr, nt)),
            np.broadcast_to(cx + dx, (stop-start, nr, nt)),
        ])
        P[start:stop, g] = map_coordinates(
            img, sampling_coords,
            order=1, mode="nearest", prefilter=False,
        )

Y = Ycube[coords[:, 0], coords[:, 1]]

valid = (
    np.isfinite(P).all(axis=(1, 2, 3))
    & np.isfinite(Y).all(axis=1)
)
coords, P, Y = coords[valid], P[valid], Y[valid]
ids = np.arange(len(Y))

if len(ids) < 40:
    raise ValueError("Too few valid centers. Reduce PATCH_SIZE.")

train_ids, test_ids = train_test_split(
    ids, test_size=TEST_FRACTION, random_state=SEED
)
fit_ids, val_ids = train_test_split(
    train_ids,
    test_size=VALIDATION_FRACTION / (1 - TEST_FRACTION),
    random_state=SEED + 1,
)

def rmse(actual, predicted):
    return np.sqrt(np.mean(
        (np.asarray(actual, dtype=np.float64) - predicted) ** 2
    ))

baseline = max(rmse(Y[val_ids], Y[fit_ids].mean(axis=0)), 1e-12)

example_ids = rng.choice(
    val_ids, size=min(N_EXAMPLES, len(val_ids)), replace=False
)

print(f"Original valid centers: {len(ids)}")
print(f"Fit / validation / test: "
      f"{len(fit_ids)} / {len(val_ids)} / {len(test_ids)}")
print(f"Augmented fit samples: {8 * len(fit_ids)}")
print(f"Polar elementary regions: {N_RADIAL * N_ANGULAR}")
print(f"Polar data memory: {P.nbytes / 2**20:.1f} MB")


# Random-point maps
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax in axes:
    ax.imshow(img, cmap="gray")

axes[0].scatter(*coords[train_ids][:, ::-1].T,
                s=5, c="blue", label="Train")
axes[0].scatter(*coords[test_ids][:, ::-1].T,
                s=5, c="red", label="Test")
axes[0].set_title("Random train/test centers")

axes[1].scatter(*coords[fit_ids][:, ::-1].T,
                s=5, c="blue", label="Fit")
axes[1].scatter(*coords[val_ids][:, ::-1].T,
                s=5, c="orange", label="Validation")
axes[1].set_title("Inner fit/validation centers")

for ax in axes:
    ax.legend()
plt.tight_layout()
plt.show()

# Same source, eight transformed polar representations
fig, axes = plt.subplots(2, 4, figsize=(13, 6))
source = fit_ids[0]
for g, ax in enumerate(axes.flat):
    ax.imshow(
        P[source, g], origin="lower", aspect="auto",
        extent=[0, 360, 0, RADIUS],
        vmin=P[source].min(), vmax=P[source].max(),
    )
    ax.set_title(augmentation_names[g])
    ax.set_xlabel("Angle (degrees)")
    ax.set_ylabel("Radius (pixels)")
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 3. CHROMOSOME
# ------------------------------------------------------------

# Leaf: ("MEAN",), ("MIN",), ("MAX",)
# Branch: ("R" or "A", boundary, child_1, child_2)
#
# Genome: (angular_offset, tree)
#
# The global offset places the angular seam anywhere on the grid.
# Regions can therefore cross the original 0°/360° boundary.
# Within one tree, mergers reverse hierarchical splits.
#
# No minimum 3×3 constraint and no DROP operator in this version:
# every polar location belongs to exactly one retained region.

OPS = ("MEAN", "MIN", "MAX")
ROOT = (0, N_RADIAL, 0, N_ANGULAR)

def children_bounds(bounds, axis, cut):
    r0, r1, a0, a1 = bounds
    if axis == "R":
        return (r0, cut, a0, a1), (cut, r1, a0, a1)
    return (r0, r1, a0, cut), (r0, r1, cut, a1)

def walk(tree, bounds=ROOT, path=()):
    yield path, bounds, tree
    if tree[0] in ("R", "A"):
        b1, b2 = children_bounds(bounds, tree[0], tree[1])
        yield from walk(tree[2], b1, path + (2,))
        yield from walk(tree[3], b2, path + (3,))

def leaves(tree):
    return [(bounds, node[0])
            for _, bounds, node in walk(tree)
            if node[0] in OPS]

def replace(tree, path, replacement):
    if not path:
        return replacement
    result = list(tree)
    result[path[0]] = replace(tree[path[0]], path[1:], replacement)
    return tuple(result)

def split_options(bounds):
    r0, r1, a0, a1 = bounds
    return (
        [("R", k) for k in range(r0 + 1, r1)]
        + [("A", k) for k in range(a0 + 1, a1)]
    )

def random_tree(bounds=ROOT, depth=0):
    options = split_options(bounds)
    if not options or depth >= 7 or rng.random() < 0.35:
        return (str(rng.choice(OPS)),)
    axis, cut = options[rng.integers(len(options))]
    b1, b2 = children_bounds(bounds, axis, cut)
    return axis, cut, random_tree(b1, depth+1), random_tree(b2, depth+1)

def full_tree(bounds=ROOT, op="MEAN"):
    r0, r1, a0, a1 = bounds
    if r1-r0 == 1 and a1-a0 == 1:
        return (op,)
    axis = "A" if a1-a0 >= r1-r0 and a1-a0 > 1 else "R"
    cut = (a0+a1)//2 if axis == "A" else (r0+r1)//2
    b1, b2 = children_bounds(bounds, axis, cut)
    return axis, cut, full_tree(b1, op), full_tree(b2, op)

def mutate(genome):
    offset, tree = genome

    if rng.random() < 0.15:
        return (offset + int(rng.choice([-1, 1]))) % N_ANGULAR, tree

    nodes = list(walk(tree))
    path, bounds, node = nodes[rng.integers(len(nodes))]

    if node[0] in OPS:
        options = split_options(bounds)
        if options and rng.random() < 0.6:
            axis, cut = options[rng.integers(len(options))]
            replacement = (axis, cut, node, node)
        else:
            replacement = (str(rng.choice(
                [op for op in OPS if op != node[0]]
            )),)
    else:
        # Coarsen a whole subtree to one pooled feature.
        replacement = (str(rng.choice(OPS)),)

    return offset, replace(tree, path, replacement)

def crossover(parent1, parent2):
    offset, t1 = parent1
    t2 = parent2[1]
    donors = {bounds: node for _, bounds, node in walk(t2)}
    compatible = [
        (path, bounds) for path, bounds, _ in walk(t1)
        if bounds in donors
    ]
    path, bounds = compatible[rng.integers(len(compatible))]
    return offset, replace(t1, path, donors[bounds])


# ------------------------------------------------------------
# 4. POLAR REGION STATISTICS AND FOREST FITTING
# ------------------------------------------------------------

# Cache reusable region statistics, not entire feature matrices.
# Cap cache size to keep memory bounded.
region_cache = {}
CACHE_LIMIT_MB = 192
cache_bytes = 0

def region_feature(offset, bounds, op):
    global cache_bytes
    key = (offset, bounds, op)
    if key in region_cache:
        return region_cache[key]

    r0, r1, a0, a1 = bounds
    s = SAMPLES_PER_BIN

    # Apply the genome's angular offset, including wraparound.
    angles = (
        np.arange(a0*s, a1*s) + offset*s
    ) % nt

    block = np.take(P[:, :, r0*s:r1*s, :], angles, axis=-1)

    if op == "MEAN":
        # Area-weighted mean: polar area element is r dr dtheta.
        weights = r[r0*s:r1*s]
        value = (
            block * weights[None, None, :, None]
        ).sum(axis=(-2, -1)) / (weights.sum() * len(angles))
    elif op == "MIN":
        value = block.min(axis=(-2, -1))
    else:
        value = block.max(axis=(-2, -1))

    value = value.astype(np.float32)

    if cache_bytes + value.nbytes > CACHE_LIMIT_MB * 2**20:
        region_cache.clear()
        cache_bytes = 0

    region_cache[key] = value
    cache_bytes += value.nbytes
    return value

def features(genome):
    offset, tree = genome
    return np.stack([
        region_feature(offset, bounds, op)
        for bounds, op in leaves(tree)
    ], axis=-1)                    # centers × 8 × features

def fit_predict(genome, training_ids):
    X = features(genome)

    model = RandomForestRegressor(**RF_SETTINGS)
    model.fit(
        X[training_ids].reshape(-1, X.shape[-1]),
        np.repeat(Y[training_ids], 8, axis=0),
    )

    # Score original orientation only, at every valid center.
    prediction = model.predict(X[:, 0])
    return model, prediction


# ------------------------------------------------------------
# 5. ARCHIVE, PARETO RANKS, TOURNAMENT SELECTION
# ------------------------------------------------------------

archive = {}

def evaluate(genome, generation):
    if genome not in archive:
        _, prediction = fit_predict(genome, fit_ids)
        pixel_rmse = np.sqrt(np.mean(
            (prediction - Y.astype(np.float64))**2, axis=1
        ))
        archive[genome] = dict(
            id=len(archive)+1,
            features=len(leaves(genome[1])),
            val_RMSE=float(np.sqrt(np.mean(pixel_rmse[val_ids]**2))),
            first_generation=generation,
            pixel_rmse=pixel_rmse.astype(np.float32),
            examples=prediction[example_ids].astype(np.float32),
        )
    return archive[genome]

def frame(genomes=None):
    if genomes is None:
        genomes = list(archive)
    return pd.DataFrame([
        dict(
            genome=g,
            **{k: v for k, v in archive[g].items()
               if k not in ("pixel_rmse", "examples")}
        )
        for g in genomes
    ])

def nondominated(values):
    # Row i dominates row j if it is no worse in both objectives
    # and strictly better in at least one.
    dominates = (
        (values[:, None, :] <= values[None, :, :]).all(axis=2)
        & (values[:, None, :] < values[None, :, :]).any(axis=2)
    )
    return ~dominates.any(axis=0)

def pareto(df):
    values = df[["features", "val_RMSE"]].to_numpy()
    return df.loc[nondominated(values)].sort_values(
        ["features", "val_RMSE", "id"]
    )

def ranked(df):
    df = df.reset_index(drop=True).copy()
    values = df[["features", "val_RMSE"]].to_numpy(float)
    ranks = np.zeros(len(df), int)
    crowd = np.zeros(len(df))
    remaining = np.arange(len(df))
    rank = 0

    while len(remaining):
        front = remaining[nondominated(values[remaining])]
        ranks[front] = rank

        if len(front) <= 2:
            crowd[front] = np.inf
        else:
            for j in range(2):
                order = front[np.argsort(values[front, j])]
                span = np.ptp(values[order, j])
                if span > 0:
                    crowd[order[[0, -1]]] = np.inf
                    crowd[order[1:-1]] += (
                        values[order[2:], j] - values[order[:-2], j]
                    ) / span

        remaining = np.setdiff1d(remaining, front)
        rank += 1

    return df.assign(rank=ranks, crowding=crowd)

def survivors(df):
    df = ranked(df)
    elite = df.sort_values(["val_RMSE", "features"]).index[0]
    selected = [elite]
    available = [i for i in df.index if i != elite]

    while len(selected) < KEEP:
        contestants = rng.choice(
            available, min(TOURNAMENT, len(available)), replace=False
        )
        winner = min(
            contestants,
            key=lambda i: (df.loc[i, "rank"], -df.loc[i, "crowding"]),
        )
        selected.append(winner)
        available.remove(winner)

    return df.loc[selected, "genome"].tolist()

def breed(parents):
    population = list(parents)
    seen = set(population)

    while len(population) < POPULATION:
        if rng.random() < IMMIGRANT_RATE:
            child = (int(rng.integers(N_ANGULAR)), random_tree())
        else:
            p1 = parents[rng.integers(len(parents))]
            p2 = parents[rng.integers(len(parents))]
            child = crossover(p1, p2) if rng.random() < CROSSOVER_RATE else p1
            child = mutate(child)

        if child not in seen:
            seen.add(child)
            population.append(child)

    return population


# ------------------------------------------------------------
# 6. VISUALIZATION HELPERS
# ------------------------------------------------------------

COLORS = {"MEAN": "tab:blue", "MIN": "tab:purple", "MAX": "tab:orange"}

def spatial_map(values):
    result = np.full((H, W), np.nan)
    result[coords[:, 0], coords[:, 1]] = values
    return result

def draw_descriptor(ax, genome, title=""):
    offset, tree = genome
    for bounds, op in leaves(tree):
        r0, r1, a0, a1 = bounds
        angle0 = (a0 + offset) * 2*np.pi / N_ANGULAR
        width = (a1-a0) * 2*np.pi / N_ANGULAR
        bottom = r0 * RADIUS / N_RADIAL
        height = (r1-r0) * RADIUS / N_RADIAL

        ax.bar(
            angle0 + width/2, height, width=width, bottom=bottom,
            color=COLORS[op], alpha=0.65,
            edgecolor="white", linewidth=0.5,
        )
    ax.set_ylim(0, RADIUS)
    ax.set_yticklabels([])
    ax.set_title(title, fontsize=10)

def show_masks(df, title, batch=12):
    for start in range(0, len(df), batch):
        part = df.iloc[start:start+batch]
        cols = min(4, len(part))
        rows = int(np.ceil(len(part)/cols))
        fig, axes = plt.subplots(
            rows, cols, figsize=(3.5*cols, 3.4*rows),
            subplot_kw={"projection": "polar"}, squeeze=False,
        )
        for ax, (_, row) in zip(axes.flat, part.iterrows()):
            draw_descriptor(
                ax, row["genome"],
                f"ID {row['id']} | {row['features']} features\n"
                f"Validation RMSE {row['val_RMSE']:.4g}",
            )
        for ax in list(axes.flat)[len(part):]:
            ax.remove()
        fig.suptitle(title + "\nBlue: mean | Purple: min | Orange: max")
        plt.tight_layout()
        plt.show()

def show_spectra(predictions, point_ids, title):
    # predictions: models × points × energy
    mean = predictions.mean(axis=0)
    std = predictions.std(axis=0)

    fig, axes = plt.subplots(
        1, len(point_ids), figsize=(5*len(point_ids), 3.5),
        squeeze=False,
    )
    for j, ax in enumerate(axes.flat):
        ax.plot(E, Y[point_ids[j]], "k", lw=2, label="Measured")
        for prediction in predictions:
            ax.plot(E, prediction[j], color="tab:blue", alpha=0.2)
        ax.plot(E, mean[j], color="tab:orange", label="Mean prediction")
        ax.fill_between(
            E, mean[j]-std[j], mean[j]+std[j],
            color="tab:orange", alpha=0.2,
        )
        ax.set_title(f"Pixel {tuple(coords[point_ids[j]])}")
        ax.set_xlabel("Energy")
        ax.set_ylabel("Intensity")
    axes[0, 0].legend()
    fig.suptitle(title + " (shading: model disagreement)")
    plt.tight_layout()
    plt.show()


# ------------------------------------------------------------
# 7. EVOLUTION
# ------------------------------------------------------------

# Include fully resolved and fully pooled baselines.
population = (
    [(0, full_tree(op=op)) for op in OPS]
    + [(0, (op,)) for op in OPS]
)
while len(population) < POPULATION:
    candidate = (int(rng.integers(N_ANGULAR)), random_tree())
    if candidate not in population:
        population.append(candidate)

history = []
error_vectors = []
winner_genomes = []
generation_tables = []

for generation in range(1, GENERATIONS+1):
    print(f"\nGENERATION {generation}/{GENERATIONS}")

    for j, genome in enumerate(population, 1):
        evaluate(genome, generation)
        if j % 5 == 0:
            print(f"  Evaluated {j}/{POPULATION}")

    current = frame(population).sort_values(
        ["val_RMSE", "features"]
    )
    best4 = current.head(4)
    winner = best4.iloc[0]["genome"]
    record = archive[winner]

    error_vectors.append(record["pixel_rmse"].copy())
    winner_genomes.append(winner)
    generation_tables.append(current.copy())
    history.append(dict(
        generation=generation,
        best_RMSE=record["val_RMSE"],
        median_RMSE=current.val_RMSE.median(),
        best_features=record["features"],
        archive_size=len(archive),
    ))

    display(best4.drop(columns="genome"))
    show_masks(best4, f"Generation {generation}: four best descriptors")

    show_spectra(
        np.stack([archive[g]["examples"] for g in best4.genome]),
        example_ids,
        f"Generation {generation}: fixed validation locations",
    )

    all_results = frame()
    front = pareto(all_results)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    im = axes[0].imshow(spatial_map(record["pixel_rmse"]), cmap="magma", vmin=0)
    axes[0].set_title("Winner RMSE at ALL original centers")
    plt.colorbar(im, ax=axes[0], label="Spectral RMSE")

    axes[1].scatter(
        all_results.features, all_results.val_RMSE,
        s=14, alpha=0.3, label="Archive",
    )
    axes[1].scatter(
        current.features, current.val_RMSE,
        s=30, label="Current generation",
    )
    axes[1].plot(front.features, front.val_RMSE, "r.-", label="Pareto front")
    axes[1].set_xlabel("Descriptor values")
    axes[1].set_ylabel("Validation spectral RMSE")
    axes[1].legend()
    plt.tight_layout()
    plt.show()

    if generation < GENERATIONS:
        population = breed(survivors(current))

ga_results = frame()
ga_history = pd.DataFrame(history)
error_vectors = np.stack(error_vectors)             # generations × centers
error_map_history = np.stack([spatial_map(v) for v in error_vectors])

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(ga_history.generation, ga_history.best_RMSE, "o-", label="Best")
axes[0].plot(ga_history.generation, ga_history.median_RMSE, "o-", label="Median")
axes[0].set_ylabel("Validation RMSE")
axes[0].legend()
axes[1].plot(ga_history.generation, ga_history.best_features, "o-")
axes[1].set_ylabel("Winner descriptor values")
for ax in axes:
    ax.set_xlabel("Generation")
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 8. PCA: SPATIAL LOADINGS AND GENERATION AMPLITUDES
# ------------------------------------------------------------

# Samples = spatial locations; variables = generations.
# Remove each generation's spatial mean before PCA.
X = error_vectors.T.astype(np.float64)
mean_error_by_generation = X.mean(axis=0)
Xc = X - mean_error_by_generation

pca_results = None

if np.linalg.norm(Xc) > 1e-12:
    k = min(3, X.shape[0]-1, X.shape[1])
    pca = PCA(n_components=k, svd_solver="full")
    scores = pca.fit_transform(Xc)

    keep = pca.singular_values_ > (
        np.finfo(float).eps * max(Xc.shape) * pca.singular_values_[0]
    )
    singular = pca.singular_values_[keep]

    spatial_loadings = scores[:, keep] / singular
    generation_amplitudes = pca.components_[keep].T * singular

    for j in range(len(singular)):
        pivot = np.argmax(np.abs(spatial_loadings[:, j]))
        if spatial_loadings[pivot, j] < 0:
            spatial_loadings[:, j] *= -1
            generation_amplitudes[:, j] *= -1

    pca_results = dict(
        model=pca,
        mean_error_by_generation=mean_error_by_generation,
        spatial_loadings=spatial_loadings,
        generation_amplitudes=generation_amplitudes,
        spatial_maps=np.stack([
            spatial_map(spatial_loadings[:, j])
            for j in range(len(singular))
        ]),
    )

    plt.figure(figsize=(6, 3))
    plt.plot(np.arange(1, GENERATIONS+1), mean_error_by_generation, "o-")
    plt.xlabel("Generation")
    plt.ylabel("Mean all-point RMSE")
    plt.tight_layout()
    plt.show()

    for j in range(len(singular)):
        fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
        limit = np.max(np.abs(spatial_loadings[:, j]))
        im = axes[0].imshow(
            spatial_map(spatial_loadings[:, j]),
            cmap="coolwarm", vmin=-limit, vmax=limit,
        )
        axes[0].set_title(f"PC {j+1}: spatial loading")
        plt.colorbar(im, ax=axes[0])

        axes[1].plot(
            np.arange(1, GENERATIONS+1),
            generation_amplitudes[:, j], "o-",
        )
        axes[1].set_xlabel("Generation")
        axes[1].set_ylabel("Component amplitude")
        axes[1].set_title(
            f"Explained variance: {pca.explained_variance_ratio_[keep][j]:.1%}"
        )
        plt.tight_layout()
        plt.show()


# ------------------------------------------------------------
# 9. ALL PARETO MASKS, KNEE, AND HIGH-COMPLEXITY MASKS
# ------------------------------------------------------------

pareto_results = pareto(ga_results)
curve = pareto_results.drop_duplicates(
    ["features", "val_RMSE"]
).reset_index(drop=True)

knee_row = None
if len(curve) >= 3:
    x = curve.features.to_numpy(float)
    y = curve.val_RMSE.to_numpy(float)

    if np.ptp(x) > 0 and np.ptp(y) > 0:
        xn = (x-x.min()) / np.ptp(x)
        yn = (y-y.min()) / np.ptp(y)
        distance = (1-xn-yn) / np.sqrt(2)
        distance[[0, -1]] = -np.inf
        k = int(np.argmax(distance))
        if distance[k] > 1e-6:
            knee_row = curve.iloc[k]

plt.figure(figsize=(7, 4))
plt.scatter(ga_results.features, ga_results.val_RMSE, s=16, alpha=0.3)
plt.plot(curve.features, curve.val_RMSE, "o-r", label="Pareto front")
if knee_row is not None:
    plt.scatter(
        knee_row.features, knee_row.val_RMSE,
        marker="*", s=220, c="black", label="Geometric knee",
    )
plt.xlabel("Descriptor values")
plt.ylabel("Validation spectral RMSE")
plt.legend()
plt.tight_layout()
plt.show()

display(pareto_results.drop(columns="genome"))
show_masks(pareto_results, "All nondominated descriptors")

# Error increase when moving toward a simpler descriptor.
steps = []
for i in range(len(curve)-1):
    simpler, richer = curve.iloc[i], curve.iloc[i+1]
    removed = richer.features - simpler.features
    if removed > 0:
        steps.append(dict(
            richer_id=richer.id,
            simpler_id=simpler.id,
            removed_features=removed,
            error_increase=simpler.val_RMSE-richer.val_RMSE,
            error_increase_per_removed_feature=(
                (simpler.val_RMSE-richer.val_RMSE) / removed
            ),
        ))
pareto_steps = pd.DataFrame(steps)
display(pareto_steps)

if knee_row is not None:
    print("Heuristic knee descriptor ID:", knee_row.id)
    knee_index = curve.index[curve.id == knee_row.id][0]
    show_masks(
        curve.iloc[max(0, knee_index-1):knee_index+2],
        "Knee and neighboring Pareto descriptors",
    )
else:
    print("No clear interior geometric knee.")

show_masks(
    ga_results.sort_values(
        ["features", "val_RMSE"], ascending=[False, True]
    ).head(12),
    "High-complexity archive descriptors",
)
show_masks(
    pareto_results.sort_values("features", ascending=False).head(8),
    "High-complexity Pareto descriptors",
)

pareto_genomes = dict(zip(pareto_results.id, pareto_results.genome))


# ------------------------------------------------------------
# 10. FINAL REFIT: AUGMENTED FIT + VALIDATION; TEST ON ORIGINALS
# ------------------------------------------------------------

selected = ga_results.sort_values(
    ["val_RMSE", "features"]
).head(4)

final_models = []
final_predictions = []
summary = []

for _, row in selected.iterrows():
    model, prediction = fit_predict(row.genome, train_ids)
    final_models.append(dict(
        id=row.id, genome=row.genome, model=model
    ))
    final_predictions.append(prediction)

    for name, subset in [
        ("Train (refitted)", train_ids),
        ("Test", test_ids),
        ("All", ids),
    ]:
        summary.append(dict(
            model_id=row.id,
            features=row.features,
            subset=name,
            RMSE=rmse(Y[subset], prediction[subset]),
        ))

ensemble_prediction = np.mean(final_predictions, axis=0)

for name, subset in [
    ("Train (refitted)", train_ids),
    ("Test", test_ids),
    ("All", ids),
]:
    summary.append(dict(
        model_id="Top-4 mean",
        features=np.nan,
        subset=name,
        RMSE=rmse(Y[subset], ensemble_prediction[subset]),
    ))

final_summary = pd.DataFrame(summary)
display(final_summary)

final_model = final_models[0]
final_prediction = final_predictions[0]

final_pixel_rmse = np.sqrt(np.mean((final_prediction-Y)**2, axis=1))
ensemble_pixel_rmse = np.sqrt(np.mean((ensemble_prediction-Y)**2, axis=1))

final_error_map = spatial_map(final_pixel_rmse)
ensemble_error_map = spatial_map(ensemble_pixel_rmse)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
vmax = max(final_pixel_rmse.max(), ensemble_pixel_rmse.max())
for ax, values, title in zip(
    axes,
    [final_error_map, ensemble_error_map],
    ["Best descriptor: all-point RMSE", "Top-4 mean: all-point RMSE"],
):
    im = ax.imshow(values, cmap="magma", vmin=0, vmax=vmax)
    ax.set_title(title)
    plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

test_examples = rng.choice(
    test_ids, min(N_EXAMPLES, len(test_ids)), replace=False
)
show_spectra(
    np.stack([prediction[test_examples] for prediction in final_predictions]),
    test_examples,
    "Final refitted models: held-out test locations",
)

plt.figure(figsize=(7, 3.5))
plt.plot(
    E,
    np.sqrt(np.mean(
        (ensemble_prediction[test_ids]-Y[test_ids])**2, axis=0
    )),
)
plt.xlabel("Energy")
plt.ylabel("Test RMSE")
plt.title("Ensemble error versus energy")
plt.tight_layout()
plt.show()

# Decoded winning descriptor
offset, tree = final_model["genome"]
feature_definitions = pd.DataFrame([
    dict(
        feature=j,
        operation=op,
        radial_start=r0 * RADIUS / N_RADIAL,
        radial_stop=r1 * RADIUS / N_RADIAL,
        angle_start_deg=((a0+offset) * 360/N_ANGULAR) % 360,
        angular_width_deg=(a1-a0) * 360/N_ANGULAR,
    )
    for j, ((r0, r1, a0, a1), op) in enumerate(leaves(tree))
])
display(feature_definitions)

print("Stored outputs:")
print("  ga_results, ga_history, generation_tables")
print("  error_map_history, pca_results")
print("  pareto_results, pareto_steps, pareto_genomes")
print("  final_models, final_summary, feature_definitions")
print("  final_error_map, ensemble_error_map")

In [ ]:
# ============================================================
# CONNECTED POLAR DESCRIPTOR GA → RIDGE → FULL SPECTRUM
#
# Required inputs:
#   image   : (H, W)
#   spectra : (H, W, energy_channels)
#   energy  : (energy_channels,)
#
# Pipeline:
#   target-centered image transformation
#   → polar resampling
#   → connected-region MIN / MAX / MEAN
#   → standardized Ridge regression
#
# Augmentation: 4 rotations × 2 reflection states = 8.
# Splits: random original centers, before training augmentation.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import OrderedDict
from scipy.ndimage import map_coordinates
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.decomposition import PCA
from IPython.display import display


# ============================================================
# 1. SETTINGS
# ============================================================

PATCH_SIZE = 12
N_RADIAL = 8
N_ANGULAR = 16
SAMPLES_PER_BIN = 2

RIDGE_ALPHA = 10.0

POPULATION = 100
GENERATIONS = 100
SURVIVORS = 20
TOURNAMENT_SIZE = 5
CROSSOVER_RATE = 0.80
IMMIGRANT_RATE = 0.15

# Radial merge, angular merge, connected split, pooling change.
MUTATION_PROBABILITIES = [0.30, 0.30, 0.25, 0.15]

TEST_FRACTION = 0.25
VALIDATION_FRACTION = 0.25       # Fraction of all original centers

N_EXAMPLES = 3
PCA_COMPONENTS = 3
PARETO_ROWS_PER_FIGURE = 4
HIGH_COMPLEXITY_MASKS = 12
REGION_CACHE_MB = 192

SEED = 0

rng = np.random.default_rng(SEED)
plot_rng = np.random.default_rng(SEED + 100)

OPS = ("MEAN", "MIN", "MAX")
COLORS = {
    "MEAN": "tab:blue",
    "MIN": "tab:purple",
    "MAX": "tab:orange",
}

assert PATCH_SIZE >= 3
assert N_RADIAL >= 1 and N_ANGULAR >= 2
assert SAMPLES_PER_BIN >= 1
assert RIDGE_ALPHA > 0
assert 1 <= SURVIVORS < POPULATION
assert GENERATIONS >= 1
assert 0 < TEST_FRACTION < 1
assert 0 < VALIDATION_FRACTION < 1 - TEST_FRACTION

mutation_probabilities = np.asarray(
    MUTATION_PROBABILITIES, dtype=float
)
assert (
    len(mutation_probabilities) == 4
    and np.all(mutation_probabilities >= 0)
    and mutation_probabilities.sum() > 0
)
mutation_probabilities /= mutation_probabilities.sum()


# ============================================================
# 2. INPUT DATA
# ============================================================

img = np.asarray(image, dtype=np.float32)
cube = np.asarray(spectra, dtype=np.float64)
E = np.asarray(energy, dtype=float).ravel()

if img.ndim != 2:
    raise ValueError("image must have shape (H, W).")
if cube.ndim != 3 or cube.shape[:2] != img.shape:
    raise ValueError("spectra must have shape (H, W, energy_channels).")
if cube.shape[-1] != len(E):
    raise ValueError("Energy axis and spectra have different lengths.")
if not np.isfinite(E).all():
    raise ValueError("Energy coordinates must be finite.")

order = np.argsort(E)
E = E[order]
cube = cube[..., order]

if np.any(np.diff(E) <= 0):
    raise ValueError("Energy coordinates must be distinct.")

H, W = img.shape
RADIUS = (PATCH_SIZE - 1) / 2
MARGIN = int(np.ceil(RADIUS))

# Center the circular neighborhood on the spectrum's target pixel.
# For even PATCH_SIZE, interpolation requires symmetric support
# extending to the integer pixels surrounding the circle.
yy, xx = np.meshgrid(
    np.arange(MARGIN, H - MARGIN),
    np.arange(MARGIN, W - MARGIN),
    indexing="ij",
)
coords = np.column_stack([yy.ravel(), xx.ravel()])

if len(coords) == 0:
    raise ValueError("PATCH_SIZE is too large for this image.")


# ============================================================
# 3. IMAGE AUGMENTATION → POLAR RESAMPLING
# ============================================================

nr = N_RADIAL * SAMPLES_PER_BIN
nt = N_ANGULAR * SAMPLES_PER_BIN

radial_samples = (np.arange(nr) + 0.5) * RADIUS / nr
angular_samples = (np.arange(nt) + 0.5) * 2*np.pi / nt

u = radial_samples[:, None] * np.cos(angular_samples)[None, :]
v = radial_samples[:, None] * np.sin(angular_samples)[None, :]

transforms = []
augmentation_names = []

# Inverse coordinate transformations:
# transformed_image(q) = original_image(T^-1 q).
# This transforms the image about the target before polar sampling,
# without introducing a second interpolation operation.
for reflected in (False, True):
    for k in range(4):
        dx, dy = u.copy(), v.copy()

        for _ in range(k):
            dx, dy = dy, -dx

        if reflected:
            dx = -dx

        transforms.append((dx, dy))
        augmentation_names.append(
            f"{'Mirror + ' if reflected else ''}{90*k}°"
        )

P = np.empty((len(coords), 8, nr, nt), dtype=np.float32)

for start in range(0, len(coords), 256):
    stop = min(start + 256, len(coords))

    cy = coords[start:stop, 0, None, None]
    cx = coords[start:stop, 1, None, None]

    for g, (dx, dy) in enumerate(transforms):
        sample_coordinates = np.stack([
            cy - dy,
            cx + dx,
        ])

        P[start:stop, g] = map_coordinates(
            img,
            sample_coordinates,
            order=1,
            mode="nearest",
            prefilter=False,
        )

Y = cube[coords[:, 0], coords[:, 1]]

valid = (
    np.isfinite(P).all(axis=(1, 2, 3))
    & np.isfinite(Y).all(axis=1)
)
coords, P, Y = coords[valid], P[valid], Y[valid]
ids = np.arange(len(Y))

if len(ids) < 40:
    raise ValueError("Too few valid centers. Reduce PATCH_SIZE.")

train_ids, test_ids = train_test_split(
    ids,
    test_size=TEST_FRACTION,
    random_state=SEED,
)
fit_ids, val_ids = train_test_split(
    train_ids,
    test_size=VALIDATION_FRACTION / (1 - TEST_FRACTION),
    random_state=SEED + 1,
)

example_ids = plot_rng.choice(
    val_ids,
    min(N_EXAMPLES, len(val_ids)),
    replace=False,
)

def rmse(actual, predicted):
    difference = np.asarray(actual, dtype=float) - predicted
    return float(np.sqrt(np.mean(difference**2)))

baseline_rmse = max(
    rmse(Y[val_ids], Y[fit_ids].mean(axis=0)),
    np.finfo(float).eps,
)

print(f"Valid original centers: {len(ids)}")
print(
    f"Fit / validation / test: "
    f"{len(fit_ids)} / {len(val_ids)} / {len(test_ids)}"
)
print(f"Augmented fit samples: {8 * len(fit_ids)}")
print(f"Polar grid: {N_RADIAL} bands × {N_ANGULAR} sectors")
print(f"Ridge alpha: {RIDGE_ALPHA}")

# Random split maps.
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax in axes:
    ax.imshow(img, cmap="gray")

axes[0].scatter(
    coords[train_ids, 1], coords[train_ids, 0],
    s=5, c="blue", label="Train",
)
axes[0].scatter(
    coords[test_ids, 1], coords[test_ids, 0],
    s=5, c="red", label="Test",
)
axes[0].set_title("Random train/test centers")

axes[1].scatter(
    coords[fit_ids, 1], coords[fit_ids, 0],
    s=5, c="blue", label="Fit",
)
axes[1].scatter(
    coords[val_ids, 1], coords[val_ids, 0],
    s=5, c="orange", label="Validation",
)
axes[1].set_title("Inner fit/validation centers")

for ax in axes:
    ax.legend()

plt.tight_layout()
plt.show()
plt.close(fig)

# Augmented polar representations of one source.
source = fit_ids[0]
fig, axes = plt.subplots(2, 4, figsize=(13, 6))

for g, ax in enumerate(axes.flat):
    ax.imshow(
        P[source, g],
        origin="lower",
        aspect="auto",
        extent=[0, 360, 0, RADIUS],
        vmin=P[source].min(),
        vmax=P[source].max(),
    )
    ax.set_title(augmentation_names[g])
    ax.set_xlabel("Angle (degrees)")
    ax.set_ylabel("Radius (pixels)")

fig.suptitle("Eight transformed polar representations")
plt.tight_layout()
plt.show()
plt.close(fig)


# ============================================================
# 4. ELEMENTARY POLAR STATISTICS
# ============================================================

NCELLS = N_RADIAL * N_ANGULAR
s = SAMPLES_PER_BIN

bin_min = np.empty((len(P), 8, NCELLS), dtype=np.float32)
bin_max = np.empty_like(bin_min)
bin_mean = np.empty_like(bin_min)
bin_weight = np.empty(NCELLS, dtype=np.float64)

for ri in range(N_RADIAL):
    radial_slice = slice(ri*s, (ri+1)*s)
    weights = radial_samples[radial_slice]
    denominator = weights.sum() * s

    for ai in range(N_ANGULAR):
        cell = ri*N_ANGULAR + ai
        angular_slice = slice(ai*s, (ai+1)*s)

        block = P[:, :, radial_slice, angular_slice]

        bin_min[:, :, cell] = block.min(axis=(-2, -1))
        bin_max[:, :, cell] = block.max(axis=(-2, -1))

        # Approximate physical-area mean using r dr dtheta.
        bin_mean[:, :, cell] = (
            block * weights[None, None, :, None]
        ).sum(axis=(-2, -1)) / denominator

        bin_weight[cell] = denominator

# Region statistics can now be calculated from elementary-bin
# statistics. These reproduce pooled values over the union,
# up to floating-point rounding.
# Release the larger resampled array.
del P


# ============================================================
# 5. POLAR ADJACENCY AND CHROMOSOME
# ============================================================

# Genome:
# (
#     ((cell_1, cell_2, ...), "MEAN"),
#     ((cell_3, cell_4, ...), "MAX"),
#     ...
# )
#
# Each region is connected.
# Regions partition the whole polar grid.
# No DROP: minimum complexity is one retained region.

EDGES = []
NEIGHBORS = [[] for _ in range(NCELLS)]

for ri in range(N_RADIAL):
    for ai in range(N_ANGULAR):
        cell = ri*N_ANGULAR + ai

        if ri + 1 < N_RADIAL:
            other = (ri+1)*N_ANGULAR + ai
            EDGES.append((cell, other, "R"))

        other = ri*N_ANGULAR + (ai+1) % N_ANGULAR
        if N_ANGULAR != 2 or ai == 0:
            EDGES.append((cell, other, "A"))

for cell, other, _ in EDGES:
    NEIGHBORS[cell].append(other)
    NEIGHBORS[other].append(cell)

def canonical(regions):
    return tuple(sorted(
        (
            (tuple(sorted(int(c) for c in cells)), str(op))
            for cells, op in regions
        ),
        key=lambda region: region[0],
    ))

def connected_components(cells):
    remaining = set(cells)
    components = []

    while remaining:
        start = min(remaining)
        remaining.remove(start)
        component = [start]
        stack = [start]

        while stack:
            cell = stack.pop()

            for other in NEIGHBORS[cell]:
                if other in remaining:
                    remaining.remove(other)
                    component.append(other)
                    stack.append(other)

        components.append(tuple(sorted(component)))

    return components

def validate_genome(genome):
    flat = [cell for cells, _ in genome for cell in cells]

    if sorted(flat) != list(range(NCELLS)):
        raise ValueError("Genome must cover all cells exactly once.")

    for cells, op in genome:
        if not cells or op not in OPS:
            raise ValueError("Invalid region.")
        if len(connected_components(cells)) != 1:
            raise ValueError("Every region must be connected.")

def region_labels(genome):
    labels = np.empty(NCELLS, dtype=int)

    for region_id, (cells, _) in enumerate(genome):
        labels[list(cells)] = region_id

    return labels.reshape(N_RADIAL, N_ANGULAR)

def adjacent_pairs(genome, direction=None):
    labels = region_labels(genome).ravel()
    pairs = set()

    for cell, other, axis in EDGES:
        if direction is not None and axis != direction:
            continue

        first = int(labels[cell])
        second = int(labels[other])

        if first != second:
            pairs.add(tuple(sorted((first, second))))

    return sorted(pairs)


# ============================================================
# 6. GENETIC OPERATORS
# ============================================================

def merge_regions(genome, direction=None):
    pairs = adjacent_pairs(genome, direction)

    if not pairs:
        return genome

    first, second = pairs[rng.integers(len(pairs))]

    merged_cells = tuple(sorted(
        genome[first][0] + genome[second][0]
    ))
    merged_operation = str(rng.choice(OPS))

    regions = [
        region for j, region in enumerate(genome)
        if j not in (first, second)
    ]
    regions.append((merged_cells, merged_operation))

    return canonical(regions)

def split_connected_region(cells):
    """Two-seed growth guarantees two connected daughter regions."""
    cells = set(cells)

    seed1, seed2 = [
        int(x)
        for x in rng.choice(sorted(cells), 2, replace=False)
    ]

    groups = [{seed1}, {seed2}]
    remaining = cells - {seed1, seed2}

    frontiers = [
        set(NEIGHBORS[seed1]) & remaining,
        set(NEIGHBORS[seed2]) & remaining,
    ]

    while remaining:
        available = [j for j in (0, 1) if frontiers[j]]

        if not available:
            raise RuntimeError("Disconnected region encountered.")

        side = available[rng.integers(len(available))]
        choices = sorted(frontiers[side])
        cell = choices[rng.integers(len(choices))]

        groups[side].add(cell)
        remaining.remove(cell)

        for frontier in frontiers:
            frontier.discard(cell)

        frontiers[side].update(
            set(NEIGHBORS[cell]) & remaining
        )

    return (
        tuple(sorted(groups[0])),
        tuple(sorted(groups[1])),
    )

def split_region(genome):
    regions = list(genome)

    candidates = [
        j for j, (cells, _) in enumerate(regions)
        if len(cells) > 1
    ]
    if not candidates:
        return genome

    index = candidates[rng.integers(len(candidates))]
    cells, op = regions.pop(index)

    first, second = split_connected_region(cells)
    regions.extend([(first, op), (second, op)])

    return canonical(regions)

def change_pooling(genome):
    regions = list(genome)
    index = int(rng.integers(len(regions)))

    cells, old_op = regions[index]
    new_op = str(rng.choice([op for op in OPS if op != old_op]))

    regions[index] = (cells, new_op)
    return canonical(regions)

def mutate(genome):
    action = int(rng.choice(4, p=mutation_probabilities))

    if action == 0:
        child = merge_regions(genome, direction="R")
    elif action == 1:
        child = merge_regions(genome, direction="A")
    elif action == 2:
        child = split_region(genome)
    else:
        child = change_pooling(genome)

    if child == genome:
        child = change_pooling(genome)

    return child

def crossover(parent1, parent2):
    """
    Transfer one connected donor region from parent2.
    Split disconnected remnants of parent1 into components.
    """
    donor_cells, donor_op = parent2[
        rng.integers(len(parent2))
    ]
    donor_set = set(donor_cells)
    regions = []

    for cells, op in parent1:
        remaining = set(cells) - donor_set

        for component in connected_components(remaining):
            regions.append((component, op))

    regions.append((donor_cells, donor_op))
    return canonical(regions)

def random_genome():
    genome = canonical([
        ((cell,), str(rng.choice(OPS)))
        for cell in range(NCELLS)
    ])

    # Include a broad range of complexities.
    target = int(np.clip(
        np.rint(np.exp(rng.uniform(0, np.log(NCELLS)))),
        1, NCELLS,
    ))

    while len(genome) > target:
        genome = merge_regions(genome)

    return genome


# ============================================================
# 7. FEATURE CACHE AND RIDGE
# ============================================================

region_cache = OrderedDict()
cache_bytes = 0
cache_limit = int(REGION_CACHE_MB * 2**20)

def region_feature(cells, op):
    global cache_bytes

    key = (cells, op)

    if key in region_cache:
        region_cache.move_to_end(key)
        return region_cache[key]

    indices = np.asarray(cells, dtype=int)

    if op == "MIN":
        value = bin_min[:, :, indices].min(axis=-1)

    elif op == "MAX":
        value = bin_max[:, :, indices].max(axis=-1)

    else:
        weights = bin_weight[indices]
        value = np.sum(
            bin_mean[:, :, indices] * weights[None, None, :],
            axis=-1,
        ) / weights.sum()

    value = value.astype(np.float32)

    if value.nbytes <= cache_limit:
        while region_cache and cache_bytes + value.nbytes > cache_limit:
            _, old = region_cache.popitem(last=False)
            cache_bytes -= old.nbytes

        region_cache[key] = value
        cache_bytes += value.nbytes

    return value

def features(genome, center_ids, augmented=False):
    columns = []

    for cells, op in genome:
        values = region_feature(cells, op)

        columns.append(
            values[center_ids].reshape(-1)
            if augmented
            else values[center_ids, 0]
        )

    return np.column_stack(columns)

def fit_model(genome, training_ids):
    model = make_pipeline(
        StandardScaler(),
        Ridge(alpha=RIDGE_ALPHA, solver="cholesky"),
    )

    model.fit(
        features(genome, training_ids, augmented=True),
        np.repeat(Y[training_ids], 8, axis=0),
    )
    return model

def predict_model(model, genome, center_ids):
    return model.predict(
        features(genome, center_ids, augmented=False)
    )


# ============================================================
# 8. ARCHIVE AND PARETO TOURNAMENT SELECTION
# ============================================================

archive = {}

def evaluate(genome, generation):
    if genome not in archive:
        model = fit_model(genome, fit_ids)
        prediction = predict_model(model, genome, val_ids)

        error = rmse(Y[val_ids], prediction)
        if not np.isfinite(error):
            raise ValueError("Nonfinite validation error.")

        archive[genome] = dict(
            id=len(archive) + 1,
            features=len(genome),
            val_RMSE=error,
            first_generation=generation,
            model=model,
        )

    return archive[genome]

def ensure_map(genome):
    """Only create full maps for winners and Pareto candidates."""
    record = archive[genome]

    if "pixel_rmse" not in record:
        prediction = predict_model(record["model"], genome, ids)

        record["pixel_rmse"] = np.sqrt(
            np.mean((prediction - Y)**2, axis=1)
        )

    return record["pixel_rmse"]

def frame(genomes=None):
    genomes = list(archive) if genomes is None else list(genomes)

    return pd.DataFrame([
        dict(
            genome=g,
            id=archive[g]["id"],
            features=archive[g]["features"],
            val_RMSE=archive[g]["val_RMSE"],
            val_NRMSE=archive[g]["val_RMSE"] / baseline_rmse,
            first_generation=archive[g]["first_generation"],
        )
        for g in genomes
    ])

def nondominated(values):
    dominates = (
        (values[:, None, :] <= values[None, :, :]).all(axis=2)
        & (values[:, None, :] < values[None, :, :]).any(axis=2)
    )
    return ~dominates.any(axis=0)

def pareto(df):
    values = df[["features", "val_RMSE"]].to_numpy(float)

    return df.loc[nondominated(values)].sort_values(
        ["features", "val_RMSE", "id"]
    )

def ranked(df):
    df = df.reset_index(drop=True).copy()
    values = df[["features", "val_RMSE"]].to_numpy(float)

    ranks = np.zeros(len(df), dtype=int)
    crowding = np.zeros(len(df))
    remaining = np.arange(len(df))
    rank = 0

    while len(remaining):
        front = remaining[nondominated(values[remaining])]
        ranks[front] = rank

        if len(front) <= 2:
            crowding[front] = np.inf
        else:
            for objective in range(2):
                order = front[np.argsort(values[front, objective])]
                span = np.ptp(values[order, objective])

                if span > 0:
                    crowding[order[[0, -1]]] = np.inf
                    crowding[order[1:-1]] += (
                        values[order[2:], objective]
                        - values[order[:-2], objective]
                    ) / span

        remaining = np.setdiff1d(remaining, front)
        rank += 1

    return df.assign(rank=ranks, crowding=crowding)

def choose_survivors(df):
    result = ranked(df)

    # One best-error elite; remaining survivors by tournament.
    elite = result.sort_values(
        ["val_RMSE", "features"]
    ).index[0]

    selected = [elite]
    available = [j for j in result.index if j != elite]

    while len(selected) < SURVIVORS:
        contestants = rng.choice(
            available,
            min(TOURNAMENT_SIZE, len(available)),
            replace=False,
        )

        winner = min(
            contestants,
            key=lambda j: (
                result.loc[j, "rank"],
                -result.loc[j, "crowding"],
            ),
        )
        selected.append(winner)
        available.remove(winner)

    return result.loc[selected, "genome"].tolist()

def breed(parents):
    population = list(parents)
    seen = set(population)
    attempts = 0

    while len(population) < POPULATION:
        attempts += 1

        if attempts > 10000:
            raise RuntimeError("Could not generate enough unique candidates.")

        if rng.random() < IMMIGRANT_RATE:
            child = random_genome()
        else:
            p1 = parents[rng.integers(len(parents))]
            p2 = parents[rng.integers(len(parents))]

            child = (
                crossover(p1, p2)
                if rng.random() < CROSSOVER_RATE else p1
            )
            child = mutate(child)

        if child not in seen:
            validate_genome(child)
            seen.add(child)
            population.append(child)

    return population


# ============================================================
# 9. PLOTTING HELPERS
# ============================================================

def spatial_map(values):
    result = np.full((H, W), np.nan)
    result[coords[:, 0], coords[:, 1]] = values
    return result

def draw_descriptor(ax, genome, title=""):
    labels = region_labels(genome)
    dr = RADIUS / N_RADIAL
    da = 2*np.pi / N_ANGULAR

    # Fill bins by their region's operation.
    for cells, op in genome:
        for cell in cells:
            ri, ai = divmod(cell, N_ANGULAR)

            ax.bar(
                (ai + 0.5)*da,
                dr,
                width=da,
                bottom=ri*dr,
                color=COLORS[op],
                alpha=0.7,
                edgecolor="none",
                linewidth=0,
            )

    # Black lines distinguish separate regions even if they
    # happen to use the same pooling operation.
    for ri in range(N_RADIAL):
        for ai in range(N_ANGULAR):
            if labels[ri, ai] != labels[ri, (ai-1) % N_ANGULAR]:
                ax.plot(
                    [ai*da, ai*da],
                    [ri*dr, (ri+1)*dr],
                    color="black", lw=0.8,
                )

            if (
                ri < N_RADIAL-1
                and labels[ri, ai] != labels[ri+1, ai]
            ):
                angles = np.linspace(ai*da, (ai+1)*da, 12)

                ax.plot(
                    angles,
                    np.full_like(angles, (ri+1)*dr),
                    color="black", lw=0.8,
                )

    ax.set_ylim(0, RADIUS)
    ax.set_yticklabels([])
    ax.grid(False)
    ax.set_title(title, fontsize=10)

def show_masks(df, title, batch=12):
    for start in range(0, len(df), batch):
        part = df.iloc[start:start+batch]
        cols = min(4, len(part))
        rows = int(np.ceil(len(part)/cols))

        fig, axes = plt.subplots(
            rows, cols,
            figsize=(3.5*cols, 3.4*rows),
            subplot_kw={"projection": "polar"},
            squeeze=False,
        )

        for ax, (_, row) in zip(axes.flat, part.iterrows()):
            draw_descriptor(
                ax, row["genome"],
                f"ID {int(row['id'])} | {int(row['features'])} regions\n"
                f"Validation RMSE: {row['val_RMSE']:.4g}",
            )

        for ax in list(axes.flat)[len(part):]:
            ax.remove()

        fig.suptitle(
            title + "\nBlue: mean | Purple: min | Orange: max"
        )
        plt.tight_layout()
        plt.show()
        plt.close(fig)

def show_spectra(predictions, point_ids, title):
    mean = predictions.mean(axis=0)
    std = predictions.std(axis=0)

    fig, axes = plt.subplots(
        1, len(point_ids),
        figsize=(5*len(point_ids), 3.5),
        squeeze=False,
    )

    for j, ax in enumerate(axes.flat):
        ax.plot(E, Y[point_ids[j]], "k", lw=2, label="Measured")

        for prediction in predictions:
            ax.plot(
                E, prediction[j],
                color="tab:blue", alpha=0.2,
            )

        ax.plot(
            E, mean[j], color="tab:orange",
            label="Mean prediction",
        )
        ax.fill_between(
            E, mean[j]-std[j], mean[j]+std[j],
            color="tab:orange", alpha=0.2,
        )

        ax.set_title(f"Pixel {tuple(coords[point_ids[j]])}")
        ax.set_xlabel("Energy")
        ax.set_ylabel("Intensity")

    axes[0, 0].legend()
    fig.suptitle(title + "\nShading: disagreement between models")
    plt.tight_layout()
    plt.show()
    plt.close(fig)


# ============================================================
# 10. EVOLUTION
# ============================================================

# Fully resolved and fully merged baselines.
population = [
    canonical([((cell,), op) for cell in range(NCELLS)])
    for op in OPS
] + [
    canonical([(tuple(range(NCELLS)), op)])
    for op in OPS
]

population = population[:POPULATION]

while len(population) < POPULATION:
    candidate = random_genome()
    if candidate not in population:
        population.append(candidate)

for genome in population:
    validate_genome(genome)

history = []
generation_tables = []
winner_genomes = []
error_vectors = []

for generation in range(1, GENERATIONS + 1):
    print(f"\nGENERATION {generation}/{GENERATIONS}")

    for genome in population:
        evaluate(genome, generation)

    current = frame(population).sort_values(
        ["val_RMSE", "features"]
    )
    best4 = current.head(4)

    winner = best4.iloc[0]["genome"]
    record = archive[winner]
    winner_errors = ensure_map(winner)

    winner_genomes.append(winner)
    error_vectors.append(winner_errors.copy())
    generation_tables.append(current.copy())

    history.append(dict(
        generation=generation,
        best_RMSE=record["val_RMSE"],
        median_RMSE=current["val_RMSE"].median(),
        best_features=record["features"],
        archive_size=len(archive),
    ))

    display(best4.drop(columns="genome"))
    show_masks(
        best4,
        f"Generation {generation}: four best descriptors",
    )

    example_predictions = np.stack([
        predict_model(archive[g]["model"], g, example_ids)
        for g in best4["genome"]
    ])
    show_spectra(
        example_predictions,
        example_ids,
        f"Generation {generation}: fixed validation locations",
    )

    all_results = frame()
    front = pareto(all_results)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    im = axes[0].imshow(
        spatial_map(winner_errors),
        cmap="magma", vmin=0,
    )
    axes[0].set_title("Generation winner: all-point RMSE")
    fig.colorbar(im, ax=axes[0], label="Spectral RMSE")

    axes[1].scatter(
        all_results["features"], all_results["val_RMSE"],
        s=14, alpha=0.3, label="Archive",
    )
    axes[1].scatter(
        current["features"], current["val_RMSE"],
        s=30, label="Current generation",
    )
    axes[1].plot(
        front["features"], front["val_RMSE"],
        "r.-", label="Pareto front",
    )
    axes[1].set_xlabel("Descriptor values / connected regions")
    axes[1].set_ylabel("Validation spectral RMSE")
    axes[1].legend()

    plt.tight_layout()
    plt.show()
    plt.close(fig)

    if generation < GENERATIONS:
        population = breed(choose_survivors(current))

ga_results = frame()
ga_history = pd.DataFrame(history)
error_vectors = np.stack(error_vectors)
error_map_history = np.stack([
    spatial_map(values) for values in error_vectors
])

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

axes[0].plot(
    ga_history["generation"], ga_history["best_RMSE"],
    "o-", label="Best",
)
axes[0].plot(
    ga_history["generation"], ga_history["median_RMSE"],
    "o-", label="Median",
)
axes[0].set_ylabel("Validation RMSE")
axes[0].legend()

axes[1].plot(
    ga_history["generation"], ga_history["best_features"], "o-"
)
axes[1].set_ylabel("Winner descriptor values")

for ax in axes:
    ax.set_xlabel("Generation")

plt.tight_layout()
plt.show()
plt.close(fig)


# ============================================================
# 11. PCA OF GENERATION-WINNER ERROR MAPS
# ============================================================

# Samples = pixels; variables = generations.
X = error_vectors.T
mean_error_by_generation = X.mean(axis=0)
Xc = X - mean_error_by_generation

pca_results = None

if np.linalg.norm(Xc) > 1e-12:
    k = min(PCA_COMPONENTS, X.shape[0]-1, X.shape[1])

    pca = PCA(n_components=k, svd_solver="full")
    scores = pca.fit_transform(Xc)

    tolerance = (
        np.finfo(float).eps
        * max(Xc.shape)
        * pca.singular_values_[0]
    )
    keep = pca.singular_values_ > tolerance
    singular = pca.singular_values_[keep]

    if len(singular):
        spatial_loadings = scores[:, keep] / singular
        generation_amplitudes = pca.components_[keep].T * singular

        for j in range(len(singular)):
            pivot = np.argmax(np.abs(spatial_loadings[:, j]))

            if spatial_loadings[pivot, j] < 0:
                spatial_loadings[:, j] *= -1
                generation_amplitudes[:, j] *= -1

        pca_results = dict(
            model=pca,
            mean_error_by_generation=mean_error_by_generation,
            spatial_loadings=spatial_loadings,
            generation_amplitudes=generation_amplitudes,
            explained_variance_ratio=pca.explained_variance_ratio_[keep],
            spatial_maps=np.stack([
                spatial_map(spatial_loadings[:, j])
                for j in range(len(singular))
            ]),
        )

        fig, ax = plt.subplots(figsize=(6, 3))
        ax.plot(
            np.arange(1, GENERATIONS+1),
            mean_error_by_generation, "o-",
        )
        ax.set_xlabel("Generation")
        ax.set_ylabel("Mean all-point RMSE")
        plt.tight_layout()
        plt.show()
        plt.close(fig)

        for j in range(len(singular)):
            fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

            limit = np.max(np.abs(spatial_loadings[:, j]))

            im = axes[0].imshow(
                spatial_map(spatial_loadings[:, j]),
                cmap="coolwarm",
                vmin=-limit,
                vmax=limit,
            )
            axes[0].set_title(f"PC {j+1}: spatial loading")
            fig.colorbar(im, ax=axes[0])

            axes[1].plot(
                np.arange(1, GENERATIONS+1),
                generation_amplitudes[:, j], "o-",
            )
            axes[1].set_xlabel("Generation")
            axes[1].set_ylabel("Component amplitude")
            axes[1].set_title(
                "Explained variance: "
                f"{pca.explained_variance_ratio_[keep][j]:.1%}"
            )

            plt.tight_layout()
            plt.show()
            plt.close(fig)


# ============================================================
# 12. FULL PARETO FRONT AND HEURISTIC KNEE
# ============================================================

pareto_results = pareto(ga_results).reset_index(drop=True)

curve = pareto_results.drop_duplicates(
    ["features", "val_RMSE"]
).reset_index(drop=True)

knee_features = None
knee_error = None

if len(curve) >= 3:
    x = curve["features"].to_numpy(float)
    y = curve["val_RMSE"].to_numpy(float)

    if np.ptp(x) > 0 and np.ptp(y) > 0:
        xn = (x-x.min()) / np.ptp(x)
        yn = (y-y.min()) / np.ptp(y)

        distance = (1-xn-yn) / np.sqrt(2)
        distance[[0, -1]] = -np.inf
        k = int(np.argmax(distance))

        if distance[k] > 1e-6:
            knee_features = int(curve.iloc[k]["features"])
            knee_error = float(curve.iloc[k]["val_RMSE"])

def front_label(row):
    if knee_features is None:
        return "Pareto"

    if row["features"] < knee_features:
        return "Low-complexity / upswing side"

    if (
        row["features"] == knee_features
        and np.isclose(row["val_RMSE"], knee_error)
    ):
        return "Knee"

    return "Higher-complexity side"

pareto_results["front_region"] = [
    front_label(row) for _, row in pareto_results.iterrows()
]

fig, ax = plt.subplots(figsize=(8, 4.5))

ax.scatter(
    ga_results["features"], ga_results["val_RMSE"],
    s=15, color="gray", alpha=0.3, label="Archive",
)
ax.plot(
    curve["features"], curve["val_RMSE"],
    "o-", label="Full Pareto front",
)

if knee_features is not None:
    upswing = curve[curve["features"] <= knee_features]

    ax.plot(
        upswing["features"], upswing["val_RMSE"],
        "o-r", label="Low-complexity / upswing side",
    )
    ax.scatter(
        knee_features, knee_error,
        marker="*", s=220, color="black",
        label="Heuristic knee",
    )

ax.set_xlabel("Descriptor values / connected regions")
ax.set_ylabel("Validation spectral RMSE")
ax.legend()

plt.tight_layout()
plt.show()
plt.close(fig)

steps = []

for i in range(len(curve)-1):
    simpler = curve.iloc[i]
    richer = curve.iloc[i+1]
    removed = richer["features"] - simpler["features"]

    if removed > 0:
        increase = simpler["val_RMSE"] - richer["val_RMSE"]

        steps.append(dict(
            richer_id=int(richer["id"]),
            simpler_id=int(simpler["id"]),
            removed_features=int(removed),
            error_increase=increase,
            error_increase_per_removed_feature=increase/removed,
        ))

pareto_steps = pd.DataFrame(steps)
display(pareto_steps)


# ============================================================
# 13. EACH PARETO MASK WITH ITS ERROR MAP
# ============================================================

# These maps use the fit-only models defining the Pareto front.
# All maps use original orientations and a common color scale.

pareto_genomes = dict(zip(
    pareto_results["id"], pareto_results["genome"]
))

pareto_error_maps = np.stack([
    spatial_map(ensure_map(g))
    for g in pareto_results["genome"]
])

pareto_error_maps_by_id = {
    int(mask_id): error_map
    for mask_id, error_map in zip(
        pareto_results["id"], pareto_error_maps
    )
}

for name, subset in [
    ("fit_RMSE", fit_ids),
    ("test_RMSE", test_ids),
    ("all_RMSE", ids),
]:
    pareto_results[name] = [
        float(np.sqrt(np.mean(ensure_map(g)[subset]**2)))
        for g in pareto_results["genome"]
    ]

display(pareto_results.drop(columns="genome"))

shared_vmax = max(
    float(np.nanmax(pareto_error_maps)),
    np.finfo(float).eps,
)

for start in range(0, len(pareto_results), PARETO_ROWS_PER_FIGURE):
    stop = min(
        start + PARETO_ROWS_PER_FIGURE,
        len(pareto_results),
    )

    fig = plt.figure(
        figsize=(11, 3.5*(stop-start)),
        constrained_layout=True,
    )
    grid = fig.add_gridspec(
        stop-start, 2, width_ratios=[1, 1.6]
    )
    map_axes = []

    for local_row, index in enumerate(range(start, stop)):
        row = pareto_results.iloc[index]

        mask_ax = fig.add_subplot(
            grid[local_row, 0], projection="polar"
        )
        draw_descriptor(
            mask_ax, row["genome"],
            f"ID {int(row['id'])} | {int(row['features'])} regions\n"
            f"{row['front_region']}",
        )

        map_ax = fig.add_subplot(grid[local_row, 1])
        map_axes.append(map_ax)

        im = map_ax.imshow(
            pareto_error_maps[index],
            cmap="magma",
            vmin=0,
            vmax=shared_vmax,
        )
        map_ax.set_title(
            f"Validation RMSE: {row['val_RMSE']:.4g} | "
            f"All-point RMSE: {row['all_RMSE']:.4g}"
        )
        map_ax.set_xlabel("Image column")
        map_ax.set_ylabel("Image row")

    fig.colorbar(
        im, ax=map_axes,
        label="Spectral RMSE — shared scale",
        shrink=0.9,
    )
    fig.suptitle(
        "Pareto masks and all-point RMSE: fit-only Ridge models\n"
        "Blue: mean | Purple: min | Orange: max"
    )

    plt.show()
    plt.close(fig)

show_masks(
    ga_results.sort_values(
        ["features", "val_RMSE"],
        ascending=[False, True],
    ).head(HIGH_COMPLEXITY_MASKS),
    "High-complexity archive descriptors",
)


# ============================================================
# 14. FINAL REFIT ON AUGMENTED FIT + VALIDATION
# ============================================================

selected = ga_results.sort_values(
    ["val_RMSE", "features"]
).head(4)

final_models = []
final_predictions = []
summary = []

for _, row in selected.iterrows():
    genome = row["genome"]
    model = fit_model(genome, train_ids)
    prediction = predict_model(model, genome, ids)

    final_models.append(dict(
        id=int(row["id"]),
        genome=genome,
        model=model,
    ))
    final_predictions.append(prediction)

    for name, subset in [
        ("Train (refitted)", train_ids),
        ("Test", test_ids),
        ("All", ids),
    ]:
        summary.append(dict(
            model_id=int(row["id"]),
            features=int(row["features"]),
            subset=name,
            RMSE=rmse(Y[subset], prediction[subset]),
        ))

ensemble_prediction = np.mean(final_predictions, axis=0)

for name, subset in [
    ("Train (refitted)", train_ids),
    ("Test", test_ids),
    ("All", ids),
]:
    summary.append(dict(
        model_id="Top-4 mean",
        features=np.nan,
        subset=name,
        RMSE=rmse(Y[subset], ensemble_prediction[subset]),
    ))

final_summary = pd.DataFrame(summary)
display(final_summary)

final_model = final_models[0]
final_prediction = final_predictions[0]

final_pixel_rmse = np.sqrt(
    np.mean((final_prediction-Y)**2, axis=1)
)
ensemble_pixel_rmse = np.sqrt(
    np.mean((ensemble_prediction-Y)**2, axis=1)
)

final_error_map = spatial_map(final_pixel_rmse)
ensemble_error_map = spatial_map(ensemble_pixel_rmse)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

vmax = max(
    final_pixel_rmse.max(),
    ensemble_pixel_rmse.max(),
    np.finfo(float).eps,
)

for ax, error_map, title in zip(
    axes,
    [final_error_map, ensemble_error_map],
    [
        "Refitted best descriptor: all-point RMSE",
        "Refitted top-4 mean: all-point RMSE",
    ],
):
    im = ax.imshow(
        error_map, cmap="magma", vmin=0, vmax=vmax
    )
    ax.set_title(title)
    fig.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()
plt.close(fig)

test_examples = plot_rng.choice(
    test_ids,
    min(N_EXAMPLES, len(test_ids)),
    replace=False,
)

show_spectra(
    np.stack([
        prediction[test_examples]
        for prediction in final_predictions
    ]),
    test_examples,
    "Final refitted models: test locations",
)

fig, ax = plt.subplots(figsize=(7, 3.5))

ax.plot(
    E,
    np.sqrt(np.mean(
        (ensemble_prediction[test_ids]-Y[test_ids])**2,
        axis=0,
    )),
)
ax.set_xlabel("Energy")
ax.set_ylabel("Test RMSE")
ax.set_title("Ensemble error versus energy")

plt.tight_layout()
plt.show()
plt.close(fig)


# ============================================================
# 15. WINNING REGION DEFINITIONS AND OUTPUTS
# ============================================================

feature_definitions = pd.DataFrame([
    dict(
        feature=j,
        operation=op,
        number_of_bins=len(cells),
        radial_angular_bins=[
            divmod(cell, N_ANGULAR)
            for cell in cells
        ],
    )
    for j, (cells, op) in enumerate(final_model["genome"])
])

final_region_labels = region_labels(final_model["genome"])

display(feature_definitions)

print("Stored outputs:")
print("  ga_results, ga_history, generation_tables")
print("  winner_genomes, error_map_history, pca_results")
print("  pareto_results, pareto_steps, pareto_genomes")
print("  pareto_error_maps, pareto_error_maps_by_id")
print("  final_models, final_summary, feature_definitions")
print("  final_region_labels")
print("  final_error_map, ensemble_error_map")